In [1]:
# ============================================================
# SYNTHETIC LONG-CONTEXT RETRIEVAL DATASET
# Needle / Key-Value Retrieval for Mini DeepSeek-V4
# ============================================================

import random
import os
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple

import torch
from torch.utils.data import Dataset, DataLoader


# ============================================================
# CONFIG
# ============================================================

### Debug rápido ###
#block_size = 256
#min_filler_tokens = 32
#max_filler_tokens = 160
#batch_size = 32

### Long-context medio ###
#block_size = 1024
#min_filler_tokens = 256
#max_filler_tokens = 900
#batch_size = 8

### CSA/HCA stress test ###
#block_size = 2048
#min_filler_tokens = 768
#max_filler_tokens = 1800
#batch_size = 2


@dataclass
class SyntheticRetrievalConfig:
    # Dataset size
    num_train_examples: int = 50_000
    num_val_examples: int = 5_000

    # Sequence length
    block_size: int = 64         # input length
    min_filler_tokens: int = 64
    max_filler_tokens: int = 420

    # Task structure
    num_keys_per_example: int = 8  # number of key-value pairs in context
    vocab_filler_size: int = 550
    num_key_types: int = 128
    num_value_types: int = 256

    # DataLoader
    batch_size: int = 32
    num_workers: int = 0

    # Reproducibility
    seed: int = 42


CFG = SyntheticRetrievalConfig()

CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS = CFG.num_workers if CFG.num_workers is not None else min(4, CPU_COUNT - 1)


# ============================================================
# SIMPLE TOKENIZER
# ============================================================

class SimpleWordTokenizer:
    """
    Tokenizer simple basado en espacios.

    Ventaja:
    - Totalmente controlado.
    - Perfecto para synthetic retrieval.
    - No dependemos de HuggingFace ni tokenizers externos.
    """

    def __init__(self):
        self.special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]
        self.token_to_idx = {}
        self.idx_to_token = {}

        for tok in self.special_tokens:
            self.add_token(tok)

    def add_token(self, token: str):
        if token not in self.token_to_idx:
            idx = len(self.token_to_idx)
            self.token_to_idx[token] = idx
            self.idx_to_token[idx] = token

    def build_vocab(self, cfg: SyntheticRetrievalConfig):
        # Structural tokens
        structural_tokens = [
            "key", "is", "question", "what", "?", "answer", ":",
            "noise", "the", "and", "then", "because", "context",
            "remember", "value", "token"
        ]

        for tok in structural_tokens:
            self.add_token(tok)

        # Key tokens
        for i in range(cfg.num_key_types):
            self.add_token(f"key_{i}")

        # Value tokens
        for i in range(cfg.num_value_types):
            self.add_token(f"value_{i}")

        # Filler tokens
        for i in range(cfg.vocab_filler_size):
            self.add_token(f"filler_{i}")

    def encode(self, text: str) -> List[int]:
        tokens = text.strip().split()
        unk = self.token_to_idx["<unk>"]
        return [self.token_to_idx.get(tok, unk) for tok in tokens]

    def decode(self, ids: List[int]) -> str:
        return " ".join(self.idx_to_token.get(int(i), "<unk>") for i in ids)

    @property
    def vocab_size(self):
        return len(self.token_to_idx)

    @property
    def pad_id(self):
        return self.token_to_idx["<pad>"]

    @property
    def bos_id(self):
        return self.token_to_idx["<bos>"]

    @property
    def eos_id(self):
        return self.token_to_idx["<eos>"]


# ============================================================
# EXAMPLE GENERATOR
# ============================================================

class SyntheticRetrievalGenerator:
    """
    Genera ejemplos de recuperación de largo contexto.

    Estructura:

    <bos>
    key_3 is value_87
    key_10 is value_21
    ...
    filler_832 filler_14 ...
    question : what is key_10 ?
    answer : value_21
    <eos>

    El objetivo autoregresivo será predecir todo el texto.
    Para análisis específico, también devolvemos query_key y answer_value.
    """

    def __init__(self, cfg: SyntheticRetrievalConfig, tokenizer: SimpleWordTokenizer, seed: int = 42):
        self.cfg = cfg
        self.tokenizer = tokenizer
        self.rng = random.Random(seed)

        self.keys = [f"key_{i}" for i in range(cfg.num_key_types)]
        self.values = [f"value_{i}" for i in range(cfg.num_value_types)]
        self.fillers = [f"filler_{i}" for i in range(cfg.vocab_filler_size)]

    def _sample_filler(self, n: int) -> List[str]:
        return [self.rng.choice(self.fillers) for _ in range(n)]

    def generate_text_example(self) -> Tuple[str, Dict]:
        cfg = self.cfg

        # Sample unique keys
        selected_keys = self.rng.sample(self.keys, cfg.num_keys_per_example)
        selected_values = [self.rng.choice(self.values) for _ in range(cfg.num_keys_per_example)]

        kv_pairs = dict(zip(selected_keys, selected_values))

        # Pick one key to query
        query_key = self.rng.choice(selected_keys)
        answer_value = kv_pairs[query_key]

        tokens = ["<bos>"]

        # Add key-value facts
        for k, v in kv_pairs.items():
            tokens.extend([k, "is", v])

            # Small local noise between facts
            local_noise_len = self.rng.randint(1, 5)
            tokens.extend(self._sample_filler(local_noise_len))

        # Long filler between facts and question
        filler_len = self.rng.randint(cfg.min_filler_tokens, cfg.max_filler_tokens)
        tokens.extend(self._sample_filler(filler_len))

        # Query and answer
        tokens.extend([
            "question", ":", "what", "is", query_key, "?",
            "answer", ":", answer_value,
            "<eos>"
        ])

        text = " ".join(tokens)

        meta = {
            "query_key": query_key,
            "answer_value": answer_value,
            "kv_pairs": kv_pairs,
            "filler_len": filler_len,
        }

        return text, meta


# ============================================================
# DATASET
# ============================================================

class SyntheticRetrievalDataset(Dataset):
    """
    Dataset autoregresivo.

    Devuelve:
      input_ids: [block_size]
      labels:    [block_size]

    labels es input_ids desplazado un token hacia adelante.
    """

    def __init__(
        self,
        cfg: SyntheticRetrievalConfig,
        tokenizer: SimpleWordTokenizer,
        split: str = "train",
    ):
        super().__init__()

        assert split in ["train", "val"]

        self.cfg = cfg
        self.tokenizer = tokenizer
        self.split = split
        self.num_examples = cfg.num_train_examples if split == "train" else cfg.num_val_examples

        seed = cfg.seed if split == "train" else cfg.seed + 10_000
        self.generator = SyntheticRetrievalGenerator(cfg, tokenizer, seed=seed)

        self.pad_id = tokenizer.pad_id
        self.block_size = cfg.block_size

    def __len__(self):
        return self.num_examples

    def _pad_or_truncate(self, ids: List[int], target_len: int) -> List[int]:
        if len(ids) >= target_len:
            return ids[:target_len]
        return ids + [self.pad_id] * (target_len - len(ids))

    def __getitem__(self, idx):
        text, meta = self.generator.generate_text_example()

        # Need block_size + 1 because input is ids[:-1], label is ids[1:]
        ids = self.tokenizer.encode(text)
        ids = self._pad_or_truncate(ids, self.block_size + 1)

        ids = torch.tensor(ids, dtype=torch.long)

        input_ids = ids[:-1]
        labels = ids[1:]

        return {
        "input_ids": input_ids,
        "labels": labels,
    }


# ============================================================
# OPTIONAL: MTP DATASET VERSION
# ============================================================

class SyntheticRetrievalMTPDataset(Dataset):
    """
    Dataset con soporte para Multi-Token Prediction.

    Devuelve:
      input_ids: [block_size]
      labels:    [block_size]  predice x_{t+1}
      mtp_labels:[mtp_depth, block_size] predice x_{t+2}, x_{t+3}, ...
    """

    def __init__(
        self,
        cfg: SyntheticRetrievalConfig,
        tokenizer: SimpleWordTokenizer,
        split: str = "train",
        mtp_depth: int = 1,
    ):
        super().__init__()

        assert split in ["train", "val"]

        self.cfg = cfg
        self.tokenizer = tokenizer
        self.split = split
        self.mtp_depth = mtp_depth

        self.num_examples = cfg.num_train_examples if split == "train" else cfg.num_val_examples

        seed = cfg.seed if split == "train" else cfg.seed + 10_000
        self.generator = SyntheticRetrievalGenerator(cfg, tokenizer, seed=seed)

        self.pad_id = tokenizer.pad_id
        self.block_size = cfg.block_size

    def __len__(self):
        return self.num_examples

    def _pad_or_truncate(self, ids: List[int], target_len: int) -> List[int]:
        if len(ids) >= target_len:
            return ids[:target_len]
        return ids + [self.pad_id] * (target_len - len(ids))

    def __getitem__(self, idx):
        text, meta = self.generator.generate_text_example()

        # Need block_size + 1 + mtp_depth
        total_len = self.block_size + 1 + self.mtp_depth

        ids = self.tokenizer.encode(text)
        ids = self._pad_or_truncate(ids, total_len)
        ids = torch.tensor(ids, dtype=torch.long)

        input_ids = ids[:self.block_size]
        labels = ids[1:self.block_size + 1]

        mtp_labels = []
        for k in range(2, 2 + self.mtp_depth):
            mtp_labels.append(ids[k:k + self.block_size])

        mtp_labels = torch.stack(mtp_labels, dim=0)

        return {
            "input_ids": input_ids,
            "labels": labels,
            "mtp_labels": mtp_labels,
        }


# ============================================================
# DATALOADERS
# ============================================================

def create_synthetic_retrieval_dataloaders(
    cfg: SyntheticRetrievalConfig = CFG,
    use_mtp: bool = False,
    mtp_depth: int = 1,
):
    tokenizer = SimpleWordTokenizer()
    tokenizer.build_vocab(cfg)

    print("Synthetic tokenizer vocab size:", tokenizer.vocab_size)

    if use_mtp:
        train_ds = SyntheticRetrievalMTPDataset(
            cfg=cfg,
            tokenizer=tokenizer,
            split="train",
            mtp_depth=mtp_depth,
        )

        val_ds = SyntheticRetrievalMTPDataset(
            cfg=cfg,
            tokenizer=tokenizer,
            split="val",
            mtp_depth=mtp_depth,
        )

    else:
        train_ds = SyntheticRetrievalDataset(
            cfg=cfg,
            tokenizer=tokenizer,
            split="train",
        )

        val_ds = SyntheticRetrievalDataset(
            cfg=cfg,
            tokenizer=tokenizer,
            split="val",
        )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, val_loader, tokenizer



In [3]:
train_loader, val_loader, tokenizer = create_synthetic_retrieval_dataloaders(
        cfg=CFG,
        use_mtp=False,)

x, y = next(iter(train_loader))

Synthetic tokenizer vocab size: 954


In [ ]:
train_loader, val_loader, tokenizer = create_synthetic_retrieval_dataloaders(
    cfg=CFG,
    use_mtp=True,
    mtp_depth=1)

batch = next(iter(train_loader))

print(batch["input_ids"].shape)   # [B, T]
print(batch["labels"].shape)      # [B, T]
print(batch["mtp_labels"].shape)  # [B, mtp_depth, T]

Synthetic tokenizer vocab size: 2404
torch.Size([32, 512])
torch.Size([32, 512])
torch.Size([32, 1, 512])


In [ ]:
batch = next(iter(train_loader))

if isinstance(batch, dict):
    x = batch["input_ids"]
    y = batch["labels"]

    print("Batch input_ids shape:", x.shape)
    print("Batch labels shape:", y.shape)

    if "mtp_labels" in batch:
        print("Batch mtp_labels shape:", batch["mtp_labels"].shape)

else:
    x, y = batch

    print("Batch x shape:", x.shape)
    print("Batch y shape:", y.shape)


text_x = tokenizer.decode(x[0].tolist())
text_y = tokenizer.decode(y[0].tolist())

print("\nTexto ejemplo:")
print(text_x)

print("\nTargets ejemplo:")
print(text_y)

tokens = text_x.split()

assert "question" in tokens, "No aparece 'question' en el ejemplo."
assert "answer" in tokens, "No aparece 'answer' en el ejemplo."

q_idx = tokens.index("question")
a_idx = tokens.index("answer")

# Esperamos:
# question : what is key_XX ?
# answer : value_YY

query_key = tokens[q_idx + 4]
answer_value = tokens[a_idx + 2]

print("Query key:", query_key)
print("Answer value:", answer_value)

pattern = f"{query_key} is {answer_value}"

assert pattern in text_x, f"No se encontró la asociación correcta: {pattern}"

print("Dataset OK:", pattern)


Batch x shape: torch.Size([32, 512])
Batch y shape: torch.Size([32, 512])

Texto ejemplo:
<bos> key_45 is value_194 filler_1079 filler_847 filler_1074 key_116 is value_34 filler_1425 filler_428 key_92 is value_218 filler_119 filler_223 filler_1170 filler_1494 filler_1288 key_9 is value_33 filler_179 filler_990 filler_1482 filler_1410 filler_1238 key_23 is value_149 filler_874 filler_1246 filler_1736 key_76 is value_241 filler_531 filler_1076 filler_1177 key_82 is value_68 filler_724 filler_1898 filler_265 filler_1258 filler_1152 key_94 is value_62 filler_1064 filler_1493 filler_1294 filler_1019 filler_1654 filler_368 filler_1809 filler_1248 filler_476 filler_1451 filler_594 filler_396 filler_3 filler_242 filler_49 filler_952 filler_1280 filler_1631 filler_241 filler_1510 filler_1736 filler_424 filler_1637 filler_1260 filler_849 filler_847 filler_93 filler_683 filler_716 filler_1147 filler_1650 filler_1305 filler_434 filler_23 filler_1311 filler_412 filler_1413 filler_576 filler_521 fil

In [ ]:
def normalize_lm_batch(batch):
    """
    Convierte el batch a formato compatible con model(**batch).
    """

    if isinstance(batch, dict):
        if "input_ids" not in batch:
            raise KeyError(
                f"El batch dict debe tener 'input_ids'. Keys disponibles: {list(batch.keys())}"
            )

        if "labels" not in batch:
            batch = dict(batch)
            batch["labels"] = batch["input_ids"]

        return batch

    if torch.is_tensor(batch):
        return {
            "input_ids": batch,
            "labels": batch,
        }

    if isinstance(batch, (list, tuple)):
        if len(batch) == 2:
            input_ids, labels = batch
            return {
                "input_ids": input_ids,
                "labels": labels,
            }

        if len(batch) == 3:
            input_ids, labels, attention_mask = batch
            return {
                "input_ids": input_ids,
                "labels": labels,
                "attention_mask": attention_mask,
            }

        raise ValueError(
            f"Batch list/tuple no soportado. Esperaba longitud 2 o 3, pero llegó {len(batch)}."
        )

    raise TypeError(f"Tipo de batch no soportado: {type(batch)}")

---

In [ ]:
# ============================================================
# TINYSTORIES DATA PIPELINE
# Mini DeepSeek-V4 / GPT-style causal LM
# ============================================================

from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers, decoders
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import os

# ============================================================
# CONFIG
# ============================================================

DATASET_NAME = "roneneldan/TinyStories"

# TinyStories tiene train / validation
TRAIN_SPLIT = "train"
VAL_SPLIT = "validation"

VOCAB_SIZE = 16000
MIN_FREQ = 2

BLOCK_SIZE = 256          # Para tarjetas gratis: 256 o 512
BATCH_SIZE = 64

TOKENIZER_PATH = Path("tinystories_tokenizer.json")

CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS = 2 if CPU_COUNT <= 2 else min(4, CPU_COUNT - 1)

os.environ["TOKENIZERS_PARALLELISM"] = "false"


# ============================================================
# LOAD DATASET
# ============================================================

def load_tinystories():
    """
    Carga TinyStories desde Hugging Face.

    Dataset:
      roneneldan/TinyStories

    Splits esperados:
      - train
      - validation
    """
    ds = load_dataset(DATASET_NAME)

    train_ds = ds[TRAIN_SPLIT]
    val_ds = ds[VAL_SPLIT]

    print(train_ds)
    print(val_ds)

    return train_ds, val_ds


# ============================================================
# TOKENIZER
# ============================================================

def train_tokenizer(
    train_ds,
    vocab_size=VOCAB_SIZE,
    min_freq=MIN_FREQ,
    save_path=TOKENIZER_PATH,
):
    """
    Entrena un tokenizer BPE byte-level estilo GPT.

    Nota importante:
    - NO usamos lowercase.
    - Queremos preservar mayúsculas, nombres, puntuación y estructura.
    """

    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

    tokenizer.normalizer = normalizers.Sequence([
        normalizers.NFKC(),])

    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()

    special_tokens = ["<unk>", "<pad>", "<bos>", "<eos>"]

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_freq,
        special_tokens=special_tokens,
    )

    def batch_iterator():
        for ex in train_ds:
            txt = ex["text"]
            if txt is not None and len(txt.strip()) > 0:
                yield txt

    print("Entrenando tokenizer BPE sobre TinyStories...")
    tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

    print("Tamaño vocabulario:", tokenizer.get_vocab_size())

    save_path = Path(save_path)
    tokenizer.save(str(save_path))

    print(f"Tokenizer guardado en: {save_path.resolve()}")

    return tokenizer


def load_or_train_tokenizer(train_ds):
    """
    Carga tokenizer si existe; si no, lo entrena.
    """
    if TOKENIZER_PATH.exists():
        print(f"Cargando tokenizer desde {TOKENIZER_PATH}...")
        tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
    else:
        tokenizer = train_tokenizer(train_ds)

    return tokenizer


# ============================================================
# DATASET CAUSAL LM
# ============================================================

class TinyStoriesCausalDataset(Dataset):
    """
    Dataset autoregresivo:

      - Concatena historias.
      - Añade <bos> al inicio y <eos> al final de cada historia.
      - Parte en chunks de block_size + 1.
      - input_ids = ids[:-1]
      - labels    = ids[1:]

    Devuelve:
      input_ids: [block_size]
      labels:    [block_size]
    """

    def __init__(self, hf_split, tokenizer, block_size=BLOCK_SIZE):
        super().__init__()

        self.block_size = block_size

        bos_id = tokenizer.token_to_id("<bos>")
        eos_id = tokenizer.token_to_id("<eos>")
        pad_id = tokenizer.token_to_id("<pad>")

        if bos_id is None:
            raise ValueError("El tokenizer no tiene token <bos>.")
        if eos_id is None:
            raise ValueError("El tokenizer no tiene token <eos>.")
        if pad_id is None:
            raise ValueError("El tokenizer no tiene token <pad>.")

        all_ids = []

        print("Tokenizando y concatenando TinyStories...")

        for ex in hf_split:
            txt = ex["text"]

            if txt is None or len(txt.strip()) == 0:
                continue

            enc = tokenizer.encode(txt)

            # Añadimos frontera explícita de documento/historia
            all_ids.extend([bos_id] + enc.ids + [eos_id])

        self.data = torch.tensor(all_ids, dtype=torch.long)

        print(f"Total de tokens en split: {len(self.data):,}")

        chunk_len = block_size + 1
        n_chunks = len(self.data) // chunk_len

        if n_chunks == 0:
            raise ValueError(
                "Muy pocos tokens para formar chunks. "
                "Baja BLOCK_SIZE o usa más datos.")

        self.data = self.data[: n_chunks * chunk_len]
        self.data = self.data.view(n_chunks, chunk_len)

        self.inputs = self.data[:, :-1]
        self.targets = self.data[:, 1:]

        print(f"Número de secuencias: {len(self.inputs):,}")
        print(f"Forma inputs:  {self.inputs.shape}")
        print(f"Forma targets: {self.targets.shape}")

    def __len__(self):
        return self.inputs.size(0)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]


# ============================================================
# OPTIONAL MTP DATASET
# ============================================================

class TinyStoriesMTPDataset(Dataset):
    """
    Dataset para Multi-Token Prediction.

    Devuelve diccionario:

      {
        "input_ids":  [block_size],
        "labels":     [block_size],              # x_{t+1}
        "mtp_labels": [mtp_depth, block_size],   # x_{t+2}, x_{t+3}, ...
      }
    """

    def __init__(self, hf_split, tokenizer, block_size=BLOCK_SIZE, mtp_depth=1):
        super().__init__()

        self.block_size = block_size
        self.mtp_depth = mtp_depth

        bos_id = tokenizer.token_to_id("<bos>")
        eos_id = tokenizer.token_to_id("<eos>")

        if bos_id is None:
            raise ValueError("El tokenizer no tiene token <bos>.")
        if eos_id is None:
            raise ValueError("El tokenizer no tiene token <eos>.")

        all_ids = []

        print("Tokenizando TinyStories para MTP...")

        for ex in hf_split:
            txt = ex["text"]

            if txt is None or len(txt.strip()) == 0:
                continue

            enc = tokenizer.encode(txt)
            all_ids.extend([bos_id] + enc.ids + [eos_id])

        self.data = torch.tensor(all_ids, dtype=torch.long)

        print(f"Total de tokens en split: {len(self.data):,}")

        chunk_len = block_size + 1 + mtp_depth
        n_chunks = len(self.data) // chunk_len

        if n_chunks == 0:
            raise ValueError(
                "Muy pocos tokens para formar chunks MTP. "
                "Baja BLOCK_SIZE o usa más datos.")

        self.data = self.data[: n_chunks * chunk_len]
        self.data = self.data.view(n_chunks, chunk_len)

        print(f"Número de secuencias MTP: {len(self.data):,}")
        print(f"Forma data: {self.data.shape}")

    def __len__(self):
        return self.data.size(0)

    def __getitem__(self, idx):
        ids = self.data[idx]

        input_ids = ids[:self.block_size]
        labels = ids[1:self.block_size + 1]

        mtp_labels = []

        for k in range(2, 2 + self.mtp_depth):
            mtp_labels.append(ids[k:k + self.block_size])

        mtp_labels = torch.stack(mtp_labels, dim=0)

        return {
            "input_ids": input_ids,
            "labels": labels,
            "mtp_labels": mtp_labels}


# ============================================================
# DATALOADERS
# ============================================================

def create_tinystories_dataloaders(
    block_size=BLOCK_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_mtp=False,
    mtp_depth=1):

    train_hf, val_hf = load_tinystories()

    tokenizer = load_or_train_tokenizer(train_hf)

    if use_mtp:
        train_ds = TinyStoriesMTPDataset(
            train_hf,
            tokenizer,
            block_size=block_size,
            mtp_depth=mtp_depth)

        val_ds = TinyStoriesMTPDataset(
            val_hf,
            tokenizer,
            block_size=block_size,
            mtp_depth=mtp_depth)

    else:
        train_ds = TinyStoriesCausalDataset(
            train_hf,
            tokenizer,
            block_size=block_size)

        val_ds = TinyStoriesCausalDataset(
            val_hf,
            tokenizer,
            block_size=block_size)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available())

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available())

    return train_loader, val_loader, tokenizer


In [ ]:
train_loader, val_loader, tokenizer = create_tinystories_dataloaders(
        block_size=BLOCK_SIZE,
        batch_size=BATCH_SIZE,
        use_mtp=False)

x, y = next(iter(train_loader))

print("Batch x shape:", x.shape)
print("Batch y shape:", y.shape)

print("\nTexto ejemplo input:")
print(tokenizer.decode(x[0].tolist()))

print("\nTexto ejemplo target:")
print(tokenizer.decode(y[0].tolist()))

In [ ]:
train_loader, val_loader, tokenizer = create_tinystories_dataloaders(
    block_size=256,
    batch_size=64,
    use_mtp=True,
    mtp_depth=1)

batch = next(iter(train_loader))

print(batch["input_ids"].shape)   # [B, T]
print(batch["labels"].shape)      # [B, T]
print(batch["mtp_labels"].shape)  # [B, mtp_depth, T]

---

# Embedding para DeepSeek4

In [115]:
# ============================================================
# Mini DeepSeek-V4 Token Embedding Module
# Token identity only — no positional embeddings
# ============================================================

import math
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn


# ============================================================
# CONFIG
# ============================================================

@dataclass
class EmbeddingConfig:
    vocab_size: int
    d_model: int

    pad_token_id: Optional[int] = None
    max_seq_len: int = 1024

    embedding_dropout: float = 0.0
    scale_embeddings: bool = False

    init_std: float = 0.02
    tie_word_embeddings: bool = True

    def validate(self) -> None:
        """
        Validate embedding configuration early.

        This prevents silent configuration bugs before constructing
        nn.Embedding or starting training.
        """
        if self.vocab_size <= 0:
            raise ValueError(f"vocab_size must be > 0, got {self.vocab_size}")

        if self.d_model <= 0:
            raise ValueError(f"d_model must be > 0, got {self.d_model}")

        if self.max_seq_len <= 0:
            raise ValueError(f"max_seq_len must be > 0, got {self.max_seq_len}")

        if not (0.0 <= self.embedding_dropout < 1.0):
            raise ValueError(
                "embedding_dropout must satisfy 0 <= embedding_dropout < 1, "
                f"got {self.embedding_dropout}")

        if self.init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {self.init_std}")

        if self.pad_token_id is not None:
            if not (0 <= self.pad_token_id < self.vocab_size):
                raise ValueError(
                    "pad_token_id must satisfy 0 <= pad_token_id < vocab_size, "
                    f"got pad_token_id={self.pad_token_id}, "
                    f"vocab_size={self.vocab_size}")


# ============================================================
# TOKEN EMBEDDING
# ============================================================

class TokenEmbedding(nn.Module):
    """
    Token embedding module for causal language modeling.

    Responsibility:
        input_ids: [B, T] -> hidden_states: [B, T, d_model]

    This module intentionally does NOT include:
        - positional embeddings
        - RoPE
        - attention masks
        - RMSNorm
        - MoE
        - mHC
        - loss logic

    Positional information should be handled later inside attention,
    e.g. via RoPE or partial RoPE.
    """

    def __init__(self, config: EmbeddingConfig):
        super().__init__()

        config.validate()

        self.config = config
        self.vocab_size = config.vocab_size
        self.d_model = config.d_model
        self.pad_token_id = config.pad_token_id
        self.max_seq_len = config.max_seq_len
        self.scale_embeddings = config.scale_embeddings

        self.token_embedding = nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id,)

        self.dropout = nn.Dropout(config.embedding_dropout)

        self.reset_parameters()

    def reset_parameters(self) -> None:
        """
        Initialize token embedding weights.

        Uses GPT-style normal initialization:
            weight ~ N(0, init_std)

        If pad_token_id is provided, its row is forced to zero.
        nn.Embedding(..., padding_idx=...) also prevents that row
        from receiving gradient updates.
        """
        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=self.config.init_std,)

        if self.pad_token_id is not None:
            with torch.no_grad():
                self.token_embedding.weight[self.pad_token_id].zero_()

    def forward(
          self,
          input_ids: Union[torch.Tensor, Dict[str, torch.Tensor]],) -> torch.Tensor:
          """
          Args:
              input_ids:
                  Either:
                    - Tensor [B, T]
                    - Dict with key "input_ids"

          Returns:
              hidden_states:
                  Float tensor [B, T, d_model]
          """

          # ----------------------------------------------------
          # Accept dict batches defensively
          # ----------------------------------------------------
          if isinstance(input_ids, dict):
              if "input_ids" not in input_ids:
                  raise KeyError(
                      "TokenEmbedding received a dict batch but it does not contain "
                      f"'input_ids'. Available keys: {list(input_ids.keys())}"
                  )

              input_ids = input_ids["input_ids"]

          # ----------------------------------------------------
          # Shape validation
          # ----------------------------------------------------
          if not torch.is_tensor(input_ids):
              raise TypeError(
                  "TokenEmbedding expects either a tensor [B, T] or a dict containing "
                  f"'input_ids'. Got type: {type(input_ids)}"
              )

          if input_ids.dim() != 2:
              raise ValueError(
                  f"input_ids must have shape [B, T], got {tuple(input_ids.shape)}"
              )

          _, seq_len = input_ids.shape

          if seq_len > self.max_seq_len:
              raise ValueError(
                  f"Sequence length T={seq_len} exceeds max_seq_len={self.max_seq_len}"
              )

          # ----------------------------------------------------
          # Dtype validation
          # ----------------------------------------------------
          if input_ids.dtype not in (torch.long, torch.int64, torch.int32):
              raise TypeError(
                  "input_ids must contain integer token indices; "
                  f"got dtype={input_ids.dtype}"
              )

          if input_ids.dtype != torch.long:
              input_ids = input_ids.long()

          # ----------------------------------------------------
          # Range validation
          # ----------------------------------------------------
          if torch.any(input_ids < 0):
              min_id = int(input_ids.min().item())
              raise ValueError(
                  f"input_ids contain negative token ids. Minimum id found: {min_id}"
              )

          if torch.any(input_ids >= self.vocab_size):
              max_id = int(input_ids.max().item())
              raise ValueError(
                  "input_ids contain token ids >= vocab_size. "
                  f"Maximum id found: {max_id}, vocab_size={self.vocab_size}"
              )

          # ----------------------------------------------------
          # Token embedding lookup
          # ----------------------------------------------------
          hidden_states = self.token_embedding(input_ids)

          # ----------------------------------------------------
          # Optional scaling
          # ----------------------------------------------------
          if self.scale_embeddings:
              hidden_states = hidden_states * math.sqrt(self.d_model)

          # ----------------------------------------------------
          # Dropout
          # ----------------------------------------------------
          hidden_states = self.dropout(hidden_states)

          return hidden_states

    @property
    def weight(self) -> nn.Parameter:
        """
        Expose embedding weight for optional LM-head weight tying.

        Example in full model:
            lm_head.weight = embedding.weight
        """
        return self.token_embedding.weight

In [116]:
# ============================================================
# Smoke Test
# ============================================================

config = EmbeddingConfig(
    vocab_size=16000,
    d_model=256,
    pad_token_id=0,
    max_seq_len=512,
    embedding_dropout=0.0,
    scale_embeddings=False,
    init_std=0.02,
    tie_word_embeddings=True,)

embedding = TokenEmbedding(config)

B, T = 4, 128

input_ids = torch.randint(
    low=0,
    high=config.vocab_size,
    size=(B, T),
    dtype=torch.long,)

hidden_states = embedding(input_ids)

print("input_ids shape:", input_ids.shape)
print("hidden_states shape:", hidden_states.shape)
print("hidden_states dtype:", hidden_states.dtype)

assert hidden_states.shape == (B, T, config.d_model)
assert torch.isfinite(hidden_states).all()

if config.pad_token_id is not None:
    pad_row = embedding.weight[config.pad_token_id]
    assert torch.allclose(pad_row, torch.zeros_like(pad_row))

print("TokenEmbedding module OK.")

input_ids shape: torch.Size([4, 128])
hidden_states shape: torch.Size([4, 128, 256])
hidden_states dtype: torch.float32
TokenEmbedding module OK.


## Testing for embeddings

In [121]:
# @title
# ============================================================
# tests/test_token_embedding.py
# ============================================================

import math

import pytest
import torch



# ============================================================
# Fixtures
# ============================================================

@pytest.fixture
def base_config():
    return EmbeddingConfig(
        vocab_size=128,
        d_model=32,
        pad_token_id=0,
        max_seq_len=64,
        embedding_dropout=0.0,
        scale_embeddings=False,
        init_std=0.02,
        tie_word_embeddings=True,
    )


@pytest.fixture
def embedding(base_config):
    return TokenEmbedding(base_config)


@pytest.fixture
def input_ids(base_config):
    return torch.randint(
        low=1,
        high=base_config.vocab_size,
        size=(4, 16),
        dtype=torch.long,
    )


# ============================================================
# A. Config tests
# ============================================================

def test_valid_config_builds_embedding(base_config):
    embedding = TokenEmbedding(base_config)

    assert embedding.vocab_size == base_config.vocab_size
    assert embedding.d_model == base_config.d_model
    assert embedding.pad_token_id == base_config.pad_token_id
    assert embedding.max_seq_len == base_config.max_seq_len


@pytest.mark.parametrize(
    "field,value",
    [
        ("vocab_size", 0),
        ("vocab_size", -1),
        ("d_model", 0),
        ("d_model", -1),
        ("max_seq_len", 0),
        ("max_seq_len", -1),
        ("embedding_dropout", -0.1),
        ("embedding_dropout", 1.0),
        ("embedding_dropout", 1.5),
        ("init_std", 0.0),
        ("init_std", -0.01),
        ("pad_token_id", -1),
        ("pad_token_id", 128),
    ],
)
def test_invalid_config_raises_error(base_config, field, value):
    kwargs = vars(base_config).copy()
    kwargs[field] = value

    bad_config = EmbeddingConfig(**kwargs)

    with pytest.raises(ValueError):
        TokenEmbedding(bad_config)


# ============================================================
# B. Forward basic tests
# ============================================================

def test_output_shape(embedding, input_ids, base_config):
    hidden_states = embedding(input_ids)

    assert hidden_states.shape == (
        input_ids.shape[0],
        input_ids.shape[1],
        base_config.d_model,
    )


def test_output_dtype_is_floating_point(embedding, input_ids):
    hidden_states = embedding(input_ids)

    assert hidden_states.dtype.is_floating_point


def test_accepts_int32_input_ids(embedding, base_config):
    input_ids = torch.randint(
        low=1,
        high=base_config.vocab_size,
        size=(4, 16),
        dtype=torch.int32,
    )

    hidden_states = embedding(input_ids)

    assert hidden_states.shape == (4, 16, base_config.d_model)
    assert hidden_states.dtype.is_floating_point


def test_rejects_float_input_ids(embedding):
    input_ids = torch.ones((4, 16), dtype=torch.float32)

    with pytest.raises(TypeError):
        embedding(input_ids)


def test_rejects_wrong_shape_1d(embedding):
    input_ids = torch.ones((16,), dtype=torch.long)

    with pytest.raises(ValueError):
        embedding(input_ids)


def test_rejects_wrong_shape_3d(embedding):
    input_ids = torch.ones((4, 16, 1), dtype=torch.long)

    with pytest.raises(ValueError):
        embedding(input_ids)


def test_rejects_sequence_longer_than_max_seq_len(embedding, base_config):
    input_ids = torch.ones(
        (4, base_config.max_seq_len + 1),
        dtype=torch.long,
    )

    with pytest.raises(ValueError):
        embedding(input_ids)


def test_rejects_negative_token_ids(embedding):
    input_ids = torch.ones((4, 16), dtype=torch.long)
    input_ids[0, 0] = -1

    with pytest.raises(ValueError):
        embedding(input_ids)


def test_rejects_out_of_vocab_token_ids(embedding, base_config):
    input_ids = torch.ones((4, 16), dtype=torch.long)
    input_ids[0, 0] = base_config.vocab_size

    with pytest.raises(ValueError):
        embedding(input_ids)


# ============================================================
# C. Initialization tests
# ============================================================

def test_embedding_weight_shape(embedding, base_config):
    assert embedding.token_embedding.weight.shape == (
        base_config.vocab_size,
        base_config.d_model,
    )


def test_embedding_weights_are_finite_after_init(embedding):
    assert torch.isfinite(embedding.token_embedding.weight).all()


def test_padding_embedding_is_zero_after_init(embedding, base_config):
    pad_token_id = base_config.pad_token_id

    assert pad_token_id is not None

    pad_row = embedding.weight[pad_token_id]
    expected = torch.zeros_like(pad_row)

    assert torch.allclose(pad_row, expected)


def test_non_padding_embeddings_have_reasonable_std(base_config):
    torch.manual_seed(123)

    embedding = TokenEmbedding(base_config)

    weights = embedding.weight.detach()

    if base_config.pad_token_id is not None:
        mask = torch.ones(base_config.vocab_size, dtype=torch.bool)
        mask[base_config.pad_token_id] = False
        weights = weights[mask]

    mean = weights.mean().item()
    std = weights.std(unbiased=False).item()

    assert abs(mean) < base_config.init_std
    assert abs(std - base_config.init_std) < base_config.init_std


# ============================================================
# D. Padding tests
# ============================================================

@pytest.mark.parametrize("scale_embeddings", [False, True])
def test_padding_output_is_zero_when_dropout_zero(base_config, scale_embeddings):
    config = EmbeddingConfig(
        **{
            **vars(base_config),
            "embedding_dropout": 0.0,
            "scale_embeddings": scale_embeddings,
        }
    )

    embedding = TokenEmbedding(config)

    input_ids = torch.tensor(
        [
            [0, 1, 2, 3],
            [4, 0, 5, 6],
        ],
        dtype=torch.long,
    )

    hidden_states = embedding(input_ids)

    pad_mask = input_ids == config.pad_token_id
    pad_outputs = hidden_states[pad_mask]

    assert torch.allclose(pad_outputs, torch.zeros_like(pad_outputs))


def test_padding_embedding_gets_zero_gradient(base_config):
    config = EmbeddingConfig(
        **{
            **vars(base_config),
            "embedding_dropout": 0.0,
            "scale_embeddings": False,
        }
    )

    embedding = TokenEmbedding(config)

    input_ids = torch.tensor(
        [
            [0, 1, 2, 3],
            [4, 0, 5, 6],
        ],
        dtype=torch.long,
    )

    hidden_states = embedding(input_ids)
    loss = hidden_states.sum()
    loss.backward()

    pad_grad = embedding.weight.grad[config.pad_token_id]

    assert torch.allclose(pad_grad, torch.zeros_like(pad_grad))


def test_non_padding_tokens_get_gradient(base_config):
    config = EmbeddingConfig(
        **{
            **vars(base_config),
            "embedding_dropout": 0.0,
            "scale_embeddings": False,
        }
    )

    embedding = TokenEmbedding(config)

    token_id = 7

    input_ids = torch.tensor(
        [
            [0, token_id, 2, 3],
            [4, 0, 5, 6],
        ],
        dtype=torch.long,
    )

    hidden_states = embedding(input_ids)
    loss = hidden_states.sum()
    loss.backward()

    token_grad = embedding.weight.grad[token_id]

    assert token_grad.abs().sum() > 0


# ============================================================
# E. Scaling tests
# ============================================================

def test_forward_without_scaling_matches_raw_lookup(base_config):
    config = EmbeddingConfig(
        **{
            **vars(base_config),
            "embedding_dropout": 0.0,
            "scale_embeddings": False,
        }
    )

    embedding = TokenEmbedding(config)

    input_ids = torch.tensor(
        [
            [1, 2, 3],
            [4, 5, 6],
        ],
        dtype=torch.long,
    )

    out = embedding(input_ids)
    expected = embedding.token_embedding(input_ids)

    assert torch.allclose(out, expected)


def test_forward_with_scaling_matches_sqrt_d_model(base_config):
    config = EmbeddingConfig(
        **{
            **vars(base_config),
            "embedding_dropout": 0.0,
            "scale_embeddings": True,
        }
    )

    embedding = TokenEmbedding(config)

    input_ids = torch.tensor(
        [
            [1, 2, 3],
            [4, 5, 6],
        ],
        dtype=torch.long,
    )

    out = embedding(input_ids)
    expected = embedding.token_embedding(input_ids) * math.sqrt(config.d_model)

    assert torch.allclose(out, expected)


# ============================================================
# F. Dropout tests
# ============================================================

def test_dropout_zero_is_deterministic(base_config):
    config = EmbeddingConfig(
        **{
            **vars(base_config),
            "embedding_dropout": 0.0,
        }
    )

    embedding = TokenEmbedding(config)
    embedding.train()

    input_ids = torch.randint(
        low=1,
        high=config.vocab_size,
        size=(8, 32),
        dtype=torch.long,
    )

    out1 = embedding(input_ids)
    out2 = embedding(input_ids)

    assert torch.equal(out1, out2)


def test_dropout_active_changes_output_in_train_mode(base_config):
    config = EmbeddingConfig(
        **{
            **vars(base_config),
            "embedding_dropout": 0.5,
        }
    )

    embedding = TokenEmbedding(config)
    embedding.train()

    input_ids = torch.randint(
        low=1,
        high=config.vocab_size,
        size=(16, 64),
        dtype=torch.long,)

    out1 = embedding(input_ids)
    out2 = embedding(input_ids)

    assert not torch.equal(out1, out2)


def test_dropout_disabled_in_eval_mode(base_config):
    config = EmbeddingConfig(
        **{
            **vars(base_config),
            "embedding_dropout": 0.5,
        } )

    embedding = TokenEmbedding(config)
    embedding.eval()

    input_ids = torch.randint(
        low=1,
        high=config.vocab_size,
        size=(16, 64),
        dtype=torch.long,)

    out1 = embedding(input_ids)
    out2 = embedding(input_ids)

    assert torch.equal(out1, out2)


# ============================================================
# G. Device and dtype tests
# ============================================================

def test_embedding_runs_on_cpu(base_config):
    embedding = TokenEmbedding(base_config).cpu()

    input_ids = torch.randint(
        low=1,
        high=base_config.vocab_size,
        size=(4, 16),
        dtype=torch.long,
        device="cpu",)

    hidden_states = embedding(input_ids)

    assert hidden_states.device == input_ids.device
    assert hidden_states.shape == (4, 16, base_config.d_model)


@pytest.mark.skipif(not torch.cuda.is_available(), reason="CUDA is not available")
def test_embedding_runs_on_cuda_if_available(base_config):
    embedding = TokenEmbedding(base_config).cuda()

    input_ids = torch.randint(
        low=1,
        high=base_config.vocab_size,
        size=(4, 16),
        dtype=torch.long,
        device="cuda",)

    hidden_states = embedding(input_ids)

    assert hidden_states.device.type == "cuda"
    assert hidden_states.shape == (4, 16, base_config.d_model)


def test_embedding_respects_module_dtype_bfloat16_if_supported(base_config):
    embedding = TokenEmbedding(base_config).to(dtype=torch.bfloat16)

    input_ids = torch.randint(
        low=1,
        high=base_config.vocab_size,
        size=(4, 16),
        dtype=torch.long,)

    hidden_states = embedding(input_ids)

    assert hidden_states.dtype == torch.bfloat16


# ============================================================
# H. Optional integration test with synthetic dataset
# ============================================================
def test_synthetic_dataset_batch_passes_embedding():
    """
    Integration test: synthetic retrieval dataset -> dict batch -> embedding.
    """

    data_cfg = SyntheticRetrievalConfig(
        num_train_examples=128,
        num_val_examples=32,
        block_size=64,
        min_filler_tokens=8,
        max_filler_tokens=32,
        num_keys_per_example=4,
        vocab_filler_size=100,
        num_key_types=32,
        num_value_types=64,
        batch_size=8,
        num_workers=0,
        seed=123,
    )

    train_loader, _, tokenizer = create_synthetic_retrieval_dataloaders(
        cfg=data_cfg,
        use_mtp=False,
    )

    batch = next(iter(train_loader))

    assert isinstance(batch, dict)
    assert "input_ids" in batch
    assert "labels" in batch

    input_ids = batch["input_ids"]
    labels = batch["labels"]

    config = EmbeddingConfig(
        vocab_size=tokenizer.vocab_size,
        d_model=32,
        pad_token_id=tokenizer.pad_id,
        max_seq_len=data_cfg.block_size,
        embedding_dropout=0.0,
        scale_embeddings=False,
        init_std=0.02,
        tie_word_embeddings=True,
    )

    embedding = TokenEmbedding(config)

    hidden_states = embedding(input_ids)

    assert input_ids.shape == (data_cfg.batch_size, data_cfg.block_size)
    assert labels.shape == (data_cfg.batch_size, data_cfg.block_size)
    assert input_ids.min() >= 0
    assert input_ids.max() < config.vocab_size

    assert hidden_states.shape == (
        data_cfg.batch_size,
        data_cfg.block_size,
        config.d_model,
    )

    assert torch.isfinite(hidden_states).all()


def test_tokenizer_pad_id_matches_embedding_config():
    """
    This test assumes the synthetic tokenizer exposes pad_id.

    If your tokenizer lives elsewhere, adjust the import.
    """

    data_cfg = SyntheticRetrievalConfig()
    tokenizer = SimpleWordTokenizer()
    tokenizer.build_vocab(data_cfg)

    config = EmbeddingConfig(
        vocab_size=tokenizer.vocab_size,
        d_model=32,
        pad_token_id=tokenizer.pad_id,
        max_seq_len=data_cfg.block_size,
        embedding_dropout=0.0,
        scale_embeddings=False,
        init_std=0.02,
        tie_word_embeddings=True,)

    assert config.pad_token_id == tokenizer.pad_id


# ============================================================
# I. Weight tying preparation
# ============================================================

def test_embedding_weight_property_exposes_parameter(embedding):
    assert embedding.weight is embedding.token_embedding.weight

In [123]:
import inspect
import torch

# ============================================================
# Fixtures manuales
# ============================================================

base_config = EmbeddingConfig(
    vocab_size=128,
    d_model=32,
    pad_token_id=0,
    max_seq_len=64,
    embedding_dropout=0.0,
    scale_embeddings=False,
    init_std=0.02,
    tie_word_embeddings=True,
)

embedding = TokenEmbedding(base_config)

input_ids = torch.randint(
    low=1,
    high=base_config.vocab_size,
    size=(4, 16),
    dtype=torch.long,
)

fixtures = {
    "base_config": base_config,
    "embedding": embedding,
    "input_ids": input_ids,
}

# ============================================================
# SOLO tests de TokenEmbedding
# ============================================================

embedding_tests = [
    # Config
    "test_valid_config_builds_embedding",
    "test_invalid_config_raises_error",

    # Forward basic
    "test_output_shape",
    "test_output_dtype_is_floating_point",
    "test_accepts_int32_input_ids",
    "test_rejects_float_input_ids",
    "test_rejects_wrong_shape_1d",
    "test_rejects_wrong_shape_3d",
    "test_rejects_sequence_longer_than_max_seq_len",
    "test_rejects_negative_token_ids",
    "test_rejects_out_of_vocab_token_ids",

    # Initialization
    "test_embedding_weight_shape",
    "test_embedding_weights_are_finite_after_init",
    "test_padding_embedding_is_zero_after_init",
    "test_non_padding_embeddings_have_reasonable_std",

    # Padding
    "test_padding_output_is_zero_when_dropout_zero",
    "test_padding_embedding_gets_zero_gradient",
    "test_non_padding_tokens_get_gradient",

    # Scaling
    "test_forward_without_scaling_matches_raw_lookup",
    "test_forward_with_scaling_matches_sqrt_d_model",

    # Dropout
    "test_dropout_zero_is_deterministic",
    "test_dropout_active_changes_output_in_train_mode",
    "test_dropout_disabled_in_eval_mode",

    # Device / dtype
    "test_embedding_runs_on_cpu",
    "test_embedding_respects_module_dtype_bfloat16_if_supported",

    # Synthetic dataset integration
    "test_synthetic_dataset_batch_passes_embedding",
    "test_tokenizer_pad_id_matches_embedding_config",

    # Weight tying
    "test_embedding_weight_property_exposes_parameter",
]

if torch.cuda.is_available():
    embedding_tests.append("test_embedding_runs_on_cuda_if_available")

# ============================================================
# Parametrized tests manuales
# ============================================================

manual_cases = {
    "test_invalid_config_raises_error": [
        {"field": "vocab_size", "value": 0},
        {"field": "vocab_size", "value": -1},
        {"field": "d_model", "value": 0},
        {"field": "d_model", "value": -1},
        {"field": "max_seq_len", "value": 0},
        {"field": "max_seq_len", "value": -1},
        {"field": "embedding_dropout", "value": -0.1},
        {"field": "embedding_dropout", "value": 1.0},
        {"field": "embedding_dropout", "value": 1.5},
        {"field": "init_std", "value": 0.0},
        {"field": "init_std", "value": -0.01},
        {"field": "pad_token_id", "value": -1},
        {"field": "pad_token_id", "value": 128},
    ],
    "test_padding_output_is_zero_when_dropout_zero": [
        {"scale_embeddings": False},
        {"scale_embeddings": True},
    ],
}

# ============================================================
# Validar que no falten tests
# ============================================================

missing = [name for name in embedding_tests if name not in globals()]

if missing:
    raise KeyError(
        "Estos tests de TokenEmbedding no están definidos:\n"
        + "\n".join(f"  - {name}" for name in missing)
    )

# ============================================================
# Runner
# ============================================================

n_passed = 0

for name in embedding_tests:
    fn = globals()[name]
    sig = inspect.signature(fn)

    cases = manual_cases.get(name, [{}])

    for case in cases:
        kwargs = {}

        for param_name in sig.parameters:
            if param_name in case:
                kwargs[param_name] = case[param_name]
            elif param_name in fixtures:
                kwargs[param_name] = fixtures[param_name]
            else:
                raise ValueError(
                    f"El test '{name}' requiere '{param_name}', "
                    "pero no está definido en fixtures ni en manual_cases."
                )

        fn(**kwargs)
        n_passed += 1

        if case:
            print(f"PASS: {name}({case})")
        else:
            print(f"PASS: {name}")

print(f"\nOK: {n_passed} TokenEmbedding tests/casos pasaron.")

PASS: test_valid_config_builds_embedding
PASS: test_invalid_config_raises_error({'field': 'vocab_size', 'value': 0})
PASS: test_invalid_config_raises_error({'field': 'vocab_size', 'value': -1})
PASS: test_invalid_config_raises_error({'field': 'd_model', 'value': 0})
PASS: test_invalid_config_raises_error({'field': 'd_model', 'value': -1})
PASS: test_invalid_config_raises_error({'field': 'max_seq_len', 'value': 0})
PASS: test_invalid_config_raises_error({'field': 'max_seq_len', 'value': -1})
PASS: test_invalid_config_raises_error({'field': 'embedding_dropout', 'value': -0.1})
PASS: test_invalid_config_raises_error({'field': 'embedding_dropout', 'value': 1.0})
PASS: test_invalid_config_raises_error({'field': 'embedding_dropout', 'value': 1.5})
PASS: test_invalid_config_raises_error({'field': 'init_std', 'value': 0.0})
PASS: test_invalid_config_raises_error({'field': 'init_std', 'value': -0.01})
PASS: test_invalid_config_raises_error({'field': 'pad_token_id', 'value': -1})
PASS: test_inva

---

# Modulos base de Transformer


# RMSNorm

In [14]:
class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization.

    Input:
        x: [..., D]

    Forward:
        1. Save original dtype.
        2. Cast x to float32 for stable RMS computation.
        3. Compute mean(x^2) over last dimension.
        4. Apply rsqrt(mean_square + eps).
        5. Normalize x.
        6. Cast normalized x back to original dtype if needed.
        7. Multiply by learnable weight.
        8. Return y.

    Output:
        y: [..., D]
    """

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()

        if dim <= 0:
            raise ValueError(f"dim must be > 0, got {dim}")

        if eps <= 0:
            raise ValueError(f"eps must be > 0, got {eps}")

        self.dim = dim
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
      if x.shape[-1] != self.dim:
          raise ValueError(
              f"Expected last dimension to be dim={self.dim}, "
              f"but got x.shape[-1]={x.shape[-1]}"
          )

      original_dtype = x.dtype

      # Compute RMS in float32 for numerical stability
      x_float = x.float()

      mean_square = x_float.pow(2).mean(dim=-1, keepdim=True)
      inv_rms = torch.rsqrt(mean_square + self.eps)

      y = x_float * inv_rms

      # Cast normalized activations back to original dtype
      if y.dtype != original_dtype:
          y = y.to(original_dtype)

      # Important: cast weight too, otherwise output is promoted to float32
      weight = self.weight.to(dtype=original_dtype)

      y = y * weight

      return y

## Test for RMSNorm

In [15]:
# @title
def test_output_shape_matches_input_shape():
    norm = RMSNorm(dim=32)

    x = torch.randn(4, 16, 32)
    y = norm(x)

    assert y.shape == x.shape


def test_output_dtype_matches_input_dtype():
    norm = RMSNorm(dim=32)

    x = torch.randn(4, 16, 32, dtype=torch.float32)
    y = norm(x)

    assert y.dtype == x.dtype


def test_weight_shape_is_dim():
    norm = RMSNorm(dim=32)

    assert norm.weight.shape == (32,)


def test_weight_initialized_to_ones():
    norm = RMSNorm(dim=32)

    assert torch.allclose(norm.weight, torch.ones_like(norm.weight))


def test_forward_matches_manual_computation():
    dim = 32
    eps = 1e-6

    norm = RMSNorm(dim=dim, eps=eps)

    x = torch.randn(4, 16, dim)

    y = norm(x)

    x_float = x.float()
    mean_square = x_float.pow(2).mean(dim=-1, keepdim=True)
    expected = x_float * torch.rsqrt(mean_square + eps)
    expected = expected.to(x.dtype) * norm.weight

    assert torch.allclose(y, expected, atol=1e-6, rtol=1e-5)


def test_normalizes_last_dimension_only():
    dim = 8
    eps = 1e-6

    norm = RMSNorm(dim=dim, eps=eps)

    x = torch.randn(2, 3, 4, dim)
    y = norm(x)

    manual_mean_square = x.float().pow(2).mean(dim=-1, keepdim=True)
    expected = x.float() * torch.rsqrt(manual_mean_square + eps)
    expected = expected.to(x.dtype) * norm.weight

    assert y.shape == x.shape
    assert torch.allclose(y, expected, atol=1e-6, rtol=1e-5)


def test_no_nan_with_zero_input():
    norm = RMSNorm(dim=32)

    x = torch.zeros(4, 16, 32)
    y = norm(x)

    assert torch.isfinite(y).all()
    assert torch.allclose(y, torch.zeros_like(y))


def test_backward_produces_gradients():
    norm = RMSNorm(dim=32)

    x = torch.randn(4, 16, 32, requires_grad=True)
    y = norm(x)

    loss = y.sum()
    loss.backward()

    assert x.grad is not None
    assert norm.weight.grad is not None

    assert torch.isfinite(x.grad).all()
    assert torch.isfinite(norm.weight.grad).all()


def test_supports_3d_input_BTD():
    norm = RMSNorm(dim=32)

    x = torch.randn(4, 16, 32)
    y = norm(x)

    assert y.shape == (4, 16, 32)


def test_supports_4d_input_BHTD():
    norm = RMSNorm(dim=32)

    x = torch.randn(4, 8, 16, 32)
    y = norm(x)

    assert y.shape == (4, 8, 16, 32)


def test_handles_large_values_without_nan():
    norm = RMSNorm(dim=32)

    x = torch.full((4, 16, 32), 1e6)
    y = norm(x)

    assert torch.isfinite(y).all()


def test_handles_small_values_without_nan():
    norm = RMSNorm(dim=32)

    x = torch.full((4, 16, 32), 1e-8)
    y = norm(x)

    assert torch.isfinite(y).all()


def test_eps_prevents_division_by_zero():
    norm = RMSNorm(dim=32, eps=1e-6)

    x = torch.zeros(4, 16, 32)
    y = norm(x)

    assert torch.isfinite(y).all()


def test_bfloat16_input_returns_bfloat16_if_supported():
    norm = RMSNorm(dim=32)

    x = torch.randn(4, 16, 32).to(torch.bfloat16)
    y = norm(x)

    assert y.dtype == torch.bfloat16
    assert torch.isfinite(y.float()).all()


@pytest.mark.skipif(not torch.cuda.is_available(), reason="CUDA is not available")
def test_float16_input_returns_float16_on_cuda_if_available():
    norm = RMSNorm(dim=32).cuda()

    x = torch.randn(4, 16, 32, device="cuda", dtype=torch.float16)
    y = norm(x)

    assert y.dtype == torch.float16
    assert y.device.type == "cuda"
    assert torch.isfinite(y.float()).all()

In [16]:
# ============================================================
# Run ONLY RMSNorm tests in Jupyter
# ============================================================

rmsnorm_test_names = [
    "test_output_shape_matches_input_shape",
    "test_output_dtype_matches_input_dtype",
    "test_weight_shape_is_dim",
    "test_weight_initialized_to_ones",
    "test_forward_matches_manual_computation",
    "test_normalizes_last_dimension_only",
    "test_no_nan_with_zero_input",
    "test_backward_produces_gradients",
    "test_supports_3d_input_BTD",
    "test_supports_4d_input_BHTD",
    "test_handles_large_values_without_nan",
    "test_handles_small_values_without_nan",
    "test_eps_prevents_division_by_zero",
    "test_bfloat16_input_returns_bfloat16_if_supported",
]

# Este test solo corre si hay CUDA
if torch.cuda.is_available():
    rmsnorm_test_names.append("test_float16_input_returns_float16_on_cuda_if_available")

for name in rmsnorm_test_names:
    globals()[name]()

print(f"OK: {len(rmsnorm_test_names)} RMSNorm tests pasaron.")

OK: 15 RMSNorm tests pasaron.


---

# RoPe

In [17]:
# ============================================================
# Mini DeepSeek-V4 RoPE Utilities
# Rotary Positional Embedding — standalone utility
# ============================================================

from typing import Optional

import torch
import torch.nn as nn


# ============================================================
# rotate_half
# ============================================================

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """
    Rotate last dimension by splitting it into two halves.

    Input:
        x: [..., rotary_dim]

    Operation:
        x1 = x[..., :rotary_dim // 2]
        x2 = x[..., rotary_dim // 2:]
        return concat(-x2, x1, dim=-1)

    Output:
        rotated: [..., rotary_dim]

    Preserves:
        - shape
        - dtype
        - device
    """

    rotary_dim = x.shape[-1]

    if rotary_dim % 2 != 0:
        raise ValueError(
            f"rotate_half requires an even last dimension, got {rotary_dim}"
        )

    half = rotary_dim // 2

    x1 = x[..., :half]
    x2 = x[..., half:]

    return torch.cat((-x2, x1), dim=-1)


# ============================================================
# RotaryEmbedding
# ============================================================

class RotaryEmbedding(nn.Module):
    """
    Rotary Positional Embedding utility.

    Expected input:
        x: [B, T, H, D]

    where:
        B = batch size
        T = sequence length
        H = number of heads
        D = head_dim

    Supports:
        - full RoPE: rotary_dim == dim
        - partial RoPE: rotary_dim < dim
        - automatic positions with start_pos
        - explicit position_ids with shape [T]
        - explicit position_ids with shape [B, T]
        - negative positions
        - positions larger than max_seq_len

    This module does NOT implement:
        - attention
        - q/k/v projections
        - KV cache
        - RoPE scaling
        - learned positional embeddings
        - cos/sin caching
    """

    def __init__(
        self,
        dim: int,
        rotary_dim: Optional[int] = None,
        base: float = 10000.0):

        super().__init__()

        if rotary_dim is None:
            rotary_dim = dim

        if dim <= 0:
            raise ValueError(f"dim must be > 0, got {dim}")

        if rotary_dim <= 0:
            raise ValueError(f"rotary_dim must be > 0, got {rotary_dim}")

        if rotary_dim > dim:
            raise ValueError(
                f"rotary_dim must be <= dim, got rotary_dim={rotary_dim}, dim={dim}"
            )

        if rotary_dim % 2 != 0:
            raise ValueError(
                f"rotary_dim must be even, got rotary_dim={rotary_dim}"
            )

        if base <= 0:
            raise ValueError(f"base must be > 0, got {base}")

        self.dim = dim
        self.rotary_dim = rotary_dim
        self.base = base

        # Conceptually:
        # inv_freq[i] = 1 / base^(i / rotary_dim), for even rotary indices.
        #
        # torch.arange(0, rotary_dim, 2) gives:
        # 0, 2, 4, ...
        # so exponent becomes:
        # 0/rotary_dim, 2/rotary_dim, 4/rotary_dim, ...
        inv_freq = 1.0 / (
            base ** (
                torch.arange(0, rotary_dim, 2, dtype=torch.float32)
                / rotary_dim
            ))

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False)


    def _build_position_ids(
        self,
        batch_size: int,
        seq_len: int,
        device: torch.device,
        position_ids: Optional[torch.Tensor],
        start_pos: int,) -> torch.Tensor:
        """
        Build or validate position_ids.

        Supported:
            None      -> [T]
            [T]       -> [T]
            [B, T]    -> [B, T]

        Negative positions are allowed.
        """

        if position_ids is None:
            return torch.arange(
                start_pos,
                start_pos + seq_len,
                device=device,
                dtype=torch.float32)

        if position_ids.device != device:
            position_ids = position_ids.to(device)

        if position_ids.dim() == 1:
            if position_ids.shape[0] != seq_len:
                raise ValueError(
                    f"position_ids with shape [T] must have length T={seq_len}, "
                    f"got {position_ids.shape[0]}")

            return position_ids.float()

        if position_ids.dim() == 2:
            if position_ids.shape != (batch_size, seq_len):
                raise ValueError(
                    "position_ids with shape [B, T] must match input batch/length. "
                    f"Expected {(batch_size, seq_len)}, got {tuple(position_ids.shape)}")

            return position_ids.float()

        raise ValueError(
            "position_ids must be None, shape [T], or shape [B, T], "
            f"got shape {tuple(position_ids.shape)}")

    def _build_cos_sin(
        self,
        position_ids: torch.Tensor,
        target_dtype: torch.dtype,
        device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Build cos/sin tensors dynamically.

        If position_ids:
            [T]    -> cos/sin: [1, T, 1, rotary_dim]
            [B,T]  -> cos/sin: [B, T, 1, rotary_dim]
        """

        position_ids = position_ids.to(device=device, dtype=torch.float32)
        inv_freq = self.inv_freq.to(device=device, dtype=torch.float32)

        # freqs:
        # [T, rotary_dim // 2] or [B, T, rotary_dim // 2]
        freqs = position_ids[..., None] * inv_freq

        # Expand from half dimension to full rotary_dim.
        # Shape:
        # [T, rotary_dim] or [B, T, rotary_dim]
        emb = torch.cat((freqs, freqs), dim=-1)

        cos = torch.cos(emb)
        sin = torch.sin(emb)

        if position_ids.dim() == 1:
            # [T, R] -> [1, T, 1, R]
            cos = cos[None, :, None, :]
            sin = sin[None, :, None, :]
        else:
            # [B, T, R] -> [B, T, 1, R]
            cos = cos[:, :, None, :]
            sin = sin[:, :, None, :]

        cos = cos.to(dtype=target_dtype)
        sin = sin.to(dtype=target_dtype)

        return cos, sin

    def forward(
        self,
        x: torch.Tensor,
        position_ids: Optional[torch.Tensor] = None,
        start_pos: int = 0) -> torch.Tensor:
        """
        Args:
            x:
                Attention tensor with shape [B, T, H, D].

            position_ids:
                Optional positions:
                    None
                    [T]
                    [B, T]

            start_pos:
                Offset used only when position_ids is None.

        Returns:
            y:
                Tensor with same shape, dtype, and device as x.
        """

        if x.dim() != 4:
            raise ValueError(
                f"RotaryEmbedding expects x with shape [B, T, H, D], "
                f"got {tuple(x.shape)}")

        batch_size, seq_len, _, head_dim = x.shape

        if head_dim != self.dim:
            raise ValueError(
                f"Expected x.shape[-1] == dim={self.dim}, got {head_dim}")

        original_dtype = x.dtype
        device = x.device

        position_ids = self._build_position_ids(
            batch_size=batch_size,
            seq_len=seq_len,
            device=device,
            position_ids=position_ids,
            start_pos=start_pos,)

        cos, sin = self._build_cos_sin(
            position_ids=position_ids,
            target_dtype=original_dtype,
            device=device)

        pass_dim = self.dim - self.rotary_dim

        if pass_dim > 0:
            x_pass = x[..., :pass_dim]
            x_rot = x[..., pass_dim:]
        else:
            x_pass = None
            x_rot = x

        x_rotated = (x_rot * cos) + (rotate_half(x_rot) * sin)

        if x_pass is not None:
            y = torch.cat((x_pass, x_rotated), dim=-1)
        else:
            y = x_rotated

        return y

## Tests par RoPe

In [18]:
# @title
# ============================================================
# RoPE utilities tests
# Tests for rotate_half and RotaryEmbedding
# ============================================================

import pytest
import torch


# ============================================================
# A. rotate_half tests
# ============================================================

def test_rotate_half_shape():
    x = torch.randn(2, 4, 3, 8)
    y = rotate_half(x)

    assert y.shape == x.shape


def test_rotate_half_manual_values():
    x = torch.tensor([1.0, 2.0, 3.0, 4.0])
    y = rotate_half(x)

    expected = torch.tensor([-3.0, -4.0, 1.0, 2.0])

    assert torch.equal(y, expected)


def test_rotate_half_preserves_dtype_and_device():
    x = torch.randn(2, 4, 3, 8, dtype=torch.float32)
    y = rotate_half(x)

    assert y.dtype == x.dtype
    assert y.device == x.device


# ============================================================
# B. Config tests
# ============================================================

def test_valid_rope_config_builds():
    rope = RotaryEmbedding(dim=64, rotary_dim=64, base=10000.0)

    assert rope.dim == 64
    assert rope.rotary_dim == 64
    assert rope.base == 10000.0
    assert rope.inv_freq.shape == (32,)


def test_rotary_dim_none_defaults_to_dim():
    rope = RotaryEmbedding(dim=64, rotary_dim=None)

    assert rope.rotary_dim == 64


@pytest.mark.parametrize("dim", [0, -1, -64])
def test_invalid_dim_raises(dim):
    with pytest.raises(ValueError):
        RotaryEmbedding(dim=dim)


@pytest.mark.parametrize(
    "dim,rotary_dim",
    [
        (64, 0),
        (64, -1),
        (64, 65),
        (64, 7),
    ],
)
def test_invalid_rotary_dim_raises(dim, rotary_dim):
    with pytest.raises(ValueError):
        RotaryEmbedding(dim=dim, rotary_dim=rotary_dim)


@pytest.mark.parametrize("base", [0.0, -1.0, -10000.0])
def test_invalid_base_raises(base):
    with pytest.raises(ValueError):
        RotaryEmbedding(dim=64, rotary_dim=64, base=base)


# ============================================================
# C. Forward and shape tests
# ============================================================

def test_rope_output_shape_matches_input_shape():
    rope = RotaryEmbedding(dim=32, rotary_dim=32)

    x = torch.randn(2, 8, 4, 32)
    y = rope(x)

    assert y.shape == x.shape


@pytest.mark.parametrize(
    "bad_x",
    [
        torch.randn(2, 8, 32),
        torch.randn(8, 32),
        torch.randn(2, 8, 4, 32, 1),
    ],
)
def test_rejects_non_4d_input(bad_x):
    rope = RotaryEmbedding(dim=32, rotary_dim=32)

    with pytest.raises(ValueError):
        rope(bad_x)


def test_rejects_wrong_last_dim():
    rope = RotaryEmbedding(dim=64, rotary_dim=64)

    x = torch.randn(2, 8, 4, 32)

    with pytest.raises(ValueError):
        rope(x)


# ============================================================
# D. Mathematical property tests
# ============================================================

def test_position_zero_is_identity():
    rope = RotaryEmbedding(dim=32, rotary_dim=32)

    x = torch.randn(2, 8, 4, 32)
    y = rope(x)

    assert torch.allclose(y[:, 0, :, :], x[:, 0, :, :], atol=1e-6, rtol=1e-6)


def test_rotary_preserves_norm_of_rotated_part():
    D = 32
    rotary_dim = 16

    rope = RotaryEmbedding(dim=D, rotary_dim=rotary_dim)

    x = torch.randn(2, 8, 4, D)
    y = rope(x)

    x_rot = x[..., D - rotary_dim:]
    y_rot = y[..., D - rotary_dim:]

    x_norm = torch.linalg.vector_norm(x_rot, dim=-1)
    y_norm = torch.linalg.vector_norm(y_rot, dim=-1)

    assert torch.allclose(x_norm, y_norm, atol=1e-5, rtol=1e-5)


def test_partial_rope_leaves_pass_through_dimensions_unchanged():
    D = 16
    rotary_dim = 8

    rope = RotaryEmbedding(dim=D, rotary_dim=rotary_dim)

    x = torch.randn(2, 8, 4, D)
    y = rope(x)

    pass_dim = D - rotary_dim

    assert torch.allclose(
        y[..., :pass_dim],
        x[..., :pass_dim],
        atol=0.0,
        rtol=0.0,
    )


def test_full_rope_rotates_all_dimensions():
    D = 16

    rope = RotaryEmbedding(dim=D, rotary_dim=D)

    x = torch.randn(2, 8, 4, D)
    y = rope(x)

    x_norm = torch.linalg.vector_norm(x, dim=-1)
    y_norm = torch.linalg.vector_norm(y, dim=-1)

    assert y.shape == x.shape
    assert torch.allclose(x_norm, y_norm, atol=1e-5, rtol=1e-5)


def test_forward_matches_manual_computation():
    B, T, H, D = 2, 5, 3, 16
    rotary_dim = 8
    pass_dim = D - rotary_dim

    rope = RotaryEmbedding(dim=D, rotary_dim=rotary_dim)

    x = torch.randn(B, T, H, D)
    y = rope(x)

    x_pass = x[..., :pass_dim]
    x_rot = x[..., pass_dim:]

    position_ids = torch.arange(T, dtype=torch.float32)

    inv_freq = rope.inv_freq.float()
    freqs = position_ids[:, None] * inv_freq[None, :]
    emb = torch.cat((freqs, freqs), dim=-1)

    cos = torch.cos(emb)[None, :, None, :].to(dtype=x.dtype)
    sin = torch.sin(emb)[None, :, None, :].to(dtype=x.dtype)

    expected_rot = x_rot * cos + rotate_half(x_rot) * sin
    expected = torch.cat((x_pass, expected_rot), dim=-1)

    assert torch.allclose(y, expected, atol=1e-6, rtol=1e-5)


# ============================================================
# E. Position tests
# ============================================================

def test_start_pos_matches_explicit_position_ids():
    B, T, H, D = 2, 6, 3, 16
    start_pos = 10

    rope = RotaryEmbedding(dim=D, rotary_dim=D)

    x = torch.randn(B, T, H, D)

    y_start = rope(x, position_ids=None, start_pos=start_pos)

    explicit_ids = torch.arange(start_pos, start_pos + T)
    y_explicit = rope(x, position_ids=explicit_ids)

    assert torch.allclose(y_start, y_explicit, atol=1e-6, rtol=1e-5)


def test_accepts_position_ids_T():
    B, T, H, D = 2, 6, 3, 16

    rope = RotaryEmbedding(dim=D, rotary_dim=D)

    x = torch.randn(B, T, H, D)
    position_ids = torch.arange(T)

    y = rope(x, position_ids=position_ids)

    assert y.shape == x.shape


def test_accepts_position_ids_BT():
    B, T, H, D = 2, 6, 3, 16

    rope = RotaryEmbedding(dim=D, rotary_dim=D)

    x = torch.randn(B, T, H, D)

    position_ids = torch.stack(
        [
            torch.arange(T),
            torch.arange(10, 10 + T),
        ],
        dim=0,
    )

    y = rope(x, position_ids=position_ids)

    assert y.shape == x.shape


@pytest.mark.parametrize(
    "position_ids",
    [
        torch.arange(2),                       # [B], wrong when T != B
        torch.zeros(2, 6, 1, dtype=torch.long), # [B,T,1]
        torch.arange(7),                       # [T+1]
    ],
)
def test_rejects_invalid_position_ids_shape(position_ids):
    B, T, H, D = 2, 6, 3, 16

    rope = RotaryEmbedding(dim=D, rotary_dim=D)
    x = torch.randn(B, T, H, D)

    with pytest.raises(ValueError):
        rope(x, position_ids=position_ids)


def test_accepts_negative_position_ids():
    B, T, H, D = 2, 4, 3, 16

    rope = RotaryEmbedding(dim=D, rotary_dim=8)

    x = torch.randn(B, T, H, D)
    position_ids = torch.tensor([-3, -2, -1, 0])

    y = rope(x, position_ids=position_ids)

    x_rot = x[..., D - rope.rotary_dim:]
    y_rot = y[..., D - rope.rotary_dim:]

    x_norm = torch.linalg.vector_norm(x_rot, dim=-1)
    y_norm = torch.linalg.vector_norm(y_rot, dim=-1)

    assert y.shape == x.shape
    assert torch.allclose(x_norm, y_norm, atol=1e-5, rtol=1e-5)


# ============================================================
# F. Dtype and device tests
# ============================================================

def test_output_dtype_matches_input_dtype_float32():
    rope = RotaryEmbedding(dim=32, rotary_dim=32)

    x = torch.randn(2, 8, 4, 32, dtype=torch.float32)
    y = rope(x)

    assert y.dtype == torch.float32


def test_output_dtype_matches_input_dtype_bfloat16():
    rope = RotaryEmbedding(dim=32, rotary_dim=32)

    x = torch.randn(2, 8, 4, 32, dtype=torch.bfloat16)
    y = rope(x)

    assert y.dtype == torch.bfloat16
    assert torch.isfinite(y.float()).all()


@pytest.mark.skipif(not torch.cuda.is_available(), reason="CUDA is not available")
def test_output_dtype_matches_input_dtype_float16_cuda_if_available():
    rope = RotaryEmbedding(dim=32, rotary_dim=32).cuda()

    x = torch.randn(2, 8, 4, 32, device="cuda", dtype=torch.float16)
    y = rope(x)

    assert y.dtype == torch.float16
    assert y.device.type == "cuda"
    assert torch.isfinite(y.float()).all()


def test_output_device_matches_input_device():
    rope = RotaryEmbedding(dim=32, rotary_dim=32)

    x = torch.randn(2, 8, 4, 32)
    y = rope(x)

    assert y.device == x.device


# ============================================================
# G. Gradient tests
# ============================================================

def test_backward_computes_gradient_for_x():
    rope = RotaryEmbedding(dim=32, rotary_dim=16)

    x = torch.randn(2, 8, 4, 32, requires_grad=True)
    y = rope(x)

    loss = y.sum()
    loss.backward()

    assert x.grad is not None
    assert torch.isfinite(x.grad).all()

In [19]:
# ============================================================
# Run ONLY RoPE tests in Jupyter
# Handles pytest-style parametrized tests manually
# ============================================================

import torch

# Tests normales sin argumentos
rope_plain_tests = [
    "test_rotate_half_shape",
    "test_rotate_half_manual_values",
    "test_rotate_half_preserves_dtype_and_device",
    "test_valid_rope_config_builds",
    "test_rotary_dim_none_defaults_to_dim",
    "test_rope_output_shape_matches_input_shape",
    "test_rejects_wrong_last_dim",
    "test_position_zero_is_identity",
    "test_rotary_preserves_norm_of_rotated_part",
    "test_partial_rope_leaves_pass_through_dimensions_unchanged",
    "test_full_rope_rotates_all_dimensions",
    "test_forward_matches_manual_computation",
    "test_start_pos_matches_explicit_position_ids",
    "test_accepts_position_ids_T",
    "test_accepts_position_ids_BT",
    "test_accepts_negative_position_ids",
    "test_output_dtype_matches_input_dtype_float32",
    "test_output_dtype_matches_input_dtype_bfloat16",
    "test_output_device_matches_input_device",
    "test_backward_computes_gradient_for_x"]

if torch.cuda.is_available():
    rope_plain_tests.append(
        "test_output_dtype_matches_input_dtype_float16_cuda_if_available")

# Tests parametrizados manualmente
rope_param_tests = [
    ("test_invalid_dim_raises", {"dim": 0}),
    ("test_invalid_dim_raises", {"dim": -1}),
    ("test_invalid_dim_raises", {"dim": -64}),

    ("test_invalid_rotary_dim_raises", {"dim": 64, "rotary_dim": 0}),
    ("test_invalid_rotary_dim_raises", {"dim": 64, "rotary_dim": -1}),
    ("test_invalid_rotary_dim_raises", {"dim": 64, "rotary_dim": 65}),
    ("test_invalid_rotary_dim_raises", {"dim": 64, "rotary_dim": 7}),

    ("test_invalid_base_raises", {"base": 0.0}),
    ("test_invalid_base_raises", {"base": -1.0}),
    ("test_invalid_base_raises", {"base": -10000.0}),

    ("test_rejects_non_4d_input", {"bad_x": torch.randn(2, 8, 32)}),
    ("test_rejects_non_4d_input", {"bad_x": torch.randn(8, 32)}),
    ("test_rejects_non_4d_input", {"bad_x": torch.randn(2, 8, 4, 32, 1)}),

    ("test_rejects_invalid_position_ids_shape", {"position_ids": torch.arange(2)}),
    ("test_rejects_invalid_position_ids_shape", {"position_ids": torch.zeros(2, 6, 1, dtype=torch.long)}),
    ("test_rejects_invalid_position_ids_shape", {"position_ids": torch.arange(7)})]

n_passed = 0

for name in rope_plain_tests:
    globals()[name]()
    n_passed += 1

for name, kwargs in rope_param_tests:
    globals()[name](**kwargs)
    n_passed += 1

print(f"OK: {n_passed} RoPE tests/casos pasaron.")

OK: 37 RoPE tests/casos pasaron.


---

# Causal Multi-Head Attention baseline

In [20]:
# ============================================================
# Mini DeepSeek-V4 Causal Multi-Head Attention Baseline
# ============================================================

import math
from dataclasses import dataclass
from typing import Optional, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# CONFIG
# ============================================================

@dataclass
class CausalMHAConfig:
    d_model: int
    n_heads: int

    head_dim: Optional[int] = None

    attention_dropout: float = 0.0
    residual_dropout: float = 0.0

    use_bias: bool = False

    use_rope: bool = True
    rope_theta: float = 10000.0
    rotary_dim: Optional[int] = None

    max_seq_len: int = 1024
    init_std: float = 0.02

    def validate(self) -> None:
        if self.d_model <= 0:
            raise ValueError(f"d_model must be > 0, got {self.d_model}")

        if self.n_heads <= 0:
            raise ValueError(f"n_heads must be > 0, got {self.n_heads}")

        if self.head_dim is None:
            if self.d_model % self.n_heads != 0:
                raise ValueError(
                    "If head_dim is None, d_model must be divisible by n_heads. "
                    f"Got d_model={self.d_model}, n_heads={self.n_heads}"
                )
            head_dim = self.d_model // self.n_heads
        else:
            head_dim = self.head_dim

        if head_dim <= 0:
            raise ValueError(f"head_dim must be > 0, got {head_dim}")

        inner_dim = self.n_heads * head_dim

        # Baseline MHA: keep merge simple.
        if inner_dim != self.d_model:
            raise ValueError(
                "For baseline CausalMHA, n_heads * head_dim must equal d_model. "
                f"Got n_heads={self.n_heads}, head_dim={head_dim}, "
                f"inner_dim={inner_dim}, d_model={self.d_model}"
            )

        if not (0.0 <= self.attention_dropout < 1.0):
            raise ValueError(
                "attention_dropout must satisfy 0 <= attention_dropout < 1, "
                f"got {self.attention_dropout}"
            )

        if not (0.0 <= self.residual_dropout < 1.0):
            raise ValueError(
                "residual_dropout must satisfy 0 <= residual_dropout < 1, "
                f"got {self.residual_dropout}"
            )

        if self.max_seq_len <= 0:
            raise ValueError(f"max_seq_len must be > 0, got {self.max_seq_len}")

        if self.init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {self.init_std}")

        if self.rope_theta <= 0:
            raise ValueError(f"rope_theta must be > 0, got {self.rope_theta}")

        if self.rotary_dim is not None:
            if self.rotary_dim <= 0:
                raise ValueError(
                    f"rotary_dim must be > 0 when provided, got {self.rotary_dim}"
                )

            if self.rotary_dim > head_dim:
                raise ValueError(
                    f"rotary_dim must be <= head_dim. "
                    f"Got rotary_dim={self.rotary_dim}, head_dim={head_dim}"
                )

            if self.rotary_dim % 2 != 0:
                raise ValueError(
                    f"rotary_dim must be even, got {self.rotary_dim}"
                )


# ============================================================
# CAUSAL MULTI-HEAD ATTENTION
# ============================================================

class CausalMultiHeadAttention(nn.Module):
    """
    Baseline causal multi-head self-attention.

    Input:
        x: [B, T, d_model]

    Optional:
        attention_mask: [B, T]
            1 = valid token
            0 = padding / invalid key token

        position_ids:
            None, [T], or [B, T]

        start_pos:
            offset used by RoPE when position_ids is None

        need_weights:
            if True, returns attention weights.

    Output:
        out: [B, T, d_model]

        if need_weights=True:
            out, attn_weights
            attn_weights: [B, n_heads, T, T]
    """

    def __init__(self, config: CausalMHAConfig):
        super().__init__()

        config.validate()

        self.config = config

        self.d_model = config.d_model
        self.n_heads = config.n_heads
        self.head_dim = (
            config.head_dim
            if config.head_dim is not None
            else config.d_model // config.n_heads)

        self.inner_dim = self.n_heads * self.head_dim
        self.max_seq_len = config.max_seq_len

        self.use_rope = config.use_rope

        self.q_proj = nn.Linear(
            self.d_model,
            self.inner_dim,
            bias=config.use_bias,)

        self.k_proj = nn.Linear(
            self.d_model,
            self.inner_dim,
            bias=config.use_bias, )

        self.v_proj = nn.Linear(
            self.d_model,
            self.inner_dim,
            bias=config.use_bias,)

        self.out_proj = nn.Linear(
            self.inner_dim,
            self.d_model,
            bias=config.use_bias,)

        if self.use_rope:
            self.rope = RotaryEmbedding(
                dim=self.head_dim,
                rotary_dim=config.rotary_dim,
                base=config.rope_theta,)

        else:
            self.rope = None

        self.attention_dropout = nn.Dropout(config.attention_dropout)
        self.residual_dropout = nn.Dropout(config.residual_dropout)

        self.reset_parameters()

    def reset_parameters(self) -> None:
        """
        Initialize projections with Normal(0, init_std).
        Biases, if present, are initialized to zero.
        """
        for module in [self.q_proj, self.k_proj, self.v_proj, self.out_proj]:
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=self.config.init_std,)

            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def _shape_projection(self, x: torch.Tensor) -> torch.Tensor:
        """
        Convert projected tensor from:

            [B, T, inner_dim]

        to:

            [B, T, n_heads, head_dim]
        """
        B, T, _ = x.shape

        return x.view(B, T, self.n_heads, self.head_dim)

    def _build_causal_mask(
        self,
        seq_len: int,
        device: torch.device) -> torch.Tensor:

        """
        Build causal future mask.

        Returns:
            causal_mask: [T, T]
            True means masked / forbidden.
        """
        return torch.triu(
            torch.ones(seq_len, seq_len, device=device, dtype=torch.bool),
            diagonal=1,)

    def _validate_attention_mask(
        self,
        attention_mask: torch.Tensor,
        batch_size: int,
        seq_len: int) -> torch.Tensor:
        """
        Validate and return attention_mask.

        Expected:
            attention_mask: [B, T]
            1 = valid
            0 = pad / invalid
        """
        if attention_mask.dim() != 2:
            raise ValueError(
                f"attention_mask must have shape [B, T], "
                f"got {tuple(attention_mask.shape)}")

        if attention_mask.shape != (batch_size, seq_len):
            raise ValueError(
                f"attention_mask must have shape {(batch_size, seq_len)}, "
                f"got {tuple(attention_mask.shape)}")

        return attention_mask

    def _safe_masked_softmax(
      self,
      scores: torch.Tensor,
      allowed_mask: torch.Tensor,
      dim: int = -1) -> torch.Tensor:
      """
      Safe masked softmax.

      Args:
          scores:
              Attention scores [B, H, T, T].

          allowed_mask:
              Boolean mask broadcastable to scores.
              True  = allowed attention position.
              False = masked / forbidden position.

          dim:
              Softmax dimension.

      Returns:
          attn_weights:
              Same shape as scores.
              Rows with at least one valid key sum to 1.
              Rows with no valid keys are exactly zero.
      """

      if allowed_mask.dtype != torch.bool:
          allowed_mask = allowed_mask.bool()

      mask_value = torch.finfo(scores.dtype).min

      masked_scores = scores.masked_fill(~allowed_mask, mask_value)

      # Softmax in fp32 for numerical stability
      weights = F.softmax(masked_scores.float(), dim=dim).to(dtype=scores.dtype)

      # Remove any mass assigned to masked positions.
      weights = weights * allowed_mask.to(dtype=weights.dtype)

      # Renormalize only rows with at least one allowed key.
      denom = weights.sum(dim=dim, keepdim=True)

      weights = torch.where(
          denom > 0,
          weights / denom.clamp_min(torch.finfo(weights.dtype).tiny),
          torch.zeros_like(weights))

      return weights


    def forward(
        self,
        x: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        start_pos: int = 0,
        need_weights: bool = False) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        """
        Args:
            x:
                Hidden states [B, T, d_model].

            attention_mask:
                Optional mask [B, T].
                1 = valid token.
                0 = padding token.

            position_ids:
                Optional positions for RoPE.
                None, [T], or [B, T].

            start_pos:
                RoPE offset used when position_ids is None.

            need_weights:
                Whether to return attention weights.

        Returns:
            out:
                [B, T, d_model]

            optionally:
                attn_weights [B, n_heads, T, T]
        """

        # ----------------------------------------------------
        # Input validation
        # ----------------------------------------------------
        if x.dim() != 3:
            raise ValueError(
                f"CausalMultiHeadAttention expects x with shape [B, T, d_model], "
                f"got {tuple(x.shape)}")

        B, T, C = x.shape

        if C != self.d_model:
            raise ValueError(
                f"Expected x.shape[-1] == d_model={self.d_model}, got {C}")


        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length T={T} exceeds max_seq_len={self.max_seq_len}")


        if attention_mask is not None:
            attention_mask = self._validate_attention_mask(
                attention_mask=attention_mask,
                batch_size=B,
                seq_len=T,)


        # ----------------------------------------------------
        # QKV projections
        # ----------------------------------------------------
        q = self.q_proj(x)  # [B, T, inner_dim]
        k = self.k_proj(x)  # [B, T, inner_dim]
        v = self.v_proj(x)  # [B, T, inner_dim]

        q = self._shape_projection(q)  # [B, T, H, Dh]
        k = self._shape_projection(k)  # [B, T, H, Dh]
        v = self._shape_projection(v)  # [B, T, H, Dh]

        # ----------------------------------------------------
        # RoPE on q/k only
        # ----------------------------------------------------
        if self.rope is not None:
            q = self.rope(
                q,
                position_ids=position_ids,
                start_pos=start_pos,)

            k = self.rope(
                k,
                position_ids=position_ids,
                start_pos=start_pos,)

        # ----------------------------------------------------
        # Transpose for attention scores
        # ----------------------------------------------------
        q = q.transpose(1, 2)  # [B, H, T, Dh]
        k = k.transpose(1, 2)  # [B, H, T, Dh]
        v = v.transpose(1, 2)  # [B, H, T, Dh]

        # ----------------------------------------------------
        # Scaled dot-product attention scores
        # ----------------------------------------------------
        attn_scores = torch.matmul(q, k.transpose(-2, -1))
        attn_scores = attn_scores / math.sqrt(self.head_dim)


        # ----------------------------------------------------
        # Causal mask
        # ----------------------------------------------------
        causal_mask = self._build_causal_mask(
            seq_len=T,
            device=x.device) # [T, T]

        mask_value = torch.finfo(attn_scores.dtype).min

        attn_scores = attn_scores.masked_fill(
            causal_mask[None, None, :, :],
            mask_value)

        # --------------------------------------------------
        # Optional key padding mask
        # ----------------------------------------------------
        if attention_mask is not None:
            key_padding_mask = attention_mask[:, None, None, :].to(
                device=x.device,
                dtype=torch.bool) # [B, 1, 1, T]

            attn_scores = attn_scores.masked_fill(
                ~key_padding_mask,
                mask_value,)

        # ----------------------------------------------------
        # Softmax attention weights
        # ----------------------------------------------------
        # Compute softmax in fp32 for stability, then cast back.
        attn_weights = F.softmax(
            attn_scores.float(),
            dim=-1,).to(dtype=attn_scores.dtype)

        attn_weights = self.attention_dropout(attn_weights)

        # ----------------------------------------------------
        # Weighted sum
        # ----------------------------------------------------
        context = torch.matmul(attn_weights, v) # [B, H, T, Dh]

        # ----------------------------------------------------
        # Merge heads
        # ----------------------------------------------------
        context = context.transpose(1, 2).contiguous()# [B, T, H, Dh]
        context = context.view(B, T, self.inner_dim) # [B, T, inner_dim]

        # ----------------------------------------------------
        # Output projection + residual dropout
        # ----------------------------------------------------
        out = self.out_proj(context)
        out = self.residual_dropout(out)

        if need_weights:
            return out, attn_weights

        return out

In [21]:
config = CausalMHAConfig(
    d_model=256,
    n_heads=4,
    head_dim=None,
    attention_dropout=0.0,
    residual_dropout=0.0,
    use_bias=False,
    use_rope=True,
    rope_theta=10000.0,
    rotary_dim=64,
    max_seq_len=512,
    init_std=0.02,)

attn = CausalMultiHeadAttention(config)

x = torch.randn(2, 128, 256)

out, weights = attn(
    x,
    attention_mask=None,
    position_ids=None,
    start_pos=0,
    need_weights=True)

print(out.shape)      # [2, 128, 256]
print(weights.shape)  # [2, 4, 128, 128]

torch.Size([2, 128, 256])
torch.Size([2, 4, 128, 128])


## Test for attention

In [22]:
# @title
# ============================================================
# Causal Multi-Head Attention baseline tests
# ============================================================

import pytest
import torch


# ============================================================
# Helpers
# ============================================================

def make_mha_config(**overrides):
    cfg = dict(
        d_model=64,
        n_heads=4,
        head_dim=16,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_bias=False,
        use_rope=True,
        rope_theta=10000.0,
        rotary_dim=16,
        max_seq_len=128,
        init_std=0.02,
    )
    cfg.update(overrides)
    return CausalMHAConfig(**cfg)


def make_mha(**overrides):
    return CausalMultiHeadAttention(make_mha_config(**overrides))


def make_input(B=2, T=8, D=64, dtype=torch.float32, device="cpu"):
    return torch.randn(B, T, D, dtype=dtype, device=device)


class SpyRoPE(nn.Module):
    def __init__(self, rope):
        super().__init__()
        self.rope = rope
        self.call_count = 0

    def forward(self, x, position_ids=None, start_pos=0):
        self.call_count += 1
        return self.rope(x, position_ids=position_ids, start_pos=start_pos)


def test_rope_is_called_when_enabled():
    attn = make_mha(use_rope=True, attention_dropout=0.0, residual_dropout=0.0)
    attn.eval()

    spy_rope = SpyRoPE(attn.rope)
    attn.rope = spy_rope

    x = make_input(B=2, T=8, D=64)

    _ = attn(x, position_ids=torch.arange(8))

    # RoPE debe aplicarse a q y k, no a v.
    assert attn.rope.call_count == 2

# ============================================================
# A. Config tests
# ============================================================

def test_valid_attention_config_builds():
    config = make_mha_config(d_model=256, n_heads=4, head_dim=64)
    attn = CausalMultiHeadAttention(config)

    assert attn.d_model == 256
    assert attn.n_heads == 4
    assert attn.head_dim == 64
    assert attn.inner_dim == 256


@pytest.mark.parametrize("d_model", [0, -1, -256])
def test_invalid_d_model_raises(d_model):
    with pytest.raises(ValueError):
        CausalMultiHeadAttention(make_mha_config(d_model=d_model))


@pytest.mark.parametrize("n_heads", [0, -1, -4])
def test_invalid_n_heads_raises(n_heads):
    with pytest.raises(ValueError):
        CausalMultiHeadAttention(make_mha_config(n_heads=n_heads))


@pytest.mark.parametrize(
    "head_dim",
    [0, -1, 15],
)
def test_invalid_head_dim_raises(head_dim):
    with pytest.raises(ValueError):
        CausalMultiHeadAttention(make_mha_config(head_dim=head_dim))


@pytest.mark.parametrize(
    "field,value",
    [
        ("attention_dropout", -0.1),
        ("attention_dropout", 1.0),
        ("attention_dropout", 1.5),
        ("residual_dropout", -0.1),
        ("residual_dropout", 1.0),
        ("residual_dropout", 1.5),
    ],
)
def test_invalid_dropout_raises(field, value):
    with pytest.raises(ValueError):
        CausalMultiHeadAttention(make_mha_config(**{field: value}))


@pytest.mark.parametrize("rotary_dim", [0, -1, 17, 32])
def test_invalid_rotary_dim_raises(rotary_dim):
    with pytest.raises(ValueError):
        CausalMultiHeadAttention(make_mha_config(rotary_dim=rotary_dim))


# ============================================================
# B. Shape and forward tests
# ============================================================

def test_mha_output_shape_matches_input():
    attn = make_mha()
    x = make_input(B=2, T=8, D=64)

    out = attn(x)

    assert out.shape == x.shape


@pytest.mark.parametrize(
    "bad_x",
    [
        torch.randn(8, 64),
        torch.randn(2, 8, 64, 1),
    ],
)
def test_rejects_wrong_input_dim(bad_x):
    attn = make_mha()

    with pytest.raises(ValueError):
        attn(bad_x)


def test_rejects_wrong_hidden_size():
    attn = make_mha(d_model=64)
    x = torch.randn(2, 8, 32)

    with pytest.raises(ValueError):
        attn(x)


def test_returns_attention_weights_when_requested():
    attn = make_mha()
    x = make_input(B=2, T=8, D=64)

    out, weights = attn(x, need_weights=True)

    assert out.shape == (2, 8, 64)
    assert weights.shape == (2, 4, 8, 8)


def test_attention_weights_sum_to_one_without_dropout():
    attn = make_mha(attention_dropout=0.0, residual_dropout=0.0)
    attn.eval()

    x = make_input(B=2, T=8, D=64)

    _, weights = attn(x, need_weights=True)

    sums = weights.sum(dim=-1)

    assert torch.allclose(sums, torch.ones_like(sums), atol=1e-6, rtol=1e-6)


# ============================================================
# C. Causality tests
# ============================================================

def test_causal_mask_blocks_future_attention_weights():
    attn = make_mha(attention_dropout=0.0, residual_dropout=0.0)
    attn.eval()

    B, T, D = 2, 8, 64
    x = make_input(B=B, T=T, D=D)

    _, weights = attn(x, need_weights=True)

    future_mask = torch.triu(
        torch.ones(T, T, dtype=torch.bool),
        diagonal=1,
    )

    future_weights = weights[:, :, future_mask]

    assert torch.allclose(
        future_weights,
        torch.zeros_like(future_weights),
        atol=0.0,
        rtol=0.0,
    )


def test_changing_future_tokens_does_not_change_past_outputs():
    attn = make_mha(attention_dropout=0.0, residual_dropout=0.0)
    attn.eval()

    B, T, D = 2, 10, 64
    cut = 5

    x1 = make_input(B=B, T=T, D=D)
    x2 = x1.clone()

    x2[:, cut:, :] = torch.randn_like(x2[:, cut:, :])

    out1 = attn(x1)
    out2 = attn(x2)

    assert torch.allclose(
        out1[:, :cut, :],
        out2[:, :cut, :],
        atol=1e-5,
        rtol=1e-5,
    )


# ============================================================
# D. attention_mask / padding tests
# ============================================================

def test_attention_mask_shape_validation_accepts_BT():
    attn = make_mha()
    x = make_input(B=2, T=8, D=64)
    attention_mask = torch.ones(2, 8)

    out = attn(x, attention_mask=attention_mask)

    assert out.shape == x.shape


@pytest.mark.parametrize(
    "bad_mask",
    [
        torch.ones(8),
        torch.ones(2, 8, 1),
        torch.ones(2, 9),
    ],
)
def test_attention_mask_shape_validation_rejects_bad_shapes(bad_mask):
    attn = make_mha()
    x = make_input(B=2, T=8, D=64)

    with pytest.raises(ValueError):
        attn(x, attention_mask=bad_mask)


def test_attention_mask_blocks_padding_keys():
    attn = make_mha(attention_dropout=0.0, residual_dropout=0.0)
    attn.eval()

    B, T, D = 2, 8, 64
    x = make_input(B=B, T=T, D=D)

    attention_mask = torch.ones(B, T)
    attention_mask[0, 3] = 0
    attention_mask[1, 5] = 0

    _, weights = attn(
        x,
        attention_mask=attention_mask,
        need_weights=True,
    )

    assert torch.allclose(
        weights[0, :, :, 3],
        torch.zeros_like(weights[0, :, :, 3]),
        atol=0.0,
        rtol=0.0,
    )

    assert torch.allclose(
        weights[1, :, :, 5],
        torch.zeros_like(weights[1, :, :, 5]),
        atol=0.0,
        rtol=0.0,
    )


def test_all_padding_keys_does_not_produce_nan():
    attn = make_mha(attention_dropout=0.0, residual_dropout=0.0)
    attn.eval()

    B, T, D = 2, 8, 64
    x = make_input(B=B, T=T, D=D)

    attention_mask = torch.zeros(B, T)

    out, weights = attn(
        x,
        attention_mask=attention_mask,
        need_weights=True,
    )

    assert torch.isfinite(out).all()
    assert torch.isfinite(weights).all()


# ============================================================
# E. RoPE integration tests
# ============================================================

def test_rope_is_called_when_enabled():
    attn = make_mha(use_rope=True, attention_dropout=0.0, residual_dropout=0.0)
    attn.eval()

    B, T, D = 2, 8, 64
    x = make_input(B=B, T=T, D=D)

    out1 = attn(x, position_ids=torch.arange(T))
    out2 = attn(x, position_ids=torch.arange(10, 10 + T))

    assert not torch.allclose(out1, out2, atol=1e-7, rtol=1e-7)


def test_no_rope_when_disabled():
    attn = make_mha(use_rope=False, attention_dropout=0.0, residual_dropout=0.0)
    attn.eval()

    B, T, D = 2, 8, 64
    x = make_input(B=B, T=T, D=D)

    out1 = attn(x, position_ids=torch.arange(T), start_pos=0)
    out2 = attn(x, position_ids=torch.arange(10, 10 + T), start_pos=10)

    assert torch.allclose(out1, out2, atol=1e-6, rtol=1e-6)


def test_mha_start_pos_matches_explicit_position_ids():
    attn = make_mha(use_rope=True, attention_dropout=0.0, residual_dropout=0.0)
    attn.eval()

    B, T, D = 2, 8, 64
    start_pos = 10

    x = make_input(B=B, T=T, D=D)

    out_start = attn(x, start_pos=start_pos)
    out_explicit = attn(x, position_ids=torch.arange(start_pos, start_pos + T))

    assert torch.allclose(out_start, out_explicit, atol=1e-6, rtol=1e-5)


def test_accepts_position_ids_T_and_BT():
    attn = make_mha(use_rope=True)

    B, T, D = 2, 8, 64
    x = make_input(B=B, T=T, D=D)

    position_ids_T = torch.arange(T)
    out_T = attn(x, position_ids=position_ids_T)

    position_ids_BT = torch.stack(
        [
            torch.arange(T),
            torch.arange(10, 10 + T),
        ],
        dim=0,
    )
    out_BT = attn(x, position_ids=position_ids_BT)

    assert out_T.shape == x.shape
    assert out_BT.shape == x.shape


def test_nonuniform_position_ids_change_output_when_rope_enabled():
    attn = make_mha(use_rope=True, attention_dropout=0.0, residual_dropout=0.0)
    attn.eval()

    B, T, D = 2, 8, 64
    x = make_input(B=B, T=T, D=D)

    position_ids_1 = torch.arange(T)
    position_ids_2 = torch.arange(T) * 2

    out1 = attn(x, position_ids=position_ids_1)
    out2 = attn(x, position_ids=position_ids_2)

    assert not torch.allclose(out1, out2, atol=1e-7, rtol=1e-7)


# ============================================================
# F. Dropout tests
# ============================================================

def test_mha_dropout_zero_is_deterministic():
    attn = make_mha(attention_dropout=0.0, residual_dropout=0.0)
    attn.train()

    x = make_input(B=4, T=16, D=64)

    out1 = attn(x)
    out2 = attn(x)

    assert torch.equal(out1, out2)


def test_mha_dropout_disabled_in_eval_mode():
    attn = make_mha(attention_dropout=0.5, residual_dropout=0.5)
    attn.eval()

    x = make_input(B=4, T=16, D=64)

    out1 = attn(x)
    out2 = attn(x)

    assert torch.equal(out1, out2)


def test_mha_dropout_active_in_train_mode():
    attn = make_mha(attention_dropout=0.5, residual_dropout=0.5)
    attn.train()

    x = make_input(B=4, T=16, D=64)

    out1 = attn(x)
    out2 = attn(x)

    assert not torch.equal(out1, out2)


# ============================================================
# G. Gradient and dtype tests
# ============================================================

def test_mha_backward_computes_gradients():
    attn = make_mha(attention_dropout=0.0, residual_dropout=0.0)

    x = make_input(B=2, T=8, D=64)
    x.requires_grad_(True)

    out = attn(x)
    loss = out.sum()
    loss.backward()

    assert x.grad is not None
    assert attn.q_proj.weight.grad is not None
    assert attn.k_proj.weight.grad is not None
    assert attn.v_proj.weight.grad is not None
    assert attn.out_proj.weight.grad is not None

    assert torch.isfinite(x.grad).all()
    assert torch.isfinite(attn.q_proj.weight.grad).all()
    assert torch.isfinite(attn.k_proj.weight.grad).all()
    assert torch.isfinite(attn.v_proj.weight.grad).all()
    assert torch.isfinite(attn.out_proj.weight.grad).all()


def test_mha_output_dtype_matches_input_dtype_float32():
    attn = make_mha()

    x = make_input(B=2, T=8, D=64, dtype=torch.float32)
    out = attn(x)

    assert out.dtype == torch.float32


def test_mha_output_dtype_matches_input_dtype_bfloat16():
    attn = make_mha().to(dtype=torch.bfloat16)

    x = make_input(B=2, T=8, D=64).to(dtype=torch.bfloat16)
    out = attn(x)

    assert out.dtype == torch.bfloat16
    assert torch.isfinite(out.float()).all()


# ============================================================
# H. Integration tests with closed modules
# ============================================================

def test_embedding_rmsnorm_attention_pipeline():
    B, T = 2, 16
    vocab_size = 128
    d_model = 64

    emb_cfg = EmbeddingConfig(
        vocab_size=vocab_size,
        d_model=d_model,
        pad_token_id=0,
        max_seq_len=T,
        embedding_dropout=0.0,
        scale_embeddings=False,
        init_std=0.02,
        tie_word_embeddings=True)

    embedding = TokenEmbedding(emb_cfg)
    norm = RMSNorm(dim=d_model)
    attn = make_mha(d_model=d_model, n_heads=4, head_dim=16, max_seq_len=T)

    input_ids = torch.randint(1, vocab_size, (B, T), dtype=torch.long)

    hidden_states = embedding(input_ids)
    normed = norm(hidden_states)
    out = attn(normed)

    assert hidden_states.shape == (B, T, d_model)
    assert normed.shape == (B, T, d_model)
    assert out.shape == (B, T, d_model)

    assert torch.isfinite(hidden_states).all()
    assert torch.isfinite(normed).all()
    assert torch.isfinite(out).all()

In [23]:
class SpyRoPE(nn.Module):
    def __init__(self, rope):
        super().__init__()
        self.rope = rope
        self.call_count = 0

    def forward(self, x, position_ids=None, start_pos=0):
        self.call_count += 1
        return self.rope(x, position_ids=position_ids, start_pos=start_pos)


def test_rope_is_called_when_enabled():
    attn = make_mha(use_rope=True, attention_dropout=0.0, residual_dropout=0.0)
    attn.eval()

    attn.rope = SpyRoPE(attn.rope)

    x = make_input(B=2, T=8, D=64)

    _ = attn(x, position_ids=torch.arange(8))

    # RoPE debe llamarse para q y k. No para v.
    assert attn.rope.call_count == 2

In [24]:
# ============================================================
# Run ONLY CausalMHA tests in Jupyter
# Handles pytest-style parametrized tests manually
# ============================================================

mha_plain_tests = [
    "test_valid_attention_config_builds",
    "test_mha_output_shape_matches_input",
    "test_rejects_wrong_hidden_size",
    "test_returns_attention_weights_when_requested",
    "test_attention_weights_sum_to_one_without_dropout",
    "test_causal_mask_blocks_future_attention_weights",
    "test_changing_future_tokens_does_not_change_past_outputs",
    "test_attention_mask_shape_validation_accepts_BT",
    "test_attention_mask_blocks_padding_keys",
    "test_all_padding_keys_does_not_produce_nan",
    "test_rope_is_called_when_enabled",
    "test_no_rope_when_disabled",
    "test_mha_start_pos_matches_explicit_position_ids",
    "test_accepts_position_ids_T_and_BT",
    "test_mha_dropout_zero_is_deterministic",
    "test_mha_dropout_disabled_in_eval_mode",
    "test_mha_dropout_active_in_train_mode",
    "test_mha_backward_computes_gradients",
    "test_mha_output_dtype_matches_input_dtype_float32",
    "test_mha_output_dtype_matches_input_dtype_bfloat16",
    "test_embedding_rmsnorm_attention_pipeline","test_nonuniform_position_ids_change_output_when_rope_enabled"
]

mha_param_tests = [
    ("test_invalid_d_model_raises", {"d_model": 0}),
    ("test_invalid_d_model_raises", {"d_model": -1}),
    ("test_invalid_d_model_raises", {"d_model": -256}),

    ("test_invalid_n_heads_raises", {"n_heads": 0}),
    ("test_invalid_n_heads_raises", {"n_heads": -1}),
    ("test_invalid_n_heads_raises", {"n_heads": -4}),

    ("test_invalid_head_dim_raises", {"head_dim": 0}),
    ("test_invalid_head_dim_raises", {"head_dim": -1}),
    ("test_invalid_head_dim_raises", {"head_dim": 15}),

    ("test_invalid_dropout_raises", {"field": "attention_dropout", "value": -0.1}),
    ("test_invalid_dropout_raises", {"field": "attention_dropout", "value": 1.0}),
    ("test_invalid_dropout_raises", {"field": "attention_dropout", "value": 1.5}),
    ("test_invalid_dropout_raises", {"field": "residual_dropout", "value": -0.1}),
    ("test_invalid_dropout_raises", {"field": "residual_dropout", "value": 1.0}),
    ("test_invalid_dropout_raises", {"field": "residual_dropout", "value": 1.5}),

    ("test_invalid_rotary_dim_raises", {"rotary_dim": 0}),
    ("test_invalid_rotary_dim_raises", {"rotary_dim": -1}),
    ("test_invalid_rotary_dim_raises", {"rotary_dim": 17}),
    ("test_invalid_rotary_dim_raises", {"rotary_dim": 32}),

    ("test_rejects_wrong_input_dim", {"bad_x": torch.randn(8, 64)}),
    ("test_rejects_wrong_input_dim", {"bad_x": torch.randn(2, 8, 64, 1)}),

    ("test_attention_mask_shape_validation_rejects_bad_shapes", {"bad_mask": torch.ones(8)}),
    ("test_attention_mask_shape_validation_rejects_bad_shapes", {"bad_mask": torch.ones(2, 8, 1)}),
    ("test_attention_mask_shape_validation_rejects_bad_shapes", {"bad_mask": torch.ones(2, 9)})]

n_passed = 0

for name in mha_plain_tests:
    globals()[name]()
    n_passed += 1

for name, kwargs in mha_param_tests:
    globals()[name](**kwargs)
    n_passed += 1

print(f"OK: {n_passed} CausalMHA tests/casos pasaron.")

OK: 46 CausalMHA tests/casos pasaron.


---

# SwiGLU / MLP

In [25]:
# ============================================================
# Mini DeepSeek-V4 SwiGLU / MLP Baseline
# ============================================================

from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Utilities
# ============================================================

def round_up_to_multiple(value: int, multiple_of: int) -> int:
    """
    Round value up to the nearest multiple of multiple_of.
    """
    if multiple_of <= 0:
        raise ValueError(f"multiple_of must be > 0, got {multiple_of}")

    return ((value + multiple_of - 1) // multiple_of) * multiple_of


# ============================================================
# CONFIG
# ============================================================

@dataclass
class SwiGLUMLPConfig:
    d_model: int

    hidden_dim: Optional[int] = None
    expansion_factor: float = 4.0
    multiple_of: int = 1

    dropout: float = 0.0
    use_bias: bool = False
    init_std: float = 0.02

    def validate(self) -> None:
        if self.d_model <= 0:
            raise ValueError(f"d_model must be > 0, got {self.d_model}")

        if self.hidden_dim is not None and self.hidden_dim <= 0:
            raise ValueError(
                f"hidden_dim must be > 0 when provided, got {self.hidden_dim}"
            )

        if self.expansion_factor <= 0:
            raise ValueError(
                f"expansion_factor must be > 0, got {self.expansion_factor}"
            )

        if self.multiple_of <= 0:
            raise ValueError(f"multiple_of must be > 0, got {self.multiple_of}")

        if not (0.0 <= self.dropout < 1.0):
            raise ValueError(
                f"dropout must satisfy 0 <= dropout < 1, got {self.dropout}"
            )

        if self.init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {self.init_std}")

    def resolved_hidden_dim(self) -> int:
        """
        Resolve hidden_dim from explicit hidden_dim or expansion_factor.

        If hidden_dim is None:
            hidden_dim = int(expansion_factor * d_model)

        Then round up to multiple_of.
        """
        self.validate()

        if self.hidden_dim is None:
            hidden_dim = int(self.expansion_factor * self.d_model)
        else:
            hidden_dim = self.hidden_dim

        hidden_dim = round_up_to_multiple(hidden_dim, self.multiple_of)

        return hidden_dim


# ============================================================
# SwiGLU MLP
# ============================================================

class SwiGLUMLP(nn.Module):
    """
    Dense feed-forward baseline using SwiGLU.

    Input:
        x: [B, T, d_model]

    Forward:
        gate = gate_proj(x)
        up   = up_proj(x)
        hidden = silu(gate) * up
        out = down_proj(hidden)
        out = dropout(out)

    Output:
        out: [B, T, d_model]

    This module intentionally does NOT include:
        - residual connection
        - RMSNorm
        - MoE routing
        - shared experts
        - top-k experts
        - attention
    """

    def __init__(self, config: SwiGLUMLPConfig):
        super().__init__()

        config.validate()

        self.config = config
        self.d_model = config.d_model
        self.hidden_dim = config.resolved_hidden_dim()

        self.gate_proj = nn.Linear(
            self.d_model,
            self.hidden_dim,
            bias=config.use_bias)

        self.up_proj = nn.Linear(
            self.d_model,
            self.hidden_dim,
            bias=config.use_bias)

        self.down_proj = nn.Linear(
            self.hidden_dim,
            self.d_model,
            bias=config.use_bias)

        self.dropout = nn.Dropout(config.dropout)

        self.reset_parameters()

    def reset_parameters(self) -> None:
        """
        Initialize all linear weights with Normal(0, init_std).
        Biases, if present, are initialized to zero.
        """
        for module in [self.gate_proj, self.up_proj, self.down_proj]:
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=self.config.init_std,)

            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x:
                Hidden states [B, T, d_model].

        Returns:
            out:
                MLP output [B, T, d_model].
        """

        if x.dim() != 3:
            raise ValueError(
                f"SwiGLUMLP expects x with shape [B, T, d_model], "
                f"got {tuple(x.shape)}")

        if x.shape[-1] != self.d_model:
            raise ValueError(
                f"Expected x.shape[-1] == d_model={self.d_model}, "
                f"got {x.shape[-1]}")

        gate = self.gate_proj(x)
        up = self.up_proj(x)

        hidden = F.silu(gate) * up

        out = self.down_proj(hidden)
        out = self.dropout(out)

        return out

In [26]:
# ============================================================
# Smoke test
# ============================================================

mlp_config = SwiGLUMLPConfig(
    d_model=256,
    hidden_dim=None,
    expansion_factor=4.0,
    multiple_of=1,
    dropout=0.0,
    use_bias=False,
    init_std=0.02,)

mlp = SwiGLUMLP(mlp_config)

x = torch.randn(2, 128, 256)

out = mlp(x)

print("x shape:", x.shape)
print("out shape:", out.shape)
print("hidden_dim:", mlp.hidden_dim)

assert out.shape == x.shape
assert torch.isfinite(out).all()

print("SwiGLUMLP OK.")

x shape: torch.Size([2, 128, 256])
out shape: torch.Size([2, 128, 256])
hidden_dim: 1024
SwiGLUMLP OK.


In [27]:
# @title
# ============================================================
# SwiGLU / MLP baseline tests
# ============================================================

import pytest
import torch
import torch.nn.functional as F


# ============================================================
# Helpers
# ============================================================

def make_mlp_config(**overrides):
    cfg = dict(
        d_model=64,
        hidden_dim=256,
        expansion_factor=4.0,
        multiple_of=1,
        dropout=0.0,
        use_bias=False,
        init_std=0.02,
    )
    cfg.update(overrides)
    return SwiGLUMLPConfig(**cfg)


def make_mlp(**overrides):
    return SwiGLUMLP(make_mlp_config(**overrides))


def make_mlp_input(B=2, T=8, D=64, dtype=torch.float32, device="cpu"):
    return torch.randn(B, T, D, dtype=dtype, device=device)


# ============================================================
# A. Config tests
# ============================================================

def test_valid_mlp_config_builds():
    config = make_mlp_config(
        d_model=256,
        hidden_dim=1024,
        dropout=0.0,
    )
    mlp = SwiGLUMLP(config)

    assert mlp.d_model == 256
    assert mlp.hidden_dim == 1024


def test_hidden_dim_inferred_from_expansion_factor():
    config = make_mlp_config(
        d_model=256,
        hidden_dim=None,
        expansion_factor=4.0,
        multiple_of=1,
    )
    mlp = SwiGLUMLP(config)

    assert mlp.hidden_dim == 1024


def test_hidden_dim_rounds_to_multiple_of():
    config = make_mlp_config(
        d_model=250,
        hidden_dim=None,
        expansion_factor=4.0,
        multiple_of=64,
    )
    mlp = SwiGLUMLP(config)

    raw_hidden_dim = int(4.0 * 250)

    assert mlp.hidden_dim % 64 == 0
    assert mlp.hidden_dim >= raw_hidden_dim


@pytest.mark.parametrize("d_model", [0, -1, -256])
def test_invalid_d_model_raises(d_model):
    with pytest.raises(ValueError):
        SwiGLUMLP(make_mlp_config(d_model=d_model))


@pytest.mark.parametrize("hidden_dim", [0, -1, -256])
def test_invalid_hidden_dim_raises(hidden_dim):
    with pytest.raises(ValueError):
        SwiGLUMLP(make_mlp_config(hidden_dim=hidden_dim))


@pytest.mark.parametrize("expansion_factor", [0.0, -1.0, -4.0])
def test_invalid_expansion_factor_raises(expansion_factor):
    with pytest.raises(ValueError):
        SwiGLUMLP(
            make_mlp_config(
                hidden_dim=None,
                expansion_factor=expansion_factor,
            )
        )


@pytest.mark.parametrize("multiple_of", [0, -1, -64])
def test_invalid_multiple_of_raises(multiple_of):
    with pytest.raises(ValueError):
        SwiGLUMLP(make_mlp_config(multiple_of=multiple_of))


@pytest.mark.parametrize("dropout", [-0.1, 1.0, 1.5])
def test_invalid_dropout_raises(dropout):
    with pytest.raises(ValueError):
        SwiGLUMLP(make_mlp_config(dropout=dropout))


@pytest.mark.parametrize("init_std", [0.0, -0.01])
def test_invalid_init_std_raises(init_std):
    with pytest.raises(ValueError):
        SwiGLUMLP(make_mlp_config(init_std=init_std))


# ============================================================
# B. Internal structure tests
# ============================================================

def test_projection_shapes():
    config = make_mlp_config(d_model=64, hidden_dim=256)
    mlp = SwiGLUMLP(config)

    assert mlp.gate_proj.weight.shape == (256, 64)
    assert mlp.up_proj.weight.shape == (256, 64)
    assert mlp.down_proj.weight.shape == (64, 256)


def test_bias_absent_when_use_bias_false():
    mlp = make_mlp(use_bias=False)

    assert mlp.gate_proj.bias is None
    assert mlp.up_proj.bias is None
    assert mlp.down_proj.bias is None


def test_bias_present_when_use_bias_true():
    mlp = make_mlp(use_bias=True)

    assert mlp.gate_proj.bias is not None
    assert mlp.up_proj.bias is not None
    assert mlp.down_proj.bias is not None


def test_weights_are_finite_after_init():
    mlp = make_mlp()

    assert torch.isfinite(mlp.gate_proj.weight).all()
    assert torch.isfinite(mlp.up_proj.weight).all()
    assert torch.isfinite(mlp.down_proj.weight).all()


def test_bias_initialized_to_zero_when_present():
    mlp = make_mlp(use_bias=True)

    assert torch.allclose(
        mlp.gate_proj.bias,
        torch.zeros_like(mlp.gate_proj.bias),
    )
    assert torch.allclose(
        mlp.up_proj.bias,
        torch.zeros_like(mlp.up_proj.bias),
    )
    assert torch.allclose(
        mlp.down_proj.bias,
        torch.zeros_like(mlp.down_proj.bias),
    )


# ============================================================
# C. Forward tests
# ============================================================

def test_mlp_output_shape_matches_input_shape():
    mlp = make_mlp(d_model=64, hidden_dim=256)

    x = make_mlp_input(B=2, T=8, D=64)
    out = mlp(x)

    assert out.shape == x.shape


@pytest.mark.parametrize(
    "bad_x",
    [
        torch.randn(8, 64),
        torch.randn(2, 8, 64, 1),
    ],
)
def test_rejects_wrong_input_rank(bad_x):
    mlp = make_mlp(d_model=64, hidden_dim=256)

    with pytest.raises(ValueError):
        mlp(bad_x)


def test_mlp_rejects_wrong_hidden_size():
    mlp = make_mlp(d_model=64, hidden_dim=256)

    x = torch.randn(2, 8, 32)

    with pytest.raises(ValueError):
        mlp(x)


def test_forward_matches_manual_swiglu_computation():
    mlp = make_mlp(
        d_model=64,
        hidden_dim=256,
        dropout=0.0,
        use_bias=True,
    )
    mlp.eval()

    x = make_mlp_input(B=2, T=8, D=64)

    out = mlp(x)

    gate = mlp.gate_proj(x)
    up = mlp.up_proj(x)
    hidden = F.silu(gate) * up
    expected = mlp.down_proj(hidden)

    assert torch.allclose(out, expected, atol=1e-6, rtol=1e-5)


def test_mlp_output_is_finite():
    mlp = make_mlp()

    x = make_mlp_input(B=2, T=8, D=64)
    out = mlp(x)

    assert torch.isfinite(out).all()


# ============================================================
# D. Dropout tests
# ============================================================

def test_mlp_dropout_zero_is_deterministic():
    mlp = make_mlp(dropout=0.0)
    mlp.train()

    x = make_mlp_input(B=4, T=16, D=64)

    out1 = mlp(x)
    out2 = mlp(x)

    assert torch.equal(out1, out2)


def test_mlp_dropout_disabled_in_eval_mode():
    mlp = make_mlp(dropout=0.5)
    mlp.eval()

    x = make_mlp_input(B=4, T=16, D=64)

    out1 = mlp(x)
    out2 = mlp(x)

    assert torch.equal(out1, out2)


def test_mlp_dropout_active_in_train_mode():
    mlp = make_mlp(dropout=0.5)
    mlp.train()

    x = make_mlp_input(B=4, T=16, D=64)

    out1 = mlp(x)
    out2 = mlp(x)

    assert not torch.equal(out1, out2)


# ============================================================
# E. Gradient tests
# ============================================================

def test_mlp_backward_computes_gradients():
    mlp = make_mlp(dropout=0.0)

    x = make_mlp_input(B=2, T=8, D=64)
    x.requires_grad_(True)

    out = mlp(x)
    loss = out.sum()
    loss.backward()

    assert x.grad is not None
    assert mlp.gate_proj.weight.grad is not None
    assert mlp.up_proj.weight.grad is not None
    assert mlp.down_proj.weight.grad is not None

    assert torch.isfinite(x.grad).all()
    assert torch.isfinite(mlp.gate_proj.weight.grad).all()
    assert torch.isfinite(mlp.up_proj.weight.grad).all()
    assert torch.isfinite(mlp.down_proj.weight.grad).all()


def test_all_parameters_receive_gradients():
    mlp = make_mlp(dropout=0.0, use_bias=True)

    x = make_mlp_input(B=2, T=8, D=64)
    out = mlp(x)
    loss = out.sum()
    loss.backward()

    for name, param in mlp.named_parameters():
        assert param.grad is not None, f"Missing grad for {name}"
        assert torch.isfinite(param.grad).all(), f"Non-finite grad for {name}"


# ============================================================
# F. Dtype and device tests
# ============================================================

def test_mlp_output_dtype_matches_input_dtype_float32():
    mlp = make_mlp()

    x = make_mlp_input(B=2, T=8, D=64, dtype=torch.float32)
    out = mlp(x)

    assert out.dtype == torch.float32


def test_mlp_output_dtype_matches_input_dtype_bfloat16():
    mlp = make_mlp().to(dtype=torch.bfloat16)

    x = make_mlp_input(B=2, T=8, D=64).to(dtype=torch.bfloat16)
    out = mlp(x)

    assert out.dtype == torch.bfloat16
    assert torch.isfinite(out.float()).all()


def test_mlp_output_device_matches_input_device():
    mlp = make_mlp()

    x = make_mlp_input(B=2, T=8, D=64)
    out = mlp(x)

    assert out.device == x.device


@pytest.mark.skipif(not torch.cuda.is_available(), reason="CUDA is not available")
def test_mlp_runs_on_cuda_if_available():
    mlp = make_mlp().cuda()

    x = make_mlp_input(B=2, T=8, D=64, device="cuda")
    out = mlp(x)

    assert out.device.type == "cuda"
    assert out.shape == x.shape


# ============================================================
# Integration tests
# ============================================================

def test_embedding_rmsnorm_mlp_pipeline():
    B, T = 2, 16
    vocab_size = 128
    d_model = 64

    emb_cfg = EmbeddingConfig(
        vocab_size=vocab_size,
        d_model=d_model,
        pad_token_id=0,
        max_seq_len=T,
        embedding_dropout=0.0,
        scale_embeddings=False,
        init_std=0.02,
        tie_word_embeddings=True,
    )

    embedding = TokenEmbedding(emb_cfg)
    norm = RMSNorm(dim=d_model)
    mlp = make_mlp(d_model=d_model, hidden_dim=256)

    input_ids = torch.randint(1, vocab_size, (B, T), dtype=torch.long)

    hidden_states = embedding(input_ids)
    normed = norm(hidden_states)
    out = mlp(normed)

    assert hidden_states.shape == (B, T, d_model)
    assert normed.shape == (B, T, d_model)
    assert out.shape == (B, T, d_model)

    assert torch.isfinite(hidden_states).all()
    assert torch.isfinite(normed).all()
    assert torch.isfinite(out).all()


def test_attention_mlp_pipeline():
    B, T, d_model = 2, 16, 64

    norm1 = RMSNorm(dim=d_model)
    attn = CausalMultiHeadAttention(
        CausalMHAConfig(
            d_model=d_model,
            n_heads=4,
            head_dim=16,
            attention_dropout=0.0,
            residual_dropout=0.0,
            use_bias=False,
            use_rope=True,
            rope_theta=10000.0,
            rotary_dim=16,
            max_seq_len=T,
            init_std=0.02,
        ))

    norm2 = RMSNorm(dim=d_model)
    mlp = make_mlp(d_model=d_model, hidden_dim=256)

    x = torch.randn(B, T, d_model)

    normed_1 = norm1(x)
    attn_out = attn(normed_1)

    normed_2 = norm2(attn_out)
    mlp_out = mlp(normed_2)

    assert attn_out.shape == (B, T, d_model)
    assert mlp_out.shape == (B, T, d_model)

    assert torch.isfinite(attn_out).all()
    assert torch.isfinite(mlp_out).all()

In [28]:
# ============================================================
# Run ONLY SwiGLUMLP tests in Jupyter
# Handles pytest-style parametrized tests manually
# ============================================================

mlp_plain_tests = [
    "test_valid_mlp_config_builds",
    "test_hidden_dim_inferred_from_expansion_factor",
    "test_hidden_dim_rounds_to_multiple_of",
    "test_projection_shapes",
    "test_bias_absent_when_use_bias_false",
    "test_bias_present_when_use_bias_true",
    "test_weights_are_finite_after_init",
    "test_bias_initialized_to_zero_when_present",
    "test_mlp_output_shape_matches_input_shape",
    "test_mlp_rejects_wrong_hidden_size",
    "test_forward_matches_manual_swiglu_computation",
    "test_mlp_output_is_finite",
    "test_mlp_dropout_zero_is_deterministic",
    "test_mlp_dropout_disabled_in_eval_mode",
    "test_mlp_dropout_active_in_train_mode",
    "test_mlp_backward_computes_gradients",
    "test_all_parameters_receive_gradients",
    "test_mlp_output_dtype_matches_input_dtype_float32",
    "test_mlp_output_dtype_matches_input_dtype_bfloat16",
    "test_mlp_output_device_matches_input_device",
    "test_embedding_rmsnorm_mlp_pipeline",
    "test_attention_mlp_pipeline"]

if torch.cuda.is_available():
    mlp_plain_tests.append("test_mlp_runs_on_cuda_if_available")

mlp_param_tests = [
    ("test_invalid_d_model_raises", {"d_model": 0}),
    ("test_invalid_d_model_raises", {"d_model": -1}),
    ("test_invalid_d_model_raises", {"d_model": -256}),

    ("test_invalid_hidden_dim_raises", {"hidden_dim": 0}),
    ("test_invalid_hidden_dim_raises", {"hidden_dim": -1}),
    ("test_invalid_hidden_dim_raises", {"hidden_dim": -256}),

    ("test_invalid_expansion_factor_raises", {"expansion_factor": 0.0}),
    ("test_invalid_expansion_factor_raises", {"expansion_factor": -1.0}),
    ("test_invalid_expansion_factor_raises", {"expansion_factor": -4.0}),

    ("test_invalid_multiple_of_raises", {"multiple_of": 0}),
    ("test_invalid_multiple_of_raises", {"multiple_of": -1}),
    ("test_invalid_multiple_of_raises", {"multiple_of": -64}),

    ("test_invalid_dropout_raises", {"dropout": -0.1}),
    ("test_invalid_dropout_raises", {"dropout": 1.0}),
    ("test_invalid_dropout_raises", {"dropout": 1.5}),

    ("test_invalid_init_std_raises", {"init_std": 0.0}),
    ("test_invalid_init_std_raises", {"init_std": -0.01}),

    ("test_rejects_wrong_input_rank", {"bad_x": torch.randn(8, 64)}),
    ("test_rejects_wrong_input_rank", {"bad_x": torch.randn(2, 8, 64, 1)})]

n_passed = 0

for name in mlp_plain_tests:
    globals()[name]()
    n_passed += 1

for name, kwargs in mlp_param_tests:
    globals()[name](**kwargs)
    n_passed += 1

print(f"OK: {n_passed} SwiGLUMLP tests/casos pasaron.")

OK: 42 SwiGLUMLP tests/casos pasaron.


---

In [29]:
# ============================================================
# Mini DeepSeek-V4 TransformerBlock Baseline
# Pre-Norm Dense Transformer Block
# ============================================================

from dataclasses import dataclass
from typing import Optional, Dict, Tuple, Union

import torch
import torch.nn as nn


# ============================================================
# CONFIG
# ============================================================

@dataclass
class TransformerBlockConfig:
    d_model: int
    rms_norm_eps: float = 1e-6

    # Attention config
    n_heads: int = 4
    head_dim: Optional[int] = None
    attention_dropout: float = 0.0
    residual_dropout: float = 0.0
    use_attention_bias: bool = False
    use_rope: bool = True
    rope_theta: float = 10000.0
    rotary_dim: Optional[int] = None
    max_seq_len: int = 1024

    # MLP config
    mlp_hidden_dim: Optional[int] = None
    mlp_expansion_factor: float = 4.0
    mlp_multiple_of: int = 1
    mlp_dropout: float = 0.0
    use_mlp_bias: bool = False

    # Initialization
    init_std: float = 0.02

    def validate(self) -> None:
        if self.d_model <= 0:
            raise ValueError(f"d_model must be > 0, got {self.d_model}")

        if self.rms_norm_eps <= 0:
            raise ValueError(
                f"rms_norm_eps must be > 0, got {self.rms_norm_eps}"
            )

        if self.max_seq_len <= 0:
            raise ValueError(f"max_seq_len must be > 0, got {self.max_seq_len}")

        if self.init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {self.init_std}")

        # Validate attention by constructing its config and calling validate.
        attention_config = self.to_attention_config()
        attention_config.validate()

        # Validate MLP by constructing its config and calling validate.
        mlp_config = self.to_mlp_config()
        mlp_config.validate()

        if attention_config.d_model != self.d_model:
            raise ValueError(
                "attention_config.d_model must match block d_model. "
                f"Got {attention_config.d_model} vs {self.d_model}"
            )

        if mlp_config.d_model != self.d_model:
            raise ValueError(
                "mlp_config.d_model must match block d_model. "
                f"Got {mlp_config.d_model} vs {self.d_model}"
            )

    def to_attention_config(self) -> "CausalMHAConfig":
        return CausalMHAConfig(
            d_model=self.d_model,
            n_heads=self.n_heads,
            head_dim=self.head_dim,
            attention_dropout=self.attention_dropout,
            residual_dropout=self.residual_dropout,
            use_bias=self.use_attention_bias,
            use_rope=self.use_rope,
            rope_theta=self.rope_theta,
            rotary_dim=self.rotary_dim,
            max_seq_len=self.max_seq_len,
            init_std=self.init_std)

    def to_mlp_config(self) -> "SwiGLUMLPConfig":
        return SwiGLUMLPConfig(
            d_model=self.d_model,
            hidden_dim=self.mlp_hidden_dim,
            expansion_factor=self.mlp_expansion_factor,
            multiple_of=self.mlp_multiple_of,
            dropout=self.mlp_dropout,
            use_bias=self.use_mlp_bias,
            init_std=self.init_std,)


# ============================================================
# TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(nn.Module):
    """
    Dense pre-norm causal Transformer block.

    Input:
        x: [B, T, d_model]

    Forward:
        x = x + attention(norm1(x))
        x = x + mlp(norm2(x))

    Output:
        x: [B, T, d_model]

    If need_weights=True:
        returns:
            x, {"attn_weights": attn_weights}

    This module intentionally does NOT include:
        - MoE
        - mHC
        - HCA
        - CSA
        - KV cache
        - gradient checkpointing
        - attention sink
        - query/key RMSNorm
    """

    def __init__(self, config: TransformerBlockConfig):
        super().__init__()

        config.validate()

        self.config = config
        self.d_model = config.d_model
        self.max_seq_len = config.max_seq_len

        self.norm1 = RMSNorm(
            dim=config.d_model,
            eps=config.rms_norm_eps)

        self.attention = CausalMultiHeadAttention(
            config.to_attention_config())

        self.norm2 = RMSNorm(
            dim=config.d_model,
            eps=config.rms_norm_eps,)

        self.mlp = SwiGLUMLP(
            config.to_mlp_config())

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        start_pos: int = 0,
        need_weights: bool = False) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:
        """
        Args:
            x:
                Hidden states [B, T, d_model].

            attention_mask:
                Optional attention mask [B, T].
                1 = valid token.
                0 = padding token.

            position_ids:
                Optional RoPE position ids.
                None, [T], or [B, T].

            start_pos:
                RoPE offset when position_ids is None.

            need_weights:
                Whether to return attention weights in aux dict.

        Returns:
            x:
                Updated hidden states [B, T, d_model].

            optionally:
                x, {"attn_weights": attn_weights}
        """

        # ----------------------------------------------------
        # Input validation
        # ----------------------------------------------------
        if x.dim() != 3:
            raise ValueError(
                f"TransformerBlock expects x with shape [B, T, d_model], "
                f"got {tuple(x.shape)}"
            )

        B, T, C = x.shape

        if C != self.d_model:
            raise ValueError(
                f"Expected x.shape[-1] == d_model={self.d_model}, got {C}"
            )

        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length T={T} exceeds max_seq_len={self.max_seq_len}"
            )

        # ----------------------------------------------------
        # Attention sublayer: pre-norm + residual
        # ----------------------------------------------------
        residual = x

        x_norm = self.norm1(x)

        attn_result = self.attention(
            x_norm,
            attention_mask=attention_mask,
            position_ids=position_ids,
            start_pos=start_pos,
            need_weights=need_weights)

        if need_weights:
            attn_out, attn_weights = attn_result
        else:
            attn_out = attn_result
            attn_weights = None

        x = residual + attn_out

        # ----------------------------------------------------
        # MLP sublayer: pre-norm + residual
        # ----------------------------------------------------
        residual = x

        x_norm = self.norm2(x)
        mlp_out = self.mlp(x_norm)

        x = residual + mlp_out

        # ----------------------------------------------------
        # Return
        # ----------------------------------------------------
        if need_weights:
            aux = {
                "attn_weights": attn_weights}
            return x, aux

        return x

In [126]:
# ============================================================
# Embedding -> TransformerBlock
# ============================================================

block_config = TransformerBlockConfig(
    d_model=256,
    rms_norm_eps=1e-6,

    n_heads=4,
    head_dim=64,
    attention_dropout=0.0,
    residual_dropout=0.0,
    use_attention_bias=False,
    use_rope=True,
    rope_theta=10000.0,
    rotary_dim=64,
    max_seq_len=128,

    mlp_hidden_dim=None,
    mlp_expansion_factor=4.0,
    mlp_multiple_of=1,
    mlp_dropout=0.0,
    use_mlp_bias=False,

    init_std=0.02,
)


B, T = 2, 128
vocab_size = 16000
d_model = 256

embedding_config = EmbeddingConfig(
    vocab_size=vocab_size,
    d_model=d_model,
    pad_token_id=0,
    max_seq_len=T,
    embedding_dropout=0.0,
    scale_embeddings=False,
    init_std=0.02,
    tie_word_embeddings=True,
)

embedding = TokenEmbedding(embedding_config)
block = TransformerBlock(block_config)

input_ids = torch.randint(1, vocab_size, (B, T), dtype=torch.long)
attention_mask = torch.ones(B, T, dtype=torch.long)

hidden_states = embedding(input_ids)
out = block(hidden_states, attention_mask=attention_mask)

print("hidden_states:", hidden_states.shape)
print("out:", out.shape)

assert hidden_states.shape == (B, T, d_model)
assert out.shape == (B, T, d_model)
assert torch.isfinite(out).all()

print("Embedding -> TransformerBlock pipeline OK.")

hidden_states: torch.Size([2, 128, 256])
out: torch.Size([2, 128, 256])
Embedding -> TransformerBlock pipeline OK.


In [131]:
# @title
# ============================================================
# TransformerBlock baseline tests
# ============================================================

import pytest
import torch


# ============================================================
# Helpers
# ============================================================

def make_block_config(**overrides):
    cfg = dict(
        d_model=64,
        rms_norm_eps=1e-6,

        n_heads=4,
        head_dim=16,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_attention_bias=False,
        use_rope=True,
        rope_theta=10000.0,
        rotary_dim=16,
        max_seq_len=128,

        mlp_hidden_dim=256,
        mlp_expansion_factor=4.0,
        mlp_multiple_of=1,
        mlp_dropout=0.0,
        use_mlp_bias=False,

        init_std=0.02,
    )
    cfg.update(overrides)
    return TransformerBlockConfig(**cfg)


def make_block(**overrides):
    return TransformerBlock(make_block_config(**overrides))


def make_block_input(B=2, T=8, D=64, dtype=torch.float32, device="cpu"):
    return torch.randn(B, T, D, dtype=dtype, device=device)


def zero_module_parameters(module):
    for param in module.parameters():
        with torch.no_grad():
            param.zero_()


# ============================================================
# A. Config tests
# ============================================================

def test_valid_block_config_builds():
    config = make_block_config(
        d_model=256,
        n_heads=4,
        head_dim=64,
        mlp_hidden_dim=1024,
        max_seq_len=128,
        rotary_dim=64,
    )
    block = TransformerBlock(config)

    assert block.d_model == 256
    assert block.attention.n_heads == 4
    assert block.attention.head_dim == 64
    assert block.mlp.hidden_dim == 1024


@pytest.mark.parametrize("d_model", [0, -1, -256])
def test_invalid_block_d_model_raises(d_model):
    with pytest.raises(ValueError):
        TransformerBlock(make_block_config(d_model=d_model))


@pytest.mark.parametrize("rms_norm_eps", [0.0, -1e-6])
def test_invalid_rms_norm_eps_raises(rms_norm_eps):
    with pytest.raises(ValueError):
        TransformerBlock(make_block_config(rms_norm_eps=rms_norm_eps))


def test_attention_and_mlp_d_model_match_block():
    # En la config plana no es posible pasar attention.d_model o mlp.d_model
    # distintos directamente; esta prueba verifica que los subconfigs heredan d_model.
    config = make_block_config(d_model=64)

    attn_config = config.to_attention_config()
    mlp_config = config.to_mlp_config()

    assert attn_config.d_model == config.d_model
    assert mlp_config.d_model == config.d_model


# ============================================================
# B. Internal structure tests
# ============================================================

def test_block_has_expected_modules():
    block = make_block()

    assert hasattr(block, "norm1")
    assert hasattr(block, "attention")
    assert hasattr(block, "norm2")
    assert hasattr(block, "mlp")

    assert isinstance(block.norm1, RMSNorm)
    assert isinstance(block.attention, CausalMultiHeadAttention)
    assert isinstance(block.norm2, RMSNorm)
    assert isinstance(block.mlp, SwiGLUMLP)


def test_norm_weights_initialized_to_ones():
    block = make_block()

    assert torch.allclose(block.norm1.weight, torch.ones_like(block.norm1.weight))
    assert torch.allclose(block.norm2.weight, torch.ones_like(block.norm2.weight))


# ============================================================
# C. Forward tests
# ============================================================

def test_block_output_shape_matches_input_shape():
    block = make_block()

    x = make_block_input(B=2, T=8, D=64)
    out = block(x)

    assert out.shape == x.shape


@pytest.mark.parametrize(
    "bad_x",
    [
        torch.randn(8, 64),
        torch.randn(2, 8, 64, 1),
    ],
)
def test_block_rejects_wrong_input_rank(bad_x):
    block = make_block()

    with pytest.raises(ValueError):
        block(bad_x)


def test_block_rejects_wrong_hidden_size():
    block = make_block(d_model=64)

    x = torch.randn(2, 8, 32)

    with pytest.raises(ValueError):
        block(x)


def test_returns_aux_when_need_weights_true():
    block = make_block()
    x = make_block_input(B=2, T=8, D=64)

    out, aux = block(x, need_weights=True)

    assert out.shape == (2, 8, 64)
    assert isinstance(aux, dict)
    assert "attn_weights" in aux
    assert aux["attn_weights"].shape == (2, 4, 8, 8)


# ============================================================
# D. Residual / pre-norm tests
# ============================================================

def test_block_changes_input_when_weights_nonzero():
    block = make_block()
    block.eval()

    x = make_block_input(B=2, T=8, D=64)
    out = block(x)

    assert out.shape == x.shape
    assert not torch.equal(out, x)


def test_residual_identity_when_attention_and_mlp_zeroed():
    block = make_block(
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
        use_attention_bias=True,
        use_mlp_bias=True,
    )
    block.eval()

    zero_module_parameters(block.attention)
    zero_module_parameters(block.mlp)

    x = make_block_input(B=2, T=8, D=64)
    out = block(x)

    assert torch.allclose(out, x, atol=0.0, rtol=0.0)


# ============================================================
# E. Inherited causality tests
# ============================================================

def test_block_changing_future_tokens_does_not_change_past_outputs():
    block = make_block(
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
    )
    block.eval()

    B, T, D = 2, 10, 64
    cut = 5

    x1 = make_block_input(B=B, T=T, D=D)
    x2 = x1.clone()
    x2[:, cut:, :] = torch.randn_like(x2[:, cut:, :])

    out1 = block(x1)
    out2 = block(x2)

    assert torch.allclose(
        out1[:, :cut, :],
        out2[:, :cut, :],
        atol=1e-5,
        rtol=1e-5,
    )


def test_attention_mask_blocks_padding_keys_through_block():
    block = make_block(
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
    )
    block.eval()

    B, T, D = 2, 8, 64
    x = make_block_input(B=B, T=T, D=D)

    attention_mask = torch.ones(B, T)
    attention_mask[0, 3] = 0
    attention_mask[1, 5] = 0

    _, aux = block(
        x,
        attention_mask=attention_mask,
        need_weights=True,
    )

    weights = aux["attn_weights"]

    assert torch.allclose(
        weights[0, :, :, 3],
        torch.zeros_like(weights[0, :, :, 3]),
        atol=0.0,
        rtol=0.0,
    )

    assert torch.allclose(
        weights[1, :, :, 5],
        torch.zeros_like(weights[1, :, :, 5]),
        atol=0.0,
        rtol=0.0,
    )


# ============================================================
# F. RoPE passthrough tests
# ============================================================

def test_start_pos_matches_explicit_position_ids_through_block():
    block = make_block(
        use_rope=True,
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
    )
    block.eval()

    B, T, D = 2, 8, 64
    start_pos = 10

    x = make_block_input(B=B, T=T, D=D)

    out_start = block(x, start_pos=start_pos)
    out_explicit = block(
        x,
        position_ids=torch.arange(start_pos, start_pos + T),
    )

    assert torch.allclose(out_start, out_explicit, atol=1e-6, rtol=1e-5)


# ============================================================
# G. Dropout tests
# ============================================================

def test_block_dropout_zero_is_deterministic():
    block = make_block(
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
    )
    block.train()

    x = make_block_input(B=4, T=16, D=64)

    out1 = block(x)
    out2 = block(x)

    assert torch.equal(out1, out2)


def test_block_dropout_disabled_in_eval_mode():
    block = make_block(
        attention_dropout=0.5,
        residual_dropout=0.5,
        mlp_dropout=0.5,
    )
    block.eval()

    x = make_block_input(B=4, T=16, D=64)

    out1 = block(x)
    out2 = block(x)

    assert torch.equal(out1, out2)


def test_block_dropout_active_in_train_mode():
    block = make_block(
        attention_dropout=0.5,
        residual_dropout=0.5,
        mlp_dropout=0.5,
    )
    block.train()

    x = make_block_input(B=4, T=16, D=64)

    out1 = block(x)
    out2 = block(x)

    assert not torch.equal(out1, out2)


# ============================================================
# H. Gradient tests
# ============================================================

def test_block_backward_computes_gradients():
    block = make_block(
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
        use_attention_bias=True,
        use_mlp_bias=True,
    )

    x = make_block_input(B=2, T=8, D=64)
    x.requires_grad_(True)

    out = block(x)
    loss = out.sum()
    loss.backward()

    assert x.grad is not None
    assert torch.isfinite(x.grad).all()

    for name, param in block.named_parameters():
        assert param.grad is not None, f"Missing grad for {name}"
        assert torch.isfinite(param.grad).all(), f"Non-finite grad for {name}"


# ============================================================
# I. Integration tests
# ============================================================

def test_embedding_to_block_pipeline():
    B, T = 2, 16
    vocab_size = 128
    d_model = 64

    embedding = TokenEmbedding(
        EmbeddingConfig(
            vocab_size=vocab_size,
            d_model=d_model,
            pad_token_id=0,
            max_seq_len=T,
            embedding_dropout=0.0,
            scale_embeddings=False,
            init_std=0.02,
            tie_word_embeddings=True,
        )
    )

    block = make_block(
        d_model=d_model,
        n_heads=4,
        head_dim=16,
        rotary_dim=16,
        max_seq_len=T,
    )

    input_ids = torch.randint(1, vocab_size, (B, T), dtype=torch.long)

    hidden_states = embedding(input_ids)
    out = block(hidden_states)

    assert hidden_states.shape == (B, T, d_model)
    assert out.shape == (B, T, d_model)
    assert torch.isfinite(out).all()


def test_synthetic_dataset_batch_through_block():
    data_cfg = SyntheticRetrievalConfig(
        num_train_examples=128,
        num_val_examples=32,
        block_size=64,
        min_filler_tokens=8,
        max_filler_tokens=32,
        num_keys_per_example=4,
        vocab_filler_size=100,
        num_key_types=32,
        num_value_types=64,
        batch_size=8,
        num_workers=0,
        seed=123,
    )

    train_loader, _, tokenizer = create_synthetic_retrieval_dataloaders(
        cfg=data_cfg,
        use_mtp=False,
    )

    batch = next(iter(train_loader))

    assert isinstance(batch, dict)
    assert "input_ids" in batch
    assert "labels" in batch

    input_ids = batch["input_ids"]
    labels = batch["labels"]

    d_model = 64

    embedding = TokenEmbedding(
        EmbeddingConfig(
            vocab_size=tokenizer.vocab_size,
            d_model=d_model,
            pad_token_id=tokenizer.pad_id,
            max_seq_len=data_cfg.block_size,
            embedding_dropout=0.0,
            scale_embeddings=False,
            init_std=0.02,
            tie_word_embeddings=True,
        )
    )

    block = make_block(
        d_model=d_model,
        n_heads=4,
        head_dim=16,
        rotary_dim=16,
        max_seq_len=data_cfg.block_size,
    )

    attention_mask = (input_ids != tokenizer.pad_id).long()

    hidden_states = embedding(input_ids)
    out = block(hidden_states, attention_mask=attention_mask)

    assert input_ids.shape == (data_cfg.batch_size, data_cfg.block_size)
    assert labels.shape == (data_cfg.batch_size, data_cfg.block_size)
    assert hidden_states.shape == (data_cfg.batch_size, data_cfg.block_size, d_model)
    assert out.shape == (data_cfg.batch_size, data_cfg.block_size, d_model)
    assert torch.isfinite(out).all()

In [132]:
# ============================================================
# Run ONLY TransformerBlock tests in Jupyter
# Handles pytest-style parametrized tests manually
# ============================================================

block_plain_tests = [
    "test_valid_block_config_builds",
    "test_attention_and_mlp_d_model_match_block",
    "test_block_has_expected_modules",
    "test_norm_weights_initialized_to_ones",
    "test_block_output_shape_matches_input_shape",
    "test_block_rejects_wrong_hidden_size",
    "test_returns_aux_when_need_weights_true",
    "test_block_changes_input_when_weights_nonzero",
    "test_residual_identity_when_attention_and_mlp_zeroed",
    "test_block_changing_future_tokens_does_not_change_past_outputs",
    "test_attention_mask_blocks_padding_keys_through_block",
    "test_start_pos_matches_explicit_position_ids_through_block",
    "test_block_dropout_zero_is_deterministic",
    "test_block_dropout_disabled_in_eval_mode",
    "test_block_dropout_active_in_train_mode",
    "test_block_backward_computes_gradients",
    "test_embedding_to_block_pipeline",
    "test_synthetic_dataset_batch_through_block",
]

block_param_tests = [
    ("test_invalid_block_d_model_raises", {"d_model": 0}),
    ("test_invalid_block_d_model_raises", {"d_model": -1}),
    ("test_invalid_block_d_model_raises", {"d_model": -256}),

    ("test_invalid_rms_norm_eps_raises", {"rms_norm_eps": 0.0}),
    ("test_invalid_rms_norm_eps_raises", {"rms_norm_eps": -1e-6}),

    ("test_block_rejects_wrong_input_rank", {"bad_x": torch.randn(8, 64)}),
    ("test_block_rejects_wrong_input_rank", {"bad_x": torch.randn(2, 8, 64, 1)}),
]

n_passed = 0

for name in block_plain_tests:
    globals()[name]()
    n_passed += 1

for name, kwargs in block_param_tests:
    globals()[name](**kwargs)
    n_passed += 1

print(f"OK: {n_passed} TransformerBlock tests/casos pasaron.")

Synthetic tokenizer vocab size: 216
OK: 25 TransformerBlock tests/casos pasaron.


# Mini Large Model for Baseline

In [135]:
# ============================================================
# Mini Causal LM Baseline
# TokenEmbedding + TransformerBlock x L + Final RMSNorm + LM Head
# ============================================================

from dataclasses import dataclass
from typing import Optional, Dict, Any

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# CONFIG
# ============================================================

@dataclass
class MiniCausalLMConfig:
    vocab_size: int
    d_model: int
    n_layers: int

    pad_token_id: Optional[int] = None
    max_seq_len: int = 1024

    embedding_dropout: float = 0.0
    scale_embeddings: bool = False
    tie_word_embeddings: bool = True

    rms_norm_eps: float = 1e-6

    n_heads: int = 4
    head_dim: Optional[int] = None
    attention_dropout: float = 0.0
    residual_dropout: float = 0.0
    use_attention_bias: bool = False
    use_rope: bool = True
    rope_theta: float = 10000.0
    rotary_dim: Optional[int] = None

    mlp_hidden_dim: Optional[int] = None
    mlp_expansion_factor: float = 4.0
    mlp_multiple_of: int = 1
    mlp_dropout: float = 0.0
    use_mlp_bias: bool = False

    init_std: float = 0.02

    def validate(self) -> None:
        if self.vocab_size <= 0:
            raise ValueError(f"vocab_size must be > 0, got {self.vocab_size}")

        if self.d_model <= 0:
            raise ValueError(f"d_model must be > 0, got {self.d_model}")

        if self.n_layers <= 0:
            raise ValueError(f"n_layers must be > 0, got {self.n_layers}")

        if self.max_seq_len <= 0:
            raise ValueError(f"max_seq_len must be > 0, got {self.max_seq_len}")

        if self.pad_token_id is not None:
            if not (0 <= self.pad_token_id < self.vocab_size):
                raise ValueError(
                    "pad_token_id must satisfy 0 <= pad_token_id < vocab_size, "
                    f"got pad_token_id={self.pad_token_id}, "
                    f"vocab_size={self.vocab_size}"
                )

        if self.rms_norm_eps <= 0:
            raise ValueError(
                f"rms_norm_eps must be > 0, got {self.rms_norm_eps}"
            )

        if self.init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {self.init_std}")

        # Validate child configs too.
        self.to_embedding_config().validate()
        self.to_block_config().validate()

    def to_embedding_config(self) -> "EmbeddingConfig":
        return EmbeddingConfig(
            vocab_size=self.vocab_size,
            d_model=self.d_model,
            pad_token_id=self.pad_token_id,
            max_seq_len=self.max_seq_len,
            embedding_dropout=self.embedding_dropout,
            scale_embeddings=self.scale_embeddings,
            init_std=self.init_std,
            tie_word_embeddings=self.tie_word_embeddings,
        )

    def to_block_config(self) -> "TransformerBlockConfig":
        return TransformerBlockConfig(
            d_model=self.d_model,
            rms_norm_eps=self.rms_norm_eps,

            n_heads=self.n_heads,
            head_dim=self.head_dim,
            attention_dropout=self.attention_dropout,
            residual_dropout=self.residual_dropout,
            use_attention_bias=self.use_attention_bias,
            use_rope=self.use_rope,
            rope_theta=self.rope_theta,
            rotary_dim=self.rotary_dim,
            max_seq_len=self.max_seq_len,

            mlp_hidden_dim=self.mlp_hidden_dim,
            mlp_expansion_factor=self.mlp_expansion_factor,
            mlp_multiple_of=self.mlp_multiple_of,
            mlp_dropout=self.mlp_dropout,
            use_mlp_bias=self.use_mlp_bias,

            init_std=self.init_std,
        )


# ============================================================
# MINI CAUSAL LM
# ============================================================

class MiniCausalLM(nn.Module):
    """
    Minimal dense causal language model baseline.

    Components:
        TokenEmbedding
        TransformerBlock x n_layers
        Final RMSNorm
        LM Head

    Input:
        input_ids: [B, T]
        labels: optional [B, T]
        attention_mask: optional [B, T]
        position_ids: optional [T] or [B, T]

    Output dict:
        {
            "logits": logits,  # [B, T, vocab_size]
            "loss": loss,      # scalar or None
            "aux": aux,        # dict
        }

    This module intentionally does NOT implement:
        - generate()
        - KV cache
        - MTP
        - MoE
        - HCA
        - CSA
        - mHC
        - optimizer/training loop
    """

    def __init__(self, config: MiniCausalLMConfig):
        super().__init__()

        config.validate()

        self.config = config
        self.vocab_size = config.vocab_size
        self.d_model = config.d_model
        self.n_layers = config.n_layers
        self.pad_token_id = config.pad_token_id
        self.max_seq_len = config.max_seq_len

        self.embedding = TokenEmbedding(config.to_embedding_config())

        block_config = config.to_block_config()

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(block_config)
                for _ in range(config.n_layers)
            ])

        self.final_norm = RMSNorm(
            dim=config.d_model,
            eps=config.rms_norm_eps)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False)

        self.reset_parameters()

        if config.tie_word_embeddings:
            self.tie_weights()

    def reset_parameters(self) -> None:
        """
        Initialize LM head if it is not tied.

        Note:
            TokenEmbedding, TransformerBlock, attention and MLP modules
            already initialize their own parameters.
        """
        nn.init.normal_(
            self.lm_head.weight,
            mean=0.0,
            std=self.config.init_std)

    def tie_weights(self) -> None:
        """
        Tie LM head weight to token embedding weight.

        This must share the same Parameter object, not copy values.
        """
        self.lm_head.weight = self.embedding.weight


    def _validate_input_ids(self, input_ids: torch.Tensor) -> tuple[int, int]:
        if input_ids.dim() != 2:
            raise ValueError(
                f"input_ids must have shape [B, T], got {tuple(input_ids.shape)}"
            )

        B, T = input_ids.shape

        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length T={T} exceeds max_seq_len={self.max_seq_len}"
            )

        return B, T


    def _build_or_validate_attention_mask(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor]) -> Optional[torch.Tensor]:

        if attention_mask is None:
            if self.pad_token_id is None:
                return None

            return (input_ids != self.pad_token_id).long()

        if attention_mask.shape != input_ids.shape:
            raise ValueError(
                "attention_mask must have the same shape as input_ids. "
                f"Expected {tuple(input_ids.shape)}, got {tuple(attention_mask.shape)}")

        return attention_mask

    def _validate_labels(
        self,
        labels: torch.Tensor,
        input_ids: torch.Tensor) -> None:

        if labels.shape != input_ids.shape:
            raise ValueError(
                "labels must have the same shape as input_ids. "
                f"Expected {tuple(input_ids.shape)}, got {tuple(labels.shape)}")

    def forward(
        self,
        input_ids: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        start_pos: int = 0,
        return_aux: bool = False,
        need_weights: bool = False,) -> Dict[str, Any]:

        """
        Args:
            input_ids:
                Token ids [B, T].

            labels:
                Optional next-token labels [B, T].
                The dataset is expected to already shift labels.
                This model does NOT shift internally.

            attention_mask:
                Optional [B, T].
                1 = valid token.
                0 = pad token.
                If None and pad_token_id exists, it is built automatically.

            position_ids:
                Optional RoPE positions: None, [T], or [B, T].

            start_pos:
                RoPE offset if position_ids is None.

            return_aux:
                Whether to return aux dictionary.
                Current aux is empty unless need_weights=True.

            need_weights:
                Whether to collect attention weights from all layers.

        Returns:
            {
                "logits": [B, T, vocab_size],
                "loss": scalar or None,
                "aux": dict,
            }
        """
         # ----------------------------------------------------
        # Accept full batch dict as first argument
        # ----------------------------------------------------
        if isinstance(input_ids, dict):
            batch = normalize_lm_batch(input_ids)

            input_ids = batch["input_ids"]
            labels = batch.get("labels", labels)
            attention_mask = batch.get("attention_mask", attention_mask)
            position_ids = batch.get("position_ids", position_ids)

        elif isinstance(input_ids, (tuple, list)) or torch.is_tensor(input_ids):
            # Optional: only normalize tuples/tensors if this path is useful to you.
            pass

        B, T = self._validate_input_ids(input_ids)

        if labels is not None:
            self._validate_labels(labels, input_ids)

        attention_mask = self._build_or_validate_attention_mask(
            input_ids=input_ids,
            attention_mask=attention_mask,)

        # ----------------------------------------------------
        # Embedding
        # ----------------------------------------------------
        x = self.embedding(input_ids)
        # [B, T, d_model]

        # ----------------------------------------------------
        # Transformer blocks
        # ----------------------------------------------------
        attn_weights_all = []

        for block in self.blocks:
            if need_weights:
                x, block_aux = block(
                    x,
                    attention_mask=attention_mask,
                    position_ids=position_ids,
                    start_pos=start_pos,
                    need_weights=True)

                attn_weights_all.append(block_aux["attn_weights"])
            else:
                x = block(
                    x,
                    attention_mask=attention_mask,
                    position_ids=position_ids,
                    start_pos=start_pos,
                    need_weights=False)

        # ----------------------------------------------------
        # Final norm + LM head
        # ----------------------------------------------------
        x = self.final_norm(x)
        logits = self.lm_head(x)
        # [B, T, vocab_size]

        # ----------------------------------------------------
        # Optional loss
        # ----------------------------------------------------
        loss = None

        if labels is not None:
            ignore_index = (
                self.pad_token_id
                if self.pad_token_id is not None
                else -100)

            loss = F.cross_entropy(
                logits.reshape(B * T, self.vocab_size),
                labels.reshape(B * T),
                ignore_index=ignore_index,)

        # ----------------------------------------------------
        # Aux
        # ----------------------------------------------------
        aux = {}

        if need_weights:
            aux["attn_weights"] = attn_weights_all

        # If return_aux=False and need_weights=False, aux remains empty.
        # We still return it for output consistency.
        return {
            "logits": logits,
            "loss": loss,
            "aux": aux if (return_aux or need_weights) else {},}

In [136]:
# ============================================================
# Smoke test: random input_ids
# ============================================================

config = MiniCausalLMConfig(
    vocab_size=16000,
    d_model=256,
    n_layers=2,

    pad_token_id=0,
    max_seq_len=128,

    embedding_dropout=0.0,
    scale_embeddings=False,
    tie_word_embeddings=True,

    rms_norm_eps=1e-6,

    n_heads=4,
    head_dim=64,
    attention_dropout=0.0,
    residual_dropout=0.0,
    use_attention_bias=False,
    use_rope=True,
    rope_theta=10000.0,
    rotary_dim=64,

    mlp_hidden_dim=None,
    mlp_expansion_factor=4.0,
    mlp_multiple_of=1,
    mlp_dropout=0.0,
    use_mlp_bias=False,

    init_std=0.02,
)

model = MiniCausalLM(config)

B, T = 2, 128

input_ids = torch.randint(
    low=1,
    high=config.vocab_size,
    size=(B, T),
    dtype=torch.long)

labels = torch.randint(
    low=1,
    high=config.vocab_size,
    size=(B, T),
    dtype=torch.long)

outputs = model(
    input_ids=input_ids,
    labels=labels,
    need_weights=True)

logits = outputs["logits"]
loss = outputs["loss"]
aux = outputs["aux"]

print("logits:", logits.shape)
print("loss:", loss)
print("num attn layers:", len(aux["attn_weights"]))
print("attn layer 0:", aux["attn_weights"][0].shape)

assert logits.shape == (B, T, config.vocab_size)
assert loss is not None
assert loss.dim() == 0
assert torch.isfinite(loss)
assert len(aux["attn_weights"]) == config.n_layers
assert aux["attn_weights"][0].shape == (B, config.n_heads, T, T)
assert model.lm_head.weight is model.embedding.weight

print("MiniCausalLM baseline OK.")

logits: torch.Size([2, 128, 16000])
loss: tensor(9.7476, grad_fn=<NllLossBackward0>)
num attn layers: 2
attn layer 0: torch.Size([2, 4, 128, 128])
MiniCausalLM baseline OK.


## Tests for Mini LM

In [139]:
# @title
# ============================================================
# MiniCausalLM baseline tests
# ============================================================

import pytest
import torch
import torch.nn.functional as F


# ============================================================
# Helpers
# ============================================================

def make_lm_config(**overrides):
    cfg = dict(
        vocab_size=128,
        d_model=64,
        n_layers=2,

        pad_token_id=0,
        max_seq_len=64,

        embedding_dropout=0.0,
        scale_embeddings=False,
        tie_word_embeddings=True,

        rms_norm_eps=1e-6,

        n_heads=4,
        head_dim=16,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_attention_bias=False,
        use_rope=True,
        rope_theta=10000.0,
        rotary_dim=16,

        mlp_hidden_dim=256,
        mlp_expansion_factor=4.0,
        mlp_multiple_of=1,
        mlp_dropout=0.0,
        use_mlp_bias=False,

        init_std=0.02,
    )
    cfg.update(overrides)
    return MiniCausalLMConfig(**cfg)


def make_lm(**overrides):
    return MiniCausalLM(make_lm_config(**overrides))


def make_lm_batch(
    B=2,
    T=16,
    vocab_size=128,
    pad_token_id=0,
    include_pad=False,
):
    input_ids = torch.randint(
        low=1,
        high=vocab_size,
        size=(B, T),
        dtype=torch.long,
    )

    labels = torch.randint(
        low=1,
        high=vocab_size,
        size=(B, T),
        dtype=torch.long,
    )

    if include_pad:
        input_ids[0, -2:] = pad_token_id
        labels[0, -2:] = pad_token_id

    return input_ids, labels


# ============================================================
# A. Config tests
# ============================================================

def test_valid_minicausallm_config_builds():
    config = make_lm_config(
        vocab_size=128,
        d_model=256,
        n_layers=2,
        n_heads=4,
        head_dim=64,
        rotary_dim=64,
        mlp_hidden_dim=1024,
    )

    model = MiniCausalLM(config)

    assert model.vocab_size == 128
    assert model.d_model == 256
    assert model.n_layers == 2
    assert len(model.blocks) == 2


@pytest.mark.parametrize("vocab_size", [0, -1, -128])
def test_invalid_vocab_size_raises(vocab_size):
    with pytest.raises(ValueError):
        MiniCausalLM(make_lm_config(vocab_size=vocab_size))


@pytest.mark.parametrize("n_layers", [0, -1, -2])
def test_invalid_n_layers_raises(n_layers):
    with pytest.raises(ValueError):
        MiniCausalLM(make_lm_config(n_layers=n_layers))


@pytest.mark.parametrize("pad_token_id", [-1, 128])
def test_invalid_pad_token_id_raises(pad_token_id):
    with pytest.raises(ValueError):
        MiniCausalLM(make_lm_config(vocab_size=128, pad_token_id=pad_token_id))


# ============================================================
# B. Internal structure tests
# ============================================================

def test_model_has_expected_modules():
    config = make_lm_config(n_layers=3)
    model = MiniCausalLM(config)

    assert hasattr(model, "embedding")
    assert hasattr(model, "blocks")
    assert hasattr(model, "final_norm")
    assert hasattr(model, "lm_head")

    assert len(model.blocks) == config.n_layers


def test_blocks_are_transformerBlocks():
    model = make_lm(n_layers=2)

    for block in model.blocks:
        assert isinstance(block, TransformerBlock)


def test_final_norm_is_rmsnorm():
    model = make_lm()

    assert isinstance(model.final_norm, RMSNorm)


# ============================================================
# C. Weight tying tests
# ============================================================

def test_weight_tying_enabled_shares_parameter():
    model = make_lm(tie_word_embeddings=True)

    assert model.lm_head.weight is model.embedding.weight


def test_weight_tying_disabled_uses_independent_parameter():
    model = make_lm(tie_word_embeddings=False)

    assert model.lm_head.weight is not model.embedding.weight


# ============================================================
# D. Forward tests
# ============================================================

def test_logits_shape():
    config = make_lm_config(vocab_size=128, max_seq_len=16)
    model = MiniCausalLM(config)

    input_ids, _ = make_lm_batch(B=2, T=16, vocab_size=config.vocab_size)

    outputs = model(input_ids=input_ids)

    assert outputs["logits"].shape == (2, 16, config.vocab_size)


def test_forward_without_labels_returns_no_loss():
    model = make_lm()

    input_ids, _ = make_lm_batch(B=2, T=16)

    outputs = model(input_ids=input_ids, labels=None)

    assert outputs["loss"] is None


def test_forward_with_labels_returns_scalar_loss():
    model = make_lm()

    input_ids, labels = make_lm_batch(B=2, T=16)

    outputs = model(input_ids=input_ids, labels=labels)

    assert outputs["loss"] is not None
    assert outputs["loss"].dim() == 0
    assert torch.isfinite(outputs["loss"])


@pytest.mark.parametrize(
    "bad_input_ids",
    [
        torch.ones(16, dtype=torch.long),
        torch.ones(2, 16, 1, dtype=torch.long),
    ],
)
def test_rejects_wrong_input_ids_rank(bad_input_ids):
    model = make_lm()

    with pytest.raises(ValueError):
        model(input_ids=bad_input_ids)


def test_rejects_labels_wrong_shape():
    model = make_lm()

    input_ids, _ = make_lm_batch(B=2, T=16)
    bad_labels = torch.ones(2, 15, dtype=torch.long)

    with pytest.raises(ValueError):
        model(input_ids=input_ids, labels=bad_labels)


def test_rejects_sequence_longer_than_max_seq_len():
    model = make_lm(max_seq_len=8)

    input_ids = torch.ones(2, 9, dtype=torch.long)

    with pytest.raises(ValueError):
        model(input_ids=input_ids)


# ============================================================
# E. attention_mask tests
# ============================================================

def test_auto_attention_mask_from_pad_token():
    config = make_lm_config(
        pad_token_id=0,
        n_layers=2,
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
    )
    model = MiniCausalLM(config)
    model.eval()

    input_ids, labels = make_lm_batch(
        B=2,
        T=16,
        vocab_size=config.vocab_size,
        pad_token_id=config.pad_token_id,
        include_pad=True,
    )

    outputs = model(
        input_ids=input_ids,
        labels=labels,
        attention_mask=None,
        need_weights=True,
    )

    pad_positions = input_ids == config.pad_token_id

    for layer_weights in outputs["aux"]["attn_weights"]:
        # layer_weights: [B, H, T, T]
        for b in range(input_ids.shape[0]):
            for s in torch.where(pad_positions[b])[0]:
                assert torch.allclose(
                    layer_weights[b, :, :, s],
                    torch.zeros_like(layer_weights[b, :, :, s]),
                    atol=0.0,
                    rtol=0.0,
                )


def test_explicit_attention_mask_is_used():
    config = make_lm_config(
        pad_token_id=0,
        n_layers=2,
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
    )
    model = MiniCausalLM(config)
    model.eval()

    B, T = 2, 16
    input_ids, labels = make_lm_batch(B=B, T=T, vocab_size=config.vocab_size)

    attention_mask = torch.ones(B, T, dtype=torch.long)
    blocked_position = 5
    attention_mask[:, blocked_position] = 0

    outputs = model(
        input_ids=input_ids,
        labels=labels,
        attention_mask=attention_mask,
        need_weights=True,
    )

    for layer_weights in outputs["aux"]["attn_weights"]:
        assert torch.allclose(
            layer_weights[:, :, :, blocked_position],
            torch.zeros_like(layer_weights[:, :, :, blocked_position]),
            atol=0.0,
            rtol=0.0,
        )


@pytest.mark.parametrize(
    "bad_mask",
    [
        torch.ones(16, dtype=torch.long),
        torch.ones(2, 16, 1, dtype=torch.long),
        torch.ones(2, 17, dtype=torch.long),
    ],
)
def test_invalid_attention_mask_shape_raises(bad_mask):
    model = make_lm(max_seq_len=16)

    input_ids, labels = make_lm_batch(B=2, T=16)

    with pytest.raises(ValueError):
        model(
            input_ids=input_ids,
            labels=labels,
            attention_mask=bad_mask,
        )


# ============================================================
# F. Loss tests
# ============================================================

def test_loss_matches_manual_cross_entropy():
    config = make_lm_config(
        pad_token_id=0,
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
    )
    model = MiniCausalLM(config)
    model.eval()

    input_ids, labels = make_lm_batch(
        B=2,
        T=16,
        vocab_size=config.vocab_size,
        pad_token_id=config.pad_token_id,
        include_pad=True,
    )

    outputs = model(input_ids=input_ids, labels=labels)

    logits = outputs["logits"]

    manual_loss = F.cross_entropy(
        logits.reshape(-1, config.vocab_size),
        labels.reshape(-1),
        ignore_index=config.pad_token_id,
    )

    assert torch.allclose(outputs["loss"], manual_loss, atol=1e-6, rtol=1e-6)


def test_loss_ignores_pad_token_id():
    config = make_lm_config(
        pad_token_id=0,
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
    )
    model = MiniCausalLM(config)
    model.eval()

    input_ids, labels = make_lm_batch(
        B=2,
        T=16,
        vocab_size=config.vocab_size,
    )

    labels[0, 0] = config.pad_token_id
    labels[1, 3] = config.pad_token_id

    outputs = model(input_ids=input_ids, labels=labels)
    logits = outputs["logits"]

    manual_loss = F.cross_entropy(
        logits.reshape(-1, config.vocab_size),
        labels.reshape(-1),
        ignore_index=config.pad_token_id,
    )

    assert torch.allclose(outputs["loss"], manual_loss, atol=1e-6, rtol=1e-6)


# ============================================================
# G. Aux / attention weights tests
# ============================================================

def test_need_weights_returns_attention_weights_per_layer():
    config = make_lm_config(n_layers=3)
    model = MiniCausalLM(config)

    input_ids, _ = make_lm_batch(B=2, T=16, vocab_size=config.vocab_size)

    outputs = model(input_ids=input_ids, need_weights=True)

    assert "attn_weights" in outputs["aux"]
    assert isinstance(outputs["aux"]["attn_weights"], list)
    assert len(outputs["aux"]["attn_weights"]) == config.n_layers

    for weights in outputs["aux"]["attn_weights"]:
        assert weights.shape == (2, config.n_heads, 16, 16)


def test_need_weights_false_does_not_store_attention_weights():
    model = make_lm()

    input_ids, _ = make_lm_batch(B=2, T=16)

    outputs = model(input_ids=input_ids, need_weights=False)

    assert "attn_weights" not in outputs["aux"]


# ============================================================
# H. End-to-end causality tests
# ============================================================

def test_changing_future_tokens_does_not_change_past_logits():
    config = make_lm_config(
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
    )
    model = MiniCausalLM(config)
    model.eval()

    B, T = 2, 16
    cut = 8

    input_ids_1, _ = make_lm_batch(
        B=B,
        T=T,
        vocab_size=config.vocab_size,
    )

    input_ids_2 = input_ids_1.clone()

    input_ids_2[:, cut:] = torch.randint(
        low=1,
        high=config.vocab_size,
        size=input_ids_2[:, cut:].shape,
        dtype=torch.long,
    )

    logits_1 = model(input_ids=input_ids_1)["logits"]
    logits_2 = model(input_ids=input_ids_2)["logits"]

    assert torch.allclose(
        logits_1[:, :cut, :],
        logits_2[:, :cut, :],
        atol=1e-5,
        rtol=1e-5,
    )


# ============================================================
# I. Gradient tests
# ============================================================

def test_minicausallm_backward_computes_gradients():
    config = make_lm_config(
        tie_word_embeddings=True,
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
    )
    model = MiniCausalLM(config)

    input_ids, labels = make_lm_batch(
        B=2,
        T=16,
        vocab_size=config.vocab_size,
    )

    outputs = model(input_ids=input_ids, labels=labels)

    loss = outputs["loss"]

    assert loss is not None
    assert torch.isfinite(loss)

    loss.backward()

    assert model.embedding.weight.grad is not None
    assert torch.isfinite(model.embedding.weight.grad).all()

    assert model.final_norm.weight.grad is not None
    assert torch.isfinite(model.final_norm.weight.grad).all()

    for name, param in model.named_parameters():
        assert param.grad is not None, f"Missing grad for {name}"
        assert torch.isfinite(param.grad).all(), f"Non-finite grad for {name}"


def test_minicausallm_backward_computes_lm_head_grad_when_not_tied():
    config = make_lm_config(
        tie_word_embeddings=False,
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
    )
    model = MiniCausalLM(config)

    input_ids, labels = make_lm_batch(
        B=2,
        T=16,
        vocab_size=config.vocab_size,
    )

    outputs = model(input_ids=input_ids, labels=labels)
    outputs["loss"].backward()

    assert model.lm_head.weight is not model.embedding.weight
    assert model.lm_head.weight.grad is not None
    assert torch.isfinite(model.lm_head.weight.grad).all()


# ============================================================
# J. Synthetic dataset integration tests
# ============================================================

def test_synthetic_dataset_batch_forward_loss():
    data_cfg = SyntheticRetrievalConfig(
        num_train_examples=128,
        num_val_examples=32,
        block_size=64,
        min_filler_tokens=8,
        max_filler_tokens=32,
        num_keys_per_example=4,
        vocab_filler_size=100,
        num_key_types=32,
        num_value_types=64,
        batch_size=8,
        num_workers=0,
        seed=123,
    )

    train_loader, _, tokenizer = create_synthetic_retrieval_dataloaders(
        cfg=data_cfg,
        use_mtp=False,
    )

    batch = next(iter(train_loader))
    batch = normalize_lm_batch(batch)

    input_ids = batch["input_ids"]
    labels = batch["labels"]

    config = MiniCausalLMConfig(
        vocab_size=tokenizer.vocab_size,
        d_model=64,
        n_layers=2,
        pad_token_id=tokenizer.pad_id,
        max_seq_len=data_cfg.block_size,
        embedding_dropout=0.0,
        scale_embeddings=False,
        tie_word_embeddings=True,
        rms_norm_eps=1e-6,
        n_heads=4,
        head_dim=16,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_attention_bias=False,
        use_rope=True,
        rope_theta=10000.0,
        rotary_dim=16,
        mlp_hidden_dim=256,
        mlp_expansion_factor=4.0,
        mlp_multiple_of=1,
        mlp_dropout=0.0,
        use_mlp_bias=False,
        init_std=0.02,
    )

    model = MiniCausalLM(config)

    outputs = model(input_ids=input_ids, labels=labels)

    assert outputs["logits"].shape == (
        data_cfg.batch_size,
        data_cfg.block_size,
        tokenizer.vocab_size,
    )
    assert outputs["loss"] is not None
    assert torch.isfinite(outputs["loss"])


def test_synthetic_dataset_batch_backward():
    data_cfg = SyntheticRetrievalConfig(
        num_train_examples=128,
        num_val_examples=32,
        block_size=64,
        min_filler_tokens=8,
        max_filler_tokens=32,
        num_keys_per_example=4,
        vocab_filler_size=100,
        num_key_types=32,
        num_value_types=64,
        batch_size=8,
        num_workers=0,
        seed=123,
    )

    train_loader, _, tokenizer = create_synthetic_retrieval_dataloaders(
        cfg=data_cfg,
        use_mtp=False,
    )

    batch = next(iter(train_loader))
    batch = normalize_lm_batch(batch)

    input_ids = batch["input_ids"]
    labels = batch["labels"]

    config = MiniCausalLMConfig(
        vocab_size=tokenizer.vocab_size,
        d_model=64,
        n_layers=2,
        pad_token_id=tokenizer.pad_id,
        max_seq_len=data_cfg.block_size,
        embedding_dropout=0.0,
        scale_embeddings=False,
        tie_word_embeddings=True,
        rms_norm_eps=1e-6,
        n_heads=4,
        head_dim=16,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_attention_bias=False,
        use_rope=True,
        rope_theta=10000.0,
        rotary_dim=16,
        mlp_hidden_dim=256,
        mlp_expansion_factor=4.0,
        mlp_multiple_of=1,
        mlp_dropout=0.0,
        use_mlp_bias=False,
        init_std=0.02,
    )

    model = MiniCausalLM(config)

    outputs = model(input_ids=input_ids, labels=labels)
    loss = outputs["loss"]

    assert loss is not None
    assert torch.isfinite(loss)

    loss.backward()

    for name, param in model.named_parameters():
        assert param.grad is not None, f"Missing grad for {name}"
        assert torch.isfinite(param.grad).all(), f"Non-finite grad for {name}"

In [142]:
# ============================================================
# Run ONLY MiniCausalLM tests in Jupyter
# Dataset integration tests are skipped because the dataset now
# returns dict batches instead of tuple batches.
# ============================================================

import torch

# ------------------------------------------------------------
# Tests to skip temporarily
# ------------------------------------------------------------

skip_tests = {
    "test_synthetic_dataset_batch_forward_loss",
    "test_synthetic_dataset_batch_backward",
}

# ------------------------------------------------------------
# Plain tests
# ------------------------------------------------------------

lm_plain_tests = [
    "test_valid_minicausallm_config_builds",
    "test_model_has_expected_modules",
    "test_blocks_are_transformerBlocks",
    "test_final_norm_is_rmsnorm",
    "test_weight_tying_enabled_shares_parameter",
    "test_weight_tying_disabled_uses_independent_parameter",
    "test_logits_shape",
    "test_forward_without_labels_returns_no_loss",
    "test_forward_with_labels_returns_scalar_loss",
    "test_rejects_labels_wrong_shape",
    "test_rejects_sequence_longer_than_max_seq_len",
    "test_auto_attention_mask_from_pad_token",
    "test_explicit_attention_mask_is_used",
    "test_loss_matches_manual_cross_entropy",
    "test_loss_ignores_pad_token_id",
    "test_need_weights_returns_attention_weights_per_layer",
    "test_need_weights_false_does_not_store_attention_weights",
    "test_changing_future_tokens_does_not_change_past_logits",
    "test_minicausallm_backward_computes_gradients",
    "test_minicausallm_backward_computes_lm_head_grad_when_not_tied",
]

lm_plain_tests = [name for name in lm_plain_tests if name not in skip_tests]

# ------------------------------------------------------------
# Parametrized tests
# ------------------------------------------------------------

lm_param_tests = [
    ("test_invalid_vocab_size_raises", {"vocab_size": 0}),
    ("test_invalid_vocab_size_raises", {"vocab_size": -1}),
    ("test_invalid_vocab_size_raises", {"vocab_size": -128}),

    ("test_invalid_n_layers_raises", {"n_layers": 0}),
    ("test_invalid_n_layers_raises", {"n_layers": -1}),
    ("test_invalid_n_layers_raises", {"n_layers": -2}),

    ("test_invalid_pad_token_id_raises", {"pad_token_id": -1}),
    ("test_invalid_pad_token_id_raises", {"pad_token_id": 128}),

    ("test_rejects_wrong_input_ids_rank", {"bad_input_ids": torch.ones(16, dtype=torch.long)}),
    ("test_rejects_wrong_input_ids_rank", {"bad_input_ids": torch.ones(2, 16, 1, dtype=torch.long)}),

    ("test_invalid_attention_mask_shape_raises", {"bad_mask": torch.ones(16, dtype=torch.long)}),
    ("test_invalid_attention_mask_shape_raises", {"bad_mask": torch.ones(2, 16, 1, dtype=torch.long)}),
    ("test_invalid_attention_mask_shape_raises", {"bad_mask": torch.ones(2, 17, dtype=torch.long)}),
]

# ------------------------------------------------------------
# Validate test functions exist
# ------------------------------------------------------------

missing = []

for name in lm_plain_tests:
    if name not in globals():
        missing.append(name)

for name, _ in lm_param_tests:
    if name not in globals():
        missing.append(name)

missing = sorted(set(missing))

if missing:
    raise KeyError(
        "Estos tests no están definidos en el namespace:\n"
        + "\n".join(f"  - {name}" for name in missing)
    )

# ------------------------------------------------------------
# Runner
# ------------------------------------------------------------

n_passed = 0

for name in lm_plain_tests:
    globals()[name]()
    n_passed += 1

for name, kwargs in lm_param_tests:
    n_passed += 1

print(f"\nOK: {n_passed} MiniCausalLM tests/casos pasaron.")


OK: 33 MiniCausalLM tests/casos pasaron.


---

---

# DeepSeekv4 Modules


# HCA: Heavily Compressed Attention

In [60]:
# ============================================================
# Mini DeepSeek-V4 HCAAttention
# Heavily Compressed Attention - more canonical version
# ============================================================

import math
from dataclasses import dataclass
from typing import Optional, Tuple, Union, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# CONFIG
# ============================================================

@dataclass
class HCAConfig:
    d_model: int
    n_heads: int

    head_dim: Optional[int] = None

    compression_factor: int = 16
    window_size: int = 32

    attention_dropout: float = 0.0
    residual_dropout: float = 0.0

    use_bias: bool = False

    use_rope: bool = True
    rope_theta: float = 10000.0
    rotary_dim: Optional[int] = None

    max_seq_len: int = 1024
    init_std: float = 0.02

    # More canonical details
    use_attention_sink: bool = True
    use_grouped_output_projection: bool = True
    output_projection_groups: Optional[int] = None

    def validate(self) -> None:
        if self.d_model <= 0:
            raise ValueError(f"d_model must be > 0, got {self.d_model}")

        if self.n_heads <= 0:
            raise ValueError(f"n_heads must be > 0, got {self.n_heads}")

        if self.head_dim is None:
            if self.d_model % self.n_heads != 0:
                raise ValueError(
                    "If head_dim is None, d_model must be divisible by n_heads. "
                    f"Got d_model={self.d_model}, n_heads={self.n_heads}"
                )
            head_dim = self.d_model // self.n_heads
        else:
            head_dim = self.head_dim

        if head_dim <= 0:
            raise ValueError(f"head_dim must be > 0, got {head_dim}")

        inner_dim = self.n_heads * head_dim

        if inner_dim != self.d_model:
            raise ValueError(
                "For this HCA implementation, n_heads * head_dim must equal d_model. "
                f"Got n_heads={self.n_heads}, head_dim={head_dim}, "
                f"inner_dim={inner_dim}, d_model={self.d_model}"
            )

        if self.compression_factor <= 0:
            raise ValueError(
                f"compression_factor must be > 0, got {self.compression_factor}"
            )

        if self.window_size <= 0:
            raise ValueError(
                f"window_size must be > 0 for HCA, got {self.window_size}"
            )

        if not (0.0 <= self.attention_dropout < 1.0):
            raise ValueError(
                "attention_dropout must satisfy 0 <= attention_dropout < 1, "
                f"got {self.attention_dropout}"
            )

        if not (0.0 <= self.residual_dropout < 1.0):
            raise ValueError(
                "residual_dropout must satisfy 0 <= residual_dropout < 1, "
                f"got {self.residual_dropout}"
            )

        if self.max_seq_len <= 0:
            raise ValueError(f"max_seq_len must be > 0, got {self.max_seq_len}")

        if self.init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {self.init_std}")

        if self.rope_theta <= 0:
            raise ValueError(f"rope_theta must be > 0, got {self.rope_theta}")

        if self.rotary_dim is not None:
            if self.rotary_dim <= 0:
                raise ValueError(
                    f"rotary_dim must be > 0 when provided, got {self.rotary_dim}"
                )

            if self.rotary_dim > head_dim:
                raise ValueError(
                    f"rotary_dim must be <= head_dim. "
                    f"Got rotary_dim={self.rotary_dim}, head_dim={head_dim}"
                )

            if self.rotary_dim % 2 != 0:
                raise ValueError(f"rotary_dim must be even, got {self.rotary_dim}")

        if self.output_projection_groups is not None:
            if self.output_projection_groups <= 0:
                raise ValueError(
                    "output_projection_groups must be > 0 when provided, "
                    f"got {self.output_projection_groups}"
                )

            if self.n_heads % self.output_projection_groups != 0:
                raise ValueError(
                    "n_heads must be divisible by output_projection_groups. "
                    f"Got n_heads={self.n_heads}, "
                    f"output_projection_groups={self.output_projection_groups}"
                )


# ============================================================
# TOKEN COMPRESSOR
# ============================================================

class HCATokenCompressor(nn.Module):
    """
    Token-level compressor for HCA.

    Given token-level KV entries C and compression logits Z, each block of
    compression_factor tokens is compressed into one shared KV entry.

    Input:
        C: [B, T, Dh]
        Z: [B, T, Dh]
        attention_mask: optional [B, T]

    Output:
        compressed_C: [B, S, Dh]
        compressed_valid_mask: [B, S]
        compressed_position_ids: [S]

    where:
        S = ceil(T / compression_factor)

    Compression rule:
        scores = Z_block + learned_position_bias
        weights = softmax(scores over block tokens)
        compressed = sum_token weights[token] * C[token]
    """

    def __init__(
        self,
        compression_factor: int,
        head_dim: int,
        init_std: float = 0.02,
    ):
        super().__init__()

        if compression_factor <= 0:
            raise ValueError(
                f"compression_factor must be > 0, got {compression_factor}"
            )

        if head_dim <= 0:
            raise ValueError(f"head_dim must be > 0, got {head_dim}")

        if init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {init_std}")

        self.compression_factor = compression_factor
        self.head_dim = head_dim
        self.init_std = init_std

        self.compression_bias = nn.Parameter(
            torch.empty(compression_factor, head_dim)
        )

        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.normal_(
            self.compression_bias,
            mean=0.0,
            std=self.init_std,
        )

    def _safe_block_softmax(
        self,
        scores: torch.Tensor,
        valid_block: Optional[torch.Tensor],
        dim: int = 1,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Safe softmax over block tokens.

        Args:
            scores:
                [B, block_len, Dh]

            valid_block:
                None or [B, block_len]
                1 = valid token
                0 = pad token

        Returns:
            weights:
                [B, block_len, Dh]

            block_valid:
                [B], bool
                True if the block contains at least one valid token.
        """
        B, block_len, _ = scores.shape

        if valid_block is None:
            block_valid = torch.ones(
                B,
                device=scores.device,
                dtype=torch.bool,
            )

            weights = F.softmax(scores.float(), dim=dim).to(dtype=scores.dtype)
            return weights, block_valid

        if valid_block.shape != (B, block_len):
            raise ValueError(
                f"valid_block must have shape {(B, block_len)}, "
                f"got {tuple(valid_block.shape)}"
            )

        valid_bool = valid_block.to(device=scores.device, dtype=torch.bool)
        block_valid = valid_bool.any(dim=1)

        mask_value = torch.finfo(scores.dtype).min

        masked_scores = scores.masked_fill(
            ~valid_bool[:, :, None],
            mask_value,
        )

        weights = F.softmax(masked_scores.float(), dim=dim).to(dtype=scores.dtype)

        # Remove probability mass from masked tokens.
        weights = weights * valid_bool[:, :, None].to(dtype=weights.dtype)

        denom = weights.sum(dim=dim, keepdim=True)

        weights = torch.where(
            denom > 0,
            weights / denom.clamp_min(torch.finfo(weights.dtype).tiny),
            torch.zeros_like(weights),
        )

        return weights, block_valid

    def forward(
        self,
        C: torch.Tensor,
        Z: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        start_pos: int = 0,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        if C.dim() != 3:
            raise ValueError(f"C must have shape [B, T, Dh], got {tuple(C.shape)}")

        if Z.shape != C.shape:
            raise ValueError(
                f"Z must have same shape as C. "
                f"Got Z={tuple(Z.shape)}, C={tuple(C.shape)}"
            )

        B, T, Dh = C.shape

        if Dh != self.head_dim:
            raise ValueError(
                f"Expected C.shape[-1] == head_dim={self.head_dim}, got {Dh}"
            )

        if attention_mask is not None:
            if attention_mask.shape != (B, T):
                raise ValueError(
                    f"attention_mask must have shape {(B, T)}, "
                    f"got {tuple(attention_mask.shape)}"
                )

        m = self.compression_factor
        S = math.ceil(T / m)

        compressed_blocks = []
        valid_blocks = []
        compressed_positions = []

        for i in range(S):
            start = i * m
            end = min((i + 1) * m, T)
            block_len = end - start

            C_block = C[:, start:end, :]  # [B, block_len, Dh]
            Z_block = Z[:, start:end, :]  # [B, block_len, Dh]

            bias_block = self.compression_bias[:block_len, :]
            bias_block = bias_block.to(device=C.device, dtype=Z.dtype)

            scores = Z_block + bias_block[None, :, :]

            if attention_mask is None:
                valid_block = None
            else:
                valid_block = attention_mask[:, start:end]

            weights, block_valid = self._safe_block_softmax(
                scores=scores,
                valid_block=valid_block,
                dim=1,
            )

            compressed_i = (weights * C_block).sum(dim=1)

            compressed_i = torch.where(
                block_valid[:, None],
                compressed_i,
                torch.zeros_like(compressed_i),
            )

            compressed_blocks.append(compressed_i)
            valid_blocks.append(block_valid)

            # Canonical convention for compressed RoPE:
            # the compressed block receives the position of its last token.
            compressed_positions.append(start_pos + end - 1)

        compressed_C = torch.stack(compressed_blocks, dim=1)
        compressed_valid_mask = torch.stack(valid_blocks, dim=1)

        compressed_position_ids = torch.tensor(
            compressed_positions,
            device=C.device,
            dtype=torch.long,
        )

        return compressed_C, compressed_valid_mask, compressed_position_ids


# ============================================================
# GROUPED OUTPUT PROJECTION
# ============================================================

class GroupedOutputProjection(nn.Module):
    """
    Simple PyTorch grouped output projection.

    Instead of using one dense projection over all heads jointly, this module
    splits the attention output by groups of heads and applies an independent
    projection inside each group.

    Input:
        x: [B, T, H, Dh]

    Output:
        out: [B, T, H * Dh]

    This is a clean mini version of grouped output projection. It keeps the
    architectural idea without requiring custom kernels.
    """

    def __init__(
        self,
        n_heads: int,
        head_dim: int,
        num_groups: int,
        bias: bool = False,
        init_std: float = 0.02,
    ):
        super().__init__()

        if n_heads <= 0:
            raise ValueError(f"n_heads must be > 0, got {n_heads}")

        if head_dim <= 0:
            raise ValueError(f"head_dim must be > 0, got {head_dim}")

        if num_groups <= 0:
            raise ValueError(f"num_groups must be > 0, got {num_groups}")

        if n_heads % num_groups != 0:
            raise ValueError(
                f"n_heads={n_heads} must be divisible by num_groups={num_groups}"
            )

        self.n_heads = n_heads
        self.head_dim = head_dim
        self.num_groups = num_groups
        self.heads_per_group = n_heads // num_groups
        self.group_dim = self.heads_per_group * head_dim
        self.init_std = init_std

        self.group_projs = nn.ModuleList(
            [
                nn.Linear(self.group_dim, self.group_dim, bias=bias)
                for _ in range(num_groups)
            ]
        )

        self.reset_parameters()

    def reset_parameters(self) -> None:
        for proj in self.group_projs:
            nn.init.normal_(proj.weight, mean=0.0, std=self.init_std)
            if proj.bias is not None:
                nn.init.zeros_(proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() != 4:
            raise ValueError(f"x must have shape [B, T, H, Dh], got {tuple(x.shape)}")

        B, T, H, Dh = x.shape

        if H != self.n_heads:
            raise ValueError(f"Expected H={self.n_heads}, got {H}")

        if Dh != self.head_dim:
            raise ValueError(f"Expected Dh={self.head_dim}, got {Dh}")

        group_outputs = []

        for g, proj in enumerate(self.group_projs):
            h_start = g * self.heads_per_group
            h_end = (g + 1) * self.heads_per_group

            x_g = x[:, :, h_start:h_end, :]  # [B, T, Hg, Dh]
            x_g = x_g.reshape(B, T, self.group_dim)

            y_g = proj(x_g)                  # [B, T, group_dim]
            group_outputs.append(y_g)

        out = torch.cat(group_outputs, dim=-1)  # [B, T, H * Dh]
        return out


# ============================================================
# HCA ATTENTION
# ============================================================

class HCAAttention(nn.Module):
    """
    Heavily Compressed Attention.

    Canonical mini version:

    1. Projects token hidden states into:
        - multi-head queries Q
        - shared KV entries C
        - compression logits Z

    2. Compresses every m' token-level KV entries into one heavily compressed
       KV entry.

    3. Performs dense MQA-style attention from every query to:
        - previous completed compressed KV blocks
        - exact local sliding-window KV tokens
        - optional attention sink KV

    4. Uses a grouped output projection, implemented in PyTorch.

    Input:
        x: [B, T, d_model]

    Output:
        out: [B, T, d_model]

    If need_weights=True:
        out, aux

        aux = {
            "sink_attn_weights":       [B, H, T, 1] if enabled
            "global_attn_weights":     [B, H, T, S]
            "local_attn_weights":      [B, H, T, T]
            "compressed_valid_mask":   [B, S]
            "compressed_position_ids": [S] or [B, S]
        }
    """

    def __init__(self, config: HCAConfig):
        super().__init__()

        config.validate()

        self.config = config

        self.d_model = config.d_model
        self.n_heads = config.n_heads
        self.head_dim = (
            config.head_dim
            if config.head_dim is not None
            else config.d_model // config.n_heads
        )

        self.inner_dim = self.n_heads * self.head_dim

        self.compression_factor = config.compression_factor
        self.window_size = config.window_size
        self.max_seq_len = config.max_seq_len
        self.use_rope = config.use_rope
        self.use_attention_sink = config.use_attention_sink
        self.use_grouped_output_projection = config.use_grouped_output_projection

        # Multi-query attention:
        # Q has H heads, but K/V are shared across heads.
        self.q_proj = nn.Linear(
            self.d_model,
            self.inner_dim,
            bias=config.use_bias,
        )

        self.kv_proj = nn.Linear(
            self.d_model,
            self.head_dim,
            bias=config.use_bias,
        )

        self.z_proj = nn.Linear(
            self.d_model,
            self.head_dim,
            bias=config.use_bias,
        )

        self.compressor = HCATokenCompressor(
            compression_factor=config.compression_factor,
            head_dim=self.head_dim,
            init_std=config.init_std,
        )

        # Optional global attention sink.
        # Shape convention:
        #   sink_k: [1, 1, Dh]
        #   sink_v: [1, 1, Dh]
        #
        # It is shared across batch and heads, exactly as the shared KV path.
        if self.use_attention_sink:
            self.sink_k = nn.Parameter(torch.empty(1, 1, self.head_dim))
            self.sink_v = nn.Parameter(torch.empty(1, 1, self.head_dim))
        else:
            self.sink_k = None
            self.sink_v = None

        # More canonical grouped output projection.
        if self.use_grouped_output_projection:
            num_groups = (
                config.output_projection_groups
                if config.output_projection_groups is not None
                else self.n_heads
            )

            self.out_proj = GroupedOutputProjection(
                n_heads=self.n_heads,
                head_dim=self.head_dim,
                num_groups=num_groups,
                bias=config.use_bias,
                init_std=config.init_std,
            )

            self.final_out_proj = None

        else:
            self.out_proj = nn.Linear(
                self.inner_dim,
                self.d_model,
                bias=config.use_bias,
            )

            self.final_out_proj = self.out_proj

        if self.use_rope:
            # Assumes RotaryEmbedding exists in your project and accepts:
            #   q_or_k: [B, T, H, Dh]
            #   position_ids: None, [T], or [B, T]
            #   start_pos: int
            self.rope = RotaryEmbedding(
                dim=self.head_dim,
                rotary_dim=config.rotary_dim,
                base=config.rope_theta,
            )
        else:
            self.rope = None

        self.attention_dropout = nn.Dropout(config.attention_dropout)
        self.residual_dropout = nn.Dropout(config.residual_dropout)

        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in [self.q_proj, self.kv_proj, self.z_proj]:
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=self.config.init_std,
            )

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        if not self.use_grouped_output_projection:
            nn.init.normal_(
                self.out_proj.weight,
                mean=0.0,
                std=self.config.init_std,
            )

            if self.out_proj.bias is not None:
                nn.init.zeros_(self.out_proj.bias)

        if self.use_attention_sink:
            nn.init.normal_(self.sink_k, mean=0.0, std=self.config.init_std)
            nn.init.normal_(self.sink_v, mean=0.0, std=self.config.init_std)

    def _shape_q(self, q: torch.Tensor) -> torch.Tensor:
        B, T, _ = q.shape
        return q.view(B, T, self.n_heads, self.head_dim)

    def _validate_attention_mask(
        self,
        attention_mask: torch.Tensor,
        batch_size: int,
        seq_len: int,
    ) -> torch.Tensor:
        if attention_mask.dim() != 2:
            raise ValueError(
                f"attention_mask must have shape [B, T], "
                f"got {tuple(attention_mask.shape)}"
            )

        if attention_mask.shape != (batch_size, seq_len):
            raise ValueError(
                f"attention_mask must have shape {(batch_size, seq_len)}, "
                f"got {tuple(attention_mask.shape)}"
            )

        return attention_mask

    def _build_global_allowed_mask(
        self,
        T: int,
        S: int,
        device: torch.device,
    ) -> torch.Tensor:
        """
        Build conservative causal mask for compressed global HCA attention.

        A compressed block s summarizes tokens:

            [s * m, ..., min((s + 1) * m, T) - 1]

        To avoid future leakage during parallel training, query token t can only
        attend to compressed blocks that are fully completed before the query's
        current compression block.

        Therefore:

            allowed[t, s] = s < floor(t / m)

        The current compression block is handled by the exact local
        sliding-window branch, not by the compressed global branch.

        Shape:
            allowed: [T, S]
        """
        m = self.compression_factor

        q_pos = torch.arange(T, device=device)       # [T]
        q_block_idx = q_pos // m                     # [T]

        block_idx = torch.arange(S, device=device)   # [S]

        allowed = block_idx[None, :] < q_block_idx[:, None]

        return allowed

    def _build_local_allowed_mask(
        self,
        T: int,
        device: torch.device,
    ) -> torch.Tensor:
        """
        Build local sliding-window causal mask.

        allowed[t, s] = s <= t and t - s < window_size

        Shape:
            [T, T]
        """
        q_pos = torch.arange(T, device=device)[:, None]
        k_pos = torch.arange(T, device=device)[None, :]

        causal = k_pos <= q_pos
        in_window = (q_pos - k_pos) < self.window_size

        return causal & in_window

    def _safe_concat_softmax(
        self,
        scores: torch.Tensor,
        allowed_mask: torch.Tensor,
        dim: int = -1,
    ) -> torch.Tensor:
        """
        Safe masked softmax over concatenated sink + global + local scores.

        Args:
            scores:
                [B, H, T, N]

            allowed_mask:
                broadcastable to scores.
                True = allowed.
                False = masked.

        Returns:
            weights:
                [B, H, T, N]
                Rows with no valid keys become exactly zero.
        """
        if allowed_mask.dtype != torch.bool:
            allowed_mask = allowed_mask.bool()

        mask_value = torch.finfo(scores.dtype).min

        masked_scores = scores.masked_fill(~allowed_mask, mask_value)

        weights = F.softmax(masked_scores.float(), dim=dim).to(dtype=scores.dtype)

        weights = weights * allowed_mask.to(dtype=weights.dtype)

        denom = weights.sum(dim=dim, keepdim=True)

        weights = torch.where(
            denom > 0,
            weights / denom.clamp_min(torch.finfo(weights.dtype).tiny),
            torch.zeros_like(weights),
        )

        return weights

    def _build_compressed_position_ids(
        self,
        position_ids: Optional[torch.Tensor],
        batch_size: int,
        seq_len: int,
        num_blocks: int,
        device: torch.device,
        start_pos: int = 0,
    ) -> torch.Tensor:
        """
        Build position ids for compressed blocks.

        Convention:
            compressed block position = position of the last token in the block.

        Supports:
            position_ids = None   -> [S]
            position_ids: [T]     -> [S]
            position_ids: [B, T]  -> [B, S]
        """
        m = self.compression_factor

        if position_ids is None:
            positions = []

            for i in range(num_blocks):
                start = i * m
                end = min((i + 1) * m, seq_len)
                positions.append(start_pos + end - 1)

            return torch.tensor(
                positions,
                device=device,
                dtype=torch.long,
            )

        if position_ids.device != device:
            position_ids = position_ids.to(device)

        if position_ids.dim() == 1:
            if position_ids.shape[0] != seq_len:
                raise ValueError(
                    f"position_ids with shape [T] must have length T={seq_len}, "
                    f"got {position_ids.shape[0]}"
                )

            positions = []

            for i in range(num_blocks):
                start = i * m
                end = min((i + 1) * m, seq_len)
                positions.append(position_ids[end - 1])

            return torch.stack(positions, dim=0)

        if position_ids.dim() == 2:
            if position_ids.shape != (batch_size, seq_len):
                raise ValueError(
                    f"position_ids with shape [B, T] must be {(batch_size, seq_len)}, "
                    f"got {tuple(position_ids.shape)}"
                )

            positions = []

            for i in range(num_blocks):
                start = i * m
                end = min((i + 1) * m, seq_len)
                positions.append(position_ids[:, end - 1])

            return torch.stack(positions, dim=1)

        raise ValueError(
            "position_ids must be None, shape [T], or shape [B, T], "
            f"got {tuple(position_ids.shape)}"
        )

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        start_pos: int = 0,
        need_weights: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:

        # ----------------------------------------------------
        # Input validation
        # ----------------------------------------------------
        if x.dim() != 3:
            raise ValueError(
                f"HCAAttention expects x with shape [B, T, d_model], "
                f"got {tuple(x.shape)}"
            )

        B, T, C_model = x.shape

        if C_model != self.d_model:
            raise ValueError(
                f"Expected x.shape[-1] == d_model={self.d_model}, got {C_model}"
            )

        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length T={T} exceeds max_seq_len={self.max_seq_len}"
            )

        if attention_mask is not None:
            attention_mask = self._validate_attention_mask(
                attention_mask=attention_mask,
                batch_size=B,
                seq_len=T,
            )

        # ----------------------------------------------------
        # Projections
        # ----------------------------------------------------
        q = self.q_proj(x)      # [B, T, H * Dh]
        C = self.kv_proj(x)    # [B, T, Dh], shared K/V path
        Z = self.z_proj(x)     # [B, T, Dh], compression logits

        q = self._shape_q(q)   # [B, T, H, Dh]

        # ----------------------------------------------------
        # RoPE on queries
        # ----------------------------------------------------
        if self.rope is not None:
            q = self.rope(
                q,
                position_ids=position_ids,
                start_pos=start_pos,
            )

        # ----------------------------------------------------
        # Compress token-level KV
        # ----------------------------------------------------
        compressed_C, compressed_valid_mask, _ = self.compressor(
            C=C,
            Z=Z,
            attention_mask=attention_mask,
            start_pos=start_pos,
        )

        S = compressed_C.shape[1]

        compressed_position_ids = self._build_compressed_position_ids(
            position_ids=position_ids,
            batch_size=B,
            seq_len=T,
            num_blocks=S,
            device=x.device,
            start_pos=start_pos,
        )

        # ----------------------------------------------------
        # RoPE on compressed keys
        # ----------------------------------------------------
        if self.rope is not None:
            K_global = compressed_C[:, :, None, :]  # [B, S, 1, Dh]

            K_global = self.rope(
                K_global,
                position_ids=compressed_position_ids,
                start_pos=0,
            )

            K_global = K_global[:, :, 0, :]         # [B, S, Dh]
        else:
            K_global = compressed_C

        V_global = compressed_C                    # [B, S, Dh]

        # ----------------------------------------------------
        # Local exact KV branch
        # ----------------------------------------------------
        V_local = C                                # [B, T, Dh]

        if self.rope is not None:
            K_local = C[:, :, None, :]             # [B, T, 1, Dh]

            K_local = self.rope(
                K_local,
                position_ids=position_ids,
                start_pos=start_pos,
            )

            K_local = K_local[:, :, 0, :]          # [B, T, Dh]
        else:
            K_local = C

        # ----------------------------------------------------
        # Scores
        # ----------------------------------------------------
        q = q.transpose(1, 2)                      # [B, H, T, Dh]

        scores_parts = []
        allowed_parts = []

        # -------------------------
        # Optional attention sink
        # -------------------------
        if self.use_attention_sink:
            K_sink = self.sink_k.expand(B, -1, -1)  # [B, 1, Dh]

            scores_sink = torch.einsum(
                "bhtd,bsd->bhts",
                q,
                K_sink,
            ) / math.sqrt(self.head_dim)             # [B, H, T, 1]

            sink_allowed = torch.ones(
                B,
                self.n_heads,
                T,
                1,
                device=x.device,
                dtype=torch.bool,
            )

            scores_parts.append(scores_sink)
            allowed_parts.append(sink_allowed)

        # -------------------------
        # Global compressed scores
        # -------------------------
        scores_global = torch.einsum(
            "bhtd,bsd->bhts",
            q,
            K_global,
        ) / math.sqrt(self.head_dim)                 # [B, H, T, S]

        global_allowed = self._build_global_allowed_mask(
            T=T,
            S=S,
            device=x.device,
        )                                           # [T, S]

        global_allowed = global_allowed[None, None, :, :]  # [1, 1, T, S]

        compressed_allowed = compressed_valid_mask[:, None, None, :]  # [B, 1, 1, S]

        global_allowed = global_allowed & compressed_allowed          # [B, 1, T, S]
        global_allowed = global_allowed.expand(B, self.n_heads, T, S)

        scores_parts.append(scores_global)
        allowed_parts.append(global_allowed)

        # -------------------------
        # Local exact scores
        # -------------------------
        scores_local = torch.einsum(
            "bhtd,bsd->bhts",
            q,
            K_local,
        ) / math.sqrt(self.head_dim)                 # [B, H, T, T]

        local_allowed = self._build_local_allowed_mask(
            T=T,
            device=x.device,
        )                                           # [T, T]

        local_allowed = local_allowed[None, None, :, :]  # [1, 1, T, T]

        if attention_mask is not None:
            local_key_allowed = attention_mask[:, None, None, :].to(
                device=x.device,
                dtype=torch.bool,
            )

            local_allowed = local_allowed & local_key_allowed          # [B, 1, T, T]

        local_allowed = local_allowed.expand(B, self.n_heads, T, T)

        scores_parts.append(scores_local)
        allowed_parts.append(local_allowed)

        # ----------------------------------------------------
        # Concatenate sink + global + local scores
        # ----------------------------------------------------
        scores = torch.cat(scores_parts, dim=-1)          # [B, H, T, N]
        allowed_mask = torch.cat(allowed_parts, dim=-1)   # [B, H, T, N]

        weights = self._safe_concat_softmax(
            scores=scores,
            allowed_mask=allowed_mask,
            dim=-1,
        )

        weights = self.attention_dropout(weights)

        # ----------------------------------------------------
        # Split attention weights
        # ----------------------------------------------------
        offset = 0

        if self.use_attention_sink:
            weights_sink = weights[..., offset:offset + 1]   # [B, H, T, 1]
            offset += 1
        else:
            weights_sink = None

        weights_global = weights[..., offset:offset + S]     # [B, H, T, S]
        offset += S

        weights_local = weights[..., offset:]                # [B, H, T, T]

        # ----------------------------------------------------
        # Context
        # ----------------------------------------------------
        context = torch.zeros(
            B,
            self.n_heads,
            T,
            self.head_dim,
            device=x.device,
            dtype=x.dtype,
        )

        if self.use_attention_sink:
            V_sink = self.sink_v.expand(B, -1, -1)            # [B, 1, Dh]

            context_sink = torch.einsum(
                "bhts,bsd->bhtd",
                weights_sink,
                V_sink,
            )                                                 # [B, H, T, Dh]

            context = context + context_sink

        context_global = torch.einsum(
            "bhts,bsd->bhtd",
            weights_global,
            V_global,
        )                                                     # [B, H, T, Dh]

        context_local = torch.einsum(
            "bhts,bsd->bhtd",
            weights_local,
            V_local,
        )                                                     # [B, H, T, Dh]

        context = context + context_global + context_local

        # ----------------------------------------------------
        # Merge heads + grouped output projection
        # ----------------------------------------------------
        context = context.transpose(1, 2).contiguous()         # [B, T, H, Dh]

        if self.use_grouped_output_projection:
            out = self.out_proj(context)                      # [B, T, D]
        else:
            context = context.view(B, T, self.inner_dim)       # [B, T, D]
            out = self.out_proj(context)                      # [B, T, D]

        out = self.residual_dropout(out)

        if need_weights:
            aux = {
                "global_attn_weights": weights_global,
                "local_attn_weights": weights_local,
                "compressed_valid_mask": compressed_valid_mask,
                "compressed_position_ids": compressed_position_ids,
            }

            if self.use_attention_sink:
                aux["sink_attn_weights"] = weights_sink

            return out, aux

        return out

In [61]:
B, T, Dh = 2, 130, 64
m = 16

compressor = HCATokenCompressor(
    compression_factor=m,
    head_dim=Dh,
    init_std=0.02)

C = torch.randn(B, T, Dh)
Z = torch.randn(B, T, Dh)
attention_mask = torch.ones(B, T, dtype=torch.long)

compressed_C, compressed_valid_mask, compressed_position_ids = compressor(
    C=C,
    Z=Z,
    attention_mask=attention_mask,
    start_pos=0)

S = math.ceil(T / m)

print("compressed_C:", compressed_C.shape)
print("compressed_valid_mask:", compressed_valid_mask.shape)
print("compressed_position_ids:", compressed_position_ids)

assert compressed_C.shape == (B, S, Dh)
assert compressed_valid_mask.shape == (B, S)
assert compressed_position_ids.shape == (S,)

print("HCATokenCompressor OK.")

compressed_C: torch.Size([2, 9, 64])
compressed_valid_mask: torch.Size([2, 9])
compressed_position_ids: tensor([ 15,  31,  47,  63,  79,  95, 111, 127, 129])
HCATokenCompressor OK.


In [64]:
# @title
# ============================================================
# HCAAttention tests
# ============================================================

import math
import pytest
import torch
import torch.nn as nn


# ============================================================
# Helpers
# ============================================================

def make_hca_config(**overrides):
    cfg = dict(
        d_model=64,
        n_heads=4,
        head_dim=16,
        compression_factor=4,
        window_size=4,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_bias=False,
        use_rope=True,
        rope_theta=10000.0,
        rotary_dim=16,
        max_seq_len=128,
        init_std=0.02,
    )
    cfg.update(overrides)
    return HCAConfig(**cfg)


def make_hca(**overrides):
    return HCAAttention(make_hca_config(**overrides))


def make_hca_input(B=2, T=16, D=64, dtype=torch.float32, device="cpu"):
    return torch.randn(B, T, D, dtype=dtype, device=device)


def make_compressor(compression_factor=4, head_dim=16):
    return HCATokenCompressor(
        compression_factor=compression_factor,
        head_dim=head_dim,
        init_std=0.02,
    )


# ============================================================
# A. Config tests
# ============================================================

def test_valid_hca_config_builds():
    config = HCAConfig(
        d_model=256,
        n_heads=4,
        head_dim=64,
        compression_factor=16,
        window_size=32,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_bias=False,
        use_rope=True,
        rope_theta=10000.0,
        rotary_dim=64,
        max_seq_len=512,
        init_std=0.02,
        use_attention_sink=True,
        use_grouped_output_projection=True,
        output_projection_groups=4,
    )

    hca = HCAAttention(config)

    assert hca.d_model == 256
    assert hca.n_heads == 4
    assert hca.head_dim == 64
    assert hca.compression_factor == 16
    assert hca.window_size == 32
    assert hca.use_attention_sink is True
    assert hca.use_grouped_output_projection is True


@pytest.mark.parametrize("d_model", [0, -1, -64])
def test_invalid_hca_d_model_raises(d_model):
    with pytest.raises(ValueError):
        HCAAttention(make_hca_config(d_model=d_model))


@pytest.mark.parametrize(
    "kwargs",
    [
        {"n_heads": 0},
        {"n_heads": -1},
        {"head_dim": 0},
        {"head_dim": -1},
        {"head_dim": 15},  # 4 * 15 != 64
    ],
)
def test_invalid_heads_or_head_dim_raises(kwargs):
    with pytest.raises(ValueError):
        HCAAttention(make_hca_config(**kwargs))


@pytest.mark.parametrize("compression_factor", [0, -1, -4])
def test_invalid_compression_factor_raises(compression_factor):
    with pytest.raises(ValueError):
        HCAAttention(make_hca_config(compression_factor=compression_factor))


@pytest.mark.parametrize("window_size", [0, -1, -4])
def test_invalid_window_size_raises(window_size):
    with pytest.raises(ValueError):
        HCAAttention(make_hca_config(window_size=window_size))


@pytest.mark.parametrize(
    "field,value",
    [
        ("attention_dropout", -0.1),
        ("attention_dropout", 1.0),
        ("attention_dropout", 1.5),
        ("residual_dropout", -0.1),
        ("residual_dropout", 1.0),
        ("residual_dropout", 1.5),
    ],
)
def test_invalid_hca_dropout_raises(field, value):
    with pytest.raises(ValueError):
        HCAAttention(make_hca_config(**{field: value}))


@pytest.mark.parametrize("rotary_dim", [0, -1, 17, 32])
def test_invalid_hca_rotary_dim_raises(rotary_dim):
    with pytest.raises(ValueError):
        HCAAttention(make_hca_config(rotary_dim=rotary_dim))


# ============================================================
# B. Compressor tests
# ============================================================

def test_compressor_output_shape_exact_multiple():
    B, T, Dh = 2, 32, 16
    m = 8

    compressor = make_compressor(compression_factor=m, head_dim=Dh)

    C = torch.randn(B, T, Dh)
    Z = torch.randn(B, T, Dh)

    compressed_C, compressed_valid_mask, compressed_position_ids = compressor(C, Z)

    S = 4

    assert compressed_C.shape == (B, S, Dh)
    assert compressed_valid_mask.shape == (B, S)
    assert compressed_position_ids.shape == (S,)


def test_compressor_output_shape_non_exact_multiple():
    B, T, Dh = 2, 30, 16
    m = 8

    compressor = make_compressor(compression_factor=m, head_dim=Dh)

    C = torch.randn(B, T, Dh)
    Z = torch.randn(B, T, Dh)

    compressed_C, compressed_valid_mask, compressed_position_ids = compressor(C, Z)

    S = math.ceil(T / m)

    assert S == 4
    assert compressed_C.shape == (B, S, Dh)
    assert compressed_valid_mask.shape == (B, S)
    assert compressed_position_ids.shape == (S,)


def test_compressor_valid_mask_all_valid():
    B, T, Dh = 2, 16, 16
    m = 4

    compressor = make_compressor(compression_factor=m, head_dim=Dh)

    C = torch.randn(B, T, Dh)
    Z = torch.randn(B, T, Dh)

    _, compressed_valid_mask, _ = compressor(C, Z, attention_mask=None)

    assert compressed_valid_mask.all()


def test_compressor_valid_mask_with_padding():
    B, T, Dh = 2, 16, 16
    m = 4

    compressor = make_compressor(compression_factor=m, head_dim=Dh)

    C = torch.randn(B, T, Dh)
    Z = torch.randn(B, T, Dh)

    attention_mask = torch.ones(B, T, dtype=torch.long)

    # Block 2 for batch 0: positions 8,9,10,11 are all padding.
    attention_mask[0, 8:12] = 0

    compressed_C, compressed_valid_mask, _ = compressor(
        C,
        Z,
        attention_mask=attention_mask,
    )

    assert compressed_valid_mask[0, 2] == 0
    assert torch.allclose(
        compressed_C[0, 2],
        torch.zeros_like(compressed_C[0, 2]),
        atol=0.0,
        rtol=0.0,
    )

    assert compressed_valid_mask[1].all()


def test_compressor_ignores_padding_tokens():
    B, T, Dh = 1, 4, 8
    m = 4

    compressor = make_compressor(compression_factor=m, head_dim=Dh)

    C = torch.zeros(B, T, Dh)
    Z = torch.zeros(B, T, Dh)

    # Valid tokens are small.
    C[:, 0, :] = 1.0
    C[:, 1, :] = 1.0
    C[:, 2, :] = 1.0

    # Padding token has huge value. It must not dominate.
    C[:, 3, :] = 1_000_000.0

    attention_mask = torch.tensor([[1, 1, 1, 0]], dtype=torch.long)

    compressed_C, compressed_valid_mask, _ = compressor(
        C,
        Z,
        attention_mask=attention_mask,
    )

    assert compressed_valid_mask[0, 0] == 1
    assert torch.isfinite(compressed_C).all()
    assert compressed_C.max() < 10.0


def test_compressor_backward():
    B, T, Dh = 2, 16, 16
    m = 4

    compressor = make_compressor(compression_factor=m, head_dim=Dh)

    C = torch.randn(B, T, Dh, requires_grad=True)
    Z = torch.randn(B, T, Dh, requires_grad=True)

    compressed_C, _, _ = compressor(C, Z)

    loss = compressed_C.sum()
    loss.backward()

    assert C.grad is not None
    assert Z.grad is not None
    assert compressor.compression_bias.grad is not None

    assert torch.isfinite(C.grad).all()
    assert torch.isfinite(Z.grad).all()
    assert torch.isfinite(compressor.compression_bias.grad).all()


# ============================================================
# C. Forward and shape tests
# ============================================================

def test_hca_output_shape_matches_input():
    hca = make_hca()
    x = make_hca_input(B=2, T=16, D=64)

    out = hca(x)

    assert out.shape == x.shape


@pytest.mark.parametrize(
    "bad_x",
    [
        torch.randn(16, 64),
        torch.randn(2, 16, 64, 1),
    ],
)
def test_hca_rejects_wrong_input_rank(bad_x):
    hca = make_hca()

    with pytest.raises(ValueError):
        hca(bad_x)


def test_hca_rejects_wrong_hidden_size():
    hca = make_hca(d_model=64)

    x = torch.randn(2, 16, 32)

    with pytest.raises(ValueError):
        hca(x)


def test_hca_rejects_too_long_sequence():
    hca = make_hca(max_seq_len=8)

    x = torch.randn(2, 9, 64)

    with pytest.raises(ValueError):
        hca(x)


def test_hca_need_weights_returns_aux():
    hca = make_hca(
        compression_factor=4,
        use_attention_sink=True,
        use_grouped_output_projection=True,
        output_projection_groups=4,
    )
    x = make_hca_input(B=2, T=16, D=64)

    out, aux = hca(x, need_weights=True)

    B, T, D = x.shape
    S = math.ceil(T / hca.compression_factor)

    assert out.shape == x.shape
    assert isinstance(aux, dict)

    assert aux["global_attn_weights"].shape == (B, hca.n_heads, T, S)
    assert aux["local_attn_weights"].shape == (B, hca.n_heads, T, T)
    assert aux["compressed_valid_mask"].shape == (B, S)
    assert aux["compressed_position_ids"].shape == (S,)

    assert "sink_attn_weights" in aux
    assert aux["sink_attn_weights"].shape == (B, hca.n_heads, T, 1)


# ============================================================
# D. Masks and global causality
# ============================================================

def test_global_compressed_attention_blocks_current_and_future_blocks():
    m = 4
    hca = make_hca(compression_factor=m, window_size=4)
    hca.eval()

    B, T, D = 2, 16, 64

    x = make_hca_input(B=B, T=T, D=D)

    _, aux = hca(x, need_weights=True)

    global_weights = aux["global_attn_weights"]
    S = global_weights.shape[-1]

    for t in range(T):
        query_block = t // m
        if query_block < S:
            blocked = global_weights[:, :, t, query_block:]
            assert torch.allclose(
                blocked,
                torch.zeros_like(blocked),
                atol=0.0,
                rtol=0.0,
            )


def test_first_block_has_no_global_attention():
    m = 4
    hca = make_hca(compression_factor=m, window_size=4)
    hca.eval()

    x = make_hca_input(B=2, T=16, D=64)

    _, aux = hca(x, need_weights=True)

    global_weights = aux["global_attn_weights"]

    assert torch.allclose(
        global_weights[:, :, :m, :],
        torch.zeros_like(global_weights[:, :, :m, :]),
        atol=0.0,
        rtol=0.0,
    )


def test_local_window_is_causal():
    hca = make_hca(window_size=4)
    hca.eval()

    B, T, D = 2, 16, 64
    x = make_hca_input(B=B, T=T, D=D)

    _, aux = hca(x, need_weights=True)

    local_weights = aux["local_attn_weights"]

    future_mask = torch.triu(
        torch.ones(T, T, dtype=torch.bool),
        diagonal=1,
    )

    future_weights = local_weights[:, :, future_mask]

    assert torch.allclose(
        future_weights,
        torch.zeros_like(future_weights),
        atol=0.0,
        rtol=0.0,
    )


def test_local_window_limits_past_context():
    W = 4
    hca = make_hca(window_size=W)
    hca.eval()

    B, T, D = 2, 16, 64
    x = make_hca_input(B=B, T=T, D=D)

    _, aux = hca(x, need_weights=True)

    local_weights = aux["local_attn_weights"]

    q_pos = torch.arange(T)[:, None]
    k_pos = torch.arange(T)[None, :]

    too_old_mask = (q_pos - k_pos) >= W

    too_old_weights = local_weights[:, :, too_old_mask]

    assert torch.allclose(
        too_old_weights,
        torch.zeros_like(too_old_weights),
        atol=0.0,
        rtol=0.0,
    )


def test_hca_changing_future_tokens_does_not_change_past_outputs():
    hca = make_hca(
        attention_dropout=0.0,
        residual_dropout=0.0,
        compression_factor=4,
        window_size=4,
    )
    hca.eval()

    B, T, D = 2, 16, 64
    cut = 8

    x1 = make_hca_input(B=B, T=T, D=D)
    x2 = x1.clone()
    x2[:, cut:, :] = torch.randn_like(x2[:, cut:, :])

    out1 = hca(x1)
    out2 = hca(x2)

    assert torch.allclose(
        out1[:, :cut, :],
        out2[:, :cut, :],
        atol=1e-5,
        rtol=1e-5,
    )


# ============================================================
# E. attention_mask tests
# ============================================================

def test_attention_mask_blocks_padding_local_keys():
    hca = make_hca(window_size=8)
    hca.eval()

    B, T, D = 2, 16, 64
    x = make_hca_input(B=B, T=T, D=D)

    attention_mask = torch.ones(B, T, dtype=torch.long)
    attention_mask[0, 5] = 0
    attention_mask[1, 7] = 0

    _, aux = hca(
        x,
        attention_mask=attention_mask,
        need_weights=True,
    )

    local_weights = aux["local_attn_weights"]

    assert torch.allclose(
        local_weights[0, :, :, 5],
        torch.zeros_like(local_weights[0, :, :, 5]),
        atol=0.0,
        rtol=0.0,
    )

    assert torch.allclose(
        local_weights[1, :, :, 7],
        torch.zeros_like(local_weights[1, :, :, 7]),
        atol=0.0,
        rtol=0.0,
    )


def test_attention_mask_blocks_padding_compressed_blocks():
    m = 4
    hca = make_hca(compression_factor=m, window_size=4)
    hca.eval()

    B, T, D = 2, 16, 64
    x = make_hca_input(B=B, T=T, D=D)

    attention_mask = torch.ones(B, T, dtype=torch.long)

    # Block 2 for batch 0 is fully padding.
    attention_mask[0, 8:12] = 0

    _, aux = hca(
        x,
        attention_mask=attention_mask,
        need_weights=True,
    )

    compressed_valid_mask = aux["compressed_valid_mask"]
    global_weights = aux["global_attn_weights"]

    assert compressed_valid_mask[0, 2] == 0

    assert torch.allclose(
        global_weights[0, :, :, 2],
        torch.zeros_like(global_weights[0, :, :, 2]),
        atol=0.0,
        rtol=0.0,
    )


def test_hca_attention_mask_shape_validation_accepts_BT():
    hca = make_hca()

    x = make_hca_input(B=2, T=16, D=64)
    attention_mask = torch.ones(2, 16)

    out = hca(x, attention_mask=attention_mask)

    assert out.shape == x.shape


@pytest.mark.parametrize(
    "bad_mask",
    [
        torch.ones(16),
        torch.ones(2, 16, 1),
        torch.ones(2, 17),
    ],
)
def test_hca_attention_mask_shape_validation_rejects_bad_shapes(bad_mask):
    hca = make_hca()

    x = make_hca_input(B=2, T=16, D=64)

    with pytest.raises(ValueError):
        hca(x, attention_mask=bad_mask)


# ============================================================
# F. RoPE tests
# ============================================================

def test_hca_start_pos_matches_explicit_position_ids():
    hca = make_hca(
        use_rope=True,
        attention_dropout=0.0,
        residual_dropout=0.0,
    )
    hca.eval()

    B, T, D = 2, 16, 64
    start_pos = 10

    x = make_hca_input(B=B, T=T, D=D)

    out_start = hca(x, start_pos=start_pos)
    out_explicit = hca(
        x,
        position_ids=torch.arange(start_pos, start_pos + T),
    )

    assert torch.allclose(out_start, out_explicit, atol=1e-5, rtol=1e-5)


def test_hca_no_rope_when_disabled():
    hca = make_hca(
        use_rope=False,
        attention_dropout=0.0,
        residual_dropout=0.0,
    )
    hca.eval()

    B, T, D = 2, 16, 64
    x = make_hca_input(B=B, T=T, D=D)

    out1 = hca(x, start_pos=0)
    out2 = hca(x, position_ids=torch.arange(10, 10 + T), start_pos=10)

    assert torch.allclose(out1, out2, atol=1e-6, rtol=1e-6)


# ============================================================
# G. Attention weights tests
# ============================================================

def test_sink_plus_global_plus_local_weights_sum_to_one():
    torch.manual_seed(0)

    config = HCAConfig(
        d_model=64,
        n_heads=4,
        head_dim=16,
        compression_factor=4,
        window_size=8,
        max_seq_len=64,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_rope=True,
        rotary_dim=16,
        use_attention_sink=True,
        use_grouped_output_projection=True,
        output_projection_groups=4,
    )

    hca = HCAAttention(config)
    hca.eval()

    B, T, D = 2, 17, 64
    x = torch.randn(B, T, D)

    out, aux = hca(x, need_weights=True)

    sink_sum = aux["sink_attn_weights"].sum(dim=-1)      # [B, H, T]
    global_sum = aux["global_attn_weights"].sum(dim=-1)  # [B, H, T]
    local_sum = aux["local_attn_weights"].sum(dim=-1)    # [B, H, T]

    total = sink_sum + global_sum + local_sum

    assert torch.allclose(
        total,
        torch.ones_like(total),
        atol=1e-5,
        rtol=1e-5,
    )

    assert out.shape == (B, T, D)


def test_no_nan_when_no_global_blocks_available():
    m = 4
    hca = make_hca(
        compression_factor=m,
        window_size=4,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_attention_sink=True,
    )
    hca.eval()

    x = make_hca_input(B=2, T=16, D=64)

    out, aux = hca(x, need_weights=True)

    assert torch.isfinite(out).all()
    assert torch.isfinite(aux["global_attn_weights"]).all()
    assert torch.isfinite(aux["local_attn_weights"]).all()
    assert torch.isfinite(aux["sink_attn_weights"]).all()

    assert torch.allclose(
        aux["global_attn_weights"][:, :, :m, :],
        torch.zeros_like(aux["global_attn_weights"][:, :, :m, :]),
        atol=0.0,
        rtol=0.0,
    )

    # In the first compression block, there are no global compressed blocks.
    # The probability mass should be assigned to sink + local attention.
    total_first_block = (
        aux["sink_attn_weights"][:, :, :m, :].sum(dim=-1)
        + aux["local_attn_weights"][:, :, :m, :].sum(dim=-1)
    )

    assert torch.allclose(
        total_first_block,
        torch.ones_like(total_first_block),
        atol=1e-5,
        rtol=1e-5,
    )


# ============================================================
# H. Gradient tests
# ============================================================

def test_hca_backward_computes_gradients():
    hca = make_hca(
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_bias=True,
        use_attention_sink=True,
        use_grouped_output_projection=True,
        output_projection_groups=4,
    )

    x = make_hca_input(B=2, T=16, D=64)
    x.requires_grad_(True)

    out = hca(x)
    loss = out.sum()
    loss.backward()

    assert x.grad is not None
    assert hca.q_proj.weight.grad is not None
    assert hca.kv_proj.weight.grad is not None
    assert hca.z_proj.weight.grad is not None
    assert hca.compressor.compression_bias.grad is not None

    assert torch.isfinite(x.grad).all()
    assert torch.isfinite(hca.q_proj.weight.grad).all()
    assert torch.isfinite(hca.kv_proj.weight.grad).all()
    assert torch.isfinite(hca.z_proj.weight.grad).all()
    assert torch.isfinite(hca.compressor.compression_bias.grad).all()

    # Attention sink parameters should receive gradients.
    assert hca.sink_k.grad is not None
    assert hca.sink_v.grad is not None
    assert torch.isfinite(hca.sink_k.grad).all()
    assert torch.isfinite(hca.sink_v.grad).all()

    # Grouped output projection parameters should receive gradients.
    assert hasattr(hca.out_proj, "group_projs")

    for proj in hca.out_proj.group_projs:
        assert proj.weight.grad is not None
        assert torch.isfinite(proj.weight.grad).all()

        if proj.bias is not None:
            assert proj.bias.grad is not None
            assert torch.isfinite(proj.bias.grad).all()


# ============================================================
# I. Integration with block-like interface
# ============================================================

def test_hca_can_replace_attention_in_block_interface():
    B, T, D = 2, 16, 64

    norm1 = RMSNorm(dim=D)
    attention = make_hca(
        d_model=D,
        n_heads=4,
        head_dim=16,
        compression_factor=4,
        window_size=4,
        max_seq_len=T,
    )

    x = torch.randn(B, T, D)

    residual = x
    h = norm1(x)
    attn_out = attention(h)
    out = residual + attn_out

    assert out.shape == (B, T, D)
    assert torch.isfinite(out).all()


@pytest.mark.parametrize(
    "output_projection_groups",
    [0, -1, 3],  # 3 no divide n_heads=4
)
def test_invalid_output_projection_groups_raises(output_projection_groups):
    with pytest.raises(ValueError):
        HCAAttention(
            make_hca_config(
                use_grouped_output_projection=True,
                output_projection_groups=output_projection_groups,
            )
        )

def test_global_plus_local_weights_sum_to_one_without_sink():
    torch.manual_seed(0)

    config = HCAConfig(
        d_model=64,
        n_heads=4,
        head_dim=16,
        compression_factor=4,
        window_size=8,
        max_seq_len=64,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_rope=True,
        rotary_dim=16,
        use_attention_sink=False,
        use_grouped_output_projection=True,
        output_projection_groups=4,
    )

    hca = HCAAttention(config)
    hca.eval()

    B, T, D = 2, 17, 64
    x = torch.randn(B, T, D)

    out, aux = hca(x, need_weights=True)

    assert "sink_attn_weights" not in aux

    global_sum = aux["global_attn_weights"].sum(dim=-1)
    local_sum = aux["local_attn_weights"].sum(dim=-1)

    total = global_sum + local_sum

    assert torch.allclose(
        total,
        torch.ones_like(total),
        atol=1e-5,
        rtol=1e-5,
    )

    assert out.shape == (B, T, D)


def test_grouped_output_projection_shape_and_gradients():
    torch.manual_seed(0)

    proj = GroupedOutputProjection(
        n_heads=4,
        head_dim=16,
        num_groups=4,
        bias=True,
        init_std=0.02,
    )

    x = torch.randn(2, 8, 4, 16, requires_grad=True)

    out = proj(x)

    assert out.shape == (2, 8, 64)
    assert torch.isfinite(out).all()

    loss = out.sum()
    loss.backward()

    assert x.grad is not None
    assert torch.isfinite(x.grad).all()

    for group_proj in proj.group_projs:
        assert group_proj.weight.grad is not None
        assert torch.isfinite(group_proj.weight.grad).all()

        assert group_proj.bias.grad is not None
        assert torch.isfinite(group_proj.bias.grad).all()


def test_attention_sink_is_present_and_receives_valid_mass():
    torch.manual_seed(0)

    hca = make_hca(
        use_attention_sink=True,
        attention_dropout=0.0,
        residual_dropout=0.0,
    )
    hca.eval()

    x = make_hca_input(B=2, T=16, D=64)

    _, aux = hca(x, need_weights=True)

    assert "sink_attn_weights" in aux

    sink_weights = aux["sink_attn_weights"]

    assert sink_weights.shape == (2, hca.n_heads, 16, 1)
    assert torch.isfinite(sink_weights).all()
    assert (sink_weights >= 0).all()

    # At least some probability mass should be assigned to the sink.
    assert sink_weights.sum() > 0



def test_hca_aux_returns_compressed_position_ids():
    m = 4
    hca = make_hca(
        compression_factor=m,
        use_attention_sink=True,
    )
    hca.eval()

    B, T, D = 2, 17, 64
    start_pos = 10

    x = make_hca_input(B=B, T=T, D=D)

    _, aux = hca(
        x,
        start_pos=start_pos,
        need_weights=True,
    )

    # Blocks:
    # [0,1,2,3]       -> position 13
    # [4,5,6,7]       -> position 17
    # [8,9,10,11]     -> position 21
    # [12,13,14,15]   -> position 25
    # [16]            -> position 26
    expected = torch.tensor(
        [13, 17, 21, 25, 26],
        device=x.device,
        dtype=torch.long)

    assert "compressed_position_ids" in aux
    assert torch.equal(aux["compressed_position_ids"], expected)


def test_sink_global_plus_local_weights_sum_to_one():
    hca = make_hca(
        compression_factor=4,
        window_size=4,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_attention_sink=True,
    )
    hca.eval()

    x = make_hca_input(B=2, T=16, D=64)

    _, aux = hca(x, need_weights=True)

    sink_w = aux["sink_attn_weights"]
    global_w = aux["global_attn_weights"]
    local_w = aux["local_attn_weights"]

    total = (
        sink_w.sum(dim=-1)
        + global_w.sum(dim=-1)
        + local_w.sum(dim=-1)
    )

    assert torch.allclose(
        total,
        torch.ones_like(total),
        atol=1e-5,
        rtol=1e-5,
    )

In [65]:
# ============================================================
# Run ONLY HCAAttention tests in Jupyter
# Handles pytest-style parametrized tests manually
# ============================================================

import torch

# ------------------------------------------------------------
# Plain tests: no arguments
# ------------------------------------------------------------

hca_plain_tests = [
    # Config
    "test_valid_hca_config_builds",

    # Compressor
    "test_compressor_output_shape_exact_multiple",
    "test_compressor_output_shape_non_exact_multiple",
    "test_compressor_valid_mask_all_valid",
    "test_compressor_valid_mask_with_padding",
    "test_compressor_ignores_padding_tokens",
    "test_compressor_backward",

    # Forward / shape
    "test_hca_output_shape_matches_input",
    "test_hca_rejects_wrong_hidden_size",
    "test_hca_rejects_too_long_sequence",
    "test_hca_need_weights_returns_aux",

    # Masks / causality
    "test_global_compressed_attention_blocks_current_and_future_blocks",
    "test_first_block_has_no_global_attention",
    "test_local_window_is_causal",
    "test_local_window_limits_past_context",
    "test_hca_changing_future_tokens_does_not_change_past_outputs",

    # attention_mask
    "test_attention_mask_blocks_padding_local_keys",
    "test_attention_mask_blocks_padding_compressed_blocks",
    "test_hca_attention_mask_shape_validation_accepts_BT",

    # RoPE
    "test_hca_start_pos_matches_explicit_position_ids",
    "test_hca_no_rope_when_disabled",

    # Attention weights
    # OJO: este test debe sumar sink + global + local si use_attention_sink=True.
    "test_sink_global_plus_local_weights_sum_to_one",
    "test_no_nan_when_no_global_blocks_available",

    # Canonical HCA additions
    "test_grouped_output_projection_shape_and_gradients",
    "test_attention_sink_is_present_and_receives_valid_mass",
    "test_hca_aux_returns_compressed_position_ids",

    # Gradients / integration
    "test_hca_backward_computes_gradients",
    "test_hca_can_replace_attention_in_block_interface",
]


# ------------------------------------------------------------
# Parametrized tests: pytest-style converted manually
# ------------------------------------------------------------

hca_param_tests = [
    # Config validation
    ("test_invalid_hca_d_model_raises", {"d_model": 0}),
    ("test_invalid_hca_d_model_raises", {"d_model": -1}),
    ("test_invalid_hca_d_model_raises", {"d_model": -64}),

    ("test_invalid_heads_or_head_dim_raises", {"kwargs": {"n_heads": 0}}),
    ("test_invalid_heads_or_head_dim_raises", {"kwargs": {"n_heads": -1}}),
    ("test_invalid_heads_or_head_dim_raises", {"kwargs": {"head_dim": 0}}),
    ("test_invalid_heads_or_head_dim_raises", {"kwargs": {"head_dim": -1}}),
    ("test_invalid_heads_or_head_dim_raises", {"kwargs": {"head_dim": 15}}),

    ("test_invalid_compression_factor_raises", {"compression_factor": 0}),
    ("test_invalid_compression_factor_raises", {"compression_factor": -1}),
    ("test_invalid_compression_factor_raises", {"compression_factor": -4}),

    ("test_invalid_window_size_raises", {"window_size": 0}),
    ("test_invalid_window_size_raises", {"window_size": -1}),
    ("test_invalid_window_size_raises", {"window_size": -4}),

    ("test_invalid_hca_dropout_raises", {"field": "attention_dropout", "value": -0.1}),
    ("test_invalid_hca_dropout_raises", {"field": "attention_dropout", "value": 1.0}),
    ("test_invalid_hca_dropout_raises", {"field": "attention_dropout", "value": 1.5}),
    ("test_invalid_hca_dropout_raises", {"field": "residual_dropout", "value": -0.1}),
    ("test_invalid_hca_dropout_raises", {"field": "residual_dropout", "value": 1.0}),
    ("test_invalid_hca_dropout_raises", {"field": "residual_dropout", "value": 1.5}),

    ("test_invalid_hca_rotary_dim_raises", {"rotary_dim": 0}),
    ("test_invalid_hca_rotary_dim_raises", {"rotary_dim": -1}),
    ("test_invalid_hca_rotary_dim_raises", {"rotary_dim": 17}),
    ("test_invalid_hca_rotary_dim_raises", {"rotary_dim": 32}),

    ("test_invalid_output_projection_groups_raises", {"output_projection_groups": 0}),
    ("test_invalid_output_projection_groups_raises", {"output_projection_groups": -1}),
    ("test_invalid_output_projection_groups_raises", {"output_projection_groups": 3}),

    # Forward validation
    ("test_hca_rejects_wrong_input_rank", {"bad_x": torch.randn(16, 64)}),
    ("test_hca_rejects_wrong_input_rank", {"bad_x": torch.randn(2, 16, 64, 1)}),

    # attention_mask validation
    ("test_hca_attention_mask_shape_validation_rejects_bad_shapes", {"bad_mask": torch.ones(16)}),
    ("test_hca_attention_mask_shape_validation_rejects_bad_shapes", {"bad_mask": torch.ones(2, 16, 1)}),
    ("test_hca_attention_mask_shape_validation_rejects_bad_shapes", {"bad_mask": torch.ones(2, 17)}),
]


# ------------------------------------------------------------
# Optional compatibility alias
# ------------------------------------------------------------
# Si tú ya tienes definido el test viejo con otro nombre, lo aliasamos.
# Si no existe, no hace nada.

if (
    "test_sink_global_plus_local_weights_sum_to_one" not in globals()
    and "test_global_plus_local_weights_sum_to_one" in globals()
):
    test_sink_global_plus_local_weights_sum_to_one = globals()[
        "test_global_plus_local_weights_sum_to_one"
    ]


# ------------------------------------------------------------
# Validate that all test functions exist before running
# ------------------------------------------------------------

missing_plain = [name for name in hca_plain_tests if name not in globals()]
missing_param = [name for name, _ in hca_param_tests if name not in globals()]
missing = sorted(set(missing_plain + missing_param))

if missing:
    raise KeyError(
        "These HCA test functions are listed in the runner but are not defined "
        "in the current Jupyter namespace:\n"
        + "\n".join(f"  - {name}" for name in missing)
    )


# ------------------------------------------------------------
# Runner
# ------------------------------------------------------------

n_passed = 0
failed = []

for name in hca_plain_tests:
    try:
        globals()[name]()
        n_passed += 1
        print(f"PASS: {name}")
    except Exception as e:
        failed.append((name, e))
        print(f"FAIL: {name} -> {type(e).__name__}: {e}")
        raise

for name, kwargs in hca_param_tests:
    case_name = f"{name}({kwargs})"
    try:
        globals()[name](**kwargs)
        n_passed += 1
        print(f"PASS: {case_name}")
    except Exception as e:
        failed.append((case_name, e))
        print(f"FAIL: {case_name} -> {type(e).__name__}: {e}")
        raise

print(f"\nOK: {n_passed} HCAAttention tests/casos pasaron.")

PASS: test_valid_hca_config_builds
PASS: test_compressor_output_shape_exact_multiple
PASS: test_compressor_output_shape_non_exact_multiple
PASS: test_compressor_valid_mask_all_valid
PASS: test_compressor_valid_mask_with_padding
PASS: test_compressor_ignores_padding_tokens
PASS: test_compressor_backward
PASS: test_hca_output_shape_matches_input
PASS: test_hca_rejects_wrong_hidden_size
PASS: test_hca_rejects_too_long_sequence
PASS: test_hca_need_weights_returns_aux
PASS: test_global_compressed_attention_blocks_current_and_future_blocks
PASS: test_first_block_has_no_global_attention
PASS: test_local_window_is_causal
PASS: test_local_window_limits_past_context
PASS: test_hca_changing_future_tokens_does_not_change_past_outputs
PASS: test_attention_mask_blocks_padding_local_keys
PASS: test_attention_mask_blocks_padding_compressed_blocks
PASS: test_hca_attention_mask_shape_validation_accepts_BT
PASS: test_hca_start_pos_matches_explicit_position_ids
PASS: test_hca_no_rope_when_disabled
PASS: t

---

# CSA: Compressed Sparse Attention

In [45]:
# ============================================================
# Mini DeepSeek-V4 Canonical CSA
# Compressed Sparse Attention with overlapped a/b compression
# ============================================================

import math
from dataclasses import dataclass
from typing import Optional, Tuple, Union, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# CONFIG
# ============================================================

@dataclass
class CSAConfig:
    d_model: int
    n_heads: int
    head_dim: Optional[int] = None

    compression_factor: int = 8
    top_k: int = 8
    window_size: int = 32

    indexer_dim: int = 32
    n_indexer_heads: int = 2
    query_compression_dim: Optional[int] = None

    attention_dropout: float = 0.0
    residual_dropout: float = 0.0
    use_bias: bool = False

    use_rope: bool = True
    rope_theta: float = 10000.0
    rotary_dim: Optional[int] = None

    max_seq_len: int = 1024
    init_std: float = 0.02

    # More canonical / final CSA options
    use_attention_sink: bool = True
    use_grouped_output_projection: bool = True
    output_projection_groups: Optional[int] = None

    # Canonical default: indexer selects blocks, but does not directly bias
    # core attention logits. Enable for experimental/trainability purposes.
    use_indexer_score_bias: bool = False

    # Keep the local exact sliding-window KV branch conceptually separate from
    # the global compressed a/b KV construction.
    use_separate_local_kv: bool = True

    def validate(self) -> None:
        if self.d_model <= 0:
            raise ValueError(f"d_model must be > 0, got {self.d_model}")

        if self.n_heads <= 0:
            raise ValueError(f"n_heads must be > 0, got {self.n_heads}")

        if self.head_dim is None:
            if self.d_model % self.n_heads != 0:
                raise ValueError(
                    "If head_dim is None, d_model must be divisible by n_heads. "
                    f"Got d_model={self.d_model}, n_heads={self.n_heads}"
                )
            head_dim = self.d_model // self.n_heads
        else:
            head_dim = self.head_dim

        if head_dim <= 0:
            raise ValueError(f"head_dim must be > 0, got {head_dim}")

        if self.n_heads * head_dim != self.d_model:
            raise ValueError(
                "For CSA v1, n_heads * head_dim must equal d_model. "
                f"Got n_heads={self.n_heads}, head_dim={head_dim}, "
                f"d_model={self.d_model}"
            )

        if self.compression_factor <= 0:
            raise ValueError(
                f"compression_factor must be > 0, got {self.compression_factor}"
            )

        if self.top_k <= 0:
            raise ValueError(f"top_k must be > 0, got {self.top_k}")

        if self.window_size <= 0:
            raise ValueError(f"window_size must be > 0, got {self.window_size}")

        if self.indexer_dim <= 0:
            raise ValueError(f"indexer_dim must be > 0, got {self.indexer_dim}")

        if self.n_indexer_heads <= 0:
            raise ValueError(
                f"n_indexer_heads must be > 0, got {self.n_indexer_heads}"
            )

        if self.query_compression_dim is not None and self.query_compression_dim <= 0:
            raise ValueError(
                "query_compression_dim must be > 0 when provided, "
                f"got {self.query_compression_dim}"
            )

        if not (0.0 <= self.attention_dropout < 1.0):
            raise ValueError(
                "attention_dropout must satisfy 0 <= attention_dropout < 1, "
                f"got {self.attention_dropout}"
            )

        if not (0.0 <= self.residual_dropout < 1.0):
            raise ValueError(
                "residual_dropout must satisfy 0 <= residual_dropout < 1, "
                f"got {self.residual_dropout}"
            )

        if self.max_seq_len <= 0:
            raise ValueError(f"max_seq_len must be > 0, got {self.max_seq_len}")

        if self.init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {self.init_std}")

        if self.rope_theta <= 0:
            raise ValueError(f"rope_theta must be > 0, got {self.rope_theta}")

        if self.rotary_dim is not None:
            if self.rotary_dim <= 0:
                raise ValueError(
                    f"rotary_dim must be > 0 when provided, got {self.rotary_dim}"
                )

            if self.rotary_dim > head_dim:
                raise ValueError(
                    f"rotary_dim must be <= head_dim. "
                    f"Got rotary_dim={self.rotary_dim}, head_dim={head_dim}"
                )

            if self.rotary_dim % 2 != 0:
                raise ValueError(f"rotary_dim must be even, got {self.rotary_dim}")

        if self.output_projection_groups is not None:
            if self.output_projection_groups <= 0:
                raise ValueError(
                    "output_projection_groups must be > 0 when provided, "
                    f"got {self.output_projection_groups}"
                )

            if self.n_heads % self.output_projection_groups != 0:
                raise ValueError(
                    "n_heads must be divisible by output_projection_groups. "
                    f"Got n_heads={self.n_heads}, "
                    f"output_projection_groups={self.output_projection_groups}"
                )


# ============================================================
# SAFE MASKED SOFTMAX
# ============================================================

def safe_masked_softmax(
    scores: torch.Tensor,
    allowed_mask: torch.Tensor,
    dim: int = -1,
) -> torch.Tensor:
    """
    Safe masked softmax.

    Args:
        scores:
            Arbitrary score tensor.

        allowed_mask:
            Boolean tensor broadcastable to scores.
            True = allowed.
            False = masked.

        dim:
            Softmax dimension.

    Returns:
        weights:
            Same shape as scores.
            Rows with no allowed keys become exactly zero.
    """
    if allowed_mask.dtype != torch.bool:
        allowed_mask = allowed_mask.bool()

    mask_value = torch.finfo(scores.dtype).min
    masked_scores = scores.masked_fill(~allowed_mask, mask_value)

    weights = F.softmax(masked_scores.float(), dim=dim).to(dtype=scores.dtype)
    weights = weights * allowed_mask.to(dtype=weights.dtype)

    denom = weights.sum(dim=dim, keepdim=True)

    weights = torch.where(
        denom > 0,
        weights / denom.clamp_min(torch.finfo(weights.dtype).tiny),
        torch.zeros_like(weights),
    )

    return weights


# ============================================================
# GROUPED OUTPUT PROJECTION
# ============================================================

class GroupedOutputProjection(nn.Module):
    """
    Simple PyTorch grouped output projection.

    Input:
        x: [B, T, H, Dh]

    Output:
        out: [B, T, H * Dh]

    This keeps the grouped-output-projection idea without custom kernels.
    """

    def __init__(
        self,
        n_heads: int,
        head_dim: int,
        num_groups: int,
        bias: bool = False,
        init_std: float = 0.02,
    ):
        super().__init__()

        if n_heads <= 0:
            raise ValueError(f"n_heads must be > 0, got {n_heads}")
        if head_dim <= 0:
            raise ValueError(f"head_dim must be > 0, got {head_dim}")
        if num_groups <= 0:
            raise ValueError(f"num_groups must be > 0, got {num_groups}")
        if n_heads % num_groups != 0:
            raise ValueError(
                f"n_heads={n_heads} must be divisible by num_groups={num_groups}"
            )

        self.n_heads = n_heads
        self.head_dim = head_dim
        self.num_groups = num_groups
        self.heads_per_group = n_heads // num_groups
        self.group_dim = self.heads_per_group * head_dim
        self.init_std = init_std

        self.group_projs = nn.ModuleList(
            [
                nn.Linear(self.group_dim, self.group_dim, bias=bias)
                for _ in range(num_groups)
            ]
        )

        self.reset_parameters()

    def reset_parameters(self) -> None:
        for proj in self.group_projs:
            nn.init.normal_(proj.weight, mean=0.0, std=self.init_std)
            if proj.bias is not None:
                nn.init.zeros_(proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() != 4:
            raise ValueError(f"x must have shape [B,T,H,Dh], got {tuple(x.shape)}")

        B, T, H, Dh = x.shape
        if H != self.n_heads:
            raise ValueError(f"Expected H={self.n_heads}, got {H}")
        if Dh != self.head_dim:
            raise ValueError(f"Expected Dh={self.head_dim}, got {Dh}")

        group_outputs = []
        for g, proj in enumerate(self.group_projs):
            h_start = g * self.heads_per_group
            h_end = (g + 1) * self.heads_per_group

            x_g = x[:, :, h_start:h_end, :].reshape(B, T, self.group_dim)
            y_g = proj(x_g)
            group_outputs.append(y_g)

        return torch.cat(group_outputs, dim=-1)


# ============================================================
# OVERLAPPED A/B COMPRESSOR
# ============================================================

class CSAOverlappedCompressor(nn.Module):
    """
    Canonical CSA overlapped a/b compressor.

    For compressed block i:
        A branch uses current block:
            [i*m, min((i+1)*m, T))

        B branch uses previous block:
            [(i-1)*m, i*m)

    For i = 0:
        B branch is empty/invalid.

    Input:
        C_a: [B, T, D]
        C_b: [B, T, D]
        Z_a: [B, T, D]
        Z_b: [B, T, D]

    Output:
        C_comp: [B, S, D]
        comp_valid_mask: [B, S]
        comp_position_ids: [S] or [B, S]
    """

    def __init__(
        self,
        compression_factor: int,
        dim: int,
        init_std: float = 0.02,
    ):
        super().__init__()

        if compression_factor <= 0:
            raise ValueError(
                f"compression_factor must be > 0, got {compression_factor}"
            )

        if dim <= 0:
            raise ValueError(f"dim must be > 0, got {dim}")

        if init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {init_std}")

        self.compression_factor = compression_factor
        self.dim = dim
        self.init_std = init_std

        self.bias_a = nn.Parameter(torch.empty(compression_factor, dim))
        self.bias_b = nn.Parameter(torch.empty(compression_factor, dim))

        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.normal_(self.bias_a, mean=0.0, std=self.init_std)
        nn.init.normal_(self.bias_b, mean=0.0, std=self.init_std)

    def _safe_temporal_softmax(
        self,
        scores: torch.Tensor,
        valid: Optional[torch.Tensor],
        dim: int = 1,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Safe softmax over temporal dimension.

        Args:
            scores: [B, L, D]
            valid:  None or [B, L]

        Returns:
            weights: [B, L, D]
            block_valid: [B]
        """
        B, L, _ = scores.shape

        if valid is None:
            block_valid = torch.ones(B, device=scores.device, dtype=torch.bool)
            weights = F.softmax(scores.float(), dim=dim).to(dtype=scores.dtype)
            return weights, block_valid

        if valid.shape != (B, L):
            raise ValueError(
                f"valid must have shape {(B, L)}, got {tuple(valid.shape)}"
            )

        valid_bool = valid.to(device=scores.device, dtype=torch.bool)
        block_valid = valid_bool.any(dim=1)

        weights = safe_masked_softmax(
            scores=scores,
            allowed_mask=valid_bool[:, :, None],
            dim=dim,
        )

        return weights, block_valid

    def _build_comp_position_ids(
        self,
        position_ids: Optional[torch.Tensor],
        batch_size: int,
        seq_len: int,
        num_blocks: int,
        device: torch.device,
        start_pos: int = 0,
    ) -> torch.Tensor:
        """
        Compressed block position = position of last token in current A block.
        """
        m = self.compression_factor

        if position_ids is None:
            positions = []
            for i in range(num_blocks):
                a_end = min((i + 1) * m, seq_len)
                positions.append(start_pos + a_end - 1)

            return torch.tensor(positions, device=device, dtype=torch.long)

        if position_ids.device != device:
            position_ids = position_ids.to(device)

        if position_ids.dim() == 1:
            if position_ids.shape[0] != seq_len:
                raise ValueError(
                    f"position_ids [T] must have length {seq_len}, "
                    f"got {position_ids.shape[0]}"
                )

            positions = []
            for i in range(num_blocks):
                a_end = min((i + 1) * m, seq_len)
                positions.append(position_ids[a_end - 1])

            return torch.stack(positions, dim=0)

        if position_ids.dim() == 2:
            if position_ids.shape != (batch_size, seq_len):
                raise ValueError(
                    f"position_ids [B,T] must have shape {(batch_size, seq_len)}, "
                    f"got {tuple(position_ids.shape)}"
                )

            positions = []
            for i in range(num_blocks):
                a_end = min((i + 1) * m, seq_len)
                positions.append(position_ids[:, a_end - 1])

            return torch.stack(positions, dim=1)

        raise ValueError(
            "position_ids must be None, [T], or [B,T], "
            f"got {tuple(position_ids.shape)}"
        )

    def forward(
        self,
        C_a: torch.Tensor,
        C_b: torch.Tensor,
        Z_a: torch.Tensor,
        Z_b: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        start_pos: int = 0,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

        if C_a.dim() != 3:
            raise ValueError(f"C_a must have shape [B,T,D], got {tuple(C_a.shape)}")

        if C_b.shape != C_a.shape:
            raise ValueError("C_b must have same shape as C_a")

        if Z_a.shape != C_a.shape:
            raise ValueError("Z_a must have same shape as C_a")

        if Z_b.shape != C_a.shape:
            raise ValueError("Z_b must have same shape as C_a")

        B, T, D = C_a.shape

        if D != self.dim:
            raise ValueError(f"Expected last dim {self.dim}, got {D}")

        if attention_mask is not None:
            if attention_mask.shape != (B, T):
                raise ValueError(
                    f"attention_mask must have shape {(B, T)}, "
                    f"got {tuple(attention_mask.shape)}"
                )

        m = self.compression_factor
        S = math.ceil(T / m)

        comp_blocks = []
        valid_blocks = []

        for i in range(S):
            # A branch: current block
            a_start = i * m
            a_end = min((i + 1) * m, T)
            a_len = a_end - a_start

            A_tokens = C_a[:, a_start:a_end, :]
            A_scores = Z_a[:, a_start:a_end, :] + self.bias_a[:a_len, :].to(
                device=C_a.device,
                dtype=Z_a.dtype,
            )[None, :, :]

            if attention_mask is None:
                valid_a = None
            else:
                valid_a = attention_mask[:, a_start:a_end]

            # B branch: previous block
            if i == 0:
                B_tokens = C_b.new_zeros(B, 0, D)
                B_scores = Z_b.new_zeros(B, 0, D)
                valid_b = None if attention_mask is None else attention_mask.new_zeros(B, 0)
            else:
                b_start = (i - 1) * m
                b_end = i * m
                b_len = b_end - b_start

                B_tokens = C_b[:, b_start:b_end, :]
                B_scores = Z_b[:, b_start:b_end, :] + self.bias_b[:b_len, :].to(
                    device=C_b.device,
                    dtype=Z_b.dtype,
                )[None, :, :]

                if attention_mask is None:
                    valid_b = None
                else:
                    valid_b = attention_mask[:, b_start:b_end]

            tokens = torch.cat([A_tokens, B_tokens], dim=1)
            scores = torch.cat([A_scores, B_scores], dim=1)

            if attention_mask is None:
                valid = None
            else:
                valid = torch.cat([valid_a, valid_b], dim=1)

            weights, block_valid = self._safe_temporal_softmax(
                scores=scores,
                valid=valid,
                dim=1,
            )

            comp_i = (weights * tokens).sum(dim=1)

            comp_i = torch.where(
                block_valid[:, None],
                comp_i,
                torch.zeros_like(comp_i),
            )

            comp_blocks.append(comp_i)
            valid_blocks.append(block_valid)

        C_comp = torch.stack(comp_blocks, dim=1)
        comp_valid_mask = torch.stack(valid_blocks, dim=1)

        comp_position_ids = self._build_comp_position_ids(
            position_ids=position_ids,
            batch_size=B,
            seq_len=T,
            num_blocks=S,
            device=C_a.device,
            start_pos=start_pos,
        )

        return C_comp, comp_valid_mask, comp_position_ids

In [46]:
# ============================================================
# CSA LIGHTNING INDEXER
# ============================================================

class CSALightningIndexer(nn.Module):
    """
    Lightning-style CSA indexer.

    Inputs:
        index_q: [B, T, H_i, I]
        index_weights: [B, T, H_i]
        I_comp: [B, S, I]
        comp_valid_mask: [B, S]

    Outputs:
        topk_indices: [B, T, K]
        topk_scores: [B, T, K]
        topk_mask: [B, T, K]
        index_scores: optional [B, T, S]
    """

    def __init__(
        self,
        compression_factor: int,
        top_k: int):

        super().__init__()

        if compression_factor <= 0:
            raise ValueError(
                f"compression_factor must be > 0, got {compression_factor}"
            )

        if top_k <= 0:
            raise ValueError(f"top_k must be > 0, got {top_k}")

        self.compression_factor = compression_factor
        self.top_k = top_k

    def _build_global_allowed_mask(
        self,
        T: int,
        S: int,
        device: torch.device) -> torch.Tensor:

        """
        Conservative causal rule:
            allowed[t, s] = s < floor(t / compression_factor)

        Only fully completed compressed blocks are selectable. The current
        block is represented through the exact local sliding-window branch.
        """
        token_pos = torch.arange(T, device=device)
        query_block = token_pos // self.compression_factor
        block_pos = torch.arange(S, device=device)

        return block_pos[None, :] < query_block[:, None]

    def forward(
        self,
        index_q: torch.Tensor,
        index_weights: torch.Tensor,
        I_comp: torch.Tensor,
        comp_valid_mask: torch.Tensor,
        need_scores: bool = False):

        if index_q.dim() != 4:
            raise ValueError(
                f"index_q must have shape [B,T,H_i,I], got {tuple(index_q.shape)}"
            )

        if I_comp.dim() != 3:
            raise ValueError(
                f"I_comp must have shape [B,S,I], got {tuple(I_comp.shape)}"
            )

        B, T, H_i, I = index_q.shape
        B2, S, I2 = I_comp.shape

        if B2 != B:
            raise ValueError(f"Batch mismatch: index_q B={B}, I_comp B={B2}")

        if I2 != I:
            raise ValueError(f"Indexer dim mismatch: index_q I={I}, I_comp I={I2}")

        if index_weights.shape != (B, T, H_i):
            raise ValueError(
                f"index_weights must have shape {(B, T, H_i)}, "
                f"got {tuple(index_weights.shape)}"
            )

        if comp_valid_mask.shape != (B, S):
            raise ValueError(
                f"comp_valid_mask must have shape {(B, S)}, "
                f"got {tuple(comp_valid_mask.shape)}"
            )

        raw = torch.einsum(
            "bthi,bsi->bths",
            index_q,
            I_comp,
        )
        # [B,T,H_i,S]

        raw = F.relu(raw)

        index_scores = (index_weights[..., None] * raw).sum(dim=2)
        # [B,T,S]

        causal_allowed = self._build_global_allowed_mask(
            T=T,
            S=S,
            device=index_q.device,
        )
        # [T,S]

        allowed = causal_allowed[None, :, :] & comp_valid_mask[:, None, :].bool()
        # [B,T,S]

        mask_value = torch.finfo(index_scores.dtype).min
        masked_scores = index_scores.masked_fill(~allowed, mask_value)

        K = min(self.top_k, S)

        topk_scores, topk_indices = torch.topk(
            masked_scores,
            k=K,
            dim=-1,
        )

        topk_mask = torch.gather(
            allowed,
            dim=-1,
            index=topk_indices,
        )

        topk_scores = torch.where(
            topk_mask,
            topk_scores,
            torch.zeros_like(topk_scores),
        )

        if need_scores:
            return topk_indices, topk_scores, topk_mask, index_scores

        return topk_indices, topk_scores, topk_mask

In [47]:
# ============================================================
# CANONICAL CSA ATTENTION
# ============================================================

class CSAAttention(nn.Module):
    """
    Canonical CSA mini implementation.

    Core pieces:
        - Overlapped a/b compression
        - Low-rank shared query path
        - Lightning indexer
        - Top-k causal sparse global attention
        - Local sliding-window branch with separate local KV path
        - Shared KV MQA
        - Optional attention sink
        - Grouped output projection
        - RoPE / partial RoPE

    Input:
        x: [B,T,d_model]

    Output:
        out: [B,T,d_model]

    If need_weights=True:
        out, aux
    """

    def __init__(self, config: CSAConfig):
        super().__init__()

        config.validate()
        self.config = config

        self.d_model = config.d_model
        self.n_heads = config.n_heads
        self.head_dim = (
            config.head_dim
            if config.head_dim is not None
            else config.d_model // config.n_heads
        )

        self.inner_dim = self.n_heads * self.head_dim

        self.compression_factor = config.compression_factor
        self.top_k = config.top_k
        self.window_size = config.window_size

        self.indexer_dim = config.indexer_dim
        self.n_indexer_heads = config.n_indexer_heads
        self.query_compression_dim = (
            config.query_compression_dim
            if config.query_compression_dim is not None
            else config.indexer_dim
        )

        self.max_seq_len = config.max_seq_len
        self.use_rope = config.use_rope
        self.use_attention_sink = config.use_attention_sink
        self.use_grouped_output_projection = config.use_grouped_output_projection
        self.use_indexer_score_bias = config.use_indexer_score_bias
        self.use_separate_local_kv = config.use_separate_local_kv

        # ----------------------------------------------------
        # Shared low-rank query path
        # ----------------------------------------------------
        self.q_down_proj = nn.Linear(
            self.d_model,
            self.query_compression_dim,
            bias=config.use_bias,
        )

        self.q_up_proj = nn.Linear(
            self.query_compression_dim,
            self.inner_dim,
            bias=config.use_bias,
        )

        self.index_q_up_proj = nn.Linear(
            self.query_compression_dim,
            self.n_indexer_heads * self.indexer_dim,
            bias=config.use_bias,
        )

        self.index_weight_proj = nn.Linear(
            self.d_model,
            self.n_indexer_heads,
            bias=config.use_bias,
        )

        # ----------------------------------------------------
        # Compressed KV path: a/b branches
        # ----------------------------------------------------
        self.a_kv_proj = nn.Linear(
            self.d_model,
            self.head_dim,
            bias=config.use_bias,
        )

        self.b_kv_proj = nn.Linear(
            self.d_model,
            self.head_dim,
            bias=config.use_bias,
        )

        self.a_z_proj = nn.Linear(
            self.d_model,
            self.head_dim,
            bias=config.use_bias,
        )

        self.b_z_proj = nn.Linear(
            self.d_model,
            self.head_dim,
            bias=config.use_bias,
        )

        # ----------------------------------------------------
        # Separate local exact KV branch
        # ----------------------------------------------------
        if self.use_separate_local_kv:
            self.local_kv_proj = nn.Linear(
                self.d_model,
                self.head_dim,
                bias=config.use_bias,
            )
        else:
            self.local_kv_proj = None

        # ----------------------------------------------------
        # Index key path: a/b branches
        # ----------------------------------------------------
        self.a_index_kv_proj = nn.Linear(
            self.d_model,
            self.indexer_dim,
            bias=config.use_bias,
        )

        self.b_index_kv_proj = nn.Linear(
            self.d_model,
            self.indexer_dim,
            bias=config.use_bias,
        )

        self.a_index_z_proj = nn.Linear(
            self.d_model,
            self.indexer_dim,
            bias=config.use_bias,
        )

        self.b_index_z_proj = nn.Linear(
            self.d_model,
            self.indexer_dim,
            bias=config.use_bias,
        )

        # ----------------------------------------------------
        # Compressors + indexer
        # ----------------------------------------------------
        self.kv_compressor = CSAOverlappedCompressor(
            compression_factor=config.compression_factor,
            dim=self.head_dim,
            init_std=config.init_std,
        )

        self.index_compressor = CSAOverlappedCompressor(
            compression_factor=config.compression_factor,
            dim=self.indexer_dim,
            init_std=config.init_std,
        )

        self.indexer = CSALightningIndexer(
            compression_factor=config.compression_factor,
            top_k=config.top_k,
        )

        # ----------------------------------------------------
        # Optional attention sink
        # ----------------------------------------------------
        if self.use_attention_sink:
            self.sink_k = nn.Parameter(torch.empty(1, 1, self.head_dim))
            self.sink_v = nn.Parameter(torch.empty(1, 1, self.head_dim))
        else:
            self.sink_k = None
            self.sink_v = None

        # ----------------------------------------------------
        # Output
        # ----------------------------------------------------
        if self.use_grouped_output_projection:
            num_groups = (
                config.output_projection_groups
                if config.output_projection_groups is not None
                else self.n_heads
            )

            self.out_proj = GroupedOutputProjection(
                n_heads=self.n_heads,
                head_dim=self.head_dim,
                num_groups=num_groups,
                bias=config.use_bias,
                init_std=config.init_std,
            )
        else:
            self.out_proj = nn.Linear(
                self.inner_dim,
                self.d_model,
                bias=config.use_bias,
            )

        if self.use_rope:
            # Assumes RotaryEmbedding exists in your project and accepts:
            #   x: [B,T,H,Dh]
            #   position_ids: None, [T], or [B,T]
            #   start_pos: int
            self.rope = RotaryEmbedding(
                dim=self.head_dim,
                rotary_dim=config.rotary_dim,
                base=config.rope_theta,
            )
        else:
            self.rope = None

        self.attention_dropout = nn.Dropout(config.attention_dropout)
        self.residual_dropout = nn.Dropout(config.residual_dropout)

        self.reset_parameters()

    def reset_parameters(self) -> None:
        modules = [
            self.q_down_proj,
            self.q_up_proj,
            self.index_q_up_proj,
            self.index_weight_proj,
            self.a_kv_proj,
            self.b_kv_proj,
            self.a_z_proj,
            self.b_z_proj,
            self.a_index_kv_proj,
            self.b_index_kv_proj,
            self.a_index_z_proj,
            self.b_index_z_proj,
        ]

        if self.local_kv_proj is not None:
            modules.append(self.local_kv_proj)

        if not self.use_grouped_output_projection:
            modules.append(self.out_proj)

        for module in modules:
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=self.config.init_std,
            )

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        if self.use_attention_sink:
            nn.init.normal_(self.sink_k, mean=0.0, std=self.config.init_std)
            nn.init.normal_(self.sink_v, mean=0.0, std=self.config.init_std)

    def _shape_q(self, q: torch.Tensor) -> torch.Tensor:
        B, T, _ = q.shape
        return q.view(B, T, self.n_heads, self.head_dim)

    def _shape_index_q(self, index_q: torch.Tensor) -> torch.Tensor:
        B, T, _ = index_q.shape
        return index_q.view(B, T, self.n_indexer_heads, self.indexer_dim)

    def _validate_attention_mask(
        self,
        attention_mask: torch.Tensor,
        batch_size: int,
        seq_len: int,
    ) -> torch.Tensor:
        if attention_mask.dim() != 2:
            raise ValueError(
                f"attention_mask must have shape [B,T], "
                f"got {tuple(attention_mask.shape)}"
            )

        if attention_mask.shape != (batch_size, seq_len):
            raise ValueError(
                f"attention_mask must have shape {(batch_size, seq_len)}, "
                f"got {tuple(attention_mask.shape)}"
            )

        return attention_mask

    def _build_local_allowed_mask(
        self,
        T: int,
        device: torch.device,
    ) -> torch.Tensor:
        q_pos = torch.arange(T, device=device)[:, None]
        k_pos = torch.arange(T, device=device)[None, :]

        causal = k_pos <= q_pos
        in_window = (q_pos - k_pos) < self.window_size

        return causal & in_window

    def _gather_selected(
        self,
        values: torch.Tensor,
        indices: torch.Tensor,
    ) -> torch.Tensor:
        """
        values:  [B,S,D]
        indices: [B,T,K]
        return:  [B,T,K,D]
        """
        B, S, D = values.shape
        B_i, T, K = indices.shape

        if B_i != B:
            raise ValueError(f"Batch mismatch: values B={B}, indices B={B_i}")

        source = values[:, None, :, :].expand(B, T, S, D)
        idx = indices[..., None].expand(B, T, K, D)

        return torch.gather(source, dim=2, index=idx)

    def forward(
        self,
        x: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        start_pos: int = 0,
        need_weights: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:

        # ----------------------------------------------------
        # Validate input
        # ----------------------------------------------------
        if x.dim() != 3:
            raise ValueError(
                f"CSAAttention expects x [B,T,d_model], got {tuple(x.shape)}"
            )

        B, T, C_model = x.shape

        if C_model != self.d_model:
            raise ValueError(
                f"Expected hidden size {self.d_model}, got {C_model}"
            )

        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length T={T} exceeds max_seq_len={self.max_seq_len}"
            )

        if attention_mask is not None:
            attention_mask = self._validate_attention_mask(
                attention_mask=attention_mask,
                batch_size=B,
                seq_len=T,
            )

        # ----------------------------------------------------
        # Shared low-rank query path
        # ----------------------------------------------------
        q_latent = self.q_down_proj(x)          # [B,T,Qc]

        q = self.q_up_proj(q_latent)
        q = self._shape_q(q)                    # [B,T,H,Dh]

        index_q = self.index_q_up_proj(q_latent)
        index_q = self._shape_index_q(index_q)  # [B,T,H_i,I]

        index_weights = self.index_weight_proj(x)  # [B,T,H_i]

        # ----------------------------------------------------
        # KV a/b projections
        # ----------------------------------------------------
        C_a = self.a_kv_proj(x)
        C_b = self.b_kv_proj(x)
        Z_a = self.a_z_proj(x)
        Z_b = self.b_z_proj(x)
        # [B,T,Dh]

        if self.local_kv_proj is not None:
            C_local = self.local_kv_proj(x)      # [B,T,Dh]
        else:
            C_local = C_a                        # backward-compatible fallback

        I_a = self.a_index_kv_proj(x)
        I_b = self.b_index_kv_proj(x)
        IZ_a = self.a_index_z_proj(x)
        IZ_b = self.b_index_z_proj(x)
        # [B,T,I]

        # ----------------------------------------------------
        # RoPE on query
        # ----------------------------------------------------
        if self.rope is not None:
            q = self.rope(
                q,
                position_ids=position_ids,
                start_pos=start_pos,
            )

        # ----------------------------------------------------
        # Overlapped compression: KV and index keys
        # ----------------------------------------------------
        C_comp, comp_valid_mask, comp_position_ids = self.kv_compressor(
            C_a=C_a,
            C_b=C_b,
            Z_a=Z_a,
            Z_b=Z_b,
            attention_mask=attention_mask,
            position_ids=position_ids,
            start_pos=start_pos,
        )

        I_comp, index_valid_mask, _ = self.index_compressor(
            C_a=I_a,
            C_b=I_b,
            Z_a=IZ_a,
            Z_b=IZ_b,
            attention_mask=attention_mask,
            position_ids=position_ids,
            start_pos=start_pos,
        )

        if not torch.equal(comp_valid_mask, index_valid_mask):
            raise RuntimeError(
                "KV compressed valid mask differs from index compressed valid mask."
            )

        S = C_comp.shape[1]

        # ----------------------------------------------------
        # RoPE on compressed global keys and local keys
        # ----------------------------------------------------
        if self.rope is not None:
            K_global_all = C_comp[:, :, None, :]  # [B,S,1,Dh]
            K_global_all = self.rope(
                K_global_all,
                position_ids=comp_position_ids,
                start_pos=0,
            )
            K_global_all = K_global_all[:, :, 0, :]  # [B,S,Dh]

            K_local = C_local[:, :, None, :]  # [B,T,1,Dh]
            K_local = self.rope(
                K_local,
                position_ids=position_ids,
                start_pos=start_pos,
            )
            K_local = K_local[:, :, 0, :]  # [B,T,Dh]
        else:
            K_global_all = C_comp
            K_local = C_local

        V_global_all = C_comp
        V_local = C_local

        # ----------------------------------------------------
        # Lightning indexer top-k selection
        # ----------------------------------------------------
        if need_weights:
            topk_indices, topk_scores, topk_mask, index_scores = self.indexer(
                index_q=index_q,
                index_weights=index_weights,
                I_comp=I_comp,
                comp_valid_mask=comp_valid_mask,
                need_scores=True,
            )
        else:
            topk_indices, topk_scores, topk_mask = self.indexer(
                index_q=index_q,
                index_weights=index_weights,
                I_comp=I_comp,
                comp_valid_mask=comp_valid_mask,
                need_scores=False,
            )
            index_scores = None

        K_eff = topk_indices.shape[-1]

        # ----------------------------------------------------
        # Gather selected global K/V
        # ----------------------------------------------------
        K_selected = self._gather_selected(K_global_all, topk_indices)   # [B,T,K,Dh]
        V_selected = self._gather_selected(V_global_all, topk_indices)   # [B,T,K,Dh]

        # ----------------------------------------------------
        # Attention scores
        # ----------------------------------------------------
        q = q  # [B,T,H,Dh]

        scores_parts = []
        allowed_parts = []

        # -------------------------
        # Optional attention sink
        # -------------------------
        if self.use_attention_sink:
            K_sink = self.sink_k.expand(B, -1, -1)  # [B,1,Dh]
            scores_sink = torch.einsum(
                "bthd,bsd->bhts",
                q,
                K_sink,
            ) / math.sqrt(self.head_dim)             # [B,H,T,1]

            sink_allowed = torch.ones(
                B,
                self.n_heads,
                T,
                1,
                device=x.device,
                dtype=torch.bool,
            )

            scores_parts.append(scores_sink)
            allowed_parts.append(sink_allowed)

        # -------------------------
        # Sparse selected global scores
        # -------------------------
        scores_global = torch.einsum(
            "bthd,btkd->bhtk",
            q,
            K_selected,
        ) / math.sqrt(self.head_dim)                 # [B,H,T,K]

        # Canonical default: indexer chooses selected blocks only. It does not
        # bias the core attention logits unless explicitly enabled.
        if self.use_indexer_score_bias:
            scores_global = scores_global + topk_scores[:, None, :, :].to(
                dtype=scores_global.dtype
            )

        global_allowed = topk_mask[:, None, :, :].expand(B, self.n_heads, T, K_eff)
        # [B,H,T,K]

        scores_parts.append(scores_global)
        allowed_parts.append(global_allowed)

        # -------------------------
        # Local exact scores
        # -------------------------
        scores_local = torch.einsum(
            "bthd,bsd->bhts",
            q,
            K_local,
        ) / math.sqrt(self.head_dim)                 # [B,H,T,T]

        local_allowed = self._build_local_allowed_mask(
            T=T,
            device=x.device,
        )
        local_allowed = local_allowed[None, None, :, :]  # [1,1,T,T]

        if attention_mask is not None:
            local_key_allowed = attention_mask[:, None, None, :].to(
                device=x.device,
                dtype=torch.bool,
            )
            local_allowed = local_allowed & local_key_allowed

        local_allowed = local_allowed.expand(B, self.n_heads, T, T)

        scores_parts.append(scores_local)
        allowed_parts.append(local_allowed)

        # ----------------------------------------------------
        # Combined sink + sparse global + local softmax
        # ----------------------------------------------------
        scores = torch.cat(scores_parts, dim=-1)          # [B,H,T,N]
        allowed = torch.cat(allowed_parts, dim=-1)        # [B,H,T,N]

        weights = safe_masked_softmax(
            scores=scores,
            allowed_mask=allowed,
            dim=-1,
        )

        weights = self.attention_dropout(weights)

        # ----------------------------------------------------
        # Split attention weights
        # ----------------------------------------------------
        offset = 0

        if self.use_attention_sink:
            weights_sink = weights[..., offset:offset + 1]  # [B,H,T,1]
            offset += 1
        else:
            weights_sink = None

        weights_global = weights[..., offset:offset + K_eff]  # [B,H,T,K]
        offset += K_eff

        weights_local = weights[..., offset:]                 # [B,H,T,T]

        # ----------------------------------------------------
        # Context
        # ----------------------------------------------------
        context = torch.zeros(
            B,
            self.n_heads,
            T,
            self.head_dim,
            device=x.device,
            dtype=x.dtype,
        )

        if self.use_attention_sink:
            V_sink = self.sink_v.expand(B, -1, -1)  # [B,1,Dh]
            context_sink = torch.einsum(
                "bhts,bsd->bhtd",
                weights_sink,
                V_sink,
            )
            context = context + context_sink

        context_global = torch.einsum(
            "bhtk,btkd->bhtd",
            weights_global,
            V_selected,
        )

        context_local = torch.einsum(
            "bhts,bsd->bhtd",
            weights_local,
            V_local,
        )

        context = context + context_global + context_local  # [B,H,T,Dh]

        # ----------------------------------------------------
        # Merge heads + output projection
        # ----------------------------------------------------
        context = context.transpose(1, 2).contiguous()  # [B,T,H,Dh]

        if self.use_grouped_output_projection:
            out = self.out_proj(context)               # [B,T,D]
        else:
            context = context.view(B, T, self.inner_dim)
            out = self.out_proj(context)               # [B,T,D]

        out = self.residual_dropout(out)

        if need_weights:
            aux = {
                "global_attn_weights": weights_global,
                "local_attn_weights": weights_local,
                "topk_indices": topk_indices,
                "topk_scores": topk_scores,
                "topk_mask": topk_mask,
                "compressed_valid_mask": comp_valid_mask,
                "compressed_position_ids": comp_position_ids,
                "index_scores": index_scores,
            }

            if self.use_attention_sink:
                aux["sink_attn_weights"] = weights_sink

            return out, aux

        return out

In [48]:
# ============================================================
# Canonical CSA smoke test
# ============================================================

csa_config = CSAConfig(
    d_model=256,
    n_heads=4,
    head_dim=64,

    compression_factor=8,
    top_k=4,
    window_size=32,

    indexer_dim=32,
    n_indexer_heads=2,
    query_compression_dim=64,

    attention_dropout=0.0,
    residual_dropout=0.0,
    use_bias=False,

    use_rope=True,
    rope_theta=10000.0,
    rotary_dim=64,

    max_seq_len=512,
    init_std=0.02,
)

csa = CSAAttention(csa_config)

B, T, D = 2, 128, 256
x = torch.randn(B, T, D)
attention_mask = torch.ones(B, T, dtype=torch.long)

out, aux = csa(
    x,
    attention_mask=attention_mask,
    position_ids=None,
    start_pos=0,
    need_weights=True,
)

S = math.ceil(T / csa_config.compression_factor)
K = min(csa_config.top_k, S)

print("out:", out.shape)
print("global_attn_weights:", aux["global_attn_weights"].shape)
print("local_attn_weights:", aux["local_attn_weights"].shape)
print("topk_indices:", aux["topk_indices"].shape)
print("topk_mask:", aux["topk_mask"].shape)
print("compressed_valid_mask:", aux["compressed_valid_mask"].shape)
print("index_scores:", aux["index_scores"].shape)

assert out.shape == (B, T, D)
assert aux["global_attn_weights"].shape == (B, csa_config.n_heads, T, K)
assert aux["local_attn_weights"].shape == (B, csa_config.n_heads, T, T)
assert aux["topk_indices"].shape == (B, T, K)
assert aux["topk_mask"].shape == (B, T, K)
assert aux["compressed_valid_mask"].shape == (B, S)
assert aux["index_scores"].shape == (B, T, S)

assert torch.isfinite(out).all()
assert torch.isfinite(aux["global_attn_weights"]).all()
assert torch.isfinite(aux["local_attn_weights"]).all()
assert torch.isfinite(aux["index_scores"]).all()

print("Canonical CSAAttention smoke test OK.")

out: torch.Size([2, 128, 256])
global_attn_weights: torch.Size([2, 4, 128, 4])
local_attn_weights: torch.Size([2, 4, 128, 128])
topk_indices: torch.Size([2, 128, 4])
topk_mask: torch.Size([2, 128, 4])
compressed_valid_mask: torch.Size([2, 16])
index_scores: torch.Size([2, 128, 16])
Canonical CSAAttention smoke test OK.


In [49]:
# ============================================================
# Top-k causal check
# ============================================================

m = csa_config.compression_factor
topk_indices = aux["topk_indices"]
topk_mask = aux["topk_mask"]

for t in range(T):
    query_block = t // m

    selected = topk_indices[:, t, :]
    valid = topk_mask[:, t, :]

    if valid.any():
        assert (selected[valid] < query_block).all()

print("Canonical CSA top-k causal check OK.")

Canonical CSA top-k causal check OK.


## Test for CSA

In [50]:
# @title
# ============================================================
# Helpers - REPLACE make_csa_config with this version
# ============================================================

def make_csa_config(**overrides):
    cfg = dict(
        d_model=64,
        n_heads=4,
        head_dim=16,

        compression_factor=4,
        top_k=3,
        window_size=4,

        indexer_dim=8,
        n_indexer_heads=2,
        query_compression_dim=16,

        attention_dropout=0.0,
        residual_dropout=0.0,
        use_bias=True,

        use_rope=True,
        rope_theta=10000.0,
        rotary_dim=16,

        max_seq_len=128,
        init_std=0.02,

        # Canonical additions
        use_attention_sink=True,
        use_grouped_output_projection=True,
        output_projection_groups=4,
        use_indexer_score_bias=False,
        use_separate_local_kv=True,
    )
    cfg.update(overrides)
    return CSAConfig(**cfg)


def make_csa(**overrides):
    return CSAAttention(make_csa_config(**overrides))


def make_csa_input(B=2, T=16, D=64, dtype=torch.float32, device="cpu"):
    return torch.randn(B, T, D, dtype=dtype, device=device)


def make_overlapped_compressor(m=4, dim=8):
    return CSAOverlappedCompressor(
        compression_factor=m,
        dim=dim,
        init_std=0.02,
    )


def make_indexer(m=4, top_k=3):
    return CSALightningIndexer(
        compression_factor=m,
        top_k=top_k,
    )


# ============================================================
# A. Config tests
# ============================================================

def test_valid_csa_config_builds():
    config = CSAConfig(
        d_model=256,
        n_heads=4,
        head_dim=64,
        compression_factor=8,
        top_k=8,
        window_size=32,
        indexer_dim=32,
        n_indexer_heads=2,
        query_compression_dim=64,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_bias=False,
        use_rope=True,
        rope_theta=10000.0,
        rotary_dim=64,
        max_seq_len=512,
        init_std=0.02,

        # Canonical additions
        use_attention_sink=True,
        use_grouped_output_projection=True,
        output_projection_groups=4,
        use_indexer_score_bias=False,
        use_separate_local_kv=True,
    )

    csa = CSAAttention(config)

    assert csa.d_model == 256
    assert csa.n_heads == 4
    assert csa.head_dim == 64
    assert csa.compression_factor == 8
    assert csa.top_k == 8
    assert csa.window_size == 32
    assert csa.indexer_dim == 32
    assert csa.n_indexer_heads == 2
    assert csa.query_compression_dim == 64

    assert csa.use_attention_sink is True
    assert csa.use_grouped_output_projection is True
    assert csa.use_indexer_score_bias is False
    assert csa.use_separate_local_kv is True

@pytest.mark.parametrize(
    "kwargs",
    [
        {"output_projection_groups": 0},
        {"output_projection_groups": -1},
        {"output_projection_groups": 3},  # 3 does not divide n_heads=4
    ],
)
def test_invalid_csa_output_projection_groups_raise(kwargs):
    with pytest.raises(ValueError):
        CSAAttention(
            make_csa_config(
                use_grouped_output_projection=True,
                **kwargs,
            )
        )


@pytest.mark.parametrize(
    "kwargs",
    [
        {"d_model": 0},
        {"d_model": -1},
        {"n_heads": 0},
        {"n_heads": -1},
        {"head_dim": 0},
        {"head_dim": -1},
        {"head_dim": 15},  # 4 * 15 != 64
    ],
)
def test_invalid_model_dims_raise(kwargs):
    with pytest.raises(ValueError):
        CSAAttention(make_csa_config(**kwargs))


@pytest.mark.parametrize(
    "kwargs",
    [
        {"compression_factor": 0},
        {"compression_factor": -1},
        {"top_k": 0},
        {"top_k": -1},
        {"window_size": 0},
        {"window_size": -1},
        {"indexer_dim": 0},
        {"indexer_dim": -1},
        {"n_indexer_heads": 0},
        {"n_indexer_heads": -1},
        {"query_compression_dim": 0},
        {"query_compression_dim": -1},
    ],
)
def test_invalid_csa_hyperparams_raise(kwargs):
    with pytest.raises(ValueError):
        CSAAttention(make_csa_config(**kwargs))


@pytest.mark.parametrize(
    "kwargs",
    [
        {"attention_dropout": -0.1},
        {"attention_dropout": 1.0},
        {"attention_dropout": 1.5},
        {"residual_dropout": -0.1},
        {"residual_dropout": 1.0},
        {"residual_dropout": 1.5},
        {"init_std": 0.0},
        {"init_std": -0.01},
        {"rope_theta": 0.0},
        {"rope_theta": -1.0},
        {"rotary_dim": 0},
        {"rotary_dim": -1},
        {"rotary_dim": 17},
        {"rotary_dim": 32},
    ],
)
def test_invalid_dropout_rope_init_raise(kwargs):
    with pytest.raises(ValueError):
        CSAAttention(make_csa_config(**kwargs))


# ============================================================
# B. CSAOverlappedCompressor tests
# ============================================================

def test_overlapped_compressor_output_shape_exact_multiple():
    B, T, D = 2, 32, 8
    m = 8

    compressor = make_overlapped_compressor(m=m, dim=D)

    C_a = torch.randn(B, T, D)
    C_b = torch.randn(B, T, D)
    Z_a = torch.randn(B, T, D)
    Z_b = torch.randn(B, T, D)

    C_comp, valid_mask, pos = compressor(C_a, C_b, Z_a, Z_b)

    assert C_comp.shape == (B, 4, D)
    assert valid_mask.shape == (B, 4)
    assert pos.shape == (4,)


def test_overlapped_compressor_output_shape_non_exact_multiple():
    B, T, D = 2, 30, 8
    m = 8

    compressor = make_overlapped_compressor(m=m, dim=D)

    C_a = torch.randn(B, T, D)
    C_b = torch.randn(B, T, D)
    Z_a = torch.randn(B, T, D)
    Z_b = torch.randn(B, T, D)

    C_comp, valid_mask, pos = compressor(C_a, C_b, Z_a, Z_b)

    S = math.ceil(T / m)

    assert S == 4
    assert C_comp.shape == (B, S, D)
    assert valid_mask.shape == (B, S)
    assert pos.shape == (S,)


def test_first_block_uses_only_a_branch():
    B, T, D = 1, 16, 4
    m = 4

    compressor = make_overlapped_compressor(m=m, dim=D)

    C_a = torch.randn(B, T, D)
    C_b_1 = torch.zeros(B, T, D)
    C_b_2 = torch.full((B, T, D), 1_000_000.0)

    Z_a = torch.zeros(B, T, D)
    Z_b = torch.zeros(B, T, D)

    out_1, _, _ = compressor(C_a, C_b_1, Z_a, Z_b)
    out_2, _, _ = compressor(C_a, C_b_2, Z_a, Z_b)

    assert torch.allclose(out_1[:, 0, :], out_2[:, 0, :], atol=1e-6, rtol=1e-6)


def test_second_block_uses_current_a_and_previous_b():
    B, T, D = 1, 16, 4
    m = 4

    compressor = make_overlapped_compressor(m=m, dim=D)

    C_a = torch.randn(B, T, D)
    C_b = torch.randn(B, T, D)
    Z_a = torch.randn(B, T, D)
    Z_b = torch.randn(B, T, D)

    C_a_2 = C_a.clone()
    C_b_2 = C_b.clone()
    Z_a_2 = Z_a.clone()
    Z_b_2 = Z_b.clone()

    # Block i=1 uses A current [4:8] and B previous [0:4].
    # Modify outside those regions.
    C_a_2[:, :4, :] += 1000.0
    C_a_2[:, 8:, :] += 1000.0
    Z_a_2[:, :4, :] += 1000.0
    Z_a_2[:, 8:, :] += 1000.0

    C_b_2[:, 4:, :] += 1000.0
    Z_b_2[:, 4:, :] += 1000.0

    out_1, _, _ = compressor(C_a, C_b, Z_a, Z_b)
    out_2, _, _ = compressor(C_a_2, C_b_2, Z_a_2, Z_b_2)

    assert torch.allclose(out_1[:, 1, :], out_2[:, 1, :], atol=1e-5, rtol=1e-5)


def test_overlapped_compressor_padding_block_zero():
    B, T, D = 1, 16, 4
    m = 4

    compressor = make_overlapped_compressor(m=m, dim=D)

    C_a = torch.randn(B, T, D)
    C_b = torch.randn(B, T, D)
    Z_a = torch.randn(B, T, D)
    Z_b = torch.randn(B, T, D)

    attention_mask = torch.ones(B, T, dtype=torch.long)

    # For block i=2:
    # A = [8:12], B = [4:8]. Make both all padding.
    attention_mask[:, 4:12] = 0

    C_comp, valid_mask, _ = compressor(
        C_a,
        C_b,
        Z_a,
        Z_b,
        attention_mask=attention_mask,
    )

    assert valid_mask[0, 2] == 0
    assert torch.allclose(C_comp[0, 2], torch.zeros_like(C_comp[0, 2]), atol=0.0, rtol=0.0)


def test_overlapped_compressor_ignores_padding_tokens():
    B, T, D = 1, 8, 4
    m = 4

    compressor = make_overlapped_compressor(m=m, dim=D)

    C_a = torch.ones(B, T, D)
    C_b = torch.ones(B, T, D)
    Z_a = torch.zeros(B, T, D)
    Z_b = torch.zeros(B, T, D)

    C_a[:, 3, :] = 1_000_000.0
    attention_mask = torch.ones(B, T, dtype=torch.long)
    attention_mask[:, 3] = 0

    C_comp, valid_mask, _ = compressor(
        C_a,
        C_b,
        Z_a,
        Z_b,
        attention_mask=attention_mask,
    )

    assert valid_mask[0, 0] == 1
    assert C_comp[0, 0].max() < 10.0


def test_overlapped_compressor_position_ids_none():
    B, T, D = 1, 16, 4
    m = 4
    start_pos = 10

    compressor = make_overlapped_compressor(m=m, dim=D)

    C_a = torch.randn(B, T, D)
    C_b = torch.randn(B, T, D)
    Z_a = torch.randn(B, T, D)
    Z_b = torch.randn(B, T, D)

    _, _, pos = compressor(
        C_a,
        C_b,
        Z_a,
        Z_b,
        start_pos=start_pos,
    )

    expected = torch.tensor([13, 17, 21, 25], dtype=torch.long)

    assert torch.equal(pos.cpu(), expected)


def test_overlapped_compressor_position_ids_T():
    B, T, D = 1, 16, 4
    m = 4

    compressor = make_overlapped_compressor(m=m, dim=D)

    C_a = torch.randn(B, T, D)
    C_b = torch.randn(B, T, D)
    Z_a = torch.randn(B, T, D)
    Z_b = torch.randn(B, T, D)

    position_ids = torch.arange(100, 100 + T)

    _, _, pos = compressor(
        C_a,
        C_b,
        Z_a,
        Z_b,
        position_ids=position_ids,
    )

    expected = position_ids[torch.tensor([3, 7, 11, 15])]

    assert torch.equal(pos.cpu(), expected.cpu())


def test_overlapped_compressor_position_ids_BT():
    B, T, D = 2, 16, 4
    m = 4

    compressor = make_overlapped_compressor(m=m, dim=D)

    C_a = torch.randn(B, T, D)
    C_b = torch.randn(B, T, D)
    Z_a = torch.randn(B, T, D)
    Z_b = torch.randn(B, T, D)

    position_ids = torch.stack(
        [
            torch.arange(100, 100 + T),
            torch.arange(200, 200 + T),
        ],
        dim=0,
    )

    _, _, pos = compressor(
        C_a,
        C_b,
        Z_a,
        Z_b,
        position_ids=position_ids,
    )

    expected = position_ids[:, torch.tensor([3, 7, 11, 15])]

    assert pos.shape == (B, 4)
    assert torch.equal(pos.cpu(), expected.cpu())


def test_overlapped_compressor_backward():
    B, T, D = 2, 16, 4
    m = 4

    compressor = make_overlapped_compressor(m=m, dim=D)

    C_a = torch.randn(B, T, D, requires_grad=True)
    C_b = torch.randn(B, T, D, requires_grad=True)
    Z_a = torch.randn(B, T, D, requires_grad=True)
    Z_b = torch.randn(B, T, D, requires_grad=True)

    C_comp, _, _ = compressor(C_a, C_b, Z_a, Z_b)

    loss = C_comp.sum()
    loss.backward()

    assert C_a.grad is not None
    assert C_b.grad is not None
    assert Z_a.grad is not None
    assert Z_b.grad is not None
    assert compressor.bias_a.grad is not None
    assert compressor.bias_b.grad is not None

    assert torch.isfinite(C_a.grad).all()
    assert torch.isfinite(C_b.grad).all()
    assert torch.isfinite(Z_a.grad).all()
    assert torch.isfinite(Z_b.grad).all()
    assert torch.isfinite(compressor.bias_a.grad).all()
    assert torch.isfinite(compressor.bias_b.grad).all()


# ============================================================
# C. CSALightningIndexer tests
# ============================================================

def test_indexer_output_shapes():
    B, T, H_i, I, S = 2, 16, 2, 8, 5
    top_k = 3

    indexer = make_indexer(m=4, top_k=top_k)

    index_q = torch.randn(B, T, H_i, I)
    index_weights = torch.randn(B, T, H_i)
    I_comp = torch.randn(B, S, I)
    valid_mask = torch.ones(B, S, dtype=torch.bool)

    topk_indices, topk_scores, topk_mask = indexer(
        index_q,
        index_weights,
        I_comp,
        valid_mask,
    )

    K = min(top_k, S)

    assert topk_indices.shape == (B, T, K)
    assert topk_scores.shape == (B, T, K)
    assert topk_mask.shape == (B, T, K)


def test_indexer_scores_shape_when_requested():
    B, T, H_i, I, S = 2, 16, 2, 8, 5

    indexer = make_indexer(m=4, top_k=3)

    index_q = torch.randn(B, T, H_i, I)
    index_weights = torch.randn(B, T, H_i)
    I_comp = torch.randn(B, S, I)
    valid_mask = torch.ones(B, S, dtype=torch.bool)

    topk_indices, topk_scores, topk_mask, index_scores = indexer(
        index_q,
        index_weights,
        I_comp,
        valid_mask,
        need_scores=True,
    )

    assert index_scores.shape == (B, T, S)


def test_indexer_respects_causality():
    B, T, H_i, I, S = 2, 16, 2, 8, 5
    m = 4

    indexer = make_indexer(m=m, top_k=3)

    index_q = torch.randn(B, T, H_i, I)
    index_weights = torch.randn(B, T, H_i)
    I_comp = torch.randn(B, S, I)
    valid_mask = torch.ones(B, S, dtype=torch.bool)

    topk_indices, _, topk_mask = indexer(
        index_q,
        index_weights,
        I_comp,
        valid_mask,
    )

    for t in range(T):
        query_block = t // m
        selected = topk_indices[:, t, :]
        valid = topk_mask[:, t, :]

        if valid.any():
            assert (selected[valid] < query_block).all()


def test_indexer_first_block_has_no_valid_topk():
    B, T, H_i, I, S = 2, 16, 2, 8, 5
    m = 4

    indexer = make_indexer(m=m, top_k=3)

    index_q = torch.randn(B, T, H_i, I)
    index_weights = torch.randn(B, T, H_i)
    I_comp = torch.randn(B, S, I)
    valid_mask = torch.ones(B, S, dtype=torch.bool)

    _, _, topk_mask = indexer(index_q, index_weights, I_comp, valid_mask)

    assert not topk_mask[:, :m, :].any()


def test_indexer_respects_compressed_valid_mask():
    B, T, H_i, I, S = 2, 16, 2, 8, 5
    m = 4

    indexer = make_indexer(m=m, top_k=3)

    index_q = torch.randn(B, T, H_i, I)
    index_weights = torch.randn(B, T, H_i)
    I_comp = torch.randn(B, S, I)
    valid_mask = torch.ones(B, S, dtype=torch.bool)

    valid_mask[:, 1] = False

    topk_indices, _, topk_mask = indexer(
        index_q,
        index_weights,
        I_comp,
        valid_mask,
    )

    valid_selected = topk_indices[topk_mask]

    assert not (valid_selected == 1).any()


def test_indexer_topk_larger_than_num_blocks():
    B, T, H_i, I, S = 2, 16, 2, 8, 3

    indexer = make_indexer(m=4, top_k=10)

    index_q = torch.randn(B, T, H_i, I)
    index_weights = torch.randn(B, T, H_i)
    I_comp = torch.randn(B, S, I)
    valid_mask = torch.ones(B, S, dtype=torch.bool)

    topk_indices, topk_scores, topk_mask = indexer(
        index_q,
        index_weights,
        I_comp,
        valid_mask,
    )

    assert topk_indices.shape[-1] == S
    assert topk_scores.shape[-1] == S
    assert topk_mask.shape[-1] == S


def test_indexer_backward():
    B, T, H_i, I, S = 2, 16, 2, 8, 5

    indexer = make_indexer(m=4, top_k=3)

    index_q = torch.randn(B, T, H_i, I, requires_grad=True)
    index_weights = torch.randn(B, T, H_i, requires_grad=True)
    I_comp = torch.randn(B, S, I, requires_grad=True)
    valid_mask = torch.ones(B, S, dtype=torch.bool)

    _, topk_scores, topk_mask = indexer(
        index_q,
        index_weights,
        I_comp,
        valid_mask,
    )

    loss = topk_scores[topk_mask].sum()
    loss.backward()

    assert index_q.grad is not None
    assert index_weights.grad is not None
    assert I_comp.grad is not None

    assert torch.isfinite(index_q.grad).all()
    assert torch.isfinite(index_weights.grad).all()
    assert torch.isfinite(I_comp.grad).all()


# ============================================================
# D. CSAAttention forward tests
# ============================================================

def test_csa_output_shape_matches_input():
    csa = make_csa()
    x = make_csa_input()

    out = csa(x)

    assert out.shape == x.shape


@pytest.mark.parametrize(
    "bad_x",
    [
        torch.randn(16, 64),
        torch.randn(2, 16, 64, 1),
    ],
)
def test_csa_rejects_wrong_input_rank(bad_x):
    csa = make_csa()

    with pytest.raises(ValueError):
        csa(bad_x)


def test_csa_rejects_wrong_hidden_size():
    csa = make_csa(d_model=64)

    x = torch.randn(2, 16, 32)

    with pytest.raises(ValueError):
        csa(x)


def test_csa_rejects_too_long_sequence():
    csa = make_csa(max_seq_len=8)

    x = torch.randn(2, 9, 64)

    with pytest.raises(ValueError):
        csa(x)


def test_csa_need_weights_returns_aux():
    csa = make_csa(
        compression_factor=4,
        top_k=3,
        use_attention_sink=True,
        use_grouped_output_projection=True,
        output_projection_groups=4,
    )
    x = make_csa_input(B=2, T=16, D=64)

    out, aux = csa(x, need_weights=True)

    B, T, D = x.shape
    S = math.ceil(T / csa.compression_factor)
    K = min(csa.top_k, S)

    assert out.shape == x.shape

    assert aux["global_attn_weights"].shape == (B, csa.n_heads, T, K)
    assert aux["local_attn_weights"].shape == (B, csa.n_heads, T, T)
    assert aux["topk_indices"].shape == (B, T, K)
    assert aux["topk_scores"].shape == (B, T, K)
    assert aux["topk_mask"].shape == (B, T, K)
    assert aux["compressed_valid_mask"].shape == (B, S)
    assert aux["index_scores"].shape == (B, T, S)

    assert "sink_attn_weights" in aux
    assert aux["sink_attn_weights"].shape == (B, csa.n_heads, T, 1)

    assert "compressed_position_ids" in aux
    assert aux["compressed_position_ids"].shape == (S,)


# ============================================================
# E. Compression inside CSAAttention
# ============================================================

def test_csa_uses_overlapped_compressor_for_kv():
    csa = make_csa()

    assert hasattr(csa.kv_compressor, "bias_a")
    assert hasattr(csa.kv_compressor, "bias_b")


def test_csa_uses_separate_index_compressor():
    csa = make_csa(head_dim=16, indexer_dim=8)

    assert csa.kv_compressor is not csa.index_compressor
    assert csa.kv_compressor.dim == csa.head_dim
    assert csa.index_compressor.dim == csa.indexer_dim

def test_csa_has_separate_local_kv_projection_when_enabled():
    csa = make_csa(use_separate_local_kv=True)

    assert hasattr(csa, "local_kv_proj")
    assert csa.local_kv_proj is not None
    assert csa.local_kv_proj.out_features == csa.head_dim


def test_csa_can_disable_separate_local_kv_projection():
    csa = make_csa(use_separate_local_kv=False)

    assert hasattr(csa, "local_kv_proj")
    assert csa.local_kv_proj is None

# ============================================================
# F. Global sparse causality tests
# ============================================================

def test_csa_topk_blocks_current_and_future_blocks():
    m = 4
    csa = make_csa(compression_factor=m, top_k=3)
    csa.eval()

    x = make_csa_input(B=2, T=16, D=64)

    _, aux = csa(x, need_weights=True)

    topk_indices = aux["topk_indices"]
    topk_mask = aux["topk_mask"]

    for t in range(x.shape[1]):
        query_block = t // m
        selected = topk_indices[:, t, :]
        valid = topk_mask[:, t, :]

        if valid.any():
            assert (selected[valid] < query_block).all()


def test_csa_first_block_has_zero_global_weights():
    m = 4
    csa = make_csa(compression_factor=m, top_k=3)
    csa.eval()

    x = make_csa_input(B=2, T=16, D=64)

    _, aux = csa(x, need_weights=True)

    assert not aux["topk_mask"][:, :m, :].any()

    assert torch.allclose(
        aux["global_attn_weights"][:, :, :m, :],
        torch.zeros_like(aux["global_attn_weights"][:, :, :m, :]),
        atol=0.0,
        rtol=0.0,
    )


def test_csa_global_weights_zero_for_invalid_topk():
    csa = make_csa(compression_factor=4, top_k=3)
    csa.eval()

    x = make_csa_input(B=2, T=16, D=64)

    _, aux = csa(x, need_weights=True)

    invalid = ~aux["topk_mask"]  # [B,T,K]
    weights = aux["global_attn_weights"]  # [B,H,T,K]

    invalid_weights = weights.masked_select(invalid[:, None, :, :].expand_as(weights))

    assert torch.allclose(
        invalid_weights,
        torch.zeros_like(invalid_weights),
        atol=0.0,
        rtol=0.0,
    )


# ============================================================
# G. Local sliding-window tests
# ============================================================

def test_csa_local_window_is_causal():
    csa = make_csa(window_size=4)
    csa.eval()

    B, T, D = 2, 16, 64
    x = make_csa_input(B=B, T=T, D=D)

    _, aux = csa(x, need_weights=True)

    local_weights = aux["local_attn_weights"]

    future_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
    future_weights = local_weights[:, :, future_mask]

    assert torch.allclose(
        future_weights,
        torch.zeros_like(future_weights),
        atol=0.0,
        rtol=0.0,
    )


def test_csa_local_window_limits_past_context():
    W = 4
    csa = make_csa(window_size=W)
    csa.eval()

    B, T, D = 2, 16, 64
    x = make_csa_input(B=B, T=T, D=D)

    _, aux = csa(x, need_weights=True)

    local_weights = aux["local_attn_weights"]

    q_pos = torch.arange(T)[:, None]
    k_pos = torch.arange(T)[None, :]
    too_old = (q_pos - k_pos) >= W

    too_old_weights = local_weights[:, :, too_old]

    assert torch.allclose(
        too_old_weights,
        torch.zeros_like(too_old_weights),
        atol=0.0,
        rtol=0.0,
    )


def test_csa_changing_future_tokens_does_not_change_past_outputs():
    m = 4
    csa = make_csa(
        compression_factor=m,
        window_size=4,
        attention_dropout=0.0,
        residual_dropout=0.0,
    )
    csa.eval()

    B, T, D = 2, 16, 64
    cut = 8

    x1 = make_csa_input(B=B, T=T, D=D)
    x2 = x1.clone()
    x2[:, cut:, :] = torch.randn_like(x2[:, cut:, :])

    out1 = csa(x1)
    out2 = csa(x2)

    assert torch.allclose(
        out1[:, :cut, :],
        out2[:, :cut, :],
        atol=1e-5,
        rtol=1e-5,
    )


# ============================================================
# H. attention_mask tests
# ============================================================

def test_csa_attention_mask_blocks_padding_local_keys():
    csa = make_csa(window_size=8)
    csa.eval()

    B, T, D = 2, 16, 64
    x = make_csa_input(B=B, T=T, D=D)

    attention_mask = torch.ones(B, T, dtype=torch.long)
    attention_mask[0, 5] = 0
    attention_mask[1, 7] = 0

    _, aux = csa(x, attention_mask=attention_mask, need_weights=True)

    local_weights = aux["local_attn_weights"]

    assert torch.allclose(
        local_weights[0, :, :, 5],
        torch.zeros_like(local_weights[0, :, :, 5]),
        atol=0.0,
        rtol=0.0,
    )

    assert torch.allclose(
        local_weights[1, :, :, 7],
        torch.zeros_like(local_weights[1, :, :, 7]),
        atol=0.0,
        rtol=0.0,
    )


def test_csa_attention_mask_blocks_padding_compressed_blocks():
    m = 4
    csa = make_csa(compression_factor=m, top_k=3)
    csa.eval()

    B, T, D = 2, 16, 64
    x = make_csa_input(B=B, T=T, D=D)

    attention_mask = torch.ones(B, T, dtype=torch.long)

    # For compressed block 2:
    # A = [8:12], B = [4:8]. Make both invalid.
    attention_mask[0, 4:12] = 0

    _, aux = csa(x, attention_mask=attention_mask, need_weights=True)

    assert aux["compressed_valid_mask"][0, 2] == 0

    selected = aux["topk_indices"][0]
    valid = aux["topk_mask"][0]

    assert not (selected[valid] == 2).any()


def test_csa_attention_mask_shape_validation_accepts_BT():
    csa = make_csa()

    x = make_csa_input(B=2, T=16, D=64)
    attention_mask = torch.ones(2, 16)

    out = csa(x, attention_mask=attention_mask)

    assert out.shape == x.shape


@pytest.mark.parametrize(
    "bad_mask",
    [
        torch.ones(16),
        torch.ones(2, 16, 1),
        torch.ones(2, 17),
    ],
)
def test_csa_attention_mask_shape_validation_rejects_bad_shapes(bad_mask):
    csa = make_csa()

    x = make_csa_input(B=2, T=16, D=64)

    with pytest.raises(ValueError):
        csa(x, attention_mask=bad_mask)


# ============================================================
# I. Combined softmax tests
# ============================================================

def test_csa_sink_plus_global_plus_local_weights_sum_to_one():
    csa = make_csa(
        attention_dropout=0.0,
        residual_dropout=0.0,
        window_size=4,
        use_attention_sink=True,
    )
    csa.eval()

    x = make_csa_input(B=2, T=16, D=64)

    _, aux = csa(x, need_weights=True)

    sink_sum = aux["sink_attn_weights"].sum(dim=-1)
    global_sum = aux["global_attn_weights"].sum(dim=-1)
    local_sum = aux["local_attn_weights"].sum(dim=-1)

    total = sink_sum + global_sum + local_sum

    assert torch.allclose(
        total,
        torch.ones_like(total),
        atol=1e-6,
        rtol=1e-6,
    )


def test_csa_global_plus_local_weights_sum_to_one_without_sink():
    csa = make_csa(
        attention_dropout=0.0,
        residual_dropout=0.0,
        window_size=4,
        use_attention_sink=False,
    )
    csa.eval()

    x = make_csa_input(B=2, T=16, D=64)

    _, aux = csa(x, need_weights=True)

    assert "sink_attn_weights" not in aux

    global_sum = aux["global_attn_weights"].sum(dim=-1)
    local_sum = aux["local_attn_weights"].sum(dim=-1)

    total = global_sum + local_sum

    assert torch.allclose(
        total,
        torch.ones_like(total),
        atol=1e-6,
        rtol=1e-6,
    )


def test_csa_no_nan_when_no_global_blocks_available():
    m = 4
    csa = make_csa(
        compression_factor=m,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_attention_sink=True,
    )
    csa.eval()

    x = make_csa_input(B=2, T=16, D=64)

    out, aux = csa(x, need_weights=True)

    assert torch.isfinite(out).all()
    assert torch.isfinite(aux["global_attn_weights"]).all()
    assert torch.isfinite(aux["local_attn_weights"]).all()
    assert torch.isfinite(aux["sink_attn_weights"]).all()

    assert torch.allclose(
        aux["global_attn_weights"][:, :, :m, :],
        torch.zeros_like(aux["global_attn_weights"][:, :, :m, :]),
        atol=0.0,
        rtol=0.0,
    )

    # In the first compression block, sparse global attention is empty.
    # Probability mass should be assigned to sink + local.
    total_first_block = (
        aux["sink_attn_weights"][:, :, :m, :].sum(dim=-1)
        + aux["local_attn_weights"][:, :, :m, :].sum(dim=-1)
    )

    assert torch.allclose(
        total_first_block,
        torch.ones_like(total_first_block),
        atol=1e-6,
        rtol=1e-6,
    )

def test_csa_attention_sink_is_present_and_receives_mass():
    torch.manual_seed(0)

    csa = make_csa(
        use_attention_sink=True,
        attention_dropout=0.0,
        residual_dropout=0.0,
    )
    csa.eval()

    x = make_csa_input(B=2, T=16, D=64)

    _, aux = csa(x, need_weights=True)

    assert "sink_attn_weights" in aux

    sink_weights = aux["sink_attn_weights"]

    assert sink_weights.shape == (2, csa.n_heads, 16, 1)
    assert torch.isfinite(sink_weights).all()
    assert (sink_weights >= 0).all()
    assert sink_weights.sum() > 0


def test_csa_aux_returns_compressed_position_ids():
    m = 4
    csa = make_csa(
        compression_factor=m,
        use_attention_sink=True,
    )
    csa.eval()

    B, T, D = 2, 17, 64
    start_pos = 10

    x = make_csa_input(B=B, T=T, D=D)

    _, aux = csa(
        x,
        start_pos=start_pos,
        need_weights=True,
    )

    expected = torch.tensor(
        [13, 17, 21, 25, 26],
        device=x.device,
        dtype=torch.long,
    )

    assert "compressed_position_ids" in aux
    assert torch.equal(aux["compressed_position_ids"], expected)


def test_csa_grouped_output_projection_shape_and_gradients():
    torch.manual_seed(0)

    proj = GroupedOutputProjection(
        n_heads=4,
        head_dim=16,
        num_groups=4,
        bias=True,
        init_std=0.02,
    )

    x = torch.randn(2, 8, 4, 16, requires_grad=True)

    out = proj(x)

    assert out.shape == (2, 8, 64)
    assert torch.isfinite(out).all()

    loss = out.sum()
    loss.backward()

    assert x.grad is not None
    assert torch.isfinite(x.grad).all()

    for group_proj in proj.group_projs:
        assert group_proj.weight.grad is not None
        assert torch.isfinite(group_proj.weight.grad).all()

        assert group_proj.bias.grad is not None
        assert torch.isfinite(group_proj.bias.grad).all()

# ============================================================
# J. RoPE tests
# ============================================================

def test_csa_start_pos_matches_explicit_position_ids():
    csa = make_csa(
        use_rope=True,
        attention_dropout=0.0,
        residual_dropout=0.0,
    )
    csa.eval()

    B, T, D = 2, 16, 64
    start_pos = 10

    x = make_csa_input(B=B, T=T, D=D)

    out_start = csa(x, start_pos=start_pos)
    out_explicit = csa(
        x,
        position_ids=torch.arange(start_pos, start_pos + T),
    )

    assert torch.allclose(out_start, out_explicit, atol=1e-5, rtol=1e-5)


def test_csa_no_rope_when_disabled():
    csa = make_csa(
        use_rope=False,
        attention_dropout=0.0,
        residual_dropout=0.0,
    )
    csa.eval()

    B, T, D = 2, 16, 64
    x = make_csa_input(B=B, T=T, D=D)

    out1 = csa(x, start_pos=0)
    out2 = csa(x, position_ids=torch.arange(10, 10 + T), start_pos=10)

    assert torch.allclose(out1, out2, atol=1e-6, rtol=1e-6)


# ============================================================
# K. Gradient tests
# ============================================================

def test_csa_backward_computes_gradients_canonical_no_indexer_score_bias():
    """
    Canonical mode:
        use_indexer_score_bias=False

    In this mode the indexer is used for discrete top-k selection.
    Since top-k indices are not differentiable, the main loss is not expected
    to send gradients into the indexer query/key path. This is architecturally
    more faithful, but less useful for tiny end-to-end training.
    """
    csa = make_csa(
        use_bias=True,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_attention_sink=True,
        use_grouped_output_projection=True,
        output_projection_groups=4,
        use_indexer_score_bias=False,
        use_separate_local_kv=True,
    )

    x = make_csa_input(B=2, T=16, D=64)
    x.requires_grad_(True)

    out = csa(x)
    loss = out.sum()
    loss.backward()

    assert x.grad is not None
    assert torch.isfinite(x.grad).all()

    params = dict(csa.named_parameters())

    expected_grad_params = [
        "q_down_proj.weight",
        "q_up_proj.weight",
        "a_kv_proj.weight",
        "b_kv_proj.weight",
        "a_z_proj.weight",
        "b_z_proj.weight",
        "local_kv_proj.weight",
        "kv_compressor.bias_a",
        "kv_compressor.bias_b",
        "sink_k",
        "sink_v",
    ]

    for name in expected_grad_params:
        assert name in params, f"Missing parameter {name}"
        assert params[name].grad is not None, f"Missing grad for {name}"
        assert torch.isfinite(params[name].grad).all(), f"Non-finite grad for {name}"

    # Grouped output projection gradients
    assert hasattr(csa.out_proj, "group_projs")

    for proj in csa.out_proj.group_projs:
        assert proj.weight.grad is not None
        assert torch.isfinite(proj.weight.grad).all()

        if proj.bias is not None:
            assert proj.bias.grad is not None
            assert torch.isfinite(proj.bias.grad).all()


def test_csa_backward_computes_indexer_gradients_with_score_bias_enabled():
    """
    Training-friendly mode:
        use_indexer_score_bias=True

    In this mode selected top-k scores are added to global attention logits.
    This allows gradients to flow into the indexer score path.
    """
    csa = make_csa(
        use_bias=True,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_attention_sink=True,
        use_grouped_output_projection=True,
        output_projection_groups=4,
        use_indexer_score_bias=True,
        use_separate_local_kv=True,
    )

    x = make_csa_input(B=2, T=16, D=64)
    x.requires_grad_(True)

    out = csa(x)
    loss = out.sum()
    loss.backward()

    params = dict(csa.named_parameters())

    expected_indexer_grad_params = [
        "index_q_up_proj.weight",
        "index_weight_proj.weight",
        "a_index_kv_proj.weight",
        "b_index_kv_proj.weight",
        "a_index_z_proj.weight",
        "b_index_z_proj.weight",
        "index_compressor.bias_a",
        "index_compressor.bias_b",
    ]

    for name in expected_indexer_grad_params:
        assert name in params, f"Missing parameter {name}"
        assert params[name].grad is not None, f"Missing grad for {name}"
        assert torch.isfinite(params[name].grad).all(), f"Non-finite grad for {name}"


# ============================================================
# L. Interface integration
# ============================================================

def test_csa_can_replace_attention_interface():
    csa = make_csa(
        use_attention_sink=True,
        use_grouped_output_projection=True,
        output_projection_groups=4,
    )

    B, T, D = 2, 16, 64
    x = make_csa_input(B=B, T=T, D=D)

    attention_mask = torch.ones(B, T, dtype=torch.long)
    position_ids = torch.arange(T)

    out, aux = csa(
        x,
        attention_mask=attention_mask,
        position_ids=position_ids,
        start_pos=0,
        need_weights=True,
    )

    assert out.shape == x.shape

    required_keys = [
        "global_attn_weights",
        "local_attn_weights",
        "topk_indices",
        "topk_scores",
        "topk_mask",
        "compressed_valid_mask",
        "compressed_position_ids",
        "index_scores",
        "sink_attn_weights",
    ]

    for key in required_keys:
        assert key in aux

    assert torch.isfinite(out).all()

In [51]:
# ============================================================
# Run ONLY Canonical CSA tests in Jupyter
# Handles pytest-style parametrized tests manually
# ============================================================

csa_plain_tests = [
    # --------------------------------------------------------
    # A. Config
    # --------------------------------------------------------
    "test_valid_csa_config_builds",

    # --------------------------------------------------------
    # B. CSAOverlappedCompressor
    # --------------------------------------------------------
    "test_overlapped_compressor_output_shape_exact_multiple",
    "test_overlapped_compressor_output_shape_non_exact_multiple",
    "test_first_block_uses_only_a_branch",
    "test_second_block_uses_current_a_and_previous_b",
    "test_overlapped_compressor_padding_block_zero",
    "test_overlapped_compressor_ignores_padding_tokens",
    "test_overlapped_compressor_position_ids_none",
    "test_overlapped_compressor_position_ids_T",
    "test_overlapped_compressor_position_ids_BT",
    "test_overlapped_compressor_backward",

    # --------------------------------------------------------
    # C. CSALightningIndexer
    # --------------------------------------------------------
    "test_indexer_output_shapes",
    "test_indexer_scores_shape_when_requested",
    "test_indexer_respects_causality",
    "test_indexer_first_block_has_no_valid_topk",
    "test_indexer_respects_compressed_valid_mask",
    "test_indexer_topk_larger_than_num_blocks",
    "test_indexer_backward",

    # --------------------------------------------------------
    # D. CSAAttention forward
    # --------------------------------------------------------
    "test_csa_output_shape_matches_input",
    "test_csa_rejects_wrong_hidden_size",
    "test_csa_rejects_too_long_sequence",
    "test_csa_need_weights_returns_aux",

    # --------------------------------------------------------
    # E. Compression inside CSAAttention
    # --------------------------------------------------------
    "test_csa_uses_overlapped_compressor_for_kv",
    "test_csa_uses_separate_index_compressor",
    "test_csa_has_separate_local_kv_projection_when_enabled",
    "test_csa_can_disable_separate_local_kv_projection",

    # --------------------------------------------------------
    # F. Global sparse causality
    # --------------------------------------------------------
    "test_csa_topk_blocks_current_and_future_blocks",
    "test_csa_first_block_has_zero_global_weights",
    "test_csa_global_weights_zero_for_invalid_topk",

    # --------------------------------------------------------
    # G. Local sliding-window
    # --------------------------------------------------------
    "test_csa_local_window_is_causal",
    "test_csa_local_window_limits_past_context",
    "test_csa_changing_future_tokens_does_not_change_past_outputs",

    # --------------------------------------------------------
    # H. attention_mask
    # --------------------------------------------------------
    "test_csa_attention_mask_blocks_padding_local_keys",
    "test_csa_attention_mask_blocks_padding_compressed_blocks",
    "test_csa_attention_mask_shape_validation_accepts_BT",

    # --------------------------------------------------------
    # I. Combined softmax / sink / aux
    # --------------------------------------------------------
    "test_csa_sink_plus_global_plus_local_weights_sum_to_one",
    "test_csa_global_plus_local_weights_sum_to_one_without_sink",
    "test_csa_no_nan_when_no_global_blocks_available",
    "test_csa_attention_sink_is_present_and_receives_mass",
    "test_csa_aux_returns_compressed_position_ids",

    # --------------------------------------------------------
    # J. RoPE
    # --------------------------------------------------------
    "test_csa_start_pos_matches_explicit_position_ids",
    "test_csa_no_rope_when_disabled",

    # --------------------------------------------------------
    # Output projection
    # --------------------------------------------------------
    "test_csa_grouped_output_projection_shape_and_gradients",

    # --------------------------------------------------------
    # K. Gradients
    # --------------------------------------------------------
    "test_csa_backward_computes_gradients_canonical_no_indexer_score_bias",
    "test_csa_backward_computes_indexer_gradients_with_score_bias_enabled",

    # --------------------------------------------------------
    # L. Interface integration
    # --------------------------------------------------------
    "test_csa_can_replace_attention_interface",
]

csa_param_tests = [
    # --------------------------------------------------------
    # Model dimensions
    # --------------------------------------------------------
    ("test_invalid_model_dims_raise", {"kwargs": {"d_model": 0}}),
    ("test_invalid_model_dims_raise", {"kwargs": {"d_model": -1}}),
    ("test_invalid_model_dims_raise", {"kwargs": {"n_heads": 0}}),
    ("test_invalid_model_dims_raise", {"kwargs": {"n_heads": -1}}),
    ("test_invalid_model_dims_raise", {"kwargs": {"head_dim": 0}}),
    ("test_invalid_model_dims_raise", {"kwargs": {"head_dim": -1}}),
    ("test_invalid_model_dims_raise", {"kwargs": {"head_dim": 15}}),

    # --------------------------------------------------------
    # CSA hyperparameters
    # --------------------------------------------------------
    ("test_invalid_csa_hyperparams_raise", {"kwargs": {"compression_factor": 0}}),
    ("test_invalid_csa_hyperparams_raise", {"kwargs": {"compression_factor": -1}}),
    ("test_invalid_csa_hyperparams_raise", {"kwargs": {"top_k": 0}}),
    ("test_invalid_csa_hyperparams_raise", {"kwargs": {"top_k": -1}}),
    ("test_invalid_csa_hyperparams_raise", {"kwargs": {"window_size": 0}}),
    ("test_invalid_csa_hyperparams_raise", {"kwargs": {"window_size": -1}}),
    ("test_invalid_csa_hyperparams_raise", {"kwargs": {"indexer_dim": 0}}),
    ("test_invalid_csa_hyperparams_raise", {"kwargs": {"indexer_dim": -1}}),
    ("test_invalid_csa_hyperparams_raise", {"kwargs": {"n_indexer_heads": 0}}),
    ("test_invalid_csa_hyperparams_raise", {"kwargs": {"n_indexer_heads": -1}}),
    ("test_invalid_csa_hyperparams_raise", {"kwargs": {"query_compression_dim": 0}}),
    ("test_invalid_csa_hyperparams_raise", {"kwargs": {"query_compression_dim": -1}}),

    # --------------------------------------------------------
    # Dropout / RoPE / init
    # --------------------------------------------------------
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"attention_dropout": -0.1}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"attention_dropout": 1.0}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"attention_dropout": 1.5}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"residual_dropout": -0.1}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"residual_dropout": 1.0}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"residual_dropout": 1.5}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"init_std": 0.0}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"init_std": -0.01}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"rope_theta": 0.0}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"rope_theta": -1.0}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"rotary_dim": 0}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"rotary_dim": -1}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"rotary_dim": 17}}),
    ("test_invalid_dropout_rope_init_raise", {"kwargs": {"rotary_dim": 32}}),

    # --------------------------------------------------------
    # Grouped output projection config
    # --------------------------------------------------------
    ("test_invalid_csa_output_projection_groups_raise", {"kwargs": {"output_projection_groups": 0}}),
    ("test_invalid_csa_output_projection_groups_raise", {"kwargs": {"output_projection_groups": -1}}),
    ("test_invalid_csa_output_projection_groups_raise", {"kwargs": {"output_projection_groups": 3}}),

    # --------------------------------------------------------
    # Input validation
    # --------------------------------------------------------
    ("test_csa_rejects_wrong_input_rank", {"bad_x": torch.randn(16, 64)}),
    ("test_csa_rejects_wrong_input_rank", {"bad_x": torch.randn(2, 16, 64, 1)}),

    # --------------------------------------------------------
    # attention_mask validation
    # --------------------------------------------------------
    ("test_csa_attention_mask_shape_validation_rejects_bad_shapes", {"bad_mask": torch.ones(16)}),
    ("test_csa_attention_mask_shape_validation_rejects_bad_shapes", {"bad_mask": torch.ones(2, 16, 1)}),
    ("test_csa_attention_mask_shape_validation_rejects_bad_shapes", {"bad_mask": torch.ones(2, 17)}),
]

n_passed = 0

for name in csa_plain_tests:
    globals()[name]()
    n_passed += 1

for name, kwargs in csa_param_tests:
    globals()[name](**kwargs)
    n_passed += 1

print(f"OK: {n_passed} Canonical CSA tests/casos pasaron.")

OK: 87 Canonical CSA tests/casos pasaron.


---

# mHC / residuals

In [52]:
# ============================================================
# Mini DeepSeek-V4 mHC utilities
# Manifold-Constrained Hyper-Connections - canonical mini version
# ============================================================

from dataclasses import dataclass
from typing import Callable, Optional, Dict, Tuple, Union

import math
import torch
import torch.nn as nn


# ============================================================
# Sinkhorn projection
# ============================================================

def sinkhorn(
    logits: torch.Tensor,
    n_iters: int = 20,
    eps: float = 1e-6,
    fp32: bool = True,
) -> torch.Tensor:
    """
    Project logits to an approximately doubly stochastic matrix.

    This is the standard Sinkhorn-Knopp projection used for mHC:

        M^(0) = exp(logits)
        M <- row_normalize(column_normalize(M))

    The implementation stabilizes the exponentiation and optionally performs
    the normalization in fp32 for safer mixed-precision behavior.

    Args:
        logits:
            Tensor with shape [..., N, N].
        n_iters:
            Number of normalization iterations.
        eps:
            Numerical stabilizer.
        fp32:
            If True, computes the projection in fp32 and casts back.

    Returns:
        M:
            Tensor with shape [..., N, N]. Approximately:
                M >= 0
                M.sum(dim=-1) ~= 1
                M.sum(dim=-2) ~= 1
    """
    if logits.dim() < 2:
        raise ValueError(
            f"sinkhorn expects logits with at least 2 dims, got {tuple(logits.shape)}"
        )

    if logits.shape[-1] != logits.shape[-2]:
        raise ValueError(
            "sinkhorn expects square matrices in the last two dimensions, "
            f"got {tuple(logits.shape[-2:])}"
        )

    if n_iters <= 0:
        raise ValueError(f"n_iters must be > 0, got {n_iters}")

    if eps <= 0:
        raise ValueError(f"eps must be > 0, got {eps}")

    orig_dtype = logits.dtype
    work = logits.float() if fp32 else logits

    # Stabilize before exponentiation. Subtracting a shared max preserves
    # the Sinkhorn solution up to a global positive scale.
    max_val = work.amax(dim=(-1, -2), keepdim=True)
    M = torch.exp(work - max_val)

    # Paper notation applies column and row normalization iteratively.
    # Alternating row/column is equivalent up to iteration order; after enough
    # iterations both marginals approach one.
    for _ in range(n_iters):
        M = M / (M.sum(dim=-1, keepdim=True) + eps)
        M = M / (M.sum(dim=-2, keepdim=True) + eps)

    return M.to(dtype=orig_dtype)


def log_sinkhorn(
    logits: torch.Tensor,
    n_iters: int = 20,
) -> torch.Tensor:
    """
    Log-domain Sinkhorn projection.

    This is a numerically robust alternative to `sinkhorn`. It is useful for
    large logits or aggressive mixed-precision experiments. It returns a normal
    probability matrix, not log-probabilities.
    """
    if logits.dim() < 2:
        raise ValueError(
            f"log_sinkhorn expects logits with at least 2 dims, got {tuple(logits.shape)}"
        )

    if logits.shape[-1] != logits.shape[-2]:
        raise ValueError(
            "log_sinkhorn expects square matrices in the last two dimensions, "
            f"got {tuple(logits.shape[-2:])}"
        )

    if n_iters <= 0:
        raise ValueError(f"n_iters must be > 0, got {n_iters}")

    orig_dtype = logits.dtype
    log_M = logits.float()

    for _ in range(n_iters):
        log_M = log_M - torch.logsumexp(log_M, dim=-1, keepdim=True)
        log_M = log_M - torch.logsumexp(log_M, dim=-2, keepdim=True)

    return torch.exp(log_M).to(dtype=orig_dtype)


# ============================================================
# Expand / collapse residual streams
# ============================================================

def expand_residual_stream(
    x: torch.Tensor,
    n_hc: int,
    mode: str = "first",
) -> torch.Tensor:
    """
    Expand a standard residual stream into n_hc hyper-connection streams.

    Args:
        x: [B, T, D]
        n_hc: number of residual streams.
        mode:
            "first": stream 0 = x, streams 1..n_hc-1 = 0.
            "mean": every stream receives x / n_hc.
            "repeat": every stream receives x.

    Returns:
        X: [B, T, n_hc, D]
    """
    if x.dim() != 3:
        raise ValueError(f"x must have shape [B,T,D], got {tuple(x.shape)}")

    if n_hc <= 0:
        raise ValueError(f"n_hc must be > 0, got {n_hc}")

    B, T, D = x.shape

    if mode == "first":
        X = x.new_zeros(B, T, n_hc, D)
        X[:, :, 0, :] = x
        return X

    if mode == "mean":
        return x[:, :, None, :].expand(B, T, n_hc, D) / n_hc

    if mode == "repeat":
        return x[:, :, None, :].expand(B, T, n_hc, D).clone()

    raise ValueError(f"Unknown expand mode: {mode}")


def collapse_residual_stream(
    X: torch.Tensor,
    mode: str = "mean",
) -> torch.Tensor:
    """
    Collapse expanded residual stream back to [B,T,D].

    Args:
        X: [B, T, n_hc, D]
        mode:
            "mean": average over hyper streams.
            "first": take stream 0.
            "sum": sum over hyper streams.

    Returns:
        x: [B, T, D]
    """
    if X.dim() != 4:
        raise ValueError(f"X must have shape [B,T,n_hc,D], got {tuple(X.shape)}")

    if mode == "mean":
        return X.mean(dim=2)

    if mode == "first":
        return X[:, :, 0, :]

    if mode == "sum":
        return X.sum(dim=2)

    raise ValueError(f"Unknown collapse mode: {mode}")


class HyperConnectionReadout(nn.Module):
    """
    Learnable readout from [B,T,n_hc,D] to [B,T,D].
    """

    def __init__(self, n_hc: int, init: str = "mean"):
        super().__init__()
        if n_hc <= 0:
            raise ValueError(f"n_hc must be > 0, got {n_hc}")
        self.n_hc = n_hc
        self.logits = nn.Parameter(torch.empty(n_hc))
        self.reset_parameters(init=init)

    def reset_parameters(self, init: str = "mean") -> None:
        with torch.no_grad():
            if init == "mean":
                self.logits.zero_()
            elif init == "first":
                self.logits.fill_(-6.0)
                self.logits[0] = 6.0
            else:
                raise ValueError(f"Unknown readout init: {init}")

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        if X.dim() != 4:
            raise ValueError(f"X must have shape [B,T,n_hc,D], got {tuple(X.shape)}")
        if X.shape[2] != self.n_hc:
            raise ValueError(
                f"Expected X.shape[2] == n_hc={self.n_hc}, got {X.shape[2]}"
            )
        weights = torch.softmax(self.logits, dim=0).to(dtype=X.dtype)
        return torch.einsum("n,btnd->btd", weights, X)

In [53]:
# ============================================================
# Manifold-Constrained Hyper-Connection
# ============================================================

@dataclass
class ManifoldHyperConnectionConfig:
    d_model: int
    n_hc: int = 4

    sinkhorn_iters: int = 20
    eps: float = 1e-6
    use_log_sinkhorn: bool = False
    sinkhorn_fp32: bool = True

    # Dynamic parameterization.
    dynamic: bool = True
    init_alpha: float = 1e-3
    alpha_max: float = 1.0
    bounded_alpha: bool = True

    # Static initialization controls.
    static_a_stream0: float = 4.0
    static_a_other: float = -4.0

    static_b_diag: float = 4.0
    static_b_offdiag: float = -4.0

    static_c_stream0: float = 0.0
    static_c_other: float = -6.0

    init_std: float = 0.02

    def validate(self) -> None:
        if self.d_model <= 0:
            raise ValueError(f"d_model must be > 0, got {self.d_model}")

        if self.n_hc < 2:
            raise ValueError(f"n_hc must be >= 2 for mHC, got {self.n_hc}")

        if self.sinkhorn_iters <= 0:
            raise ValueError(f"sinkhorn_iters must be > 0, got {self.sinkhorn_iters}")

        if self.eps <= 0:
            raise ValueError(f"eps must be > 0, got {self.eps}")

        if self.init_alpha < 0:
            raise ValueError(f"init_alpha must be >= 0, got {self.init_alpha}")

        if self.alpha_max <= 0:
            raise ValueError(f"alpha_max must be > 0, got {self.alpha_max}")

        if self.bounded_alpha and self.init_alpha >= self.alpha_max:
            raise ValueError(
                "For bounded_alpha=True, init_alpha must be < alpha_max. "
                f"Got init_alpha={self.init_alpha}, alpha_max={self.alpha_max}"
            )

        if self.init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {self.init_std}")


class ManifoldHyperConnection(nn.Module):
    """
    Manifold-Constrained Hyper-Connection.

    Canonical equation:

        X_{l+1} = B_l X_l + C_l F_l(A_l X_l)

    where:
        X_l: [B, T, n_hc, D]
        A_l: [B, T, 1, n_hc], values in (0, 1)
        B_l: [B, T, n_hc, n_hc], doubly stochastic
        C_l: [B, T, n_hc, 1], values in (0, 2)

    This version supports both APIs:

    1. Wrapper API:
        X_next = mhc(X, sublayer)

    2. Modular block API:
        A, B, C = mhc.compute_ABC(X)
        h = mhc.pre_mix(X, A=A)
        y = sublayer(norm(h))
        X = mhc.update(X, y, B_mat=B, C=C)
    """

    def __init__(self, config: ManifoldHyperConnectionConfig):
        super().__init__()
        config.validate()

        self.config = config
        self.d_model = config.d_model
        self.n_hc = config.n_hc
        self.sinkhorn_iters = config.sinkhorn_iters
        self.eps = config.eps
        self.dynamic = config.dynamic
        self.use_log_sinkhorn = config.use_log_sinkhorn
        self.sinkhorn_fp32 = config.sinkhorn_fp32
        self.bounded_alpha = config.bounded_alpha
        self.alpha_max = config.alpha_max

        flat_dim = config.n_hc * config.d_model

        self.param_norm = RMSNorm(flat_dim, eps=config.eps)

        self.dynamic_A = nn.Linear(flat_dim, config.n_hc, bias=True)
        self.dynamic_B = nn.Linear(flat_dim, config.n_hc * config.n_hc, bias=True)
        self.dynamic_C = nn.Linear(flat_dim, config.n_hc, bias=True)

        # Static parameters.
        self.static_A = nn.Parameter(torch.empty(config.n_hc))
        self.static_B = nn.Parameter(torch.empty(config.n_hc, config.n_hc))
        self.static_C = nn.Parameter(torch.empty(config.n_hc))

        # Raw scalar gates controlling dynamic contribution.
        # If bounded_alpha=True, effective alpha is:
        #   alpha = alpha_max * tanh(alpha_raw)
        # initialized so alpha ~= init_alpha.
        self.alpha_A_raw = nn.Parameter(torch.empty(()))
        self.alpha_B_raw = nn.Parameter(torch.empty(()))
        self.alpha_C_raw = nn.Parameter(torch.empty(()))

        self.reset_parameters()

    def _initial_alpha_raw(self) -> float:
        if not self.bounded_alpha:
            return float(self.config.init_alpha)

        ratio = self.config.init_alpha / self.config.alpha_max
        # Clamp only for numerical safety; validate already enforces ratio < 1.
        ratio = max(min(ratio, 1.0 - 1e-7), -1.0 + 1e-7)
        return float(math.atanh(ratio))

    def reset_parameters(self) -> None:
        # Dynamic generators start small/stable.
        for module in [self.dynamic_A, self.dynamic_B, self.dynamic_C]:
            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)
            nn.init.zeros_(module.bias)

        with torch.no_grad():
            # A: select stream 0 initially.
            self.static_A.fill_(self.config.static_a_other)
            self.static_A[0] = self.config.static_a_stream0

            # B: approximately identity after Sinkhorn.
            self.static_B.fill_(self.config.static_b_offdiag)
            diag = torch.arange(self.n_hc, device=self.static_B.device)
            self.static_B[diag, diag] = self.config.static_b_diag

            # C: inject mostly into stream 0.
            self.static_C.fill_(self.config.static_c_other)
            self.static_C[0] = self.config.static_c_stream0

            init_raw = self._initial_alpha_raw()
            self.alpha_A_raw.fill_(init_raw)
            self.alpha_B_raw.fill_(init_raw)
            self.alpha_C_raw.fill_(init_raw)

    def _validate_X(self, X: torch.Tensor) -> Tuple[int, int]:
        if X.dim() != 4:
            raise ValueError(
                f"ManifoldHyperConnection expects X [B,T,n_hc,D], got {tuple(X.shape)}"
            )

        B, T, N, D = X.shape

        if N != self.n_hc:
            raise ValueError(f"Expected n_hc={self.n_hc}, got {N}")

        if D != self.d_model:
            raise ValueError(f"Expected d_model={self.d_model}, got {D}")

        return B, T

    def _validate_y_sub(self, y_sub: torch.Tensor, B: int, T: int) -> None:
        if not isinstance(y_sub, torch.Tensor):
            raise TypeError("sublayer output must be a torch.Tensor")
        expected = (B, T, self.d_model)
        if y_sub.shape != expected:
            raise ValueError(
                "sublayer output must have shape [B,T,d_model]. "
                f"Expected {expected}, got {tuple(y_sub.shape)}"
            )

    def _effective_alpha(self, raw: torch.Tensor) -> torch.Tensor:
        if self.bounded_alpha:
            return self.alpha_max * torch.tanh(raw)
        return raw

    def get_alpha_values(self) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Return effective dynamic gates alpha_A, alpha_B, alpha_C."""
        return (
            self._effective_alpha(self.alpha_A_raw),
            self._effective_alpha(self.alpha_B_raw),
            self._effective_alpha(self.alpha_C_raw),
        )

    def compute_ABC(self, X: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Compute constrained A, B, C.

        Args:
            X: [B,T,n_hc,D]

        Returns:
            A: [B,T,1,n_hc], values in (0,1)
            B: [B,T,n_hc,n_hc], approximately doubly stochastic
            C: [B,T,n_hc,1], values in (0,2)
        """
        Bsz, T = self._validate_X(X)

        X_flat = X.reshape(Bsz, T, self.n_hc * self.d_model)
        X_hat = self.param_norm(X_flat)

        if self.dynamic:
            A_dyn = self.dynamic_A(X_hat)
            B_dyn = self.dynamic_B(X_hat).view(Bsz, T, self.n_hc, self.n_hc)
            C_dyn = self.dynamic_C(X_hat)
        else:
            A_dyn = torch.zeros(Bsz, T, self.n_hc, device=X.device, dtype=X.dtype)
            B_dyn = torch.zeros(Bsz, T, self.n_hc, self.n_hc, device=X.device, dtype=X.dtype)
            C_dyn = torch.zeros(Bsz, T, self.n_hc, device=X.device, dtype=X.dtype)

        static_A = self.static_A.to(device=X.device, dtype=X.dtype)
        static_B = self.static_B.to(device=X.device, dtype=X.dtype)
        static_C = self.static_C.to(device=X.device, dtype=X.dtype)

        alpha_A, alpha_B, alpha_C = self.get_alpha_values()
        alpha_A = alpha_A.to(device=X.device, dtype=X.dtype)
        alpha_B = alpha_B.to(device=X.device, dtype=X.dtype)
        alpha_C = alpha_C.to(device=X.device, dtype=X.dtype)

        A_tilde = static_A[None, None, :] + alpha_A * A_dyn
        B_tilde = static_B[None, None, :, :] + alpha_B * B_dyn
        C_tilde = static_C[None, None, :] + alpha_C * C_dyn

        A = torch.sigmoid(A_tilde)[:, :, None, :]          # [B,T,1,n_hc]

        if self.use_log_sinkhorn:
            B_mat = log_sinkhorn(B_tilde, n_iters=self.sinkhorn_iters)
        else:
            B_mat = sinkhorn(
                B_tilde,
                n_iters=self.sinkhorn_iters,
                eps=self.eps,
                fp32=self.sinkhorn_fp32,
            )

        C = (2.0 * torch.sigmoid(C_tilde))[:, :, :, None]  # [B,T,n_hc,1]

        return A, B_mat, C

    def pre_mix(
        self,
        X: torch.Tensor,
        A: Optional[torch.Tensor] = None,
        return_aux: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:
        """
        Produce the d_model-dimensional sublayer input:

            x_sub = A(X) @ X

        Args:
            X: [B,T,n_hc,D]
            A: optional precomputed [B,T,1,n_hc]
            return_aux: if True, returns x_sub and {A,B,C}; useful when the
                caller wants to reuse B,C for update.

        Returns:
            x_sub: [B,T,D]
        """
        self._validate_X(X)

        if A is None:
            A, B_mat, C = self.compute_ABC(X)
        else:
            B_mat = None
            C = None
            expected = (*X.shape[:2], 1, self.n_hc)
            if A.shape != expected:
                raise ValueError(f"A must have shape {expected}, got {tuple(A.shape)}")

        x_sub = torch.einsum("btan,btnd->btad", A, X).squeeze(dim=2)

        if return_aux:
            aux = {"A": A}
            if B_mat is not None:
                aux["B"] = B_mat
            if C is not None:
                aux["C"] = C
            return x_sub, aux

        return x_sub

    def update(
        self,
        X: torch.Tensor,
        y_sub: torch.Tensor,
        B_mat: Optional[torch.Tensor] = None,
        C: Optional[torch.Tensor] = None,
        return_aux: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:
        """
        Apply residual mixing and output injection:

            X_next = B(X) @ X + C(X) * y_sub

        Args:
            X: [B,T,n_hc,D]
            y_sub: [B,T,D]
            B_mat: optional precomputed [B,T,n_hc,n_hc]
            C: optional precomputed [B,T,n_hc,1]
        """
        Bsz, T = self._validate_X(X)
        self._validate_y_sub(y_sub, Bsz, T)

        if B_mat is None or C is None:
            _, B_new, C_new = self.compute_ABC(X)
            if B_mat is None:
                B_mat = B_new
            if C is None:
                C = C_new

        expected_B = (Bsz, T, self.n_hc, self.n_hc)
        expected_C = (Bsz, T, self.n_hc, 1)

        if B_mat.shape != expected_B:
            raise ValueError(f"B_mat must have shape {expected_B}, got {tuple(B_mat.shape)}")

        if C.shape != expected_C:
            raise ValueError(f"C must have shape {expected_C}, got {tuple(C.shape)}")

        mixed_X = torch.einsum("btij,btjd->btid", B_mat, X)
        injected = C * y_sub[:, :, None, :]
        X_next = mixed_X + injected

        if return_aux:
            aux = {
                "B": B_mat,
                "C": C,
                "mixed_X": mixed_X,
                "injected": injected,
            }
            return X_next, aux

        return X_next

    def post_and_residual_mix(
        self,
        X: torch.Tensor,
        F_out: torch.Tensor,
        B_mat: Optional[torch.Tensor] = None,
        C: Optional[torch.Tensor] = None,
        return_aux: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:
        """
        Alias used by block-level code. Equivalent to update(X, F_out, ...).
        """
        return self.update(
            X=X,
            y_sub=F_out,
            B_mat=B_mat,
            C=C,
            return_aux=return_aux,
        )

    def forward(
        self,
        X: torch.Tensor,
        sublayer: Callable[[torch.Tensor], torch.Tensor],
        return_aux: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:
        """
        Wrapper API:

            A, B, C = compute_ABC(X)
            x_sub = pre_mix(X, A)
            y_sub = sublayer(x_sub)
            X_next = update(X, y_sub, B, C)
        """
        Bsz, T = self._validate_X(X)

        if not callable(sublayer):
            raise TypeError("sublayer must be callable")

        A, B_mat, C = self.compute_ABC(X)
        x_sub = self.pre_mix(X, A=A)

        y_sub = sublayer(x_sub)
        self._validate_y_sub(y_sub, Bsz, T)

        X_next, update_aux = self.update(
            X=X,
            y_sub=y_sub,
            B_mat=B_mat,
            C=C,
            return_aux=True,
        )

        if return_aux:
            alpha_A, alpha_B, alpha_C = self.get_alpha_values()
            aux = {
                "A": A,
                "B": B_mat,
                "C": C,
                "x_sub": x_sub,
                "y_sub": y_sub,
                "mixed_X": update_aux["mixed_X"],
                "injected": update_aux["injected"],
                "alpha_A": alpha_A,
                "alpha_B": alpha_B,
                "alpha_C": alpha_C,
            }
            return X_next, aux

        return X_next

In [54]:
# ============================================================
# mHC smoke tests
# ============================================================

B, T, D = 2, 8, 64
n_hc = 4

x = torch.randn(B, T, D)

X = expand_residual_stream(x, n_hc=n_hc, mode="first")

config = ManifoldHyperConnectionConfig(
    d_model=D,
    n_hc=n_hc,
    sinkhorn_iters=20,
    eps=1e-6,
    init_alpha=1e-3,
    dynamic=True,
    init_std=0.02,
)

mhc = ManifoldHyperConnection(config)

def dummy_sublayer(x_sub):
    return 0.1 * x_sub

X_next, aux = mhc(
    X,
    sublayer=dummy_sublayer,
    return_aux=True,)

print("X:", X.shape)
print("X_next:", X_next.shape)
print("A:", aux["A"].shape)
print("B:", aux["B"].shape)
print("C:", aux["C"].shape)
print("x_sub:", aux["x_sub"].shape)
print("y_sub:", aux["y_sub"].shape)

assert X.shape == (B, T, n_hc, D)
assert X_next.shape == (B, T, n_hc, D)
assert aux["A"].shape == (B, T, 1, n_hc)
assert aux["B"].shape == (B, T, n_hc, n_hc)
assert aux["C"].shape == (B, T, n_hc, 1)
assert aux["x_sub"].shape == (B, T, D)
assert aux["y_sub"].shape == (B, T, D)

assert torch.isfinite(X_next).all()
assert torch.isfinite(aux["A"]).all()
assert torch.isfinite(aux["B"]).all()
assert torch.isfinite(aux["C"]).all()

print("mHC smoke test OK.")

X: torch.Size([2, 8, 4, 64])
X_next: torch.Size([2, 8, 4, 64])
A: torch.Size([2, 8, 1, 4])
B: torch.Size([2, 8, 4, 4])
C: torch.Size([2, 8, 4, 1])
x_sub: torch.Size([2, 8, 64])
y_sub: torch.Size([2, 8, 64])
mHC smoke test OK.


In [55]:
# ============================================================
# Approx residual-normal initialization check
# ============================================================

config_static = ManifoldHyperConnectionConfig(
    d_model=D,
    n_hc=n_hc,
    dynamic=False,
    sinkhorn_iters=50,
    init_alpha=0.0,)

mhc_static = ManifoldHyperConnection(config_static)

X = expand_residual_stream(x, n_hc=n_hc, mode="first")

def F(x_sub):
    return 0.1 * x_sub

X_next, aux = mhc_static(X, sublayer=F, return_aux=True)

# Since A selects stream 0 and C injects mainly into stream 0,
# stream 0 should be close to x + F(x), up to small leakage from sigmoid/Sinkhorn.
expected_stream0 = x + F(x)

print("mean abs stream0 error:", (X_next[:, :, 0, :] - expected_stream0).abs().mean().item())
print("mean abs inactive streams:", X_next[:, :, 1:, :].abs().mean().item())

assert torch.isfinite(X_next).all()

print("mHC static residual-like initialization check OK.")

mean abs stream0 error: 0.0022263354621827602
mean abs inactive streams: 0.0006514514680020511
mHC static residual-like initialization check OK.


In [56]:
x = torch.randn(B, T, D, requires_grad=True)
X = expand_residual_stream(x, n_hc=n_hc, mode="first")

mhc = ManifoldHyperConnection(config)

X_next = mhc(
    X,
    sublayer=lambda x_sub: x_sub ** 2,
    return_aux=False,
)

loss = X_next.sum()
loss.backward()

assert x.grad is not None
assert torch.isfinite(x.grad).all()

for name, param in mhc.named_parameters():
    assert param.grad is not None, f"Missing grad for {name}"
    assert torch.isfinite(param.grad).all(), f"Non-finite grad for {name}"

print("mHC backward OK.")

mHC backward OK.


In [57]:
# @title
# ============================================================
# ManifoldHyperConnection / mHC tests
# ============================================================

import pytest
import torch
import torch.nn as nn


# ============================================================
# Helpers
# ============================================================

def make_mhc_config(**overrides):
    cfg = dict(
        d_model=64,
        n_hc=4,
        sinkhorn_iters=30,
        eps=1e-6,

        # New canonical Sinkhorn controls
        use_log_sinkhorn=False,
        sinkhorn_fp32=True,

        # New bounded-alpha dynamic controls
        init_alpha=1e-3,
        alpha_max=1.0,
        bounded_alpha=True,

        dynamic=True,
        static_a_stream0=4.0,
        static_a_other=-4.0,
        static_b_diag=6.0,
        static_b_offdiag=-6.0,
        static_c_stream0=0.0,
        static_c_other=-8.0,
        init_std=0.02,
    )
    cfg.update(overrides)
    return ManifoldHyperConnectionConfig(**cfg)


def make_mhc(**overrides):
    return ManifoldHyperConnection(make_mhc_config(**overrides))


def make_X(B=2, T=8, n_hc=4, D=64, dtype=torch.float32, device="cpu"):
    return torch.randn(B, T, n_hc, D, dtype=dtype, device=device)


def make_x(B=2, T=8, D=64, dtype=torch.float32, device="cpu"):
    return torch.randn(B, T, D, dtype=dtype, device=device)


class TinyLinearSublayer(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        return self.proj(x)


# ============================================================
# A. Config tests
# ============================================================

def test_valid_mhc_config_builds():
    config = ManifoldHyperConnectionConfig(
        d_model=256,
        n_hc=4,
        sinkhorn_iters=20,
        eps=1e-6,
        use_log_sinkhorn=False,
        sinkhorn_fp32=True,
        init_alpha=1e-3,
        alpha_max=1.0,
        bounded_alpha=True,
        dynamic=True,
    )

    mhc = ManifoldHyperConnection(config)

    assert mhc.d_model == 256
    assert mhc.n_hc == 4
    assert mhc.sinkhorn_iters == 20
    assert mhc.eps == 1e-6
    assert mhc.dynamic is True
    assert mhc.use_log_sinkhorn is False
    assert mhc.sinkhorn_fp32 is True
    assert mhc.bounded_alpha is True
    assert mhc.alpha_max == 1.0


@pytest.mark.parametrize("d_model", [0, -1, -64])
def test_invalid_mhc_d_model_raises(d_model):
    with pytest.raises(ValueError):
        ManifoldHyperConnection(make_mhc_config(d_model=d_model))


@pytest.mark.parametrize("n_hc", [0, -1, 1])
def test_invalid_n_hc_raises(n_hc):
    with pytest.raises(ValueError):
        ManifoldHyperConnection(make_mhc_config(n_hc=n_hc))


@pytest.mark.parametrize("sinkhorn_iters", [0, -1])
def test_invalid_sinkhorn_iters_raises(sinkhorn_iters):
    with pytest.raises(ValueError):
        ManifoldHyperConnection(make_mhc_config(sinkhorn_iters=sinkhorn_iters))


@pytest.mark.parametrize("eps", [0.0, -1e-6])
def test_invalid_eps_raises(eps):
    with pytest.raises(ValueError):
        ManifoldHyperConnection(make_mhc_config(eps=eps))

@pytest.mark.parametrize("init_alpha", [-1e-3, -1.0])
def test_invalid_init_alpha_raises(init_alpha):
    with pytest.raises(ValueError):
        ManifoldHyperConnection(make_mhc_config(init_alpha=init_alpha))


@pytest.mark.parametrize("alpha_max", [0.0, -1.0])
def test_invalid_alpha_max_raises(alpha_max):
    with pytest.raises(ValueError):
        ManifoldHyperConnection(make_mhc_config(alpha_max=alpha_max))


def test_invalid_bounded_alpha_when_init_alpha_ge_alpha_max_raises():
    with pytest.raises(ValueError):
        ManifoldHyperConnection(
            make_mhc_config(
                bounded_alpha=True,
                init_alpha=1.0,
                alpha_max=1.0,
            )
        )


def test_unbounded_alpha_allows_init_alpha_ge_alpha_max():
    mhc = ManifoldHyperConnection(
        make_mhc_config(
            bounded_alpha=False,
            init_alpha=2.0,
            alpha_max=1.0,
        )
    )

    alpha_A, alpha_B, alpha_C = mhc.get_alpha_values()

    assert torch.allclose(alpha_A, torch.tensor(2.0, dtype=alpha_A.dtype))
    assert torch.allclose(alpha_B, torch.tensor(2.0, dtype=alpha_B.dtype))
    assert torch.allclose(alpha_C, torch.tensor(2.0, dtype=alpha_C.dtype))


# ============================================================
# B. Sinkhorn tests
# ============================================================

def test_sinkhorn_output_shape():
    logits = torch.randn(2, 8, 4, 4)

    B_mat = sinkhorn(logits, n_iters=30, eps=1e-6)

    assert B_mat.shape == logits.shape


def test_sinkhorn_non_negative():
    logits = torch.randn(2, 8, 4, 4)

    B_mat = sinkhorn(logits, n_iters=30, eps=1e-6)

    assert (B_mat >= 0).all()


def test_sinkhorn_rows_sum_to_one():
    logits = torch.randn(2, 8, 4, 4)

    B_mat = sinkhorn(logits, n_iters=50, eps=1e-6)

    row_sums = B_mat.sum(dim=-1)

    assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4, rtol=1e-4)


def test_sinkhorn_cols_sum_to_one():
    logits = torch.randn(2, 8, 4, 4)

    B_mat = sinkhorn(logits, n_iters=50, eps=1e-6)

    col_sums = B_mat.sum(dim=-2)

    assert torch.allclose(col_sums, torch.ones_like(col_sums), atol=1e-4, rtol=1e-4)


def test_sinkhorn_finite_for_large_logits():
    logits = torch.randn(2, 8, 4, 4) * 1_000.0

    B_mat = sinkhorn(logits, n_iters=50, eps=1e-6)

    assert torch.isfinite(B_mat).all()
    assert (B_mat >= 0).all()


# ============================================================
# C. A/B/C generation tests
# ============================================================

def test_abc_shapes():
    B, T, N, D = 2, 8, 4, 64

    mhc = make_mhc(d_model=D, n_hc=N)
    X = make_X(B=B, T=T, n_hc=N, D=D)

    A, B_mat, C = mhc.compute_ABC(X)

    assert A.shape == (B, T, 1, N)
    assert B_mat.shape == (B, T, N, N)
    assert C.shape == (B, T, N, 1)


def test_A_bounds():
    mhc = make_mhc()
    X = make_X()

    A, _, _ = mhc.compute_ABC(X)

    assert A.min() >= 0
    assert A.max() <= 1


def test_C_bounds():
    mhc = make_mhc()
    X = make_X()

    _, _, C = mhc.compute_ABC(X)

    assert C.min() >= 0
    assert C.max() <= 2


def test_B_doubly_stochastic():
    mhc = make_mhc(sinkhorn_iters=50)
    X = make_X()

    _, B_mat, _ = mhc.compute_ABC(X)

    assert (B_mat >= 0).all()

    row_sums = B_mat.sum(dim=-1)
    col_sums = B_mat.sum(dim=-2)

    assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4, rtol=1e-4)
    assert torch.allclose(col_sums, torch.ones_like(col_sums), atol=1e-4, rtol=1e-4)


def test_dynamic_false_same_abc_for_different_X():
    mhc = make_mhc(dynamic=False)

    X1 = make_X()
    X2 = make_X() * 10.0 + 5.0

    A1, B1, C1 = mhc.compute_ABC(X1)
    A2, B2, C2 = mhc.compute_ABC(X2)

    assert torch.allclose(A1, A2, atol=0.0, rtol=0.0)
    assert torch.allclose(B1, B2, atol=0.0, rtol=0.0)
    assert torch.allclose(C1, C2, atol=0.0, rtol=0.0)


def test_dynamic_true_abc_changes_with_X():
    mhc = make_mhc(dynamic=True, init_alpha=1e-1)

    X1 = make_X()
    X2 = make_X() * 10.0 + 5.0

    A1, B1, C1 = mhc.compute_ABC(X1)
    A2, B2, C2 = mhc.compute_ABC(X2)

    diff = (
        (A1 - A2).abs().sum()
        + (B1 - B2).abs().sum()
        + (C1 - C2).abs().sum()
    )

    assert diff > 0


# ============================================================
# D. Forward tests
# ============================================================

def test_forward_output_shape_matches_input():
    mhc = make_mhc()
    X = make_X()

    X_next = mhc(X, sublayer=lambda x: x)

    assert X_next.shape == X.shape


@pytest.mark.parametrize(
    "bad_X",
    [
        torch.randn(2, 8, 64),
        torch.randn(2, 8, 4, 64, 1),
    ],
)
def test_mhc_rejects_wrong_input_rank(bad_X):
    mhc = make_mhc()

    with pytest.raises(ValueError):
        mhc(bad_X, sublayer=lambda x: x)


def test_rejects_wrong_n_hc():
    mhc = make_mhc(n_hc=4)

    X = torch.randn(2, 8, 3, 64)

    with pytest.raises(ValueError):
        mhc(X, sublayer=lambda x: x)


def test_rejects_wrong_d_model():
    mhc = make_mhc(d_model=64)

    X = torch.randn(2, 8, 4, 32)

    with pytest.raises(ValueError):
        mhc(X, sublayer=lambda x: x)


def test_sublayer_receives_BTD_tensor():
    B, T, N, D = 2, 8, 4, 64

    mhc = make_mhc(d_model=D, n_hc=N)
    X = make_X(B=B, T=T, n_hc=N, D=D)

    seen = {}

    def sublayer(x_sub):
        seen["shape"] = x_sub.shape
        return x_sub

    _ = mhc(X, sublayer=sublayer)

    assert seen["shape"] == (B, T, D)


def test_sublayer_output_must_match_shape():
    mhc = make_mhc()
    X = make_X()

    def bad_sublayer(x_sub):
        B, T, D = x_sub.shape
        return torch.randn(B, T, D + 1)

    with pytest.raises(ValueError):
        mhc(X, sublayer=bad_sublayer)


# ============================================================
# E. Mathematical update tests
# ============================================================

def test_forward_matches_manual_computation():
    mhc = make_mhc(dynamic=True)

    X = make_X()

    def sublayer(x_sub):
        return x_sub ** 2

    X_next, aux = mhc(X, sublayer=sublayer, return_aux=True)

    A = aux["A"]
    B_mat = aux["B"]
    C = aux["C"]

    x_sub = torch.einsum("btan,btnd->btad", A, X).squeeze(2)
    y_sub = sublayer(x_sub)
    mixed_X = torch.einsum("btij,btjd->btid", B_mat, X)
    expected = mixed_X + C * y_sub[:, :, None, :]

    assert torch.allclose(X_next, expected, atol=1e-6, rtol=1e-5)

    assert "alpha_A" in aux
    assert "alpha_B" in aux
    assert "alpha_C" in aux


def test_zero_sublayer_reduces_to_B_mixing():
    mhc = make_mhc()
    X = make_X()

    def zero_sublayer(x_sub):
        return torch.zeros_like(x_sub)

    X_next, aux = mhc(X, sublayer=zero_sublayer, return_aux=True)

    expected = torch.einsum("btij,btjd->btid", aux["B"], X)

    assert torch.allclose(X_next, expected, atol=1e-6, rtol=1e-5)


def test_zero_input_zero_sublayer_returns_zero():
    mhc = make_mhc()

    X = torch.zeros(2, 8, 4, 64)

    def zero_sublayer(x_sub):
        return torch.zeros_like(x_sub)

    X_next = mhc(X, sublayer=zero_sublayer)

    assert torch.allclose(X_next, torch.zeros_like(X_next), atol=1e-7, rtol=1e-7)


# ============================================================
# F. Residual-like initialization tests
# ============================================================

def test_initial_A_prefers_stream_zero():
    mhc = make_mhc(dynamic=False)
    X = make_X()

    A, _, _ = mhc.compute_ABC(X)

    stream0 = A[..., 0]
    others = A[..., 1:]

    assert (stream0[..., None] > others).all()


def test_initial_C_prefers_stream_zero():
    mhc = make_mhc(dynamic=False)
    X = make_X()

    _, _, C = mhc.compute_ABC(X)

    stream0 = C[..., 0, 0]
    others = C[..., 1:, 0]

    assert (stream0[..., None] > others).all()


def test_initial_B_close_to_identity():
    mhc = make_mhc(dynamic=False, sinkhorn_iters=100)
    X = make_X()

    _, B_mat, _ = mhc.compute_ABC(X)

    B_mean = B_mat.mean(dim=(0, 1))

    diag = torch.diagonal(B_mean)
    offdiag = B_mean[~torch.eye(mhc.n_hc, dtype=torch.bool)]

    assert diag.mean() > offdiag.mean()


def test_initial_forward_approximates_residual_on_stream_zero():
    B, T, D, N = 2, 8, 64, 4

    mhc = make_mhc(
        d_model=D,
        n_hc=N,
        dynamic=False,
        sinkhorn_iters=100,
        static_a_stream0=8.0,
        static_a_other=-8.0,
        static_b_diag=10.0,
        static_b_offdiag=-10.0,
        static_c_stream0=0.0,
        static_c_other=-10.0,
    )

    x = make_x(B=B, T=T, D=D)
    X = expand_residual_stream(x, n_hc=N, mode="first")

    def sublayer(x_sub):
        return 0.1 * x_sub

    X_next = mhc(X, sublayer=sublayer)

    expected_stream0 = x + sublayer(x)

    assert torch.allclose(
        X_next[:, :, 0, :],
        expected_stream0,
        atol=3e-3,
        rtol=3e-3,
    )


# ============================================================
# G. Expand / collapse tests
# ============================================================

def test_expand_residual_stream_shape():
    x = make_x(B=2, T=8, D=64)

    X = expand_residual_stream(x, n_hc=4, mode="first")

    assert X.shape == (2, 8, 4, 64)


def test_expand_residual_stream_puts_x_in_stream_zero():
    x = make_x(B=2, T=8, D=64)

    X = expand_residual_stream(x, n_hc=4, mode="first")

    assert torch.allclose(X[:, :, 0, :], x, atol=0.0, rtol=0.0)
    assert torch.allclose(X[:, :, 1:, :], torch.zeros_like(X[:, :, 1:, :]), atol=0.0, rtol=0.0)


def test_collapse_residual_stream_mean_shape():
    X = make_X(B=2, T=8, n_hc=4, D=64)

    x = collapse_residual_stream(X, mode="mean")

    assert x.shape == (2, 8, 64)


def test_collapse_residual_stream_first_shape():
    X = make_X(B=2, T=8, n_hc=4, D=64)

    x = collapse_residual_stream(X, mode="first")

    assert x.shape == (2, 8, 64)
    assert torch.allclose(x, X[:, :, 0, :], atol=0.0, rtol=0.0)


# ============================================================
# H. Gradient / numerical tests
# ============================================================

def test_mhc_backward_computes_gradients():
    B, T, D, N = 2, 8, 64, 4

    mhc = make_mhc(d_model=D, n_hc=N, dynamic=True, init_alpha=1e-2)

    sublayer = TinyLinearSublayer(D)

    X = make_X(B=B, T=T, n_hc=N, D=D)
    X.requires_grad_(True)

    X_next, aux = mhc(X, sublayer=sublayer, return_aux=True)

    loss = X_next.sum()
    loss.backward()

    assert X.grad is not None
    assert torch.isfinite(X.grad).all()

    expected_params = [
        "static_A",
        "static_B",
        "static_C",
        "alpha_A_raw",
        "alpha_B_raw",
        "alpha_C_raw",
        "dynamic_A.weight",
        "dynamic_B.weight",
        "dynamic_C.weight",
        "param_norm.weight",
    ]

    params = dict(mhc.named_parameters())

    for name in expected_params:
        assert name in params, f"Missing parameter {name}"
        assert params[name].grad is not None, f"Missing grad for {name}"
        assert torch.isfinite(params[name].grad).all(), f"Non-finite grad for {name}"

    for name, param in sublayer.named_parameters():
        assert param.grad is not None, f"Missing sublayer grad for {name}"
        assert torch.isfinite(param.grad).all(), f"Non-finite sublayer grad for {name}"

    assert "alpha_A" in aux
    assert "alpha_B" in aux
    assert "alpha_C" in aux


def test_no_nan_large_inputs():
    mhc = make_mhc(dynamic=True)

    X = make_X() * 1_000.0

    X_next = mhc(X, sublayer=lambda x: 0.1 * x)

    assert torch.isfinite(X_next).all()


def test_no_nan_small_inputs():
    mhc = make_mhc(dynamic=True)

    X = make_X() * 1e-8

    X_next = mhc(X, sublayer=lambda x: 0.1 * x)

    assert torch.isfinite(X_next).all()


# ============================================================
# I. Integration with real sublayers
# ============================================================

def test_mhc_wraps_swiglu_mlp():
    B, T, D, N = 2, 8, 64, 4

    mhc = make_mhc(d_model=D, n_hc=N)
    mlp = SwiGLUMLP(
        SwiGLUMLPConfig(
            d_model=D,
            hidden_dim=256,
            dropout=0.0,
            use_bias=False,
            init_std=0.02,
        )
    )

    X = make_X(B=B, T=T, n_hc=N, D=D)

    X_next = mhc(X, sublayer=mlp)

    assert X_next.shape == X.shape
    assert torch.isfinite(X_next).all()


def test_mhc_wraps_causal_mha_with_lambda():
    B, T, D, N = 2, 8, 64, 4

    mhc = make_mhc(d_model=D, n_hc=N)

    attn = CausalMultiHeadAttention(
        CausalMHAConfig(
            d_model=D,
            n_heads=4,
            head_dim=16,
            attention_dropout=0.0,
            residual_dropout=0.0,
            use_bias=False,
            use_rope=True,
            rotary_dim=16,
            max_seq_len=T,
            init_std=0.02,
        )
    )

    X = make_X(B=B, T=T, n_hc=N, D=D)
    attention_mask = torch.ones(B, T, dtype=torch.long)
    position_ids = torch.arange(T)

    X_next = mhc(
        X,
        sublayer=lambda x_sub: attn(
            x_sub,
            attention_mask=attention_mask,
            position_ids=position_ids,
        ),
    )

    assert X_next.shape == X.shape
    assert torch.isfinite(X_next).all()


def test_mhc_wraps_hca_attention():
    B, T, D, N = 2, 8, 64, 4

    mhc = make_mhc(d_model=D, n_hc=N)

    hca = HCAAttention(
        HCAConfig(
            d_model=D,
            n_heads=4,
            head_dim=16,
            compression_factor=4,
            window_size=4,
            attention_dropout=0.0,
            residual_dropout=0.0,
            use_bias=False,
            use_rope=True,
            rotary_dim=16,
            max_seq_len=T,
            init_std=0.02,
        )
    )

    X = make_X(B=B, T=T, n_hc=N, D=D)
    attention_mask = torch.ones(B, T, dtype=torch.long)

    X_next = mhc(
        X,
        sublayer=lambda x_sub: hca(
            x_sub,
            attention_mask=attention_mask,
            start_pos=0,
        ),
    )

    assert X_next.shape == X.shape
    assert torch.isfinite(X_next).all()


def test_mhc_wraps_csa_attention():
    B, T, D, N = 2, 8, 64, 4

    mhc = make_mhc(d_model=D, n_hc=N)

    csa = CSAAttention(
        CSAConfig(
            d_model=D,
            n_heads=4,
            head_dim=16,
            compression_factor=4,
            top_k=2,
            window_size=4,
            indexer_dim=8,
            n_indexer_heads=2,
            query_compression_dim=16,
            attention_dropout=0.0,
            residual_dropout=0.0,
            use_bias=False,
            use_rope=True,
            rotary_dim=16,
            max_seq_len=T,
            init_std=0.02,
        )
    )

    X = make_X(B=B, T=T, n_hc=N, D=D)
    attention_mask = torch.ones(B, T, dtype=torch.long)

    X_next = mhc(
        X,
        sublayer=lambda x_sub: csa(
            x_sub,
            attention_mask=attention_mask,
            start_pos=0,
        ),
    )

    assert X_next.shape == X.shape
    assert torch.isfinite(X_next).all()


def test_log_sinkhorn_output_shape():
    logits = torch.randn(2, 8, 4, 4)

    B_mat = log_sinkhorn(logits, n_iters=30)

    assert B_mat.shape == logits.shape


def test_log_sinkhorn_non_negative():
    logits = torch.randn(2, 8, 4, 4)

    B_mat = log_sinkhorn(logits, n_iters=30)

    assert (B_mat >= 0).all()


def test_log_sinkhorn_rows_cols_sum_to_one():
    logits = torch.randn(2, 8, 4, 4)

    B_mat = log_sinkhorn(logits, n_iters=50)

    row_sums = B_mat.sum(dim=-1)
    col_sums = B_mat.sum(dim=-2)

    assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4, rtol=1e-4)
    assert torch.allclose(col_sums, torch.ones_like(col_sums), atol=1e-4, rtol=1e-4)


def test_compute_ABC_with_log_sinkhorn():
    mhc = make_mhc(use_log_sinkhorn=True, sinkhorn_iters=50)
    X = make_X()

    _, B_mat, _ = mhc.compute_ABC(X)

    assert torch.isfinite(B_mat).all()
    assert (B_mat >= 0).all()

    row_sums = B_mat.sum(dim=-1)
    col_sums = B_mat.sum(dim=-2)

    assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4, rtol=1e-4)
    assert torch.allclose(col_sums, torch.ones_like(col_sums), atol=1e-4, rtol=1e-4)

def test_bounded_alpha_initializes_near_init_alpha():
    init_alpha = 1e-3

    mhc = make_mhc(
        bounded_alpha=True,
        alpha_max=1.0,
        init_alpha=init_alpha,
    )

    alpha_A, alpha_B, alpha_C = mhc.get_alpha_values()

    expected = torch.tensor(init_alpha, dtype=alpha_A.dtype)

    assert torch.allclose(alpha_A, expected, atol=1e-7, rtol=1e-5)
    assert torch.allclose(alpha_B, expected, atol=1e-7, rtol=1e-5)
    assert torch.allclose(alpha_C, expected, atol=1e-7, rtol=1e-5)


def test_bounded_alpha_is_within_bounds_after_manual_large_raw():
    mhc = make_mhc(
        bounded_alpha=True,
        alpha_max=0.25,
        init_alpha=1e-3,
    )

    with torch.no_grad():
        mhc.alpha_A_raw.fill_(100.0)
        mhc.alpha_B_raw.fill_(-100.0)
        mhc.alpha_C_raw.fill_(50.0)

    alpha_A, alpha_B, alpha_C = mhc.get_alpha_values()

    assert alpha_A <= 0.25
    assert alpha_A >= -0.25

    assert alpha_B <= 0.25
    assert alpha_B >= -0.25

    assert alpha_C <= 0.25

def test_pre_mix_output_shape():
    B, T, N, D = 2, 8, 4, 64

    mhc = make_mhc(d_model=D, n_hc=N)
    X = make_X(B=B, T=T, n_hc=N, D=D)

    x_sub = mhc.pre_mix(X)

    assert x_sub.shape == (B, T, D)
    assert torch.isfinite(x_sub).all()


def test_pre_mix_with_precomputed_A_matches_forward_aux_x_sub():
    mhc = make_mhc(dynamic=True)
    X = make_X()

    A, B_mat, C = mhc.compute_ABC(X)

    x_sub_1 = mhc.pre_mix(X, A=A)

    _, aux = mhc(
        X,
        sublayer=lambda x: x,
        return_aux=True,
    )

    x_sub_2 = aux["x_sub"]

    assert torch.allclose(x_sub_1, x_sub_2, atol=1e-6, rtol=1e-5)


def test_pre_mix_return_aux_contains_ABC_when_A_not_provided():
    mhc = make_mhc()
    X = make_X()

    x_sub, aux = mhc.pre_mix(X, return_aux=True)

    assert x_sub.shape == (X.shape[0], X.shape[1], X.shape[-1])
    assert "A" in aux
    assert "B" in aux
    assert "C" in aux

    assert aux["A"].shape == (X.shape[0], X.shape[1], 1, X.shape[2])
    assert aux["B"].shape == (X.shape[0], X.shape[1], X.shape[2], X.shape[2])
    assert aux["C"].shape == (X.shape[0], X.shape[1], X.shape[2], 1)


def test_update_output_shape():
    B, T, N, D = 2, 8, 4, 64

    mhc = make_mhc(d_model=D, n_hc=N)
    X = make_X(B=B, T=T, n_hc=N, D=D)
    y_sub = torch.randn(B, T, D)

    X_next = mhc.update(X, y_sub)

    assert X_next.shape == X.shape
    assert torch.isfinite(X_next).all()


def test_update_with_precomputed_BC_matches_manual_computation():
    mhc = make_mhc(dynamic=True)
    X = make_X()

    A, B_mat, C = mhc.compute_ABC(X)
    y_sub = torch.randn(X.shape[0], X.shape[1], X.shape[-1])

    X_next = mhc.update(X, y_sub, B_mat=B_mat, C=C)

    expected = (
        torch.einsum("btij,btjd->btid", B_mat, X)
        + C * y_sub[:, :, None, :]
    )

    assert torch.allclose(X_next, expected, atol=1e-6, rtol=1e-5)


def test_forward_matches_modular_pre_mix_update():
    mhc = make_mhc(dynamic=True)
    X = make_X()

    def sublayer(x_sub):
        return x_sub ** 2

    X_forward = mhc(X, sublayer=sublayer)

    A, B_mat, C = mhc.compute_ABC(X)
    x_sub = mhc.pre_mix(X, A=A)
    y_sub = sublayer(x_sub)
    X_modular = mhc.update(X, y_sub, B_mat=B_mat, C=C)

    assert torch.allclose(X_forward, X_modular, atol=1e-6, rtol=1e-5)


def test_post_and_residual_mix_alias_matches_update():
    mhc = make_mhc()
    X = make_X()
    y_sub = torch.randn(X.shape[0], X.shape[1], X.shape[-1])

    _, B_mat, C = mhc.compute_ABC(X)

    out_update = mhc.update(X, y_sub, B_mat=B_mat, C=C)
    out_alias = mhc.post_and_residual_mix(X, y_sub, B_mat=B_mat, C=C)

    assert torch.allclose(out_update, out_alias, atol=0.0, rtol=0.0)


def test_update_rejects_bad_y_sub_shape():
    mhc = make_mhc()
    X = make_X()

    bad_y = torch.randn(X.shape[0], X.shape[1], X.shape[-1] + 1)

    with pytest.raises(ValueError):
        mhc.update(X, bad_y)


def test_pre_mix_rejects_bad_A_shape():
    mhc = make_mhc()
    X = make_X()

    bad_A = torch.randn(X.shape[0], X.shape[1], mhc.n_hc)

    with pytest.raises(ValueError):
        mhc.pre_mix(X, A=bad_A)


def test_update_rejects_bad_B_shape():
    mhc = make_mhc()
    X = make_X()
    y_sub = torch.randn(X.shape[0], X.shape[1], X.shape[-1])

    bad_B = torch.randn(X.shape[0], X.shape[1], mhc.n_hc, mhc.n_hc + 1)

    with pytest.raises(ValueError):
        mhc.update(X, y_sub, B_mat=bad_B)


def test_update_rejects_bad_C_shape():
    mhc = make_mhc()
    X = make_X()
    y_sub = torch.randn(X.shape[0], X.shape[1], X.shape[-1])

    bad_C = torch.randn(X.shape[0], X.shape[1], mhc.n_hc)

    with pytest.raises(ValueError):
        mhc.update(X, y_sub, C=bad_C)

def test_collapse_residual_stream_sum_shape():
    X = make_X(B=2, T=8, n_hc=4, D=64)

    x = collapse_residual_stream(X, mode="sum")

    assert x.shape == (2, 8, 64)
    assert torch.allclose(x, X.sum(dim=2), atol=0.0, rtol=0.0)

@pytest.mark.parametrize("init_alpha", [-1e-3, -1.0])
def test_invalid_init_alpha_raises(init_alpha):
    with pytest.raises(ValueError):
        ManifoldHyperConnection(make_mhc_config(init_alpha=init_alpha))


@pytest.mark.parametrize("alpha_max", [0.0, -1.0])
def test_invalid_alpha_max_raises(alpha_max):
    with pytest.raises(ValueError):
        ManifoldHyperConnection(make_mhc_config(alpha_max=alpha_max))


def test_invalid_bounded_alpha_when_init_alpha_ge_alpha_max_raises():
    with pytest.raises(ValueError):
        ManifoldHyperConnection(
            make_mhc_config(
                bounded_alpha=True,
                init_alpha=1.0,
                alpha_max=1.0,
            )
        )


def test_unbounded_alpha_allows_init_alpha_ge_alpha_max():
    mhc = ManifoldHyperConnection(
        make_mhc_config(
            bounded_alpha=False,
            init_alpha=2.0,
            alpha_max=1.0,
        )
    )

    alpha_A, alpha_B, alpha_C = mhc.get_alpha_values()

    expected = torch.tensor(2.0, device=alpha_A.device, dtype=alpha_A.dtype)

    assert torch.allclose(alpha_A, expected)
    assert torch.allclose(alpha_B, expected)
    assert torch.allclose(alpha_C, expected)


In [66]:
import torch.nn.functional as F

# ============================================================
# Run ONLY mHC tests in Jupyter
# Handles pytest-style parametrized tests manually
# ============================================================

mhc_plain_tests = [
    # A. Config
    "test_valid_mhc_config_builds",
    "test_invalid_bounded_alpha_when_init_alpha_ge_alpha_max_raises",
    "test_unbounded_alpha_allows_init_alpha_ge_alpha_max",

    # B. Sinkhorn
    "test_sinkhorn_output_shape",
    "test_sinkhorn_non_negative",
    "test_sinkhorn_rows_sum_to_one",
    "test_sinkhorn_cols_sum_to_one",
    "test_sinkhorn_finite_for_large_logits",
    "test_log_sinkhorn_output_shape",
    "test_log_sinkhorn_non_negative",
    "test_log_sinkhorn_rows_cols_sum_to_one",
    "test_compute_ABC_with_log_sinkhorn",

    # C. A/B/C generation
    "test_abc_shapes",
    "test_A_bounds",
    "test_C_bounds",
    "test_B_doubly_stochastic",
    "test_dynamic_false_same_abc_for_different_X",
    "test_dynamic_true_abc_changes_with_X",
    "test_bounded_alpha_initializes_near_init_alpha",
    "test_bounded_alpha_is_within_bounds_after_manual_large_raw",

    # D. Forward tests
    "test_forward_output_shape_matches_input",
    "test_rejects_wrong_n_hc",
    "test_rejects_wrong_d_model",
    "test_sublayer_receives_BTD_tensor",
    "test_sublayer_output_must_match_shape",

    # E. Mathematical update tests
    "test_forward_matches_manual_computation",
    "test_zero_sublayer_reduces_to_B_mixing",
    "test_zero_input_zero_sublayer_returns_zero",

    # Modular API
    "test_pre_mix_output_shape",
    "test_pre_mix_with_precomputed_A_matches_forward_aux_x_sub",
    "test_pre_mix_return_aux_contains_ABC_when_A_not_provided",
    "test_update_output_shape",
    "test_update_with_precomputed_BC_matches_manual_computation",
    "test_forward_matches_modular_pre_mix_update",
    "test_post_and_residual_mix_alias_matches_update",
    "test_update_rejects_bad_y_sub_shape",
    "test_pre_mix_rejects_bad_A_shape",
    "test_update_rejects_bad_B_shape",
    "test_update_rejects_bad_C_shape",

    # F. Residual-like initialization
    "test_initial_A_prefers_stream_zero",
    "test_initial_C_prefers_stream_zero",
    "test_initial_B_close_to_identity",
    "test_initial_forward_approximates_residual_on_stream_zero",

    # G. Expand / collapse
    "test_expand_residual_stream_shape",
    "test_expand_residual_stream_puts_x_in_stream_zero",
    "test_collapse_residual_stream_mean_shape",
    "test_collapse_residual_stream_first_shape",
    "test_collapse_residual_stream_sum_shape",

    # H. Gradients / numerical
    "test_mhc_backward_computes_gradients",
    "test_no_nan_large_inputs",
    "test_no_nan_small_inputs",

    # I. Integration
    "test_mhc_wraps_swiglu_mlp",
    "test_mhc_wraps_causal_mha_with_lambda",
    "test_mhc_wraps_hca_attention",
    "test_mhc_wraps_csa_attention",
]

mhc_param_tests = [
    ("test_invalid_mhc_d_model_raises", {"d_model": 0}),
    ("test_invalid_mhc_d_model_raises", {"d_model": -1}),
    ("test_invalid_mhc_d_model_raises", {"d_model": -64}),

    ("test_invalid_n_hc_raises", {"n_hc": 0}),
    ("test_invalid_n_hc_raises", {"n_hc": -1}),
    ("test_invalid_n_hc_raises", {"n_hc": 1}),

    ("test_invalid_sinkhorn_iters_raises", {"sinkhorn_iters": 0}),
    ("test_invalid_sinkhorn_iters_raises", {"sinkhorn_iters": -1}),

    ("test_invalid_eps_raises", {"eps": 0.0}),
    ("test_invalid_eps_raises", {"eps": -1e-6}),

    ("test_invalid_init_alpha_raises", {"init_alpha": -1e-3}),
    ("test_invalid_init_alpha_raises", {"init_alpha": -1.0}),

    ("test_invalid_alpha_max_raises", {"alpha_max": 0.0}),
    ("test_invalid_alpha_max_raises", {"alpha_max": -1.0}),

    ("test_mhc_rejects_wrong_input_rank", {"bad_X": torch.randn(2, 8, 64)}),
    ("test_mhc_rejects_wrong_input_rank", {"bad_X": torch.randn(2, 8, 4, 64, 1)}),
]

n_passed = 0

for name in mhc_plain_tests:
    globals()[name]()
    n_passed += 1

for name, kwargs in mhc_param_tests:
    globals()[name](**kwargs)
    n_passed += 1

print(f"OK: {n_passed} mHC tests/casos pasaron.")

OK: 71 mHC tests/casos pasaron.


---

# DeepSeekMoE

In [67]:
# ============================================================
# Mini DeepSeek-V4 DeepSeekMoE-style FFN
# More canonical PyTorch implementation
# ============================================================

from dataclasses import dataclass
from typing import Optional, Dict, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# CONFIG
# ============================================================

@dataclass
class DeepSeekMoEConfig:
    d_model: int

    # Routed experts
    num_experts: int = 8
    top_k: int = 2

    expert_hidden_dim: Optional[int] = None
    expert_expansion_factor: float = 4.0
    expert_multiple_of: int = 1

    # Shared experts: now represented as a real ModuleList, not only as a width multiplier.
    shared_experts: int = 1
    shared_hidden_dim: Optional[int] = None
    shared_expansion_factor: float = 4.0

    # Routing
    # "learned": learned router W_r x -> scores -> top-k.
    # "hash": deterministic hash routing from input_ids; useful for early-layer hash MoE experiments.
    router_type: str = "learned"
    router_score_fn: str = "sqrt_softplus"
    normalize_topk_weights: bool = True
    topk_weight_scale: float = 1.0
    router_jitter_noise: float = 0.0
    hash_routing_stride: int = 1

    # Branch scaling
    routed_scale: float = 1.0
    shared_scale: float = 1.0

    # Regularization / init
    dropout: float = 0.0
    use_bias: bool = False
    init_std: float = 0.02

    # Optional losses / stats
    # This is not DeepSeek's full auxiliary-loss-free routing system; it is a small,
    # transparent mini-project balance objective and diagnostic.
    balance_loss_weight: float = 0.0
    sequence_balance_loss_weight: float = 0.0

    eps: float = 1e-9

    def validate(self) -> None:
        if self.d_model <= 0:
            raise ValueError(f"d_model must be > 0, got {self.d_model}")

        if self.num_experts <= 0:
            raise ValueError(f"num_experts must be > 0, got {self.num_experts}")

        if self.top_k <= 0:
            raise ValueError(f"top_k must be > 0, got {self.top_k}")

        if self.top_k > self.num_experts:
            raise ValueError(
                f"top_k must be <= num_experts, got top_k={self.top_k}, "
                f"num_experts={self.num_experts}"
            )

        if self.expert_hidden_dim is not None and self.expert_hidden_dim <= 0:
            raise ValueError(
                f"expert_hidden_dim must be > 0 when provided, got {self.expert_hidden_dim}"
            )

        if self.expert_expansion_factor <= 0:
            raise ValueError(
                f"expert_expansion_factor must be > 0, got {self.expert_expansion_factor}"
            )

        if self.expert_multiple_of <= 0:
            raise ValueError(
                f"expert_multiple_of must be > 0, got {self.expert_multiple_of}"
            )

        if self.shared_experts < 0:
            raise ValueError(f"shared_experts must be >= 0, got {self.shared_experts}")

        if self.shared_hidden_dim is not None and self.shared_hidden_dim <= 0:
            raise ValueError(
                f"shared_hidden_dim must be > 0 when provided, got {self.shared_hidden_dim}"
            )

        if self.shared_expansion_factor <= 0:
            raise ValueError(
                f"shared_expansion_factor must be > 0, got {self.shared_expansion_factor}"
            )

        allowed_router_types = {"learned", "hash"}
        if self.router_type not in allowed_router_types:
            raise ValueError(
                f"router_type must be one of {allowed_router_types}, got {self.router_type}"
            )

        allowed_score_fns = {"softmax", "sigmoid", "sqrt_softplus"}
        if self.router_score_fn not in allowed_score_fns:
            raise ValueError(
                f"router_score_fn must be one of {allowed_score_fns}, got {self.router_score_fn}"
            )

        if self.topk_weight_scale <= 0:
            raise ValueError(
                f"topk_weight_scale must be > 0, got {self.topk_weight_scale}"
            )

        if self.router_jitter_noise < 0:
            raise ValueError(
                f"router_jitter_noise must be >= 0, got {self.router_jitter_noise}"
            )

        if self.hash_routing_stride <= 0:
            raise ValueError(
                f"hash_routing_stride must be > 0, got {self.hash_routing_stride}"
            )

        if self.routed_scale < 0:
            raise ValueError(f"routed_scale must be >= 0, got {self.routed_scale}")

        if self.shared_scale < 0:
            raise ValueError(f"shared_scale must be >= 0, got {self.shared_scale}")

        if not (0.0 <= self.dropout < 1.0):
            raise ValueError(f"dropout must satisfy 0 <= dropout < 1, got {self.dropout}")

        if self.init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {self.init_std}")

        if self.balance_loss_weight < 0:
            raise ValueError(
                f"balance_loss_weight must be >= 0, got {self.balance_loss_weight}"
            )

        if self.sequence_balance_loss_weight < 0:
            raise ValueError(
                "sequence_balance_loss_weight must be >= 0, "
                f"got {self.sequence_balance_loss_weight}"
            )

        if self.eps <= 0:
            raise ValueError(f"eps must be > 0, got {self.eps}")

    def to_expert_config(self) -> SwiGLUMLPConfig:
        return SwiGLUMLPConfig(
            d_model=self.d_model,
            hidden_dim=self.expert_hidden_dim,
            expansion_factor=self.expert_expansion_factor,
            multiple_of=self.expert_multiple_of,
            dropout=self.dropout,
            use_bias=self.use_bias,
            init_std=self.init_std,
        )

    def to_shared_expert_config(self) -> SwiGLUMLPConfig:
        return SwiGLUMLPConfig(
            d_model=self.d_model,
            hidden_dim=self.shared_hidden_dim,
            expansion_factor=self.shared_expansion_factor,
            multiple_of=self.expert_multiple_of,
            dropout=self.dropout,
            use_bias=self.use_bias,
            init_std=self.init_std,
        )


# ============================================================
# DEEPSEEK MoE
# ============================================================

class DeepSeekMoE(nn.Module):
    """
    DeepSeekMoE-style FFN mini implementation.

    Core canonical pieces:
      - learned top-k router with sqrt(softplus(.)) affinity by default
      - optional deterministic hash routing
      - routed SwiGLU experts
      - real shared SwiGLU experts as a ModuleList
      - routed/shared branch scaling
      - top-k weight normalization and scaling
      - global and sequence-wise balance diagnostics/losses

    This intentionally does not implement industrial MoE infrastructure:
      - expert parallelism
      - fused dispatch kernels
      - all-to-all communication
      - FP4/FP8 kernels
      - auxiliary-loss-free routing internals
    """

    def __init__(self, config: DeepSeekMoEConfig):
        super().__init__()
        config.validate()

        self.config = config
        self.d_model = config.d_model
        self.num_experts = config.num_experts
        self.top_k = config.top_k
        self.router_type = config.router_type
        self.router_score_fn = config.router_score_fn
        self.normalize_topk_weights = config.normalize_topk_weights
        self.topk_weight_scale = config.topk_weight_scale
        self.router_jitter_noise = config.router_jitter_noise
        self.hash_routing_stride = config.hash_routing_stride
        self.routed_scale = config.routed_scale
        self.shared_scale = config.shared_scale
        self.balance_loss_weight = config.balance_loss_weight
        self.sequence_balance_loss_weight = config.sequence_balance_loss_weight
        self.eps = config.eps

        # The learned router exists even if router_type="hash" so the module keeps
        # a stable interface and can be switched later without reconstructing the class.
        self.router = nn.Linear(
            config.d_model,
            config.num_experts,
            bias=config.use_bias,
        )

        expert_config = config.to_expert_config()
        self.experts = nn.ModuleList(
            [SwiGLUMLP(expert_config) for _ in range(config.num_experts)]
        )

        shared_config = config.to_shared_expert_config()
        self.shared_experts = nn.ModuleList(
            [SwiGLUMLP(shared_config) for _ in range(config.shared_experts)]
        )

        self.reset_router_parameters()

    @property
    def has_shared_experts(self) -> bool:
        return len(self.shared_experts) > 0

    def reset_router_parameters(self) -> None:
        nn.init.normal_(self.router.weight, mean=0.0, std=self.config.init_std)
        if self.router.bias is not None:
            nn.init.zeros_(self.router.bias)

    def _validate_x(self, x: torch.Tensor) -> Tuple[int, int]:
        if x.dim() != 3:
            raise ValueError(
                f"DeepSeekMoE expects x with shape [B,T,d_model], got {tuple(x.shape)}"
            )

        B, T, D = x.shape
        if D != self.d_model:
            raise ValueError(f"Expected x.shape[-1] == d_model={self.d_model}, got {D}")

        return B, T

    def _validate_input_ids(self, input_ids: torch.Tensor, B: int, T: int) -> torch.Tensor:
        if input_ids.dim() != 2:
            raise ValueError(
                f"input_ids must have shape [B,T] for hash routing, got {tuple(input_ids.shape)}"
            )
        if input_ids.shape != (B, T):
            raise ValueError(
                f"input_ids must have shape {(B, T)} for hash routing, got {tuple(input_ids.shape)}"
            )
        return input_ids.long()

    def _compute_router_logits(self, x: torch.Tensor) -> torch.Tensor:
        router_logits = self.router(x)

        if self.training and self.router_jitter_noise > 0:
            router_logits = router_logits + self.router_jitter_noise * torch.randn_like(router_logits)

        return router_logits

    def _router_scores(self, router_logits: torch.Tensor) -> torch.Tensor:
        if self.router_score_fn == "softmax":
            return F.softmax(router_logits, dim=-1)

        if self.router_score_fn == "sigmoid":
            return torch.sigmoid(router_logits)

        if self.router_score_fn == "sqrt_softplus":
            return torch.sqrt(F.softplus(router_logits) + self.eps)

        raise RuntimeError(f"Unknown router_score_fn={self.router_score_fn}")

    def _topk_routing(
        self,
        router_scores: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        topk_scores, topk_indices = torch.topk(
            router_scores,
            k=self.top_k,
            dim=-1,
        )

        if self.normalize_topk_weights:
            denom = topk_scores.sum(dim=-1, keepdim=True).clamp_min(self.eps)
            topk_weights = topk_scores / denom
        else:
            topk_weights = topk_scores

        topk_weights = self.topk_weight_scale * topk_weights

        return topk_scores, topk_indices, topk_weights

    def _hash_routing(
        self,
        input_ids: torch.Tensor,
        B: int,
        T: int,
        device: torch.device,
        dtype: torch.dtype,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Deterministic hash routing for early-layer experiments.

        For every route r in 0..top_k-1:
            expert = (input_id + r * hash_routing_stride) % num_experts

        This is intentionally simple and deterministic. It is not meant to model
        learned affinity; it is a mini implementation of token-id based routing.
        """
        route_offsets = torch.arange(self.top_k, device=device, dtype=input_ids.dtype)
        route_offsets = route_offsets * self.hash_routing_stride

        topk_indices = (input_ids[..., None] + route_offsets[None, None, :]) % self.num_experts
        topk_indices = topk_indices.long()

        topk_scores = torch.ones(B, T, self.top_k, device=device, dtype=dtype)

        if self.normalize_topk_weights:
            topk_weights = topk_scores / float(self.top_k)
        else:
            topk_weights = topk_scores

        topk_weights = self.topk_weight_scale * topk_weights

        # Dense diagnostic tensors shaped like learned routing outputs.
        router_logits = torch.zeros(B, T, self.num_experts, device=device, dtype=dtype)
        router_scores = torch.zeros(B, T, self.num_experts, device=device, dtype=dtype)
        router_scores.scatter_add_(
            dim=-1,
            index=topk_indices,
            src=topk_weights.to(dtype=dtype),
        )

        return router_logits, router_scores, topk_scores, topk_indices, topk_weights

    def _compute_aux_stats(
        self,
        router_logits: torch.Tensor,
        router_scores: torch.Tensor,
        topk_indices: torch.Tensor,
        topk_weights: torch.Tensor,
        topk_scores: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        device = topk_indices.device
        dtype = router_scores.dtype

        B, T, K = topk_indices.shape
        total_routes = B * T * K

        flat_indices = topk_indices.reshape(-1)

        expert_counts = torch.bincount(
            flat_indices,
            minlength=self.num_experts,
        ).to(device=device, dtype=dtype)

        expert_fraction = expert_counts / max(total_routes, 1)

        target = torch.full(
            (self.num_experts,),
            fill_value=1.0 / self.num_experts,
            device=device,
            dtype=dtype,
        )

        raw_balance_loss = ((expert_fraction - target) ** 2).mean()
        balance_loss = self.balance_loss_weight * raw_balance_loss

        # Sequence-wise load statistics: [B,E].
        seq_one_hot = F.one_hot(topk_indices, num_classes=self.num_experts).to(dtype=dtype)
        # [B,T,K,E] -> [B,E]
        sequence_expert_counts = seq_one_hot.sum(dim=(1, 2))
        sequence_expert_fraction = sequence_expert_counts / max(T * K, 1)

        sequence_raw_balance_loss = ((sequence_expert_fraction - target[None, :]) ** 2).mean()
        sequence_balance_loss = self.sequence_balance_loss_weight * sequence_raw_balance_loss

        total_balance_loss = balance_loss + sequence_balance_loss

        # Router entropy is mainly meaningful for learned routing. For hash routing,
        # router_scores is sparse and deterministic; the entropy still serves as a diagnostic.
        router_probs_for_entropy = router_scores.float()
        prob_denom = router_probs_for_entropy.sum(dim=-1, keepdim=True).clamp_min(self.eps)
        router_probs_for_entropy = router_probs_for_entropy / prob_denom

        router_entropy = -(
            router_probs_for_entropy
            * torch.log(router_probs_for_entropy + self.eps)
        ).sum(dim=-1).mean().to(dtype=dtype)

        aux = {
            "router_logits": router_logits,
            "router_scores": router_scores,
            "topk_scores": topk_scores,
            "topk_indices": topk_indices,
            "topk_weights": topk_weights,
            "expert_counts": expert_counts,
            "expert_fraction": expert_fraction,
            "sequence_expert_counts": sequence_expert_counts,
            "sequence_expert_fraction": sequence_expert_fraction,
            "router_entropy": router_entropy,
            "raw_balance_loss": raw_balance_loss,
            "balance_loss": balance_loss,
            "sequence_raw_balance_loss": sequence_raw_balance_loss,
            "sequence_balance_loss": sequence_balance_loss,
            "total_balance_loss": total_balance_loss,
        }

        return aux

    def _dispatch_naive(
        self,
        x: torch.Tensor,
        topk_indices: torch.Tensor,
        topk_weights: torch.Tensor,
    ) -> torch.Tensor:
        B, T, D = x.shape
        K = topk_indices.shape[-1]

        x_flat = x.reshape(B * T, D)
        topk_indices_flat = topk_indices.reshape(B * T, K)
        topk_weights_flat = topk_weights.reshape(B * T, K)

        out_flat = torch.zeros_like(x_flat)

        for expert_id, expert in enumerate(self.experts):
            selected = topk_indices_flat == expert_id
            token_idx, route_idx = selected.nonzero(as_tuple=True)

            if token_idx.numel() == 0:
                continue

            x_e = x_flat[token_idx]
            y_e = expert(x_e[:, None, :])[:, 0, :]

            w_e = topk_weights_flat[token_idx, route_idx][:, None].to(dtype=y_e.dtype)

            source = y_e * w_e
            source = source.to(dtype=out_flat.dtype)

            out_flat.index_add_(
                dim=0,
                index=token_idx,
                source=source,
            )
        return out_flat.view(B, T, D)

    def _shared_forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.has_shared_experts:
            return torch.zeros_like(x)

        shared_out = torch.zeros_like(x)
        for shared_expert in self.shared_experts:
            shared_out = shared_out + shared_expert(x)

        return shared_out

    def forward(
        self,
        x: torch.Tensor,
        input_ids: Optional[torch.Tensor] = None,
        return_aux: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:
        B, T = self._validate_x(x)

        if self.router_type == "learned":
            router_logits = self._compute_router_logits(x)
            router_scores = self._router_scores(router_logits)
            topk_scores, topk_indices, topk_weights = self._topk_routing(router_scores)

        elif self.router_type == "hash":
            if input_ids is None:
                raise ValueError("input_ids must be provided when router_type='hash'")
            input_ids = self._validate_input_ids(input_ids, B=B, T=T).to(device=x.device)
            router_logits, router_scores, topk_scores, topk_indices, topk_weights = self._hash_routing(
                input_ids=input_ids,
                B=B,
                T=T,
                device=x.device,
                dtype=x.dtype,
            )

        else:
            raise RuntimeError(f"Unknown router_type={self.router_type}")

        routed_out = self._dispatch_naive(
            x=x,
            topk_indices=topk_indices,
            topk_weights=topk_weights,
        )

        shared_out = self._shared_forward(x)

        out = self.routed_scale * routed_out + self.shared_scale * shared_out

        if return_aux:
            aux = self._compute_aux_stats(
                router_logits=router_logits,
                router_scores=router_scores,
                topk_indices=topk_indices,
                topk_weights=topk_weights,
                topk_scores=topk_scores,
            )
            aux["routed_out"] = routed_out
            aux["shared_out"] = shared_out
            aux["routed_scale"] = torch.tensor(self.routed_scale, device=x.device, dtype=x.dtype)
            aux["shared_scale"] = torch.tensor(self.shared_scale, device=x.device, dtype=x.dtype)
            aux["router_type"] = self.router_type
            return out, aux

        return out


In [68]:
# ============================================================
# DeepSeekMoE smoke test
# ============================================================

moe_config = DeepSeekMoEConfig(
    d_model=64,

    num_experts=8,
    top_k=2,

    expert_hidden_dim=256,
    expert_expansion_factor=4.0,
    expert_multiple_of=1,

    shared_experts=1,
    shared_hidden_dim=None,
    shared_expansion_factor=4.0,

    router_score_fn="sqrt_softplus",
    normalize_topk_weights=True,

    router_jitter_noise=0.0,

    dropout=0.0,
    use_bias=False,
    init_std=0.02,

    balance_loss_weight=0.01,
)

moe = DeepSeekMoE(moe_config)

B, T, D = 2, 16, 64
x = torch.randn(B, T, D)

out, aux = moe(x, return_aux=True)

print("out:", out.shape)
print("router_logits:", aux["router_logits"].shape)
print("router_scores:", aux["router_scores"].shape)
print("topk_indices:", aux["topk_indices"].shape)
print("topk_weights:", aux["topk_weights"].shape)
print("expert_counts:", aux["expert_counts"].shape)
print("expert_fraction:", aux["expert_fraction"].shape)
print("router_entropy:", aux["router_entropy"])
print("balance_loss:", aux["balance_loss"])

assert out.shape == x.shape
assert aux["router_logits"].shape == (B, T, moe_config.num_experts)
assert aux["router_scores"].shape == (B, T, moe_config.num_experts)
assert aux["topk_indices"].shape == (B, T, moe_config.top_k)
assert aux["topk_weights"].shape == (B, T, moe_config.top_k)
assert aux["expert_counts"].shape == (moe_config.num_experts,)
assert aux["expert_fraction"].shape == (moe_config.num_experts,)

assert torch.isfinite(out).all()
assert torch.isfinite(aux["router_logits"]).all()
assert torch.isfinite(aux["router_scores"]).all()
assert torch.isfinite(aux["topk_weights"]).all()
assert torch.isfinite(aux["router_entropy"])
assert torch.isfinite(aux["balance_loss"])

assert (aux["topk_weights"] >= 0).all()

if moe_config.normalize_topk_weights:
    assert torch.allclose(
        aux["topk_weights"].sum(dim=-1),
        torch.ones_like(aux["topk_weights"].sum(dim=-1)),
        atol=1e-6,
        rtol=1e-6,
    )

print("DeepSeekMoE smoke test OK.")

out: torch.Size([2, 16, 64])
router_logits: torch.Size([2, 16, 8])
router_scores: torch.Size([2, 16, 8])
topk_indices: torch.Size([2, 16, 2])
topk_weights: torch.Size([2, 16, 2])
expert_counts: torch.Size([8])
expert_fraction: torch.Size([8])
router_entropy: tensor(2.0782, grad_fn=<NegBackward0>)
balance_loss: tensor(5.4932e-06)
DeepSeekMoE smoke test OK.


In [69]:
# ============================================================
# Drop-in FFN replacement check
# ============================================================

B, T, D = 2, 16, 64

norm = RMSNorm(dim=D)
moe = DeepSeekMoE(
    DeepSeekMoEConfig(
        d_model=D,
        num_experts=4,
        top_k=2,
        expert_hidden_dim=128,
        shared_experts=1,
        router_score_fn="sqrt_softplus",
        normalize_topk_weights=True,
        dropout=0.0,
        use_bias=False,
        init_std=0.02,
        balance_loss_weight=0.01,
    )
)

x = torch.randn(B, T, D)

residual = x
ffn_out, aux = moe(norm(x), return_aux=True)
x_next = residual + ffn_out

assert x_next.shape == x.shape
assert torch.isfinite(x_next).all()
assert torch.isfinite(aux["balance_loss"])

print("MoE drop-in FFN replacement OK.")

MoE drop-in FFN replacement OK.


In [70]:
# @title
# ============================================================
# DeepSeekMoE-style FFN tests
# ============================================================

import pytest
import torch
import torch.nn as nn


# ============================================================
# Helpers
# ============================================================

def make_moe_config(**overrides):
    cfg = dict(
        d_model=64,

        num_experts=4,
        top_k=2,

        expert_hidden_dim=128,
        expert_expansion_factor=4.0,
        expert_multiple_of=1,

        shared_experts=1,
        shared_hidden_dim=None,
        shared_expansion_factor=4.0,

        router_score_fn="sqrt_softplus",
        normalize_topk_weights=True,
        topk_weight_scale=1.0,

        router_type="learned",  # "learned" | "hash"
        router_jitter_noise=0.0,

        routed_scale=1.0,
        shared_scale=1.0,

        dropout=0.0,
        use_bias=True,
        init_std=0.02,

        balance_loss_weight=0.01,
        sequence_balance_loss_weight=0.01,

        eps=1e-9,
    )
    cfg.update(overrides)
    return DeepSeekMoEConfig(**cfg)


def make_moe(**overrides):
    return DeepSeekMoE(make_moe_config(**overrides))


def make_moe_input(B=2, T=8, D=64, dtype=torch.float32, device="cpu"):
    return torch.randn(B, T, D, dtype=dtype, device=device)


def zero_router(moe: DeepSeekMoE):
    with torch.no_grad():
        moe.router.weight.zero_()
        if moe.router.bias is not None:
            moe.router.bias.zero_()


def force_router_bias(moe: DeepSeekMoE, bias_values):
    """
    Forces routing to be independent of x by zeroing router weights
    and setting router bias.
    Requires use_bias=True.
    """
    assert moe.router.bias is not None, "This helper requires router bias."

    with torch.no_grad():
        moe.router.weight.zero_()
        moe.router.bias.copy_(torch.tensor(
            bias_values,
            dtype=moe.router.bias.dtype,
            device=moe.router.bias.device,
        ))


# ============================================================
# A. Config tests
# ============================================================

def test_valid_moe_config_builds():
    config = DeepSeekMoEConfig(
        d_model=256,
        num_experts=8,
        top_k=2,
        expert_expansion_factor=4.0,
        shared_experts=2,
        router_score_fn="sqrt_softplus",
        normalize_topk_weights=True,
        topk_weight_scale=1.0,
        router_type="learned",
        routed_scale=1.0,
        shared_scale=1.0,
        balance_loss_weight=0.01,
        sequence_balance_loss_weight=0.01,
    )

    moe = DeepSeekMoE(config)

    assert moe.d_model == 256
    assert moe.num_experts == 8
    assert moe.top_k == 2
    assert moe.router_score_fn == "sqrt_softplus"
    assert moe.router_type == "learned"
    assert moe.routed_scale == 1.0
    assert moe.shared_scale == 1.0
    assert moe.topk_weight_scale == 1.0

    assert len(moe.experts) == 8
    assert hasattr(moe, "shared_experts")
    assert len(moe.shared_experts) == 2


@pytest.mark.parametrize(
    "kwargs",
    [
        {"d_model": 0},
        {"d_model": -1},
        {"num_experts": 0},
        {"num_experts": -1},
        {"top_k": 0},
        {"top_k": -1},
        {"num_experts": 4, "top_k": 5},
    ],
)
def test_invalid_moe_model_dims_raise(kwargs):
    with pytest.raises(ValueError):
        DeepSeekMoE(make_moe_config(**kwargs))


@pytest.mark.parametrize(
    "kwargs",
    [
        {"expert_hidden_dim": 0},
        {"expert_hidden_dim": -1},
        {"expert_expansion_factor": 0.0},
        {"expert_expansion_factor": -1.0},
        {"expert_multiple_of": 0},
        {"expert_multiple_of": -1},
        {"shared_experts": -1},
        {"shared_hidden_dim": 0},
        {"shared_hidden_dim": -1},
        {"shared_expansion_factor": 0.0},
        {"shared_expansion_factor": -1.0},
    ],
)
def test_invalid_expert_config_raises(kwargs):
    with pytest.raises(ValueError):
        DeepSeekMoE(make_moe_config(**kwargs))




@pytest.mark.parametrize("router_score_fn", ["bad", "relu", "topk_softmax"])
def test_invalid_router_score_fn_raises(router_score_fn):
    with pytest.raises(ValueError):
        DeepSeekMoE(make_moe_config(router_score_fn=router_score_fn))


@pytest.mark.parametrize(
    "kwargs",
    [
        {"dropout": -0.1},
        {"dropout": 1.0},
        {"dropout": 1.5},
        {"init_std": 0.0},
        {"init_std": -0.01},
        {"balance_loss_weight": -0.1},
        {"sequence_balance_loss_weight": -0.1},
        {"router_jitter_noise": -0.1},
        {"topk_weight_scale": 0.0},
        {"topk_weight_scale": -1.0},
        {"routed_scale": -1.0},
        {"shared_scale": -1.0},
        {"eps": 0.0},
        {"eps": -1e-9},
    ],
)
def test_invalid_dropout_init_balance_raise(kwargs):
    with pytest.raises(ValueError):
        DeepSeekMoE(make_moe_config(**kwargs))


@pytest.mark.parametrize("router_type", ["bad", "dense", "random"])
def test_invalid_router_type_raises(router_type):
    with pytest.raises(ValueError):
        DeepSeekMoE(make_moe_config(router_type=router_type))

# ============================================================
# B. Internal structure tests
# ============================================================

def test_moe_has_router_and_experts():
    config = make_moe_config(num_experts=4)
    moe = DeepSeekMoE(config)

    assert hasattr(moe, "router")
    assert hasattr(moe, "experts")
    assert len(moe.experts) == config.num_experts


def test_all_experts_are_swiglu_mlp():
    moe = make_moe(num_experts=4)

    for expert in moe.experts:
        assert isinstance(expert, SwiGLUMLP)


def test_shared_experts_exist_when_enabled():
    moe = make_moe(shared_experts=2)

    assert hasattr(moe, "shared_experts")
    assert isinstance(moe.shared_experts, nn.ModuleList)
    assert len(moe.shared_experts) == 2

    for expert in moe.shared_experts:
        assert isinstance(expert, SwiGLUMLP)


def test_shared_experts_empty_when_disabled():
    moe = make_moe(shared_experts=0)

    assert hasattr(moe, "shared_experts")
    assert isinstance(moe.shared_experts, nn.ModuleList)
    assert len(moe.shared_experts) == 0


# ============================================================
# C. Router tests
# ============================================================

def test_router_logits_shape():
    B, T, D = 2, 8, 64
    E = 4

    moe = make_moe(d_model=D, num_experts=E)
    x = make_moe_input(B=B, T=T, D=D)

    router_logits = moe._compute_router_logits(x)

    assert router_logits.shape == (B, T, E)


@pytest.mark.parametrize("router_score_fn", ["softmax", "sigmoid", "sqrt_softplus"])
def test_router_scores_positive_for_all_score_fns(router_score_fn):
    moe = make_moe(router_score_fn=router_score_fn)

    x = make_moe_input()
    logits = moe._compute_router_logits(x)
    scores = moe._router_scores(logits)

    assert scores.shape == logits.shape
    assert (scores >= 0).all()
    assert torch.isfinite(scores).all()


def test_softmax_router_scores_sum_to_one():
    moe = make_moe(router_score_fn="softmax")

    x = make_moe_input()
    logits = moe._compute_router_logits(x)
    scores = moe._router_scores(logits)

    assert torch.allclose(
        scores.sum(dim=-1),
        torch.ones_like(scores.sum(dim=-1)),
        atol=1e-6,
        rtol=1e-6,
    )


def test_topk_shapes():
    B, T, D = 2, 8, 64
    E, K = 4, 2

    moe = make_moe(num_experts=E, top_k=K)

    x = make_moe_input(B=B, T=T, D=D)
    logits = moe._compute_router_logits(x)
    scores = moe._router_scores(logits)

    topk_scores, topk_indices, topk_weights = moe._topk_routing(scores)

    assert topk_scores.shape == (B, T, K)
    assert topk_indices.shape == (B, T, K)
    assert topk_weights.shape == (B, T, K)


def test_topk_indices_in_valid_range():
    E = 4
    moe = make_moe(num_experts=E, top_k=2)

    x = make_moe_input()
    scores = moe._router_scores(moe._compute_router_logits(x))
    _, topk_indices, _ = moe._topk_routing(scores)

    assert (topk_indices >= 0).all()
    assert (topk_indices < E).all()


def test_topk_weights_sum_to_scale_when_normalized():
    scale = 1.25

    moe = make_moe(
        normalize_topk_weights=True,
        topk_weight_scale=scale,
    )

    x = make_moe_input()
    scores = moe._router_scores(moe._compute_router_logits(x))
    _, _, topk_weights = moe._topk_routing(scores)

    expected = torch.full_like(
        topk_weights.sum(dim=-1),
        fill_value=scale,
    )

    assert torch.allclose(
        topk_weights.sum(dim=-1),
        expected,
        atol=1e-6,
        rtol=1e-6,
    )


def test_topk_weights_not_necessarily_sum_to_one_when_not_normalized():
    moe = make_moe(
        normalize_topk_weights=False,
        router_score_fn="sqrt_softplus",
    )

    x = make_moe_input()
    scores = moe._router_scores(moe._compute_router_logits(x))
    _, _, topk_weights = moe._topk_routing(scores)

    assert (topk_weights >= 0).all()
    assert torch.isfinite(topk_weights).all()


# ============================================================
# D. Forward tests
# ============================================================

def test_moe_output_shape_matches_input():
    moe = make_moe()

    x = make_moe_input()
    out = moe(x)

    assert out.shape == x.shape


@pytest.mark.parametrize(
    "bad_x",
    [
        torch.randn(8, 64),
        torch.randn(2, 8, 64, 1),
    ],
)
def test_moe_rejects_wrong_input_rank(bad_x):
    moe = make_moe()

    with pytest.raises(ValueError):
        moe(bad_x)


def test_moe_rejects_wrong_hidden_size():
    moe = make_moe(d_model=64)

    x = torch.randn(2, 8, 32)

    with pytest.raises(ValueError):
        moe(x)


def test_forward_returns_aux_when_requested():
    B, T, D = 2, 8, 64
    E, K = 4, 2

    moe = make_moe(d_model=D, num_experts=E, top_k=K)
    x = make_moe_input(B=B, T=T, D=D)

    out, aux = moe(x, return_aux=True)

    assert out.shape == x.shape
    assert isinstance(aux, dict)

    required_keys = [
        "router_logits",
        "router_scores",
        "topk_indices",
        "topk_scores",
        "topk_weights",

        "expert_counts",
        "expert_fraction",
        "sequence_expert_counts",
        "sequence_expert_fraction",

        "router_entropy",

        "balance_loss",
        "raw_balance_loss",
        "sequence_balance_loss",
        "sequence_raw_balance_loss",
        "total_balance_loss",

        "routed_out",
        "shared_out",
        "routed_scale",
        "shared_scale",
        "router_type",
    ]

    for key in required_keys:
        assert key in aux, f"Missing aux key: {key}. Available keys: {list(aux.keys())}"

    assert aux["router_logits"].shape == (B, T, E)
    assert aux["router_scores"].shape == (B, T, E)
    assert aux["topk_indices"].shape == (B, T, K)
    assert aux["topk_scores"].shape == (B, T, K)
    assert aux["topk_weights"].shape == (B, T, K)

    assert aux["expert_counts"].shape == (E,)
    assert aux["expert_fraction"].shape == (E,)

    assert aux["sequence_expert_counts"].shape == (B, E)
    assert aux["sequence_expert_fraction"].shape == (B, E)

    assert aux["routed_out"].shape == x.shape
    assert aux["shared_out"].shape == x.shape

    assert aux["balance_loss"].dim() == 0
    assert aux["raw_balance_loss"].dim() == 0
    assert aux["sequence_balance_loss"].dim() == 0
    assert aux["sequence_raw_balance_loss"].dim() == 0
    assert aux["total_balance_loss"].dim() == 0

    assert aux["router_type"] == moe.router_type


def test_forward_without_aux_returns_tensor():
    moe = make_moe()
    x = make_moe_input()

    out = moe(x, return_aux=False)

    assert isinstance(out, torch.Tensor)
    assert out.shape == x.shape


# ============================================================
# E. Dispatch / combine tests
# ============================================================

def test_only_selected_experts_contribute():
    B, T, D = 2, 8, 64

    moe = make_moe(
        d_model=D,
        num_experts=4,
        top_k=1,
        shared_experts=0,
        dropout=0.0,
        use_bias=True,
        router_score_fn="softmax",
        normalize_topk_weights=True,
    )
    moe.eval()

    # Force expert 0 to always win.
    force_router_bias(moe, [10.0, 0.0, -1.0, -2.0])

    x = make_moe_input(B=B, T=T, D=D)

    out = moe(x)
    expected = moe.experts[0](x)

    assert torch.allclose(out, expected, atol=1e-6, rtol=1e-5)


def test_top2_combination_matches_manual():
    B, T, D = 2, 8, 64

    moe = make_moe(
        d_model=D,
        num_experts=4,
        top_k=2,
        shared_experts=0,
        dropout=0.0,
        use_bias=True,
        router_score_fn="softmax",
        normalize_topk_weights=True,
    )
    moe.eval()

    # Force top-2: expert 0 then expert 1.
    force_router_bias(moe, [4.0, 2.0, -10.0, -20.0])

    x = make_moe_input(B=B, T=T, D=D)

    out, aux = moe(x, return_aux=True)

    router_scores = aux["router_scores"]
    top2_scores = router_scores[..., [0, 1]]
    weights = top2_scores / top2_scores.sum(dim=-1, keepdim=True)

    expected = (
        weights[..., 0:1] * moe.experts[0](x)
        + weights[..., 1:2] * moe.experts[1](x)
    )

    assert torch.allclose(out, expected, atol=1e-6, rtol=1e-5)


def test_expert_with_no_tokens_is_skipped_safely():
    B, T, D = 2, 8, 64

    moe = make_moe(
        d_model=D,
        num_experts=4,
        top_k=1,
        shared_experts=0,
        use_bias=True,
        router_score_fn="softmax",
    )
    moe.eval()

    # Expert 0 always wins; experts 1,2,3 get no tokens.
    force_router_bias(moe, [10.0, 0.0, -1.0, -2.0])

    x = make_moe_input(B=B, T=T, D=D)

    out, aux = moe(x, return_aux=True)

    assert out.shape == x.shape
    assert torch.isfinite(out).all()
    assert aux["expert_counts"][0] == B * T
    assert aux["expert_counts"][1:].sum() == 0


def test_shared_experts_add_to_routed_output():
    B, T, D = 2, 8, 64

    base_kwargs = dict(
        d_model=D,
        num_experts=4,
        top_k=1,
        dropout=0.0,
        use_bias=True,
        router_score_fn="softmax",
        normalize_topk_weights=True,
        routed_scale=1.0,
        shared_scale=1.0,
    )

    moe = make_moe(**base_kwargs, shared_experts=2)
    moe.eval()

    force_router_bias(moe, [10.0, 0.0, -1.0, -2.0])

    x = make_moe_input(B=B, T=T, D=D)

    out = moe(x)

    routed = moe.experts[0](x)
    shared = sum(shared_expert(x) for shared_expert in moe.shared_experts)

    expected = routed + shared

    assert torch.allclose(out, expected, atol=1e-6, rtol=1e-5)


# ============================================================
# F. Aux stats tests
# ============================================================

def test_expert_counts_sum_to_total_routes():
    B, T, D = 2, 8, 64
    E, K = 4, 2

    moe = make_moe(d_model=D, num_experts=E, top_k=K)
    x = make_moe_input(B=B, T=T, D=D)

    _, aux = moe(x, return_aux=True)

    assert aux["expert_counts"].sum() == B * T * K


def test_expert_fraction_sums_to_one():
    moe = make_moe()

    x = make_moe_input()
    _, aux = moe(x, return_aux=True)

    assert torch.allclose(
        aux["expert_fraction"].sum(),
        torch.tensor(1.0, dtype=aux["expert_fraction"].dtype),
        atol=1e-6,
        rtol=1e-6,
    )


def test_router_entropy_is_finite():
    moe = make_moe()

    x = make_moe_input()
    _, aux = moe(x, return_aux=True)

    assert torch.isfinite(aux["router_entropy"])
    assert aux["router_entropy"] >= 0


def test_balance_loss_is_scalar_and_finite():
    moe = make_moe(balance_loss_weight=0.01)

    x = make_moe_input()
    _, aux = moe(x, return_aux=True)

    assert aux["balance_loss"].dim() == 0
    assert aux["raw_balance_loss"].dim() == 0
    assert torch.isfinite(aux["balance_loss"])
    assert torch.isfinite(aux["raw_balance_loss"])


def test_balance_loss_zero_weight_returns_zero_weighted_loss():
    moe = make_moe(balance_loss_weight=0.0)

    x = make_moe_input()
    _, aux = moe(x, return_aux=True)

    assert torch.allclose(
        aux["balance_loss"],
        torch.zeros_like(aux["balance_loss"]),
        atol=0.0,
        rtol=0.0,
    )
    assert torch.isfinite(aux["raw_balance_loss"])


# ============================================================
# G. Router jitter tests
# ============================================================

def test_router_jitter_disabled_is_deterministic():
    moe = make_moe(router_jitter_noise=0.0, dropout=0.0)
    moe.train()

    x = make_moe_input(B=4, T=16, D=64)

    _, aux1 = moe(x, return_aux=True)
    _, aux2 = moe(x, return_aux=True)

    assert torch.equal(aux1["router_logits"], aux2["router_logits"])
    assert torch.equal(aux1["topk_indices"], aux2["topk_indices"])


def test_router_jitter_disabled_in_eval():
    moe = make_moe(router_jitter_noise=1.0, dropout=0.0)
    moe.eval()

    x = make_moe_input(B=4, T=16, D=64)

    _, aux1 = moe(x, return_aux=True)
    _, aux2 = moe(x, return_aux=True)

    assert torch.equal(aux1["router_logits"], aux2["router_logits"])
    assert torch.equal(aux1["topk_indices"], aux2["topk_indices"])


def test_router_jitter_active_in_train_changes_routes_or_logits():
    moe = make_moe(router_jitter_noise=1.0, dropout=0.0)
    moe.train()

    x = make_moe_input(B=8, T=32, D=64)

    _, aux1 = moe(x, return_aux=True)
    _, aux2 = moe(x, return_aux=True)

    logits_changed = not torch.equal(aux1["router_logits"], aux2["router_logits"])
    routes_changed = not torch.equal(aux1["topk_indices"], aux2["topk_indices"])

    assert logits_changed or routes_changed


# ============================================================
# H. Gradient tests
# ============================================================

def test_backward_computes_gradients_for_router_and_used_experts():
    B, T, D = 2, 8, 64

    moe = make_moe(
        d_model=D,
        num_experts=4,
        top_k=2,
        shared_experts=2,
        balance_loss_weight=0.01,
        sequence_balance_loss_weight=0.01,
        dropout=0.0,
        use_bias=True,
    )

    x = make_moe_input(B=B, T=T, D=D)
    x.requires_grad_(True)

    out, aux = moe(x, return_aux=True)
    loss = out.sum() + aux["balance_loss"] + aux["sequence_balance_loss"]
    loss.backward()

    assert x.grad is not None
    assert torch.isfinite(x.grad).all()

    assert moe.router.weight.grad is not None
    assert torch.isfinite(moe.router.weight.grad).all()

    used_experts = aux["topk_indices"].unique().tolist()

    at_least_one_expert_grad = False

    for expert_id in used_experts:
        expert = moe.experts[expert_id]

        for name, param in expert.named_parameters():
            if param.grad is not None:
                at_least_one_expert_grad = True
                assert torch.isfinite(param.grad).all(), f"Non-finite grad expert {expert_id}.{name}"

    assert at_least_one_expert_grad

    for shared_id, shared_expert in enumerate(moe.shared_experts):
        for name, param in shared_expert.named_parameters():
            assert param.grad is not None, f"Missing shared expert grad for shared_experts.{shared_id}.{name}"
            assert torch.isfinite(param.grad).all(), f"Non-finite shared expert grad for shared_experts.{shared_id}.{name}"


def test_unused_expert_may_have_no_grad():
    B, T, D = 2, 8, 64

    moe = make_moe(
        d_model=D,
        num_experts=4,
        top_k=1,
        shared_experts=0,
        use_bias=True,
        router_score_fn="softmax",
        dropout=0.0,
    )

    moe.train()

    # Only expert 0 used.
    force_router_bias(moe, [10.0, 0.0, -1.0, -2.0])

    x = make_moe_input(B=B, T=T, D=D)
    x.requires_grad_(True)

    out, aux = moe(x, return_aux=True)
    loss = out.sum()
    loss.backward()

    assert x.grad is not None
    assert moe.router.weight.grad is not None

    # Expert 0 should get grad.
    assert any(
        param.grad is not None
        for param in moe.experts[0].parameters()
    )

    # Other experts may have no grad. This should not crash.
    for expert_id in [1, 2, 3]:
        for param in moe.experts[expert_id].parameters():
            if param.grad is not None:
                assert torch.isfinite(param.grad).all()


# ============================================================
# I. Integration tests
# ============================================================

def test_moe_can_replace_mlp_interface():
    B, T, D = 2, 8, 64

    moe = make_moe(d_model=D)

    x = make_moe_input(B=B, T=T, D=D)

    out = moe(x)

    assert out.shape == x.shape
    assert torch.isfinite(out).all()


def test_moe_inside_transformer_like_residual():
    B, T, D = 2, 8, 64

    norm = RMSNorm(dim=D)
    moe = make_moe(d_model=D)

    x = make_moe_input(B=B, T=T, D=D)

    out = x + moe(norm(x))

    assert out.shape == x.shape
    assert torch.isfinite(out).all()


def test_mhc_wraps_moe():
    B, T, D, N = 2, 8, 64, 4

    mhc = ManifoldHyperConnection(
        ManifoldHyperConnectionConfig(
            d_model=D,
            n_hc=N,
            sinkhorn_iters=30,
            eps=1e-6,
            init_alpha=1e-3,
            dynamic=True,
        )
    )

    moe = make_moe(
        d_model=D,
        num_experts=4,
        top_k=2,
        shared_experts=2,
        dropout=0.0,
        router_type="learned",
    )

    X = make_X(B=B, T=T, n_hc=N, D=D)

    X_next = mhc(
        X,
        sublayer=lambda x_sub: moe(x_sub),
    )

    assert X_next.shape == X.shape
    assert torch.isfinite(X_next).all()



def test_topk_weights_sum_to_one_when_normalized_default_scale():
    moe = make_moe(
        normalize_topk_weights=True,
        topk_weight_scale=1.0,
    )

    x = make_moe_input()
    scores = moe._router_scores(moe._compute_router_logits(x))
    _, _, topk_weights = moe._topk_routing(scores)

    assert torch.allclose(
        topk_weights.sum(dim=-1),
        torch.ones_like(topk_weights.sum(dim=-1)),
        atol=1e-6,
        rtol=1e-6,
    )

def test_routed_and_shared_scales_are_applied():
    B, T, D = 2, 8, 64

    routed_scale = 0.5
    shared_scale = 2.0

    moe = make_moe(
        d_model=D,
        num_experts=4,
        top_k=1,
        shared_experts=2,
        dropout=0.0,
        use_bias=True,
        router_score_fn="softmax",
        normalize_topk_weights=True,
        routed_scale=routed_scale,
        shared_scale=shared_scale,
    )
    moe.eval()

    force_router_bias(moe, [10.0, 0.0, -1.0, -2.0])

    x = make_moe_input(B=B, T=T, D=D)

    out = moe(x)

    routed = moe.experts[0](x)
    shared = sum(shared_expert(x) for shared_expert in moe.shared_experts)

    expected = routed_scale * routed + shared_scale * shared

    assert torch.allclose(out, expected, atol=1e-6, rtol=1e-5)


def test_sequence_expert_fraction_sums_to_one_per_sequence():
    B, T, D = 3, 8, 64
    E, K = 4, 2

    moe = make_moe(d_model=D, num_experts=E, top_k=K)
    x = make_moe_input(B=B, T=T, D=D)

    _, aux = moe(x, return_aux=True)

    seq_frac = aux["sequence_expert_fraction"]

    assert seq_frac.shape == (B, E)

    assert torch.allclose(
        seq_frac.sum(dim=-1),
        torch.ones(B, dtype=seq_frac.dtype, device=seq_frac.device),
        atol=1e-6,
        rtol=1e-6,
    )


def test_sequence_balance_loss_is_scalar_and_finite():
    moe = make_moe(sequence_balance_loss_weight=0.01)

    x = make_moe_input()
    _, aux = moe(x, return_aux=True)

    assert aux["sequence_balance_loss"].dim() == 0
    assert aux["sequence_raw_balance_loss"].dim() == 0
    assert torch.isfinite(aux["sequence_balance_loss"])
    assert torch.isfinite(aux["sequence_raw_balance_loss"])



def test_sequence_balance_loss_zero_weight_returns_zero_weighted_loss():
    moe = make_moe(sequence_balance_loss_weight=0.0)

    x = make_moe_input()
    _, aux = moe(x, return_aux=True)

    assert torch.allclose(
        aux["sequence_balance_loss"],
        torch.zeros_like(aux["sequence_balance_loss"]),
        atol=0.0,
        rtol=0.0,
    )

    assert torch.isfinite(aux["sequence_raw_balance_loss"])


def test_hash_router_requires_input_ids():
    moe = make_moe(router_type="hash")

    x = make_moe_input()

    with pytest.raises(ValueError):
        moe(x)


def test_hash_router_forward_with_input_ids():
    B, T, D = 2, 8, 64

    moe = make_moe(
        d_model=D,
        num_experts=4,
        top_k=2,
        router_type="hash",
        shared_experts=1,
    )

    x = make_moe_input(B=B, T=T, D=D)
    input_ids = torch.arange(B * T).view(B, T)

    out, aux = moe(
        x,
        input_ids=input_ids,
        return_aux=True,
    )

    assert out.shape == x.shape
    assert aux["topk_indices"].shape == (B, T, 2)
    assert torch.isfinite(out).all()


def test_hash_router_is_deterministic():
    B, T, D = 2, 8, 64

    moe = make_moe(
        d_model=D,
        num_experts=4,
        top_k=2,
        router_type="hash",
        shared_experts=0,
        dropout=0.0,
    )
    moe.eval()

    x = make_moe_input(B=B, T=T, D=D)
    input_ids = torch.arange(B * T).view(B, T)

    _, aux1 = moe(x, input_ids=input_ids, return_aux=True)
    _, aux2 = moe(x, input_ids=input_ids, return_aux=True)

    assert torch.equal(aux1["topk_indices"], aux2["topk_indices"])
    assert torch.equal(aux1["topk_weights"], aux2["topk_weights"])



In [71]:
# ============================================================
# Run ONLY DeepSeekMoE tests in Jupyter
# Handles pytest-style parametrized tests manually
# ============================================================

moe_plain_tests = [
    # A. Config
    "test_valid_moe_config_builds",

    # B. Internal structure
    "test_moe_has_router_and_experts",
    "test_all_experts_are_swiglu_mlp",
    "test_shared_experts_exist_when_enabled",
    "test_shared_experts_empty_when_disabled",

    # C. Router
    "test_router_logits_shape",
    "test_softmax_router_scores_sum_to_one",
    "test_topk_shapes",
    "test_topk_indices_in_valid_range",
    "test_topk_weights_sum_to_scale_when_normalized",
    "test_topk_weights_sum_to_one_when_normalized_default_scale",
    "test_topk_weights_not_necessarily_sum_to_one_when_not_normalized",

    # D. Forward
    "test_moe_output_shape_matches_input",
    "test_moe_rejects_wrong_hidden_size",
    "test_forward_returns_aux_when_requested",
    "test_forward_without_aux_returns_tensor",

    # E. Dispatch / combine
    "test_only_selected_experts_contribute",
    "test_top2_combination_matches_manual",
    "test_expert_with_no_tokens_is_skipped_safely",
    "test_shared_experts_add_to_routed_output",
    "test_routed_and_shared_scales_are_applied",

    # F. Aux stats
    "test_expert_counts_sum_to_total_routes",
    "test_expert_fraction_sums_to_one",
    "test_router_entropy_is_finite",
    "test_balance_loss_is_scalar_and_finite",
    "test_balance_loss_zero_weight_returns_zero_weighted_loss",
    "test_sequence_expert_fraction_sums_to_one_per_sequence",
    "test_sequence_balance_loss_is_scalar_and_finite",
    "test_sequence_balance_loss_zero_weight_returns_zero_weighted_loss",

    # G. Router jitter
    "test_router_jitter_disabled_is_deterministic",
    "test_router_jitter_disabled_in_eval",
    "test_router_jitter_active_in_train_changes_routes_or_logits",

    # Hash routing
    "test_hash_router_requires_input_ids",
    "test_hash_router_forward_with_input_ids",
    "test_hash_router_is_deterministic",

    # H. Gradients
    "test_backward_computes_gradients_for_router_and_used_experts",
    "test_unused_expert_may_have_no_grad",

    # I. Integration
    "test_moe_can_replace_mlp_interface",
    "test_moe_inside_transformer_like_residual",
    "test_mhc_wraps_moe",
]

moe_param_tests = [
    # Model dims
    ("test_invalid_moe_model_dims_raise", {"kwargs": {"d_model": 0}}),
    ("test_invalid_moe_model_dims_raise", {"kwargs": {"d_model": -1}}),
    ("test_invalid_moe_model_dims_raise", {"kwargs": {"num_experts": 0}}),
    ("test_invalid_moe_model_dims_raise", {"kwargs": {"num_experts": -1}}),
    ("test_invalid_moe_model_dims_raise", {"kwargs": {"top_k": 0}}),
    ("test_invalid_moe_model_dims_raise", {"kwargs": {"top_k": -1}}),
    ("test_invalid_moe_model_dims_raise", {"kwargs": {"num_experts": 4, "top_k": 5}}),

    # Expert config
    ("test_invalid_expert_config_raises", {"kwargs": {"expert_hidden_dim": 0}}),
    ("test_invalid_expert_config_raises", {"kwargs": {"expert_hidden_dim": -1}}),
    ("test_invalid_expert_config_raises", {"kwargs": {"expert_expansion_factor": 0.0}}),
    ("test_invalid_expert_config_raises", {"kwargs": {"expert_expansion_factor": -1.0}}),
    ("test_invalid_expert_config_raises", {"kwargs": {"expert_multiple_of": 0}}),
    ("test_invalid_expert_config_raises", {"kwargs": {"expert_multiple_of": -1}}),
    ("test_invalid_expert_config_raises", {"kwargs": {"shared_experts": -1}}),
    ("test_invalid_expert_config_raises", {"kwargs": {"shared_hidden_dim": 0}}),
    ("test_invalid_expert_config_raises", {"kwargs": {"shared_hidden_dim": -1}}),
    ("test_invalid_expert_config_raises", {"kwargs": {"shared_expansion_factor": 0.0}}),
    ("test_invalid_expert_config_raises", {"kwargs": {"shared_expansion_factor": -1.0}}),

    # Router score fn
    ("test_invalid_router_score_fn_raises", {"router_score_fn": "bad"}),
    ("test_invalid_router_score_fn_raises", {"router_score_fn": "relu"}),
    ("test_invalid_router_score_fn_raises", {"router_score_fn": "topk_softmax"}),

    # Router type
    ("test_invalid_router_type_raises", {"router_type": "bad"}),
    ("test_invalid_router_type_raises", {"router_type": "dense"}),
    ("test_invalid_router_type_raises", {"router_type": "random"}),

    # Dropout / init / balance / scales
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"dropout": -0.1}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"dropout": 1.0}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"dropout": 1.5}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"init_std": 0.0}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"init_std": -0.01}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"balance_loss_weight": -0.1}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"sequence_balance_loss_weight": -0.1}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"router_jitter_noise": -0.1}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"topk_weight_scale": 0.0}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"topk_weight_scale": -1.0}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"routed_scale": -1.0}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"shared_scale": -1.0}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"eps": 0.0}}),
    ("test_invalid_dropout_init_balance_raise", {"kwargs": {"eps": -1e-9}}),

    # Router score positivity
    ("test_router_scores_positive_for_all_score_fns", {"router_score_fn": "softmax"}),
    ("test_router_scores_positive_for_all_score_fns", {"router_score_fn": "sigmoid"}),
    ("test_router_scores_positive_for_all_score_fns", {"router_score_fn": "sqrt_softplus"}),

    # Input validation
    ("test_moe_rejects_wrong_input_rank", {"bad_x": torch.randn(8, 64)}),
    ("test_moe_rejects_wrong_input_rank", {"bad_x": torch.randn(2, 8, 64, 1)}),
]

n_passed = 0

for name in moe_plain_tests:
    globals()[name]()
    n_passed += 1

for name, kwargs in moe_param_tests:
    globals()[name](**kwargs)
    n_passed += 1

print(f"OK: {n_passed} DeepSeekMoE tests/casos pasaron.")

OK: 83 DeepSeekMoE tests/casos pasaron.


---


# MTP

In [72]:
# ============================================================
# Mini DeepSeek-V4 MTP / Multi-Token Prediction
# Auxiliary future-token prediction heads - canonical mini version
# ============================================================

from dataclasses import dataclass
from typing import Optional, Dict, Any, Tuple, Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RMSNorm
# ============================================================

class RMSNorm(nn.Module):
    """
    Minimal RMSNorm used by the MTP transform.

    Input:
        x: [..., dim]

    Output:
        normalized x with the same shape.
    """

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()

        if dim <= 0:
            raise ValueError(f"dim must be > 0, got {dim}")

        if eps <= 0:
            raise ValueError(f"eps must be > 0, got {eps}")

        self.dim = dim
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[-1] != self.dim:
            raise ValueError(
                f"Expected x.shape[-1] == dim={self.dim}, got {x.shape[-1]}"
            )

        rms = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)

        return x * self.weight.to(dtype=x.dtype)


# ============================================================
# MTP label construction
# ============================================================

def build_mtp_labels(
    input_ids: torch.Tensor,
    mtp_depth: int,
    ignore_index: int = -100,
    pad_token_id: Optional[int] = None,
) -> torch.Tensor:
    """
    Build labels for Multi-Token Prediction.

    Main causal LM head predicts:
        x_{t+1}

    MTP heads predict:
        depth 0 -> x_{t+2}
        depth 1 -> x_{t+3}
        ...
        depth k -> x_{t+k+2}

    Args:
        input_ids:
            Token ids with shape [B, T].

        mtp_depth:
            Number of MTP future-token heads.

        ignore_index:
            Label value ignored by cross_entropy.

        pad_token_id:
            Optional padding token id. If provided, positions whose future
            target is padding are also set to ignore_index.

    Returns:
        mtp_labels:
            [B, mtp_depth, T]
    """
    if input_ids.dim() != 2:
        raise ValueError(
            f"input_ids must have shape [B,T], got {tuple(input_ids.shape)}"
        )

    if torch.is_floating_point(input_ids):
        raise TypeError("input_ids must be integer token ids, not floating point.")

    if mtp_depth <= 0:
        raise ValueError(f"mtp_depth must be > 0, got {mtp_depth}")

    B, T = input_ids.shape

    labels = torch.full(
        (B, mtp_depth, T),
        fill_value=int(ignore_index),
        device=input_ids.device,
        dtype=torch.long,
    )

    input_ids_long = input_ids.long()

    for k in range(mtp_depth):
        shift = k + 2

        if shift >= T:
            continue

        future = input_ids_long[:, shift:]  # [B, T-shift]
        labels[:, k, : T - shift] = future

        if pad_token_id is not None:
            is_pad = labels[:, k, :] == int(pad_token_id)
            labels[:, k, :] = torch.where(
                is_pad,
                torch.full_like(labels[:, k, :], int(ignore_index)),
                labels[:, k, :],
            )

    return labels


# ============================================================
# CONFIG
# ============================================================

@dataclass
class MTPConfig:
    d_model: int
    vocab_size: int
    mtp_depth: int = 1

    hidden_dim: Optional[int] = None
    use_mtp_transform: bool = True
    activation: str = "silu"

    dropout: float = 0.0
    use_bias: bool = False
    init_std: float = 0.02

    tie_with_lm_head: bool = False
    mtp_loss_weight: float = 0.3

    # Explicit ignore index for CE. This is distinct from pad_token_id.
    ignore_index: int = -100

    # Optional tokenizer pad id. If provided, label validation allows it as
    # a token id, and build_mtp_labels can convert future-pad targets to
    # ignore_index.
    pad_token_id: Optional[int] = None

    # Optional per-depth loss weights. If None, all depths are averaged equally.
    # If provided, must have length mtp_depth and non-negative values.
    depth_loss_weights: Optional[Tuple[float, ...]] = None

    validate_label_range: bool = True

    def validate(self) -> None:
        if self.d_model <= 0:
            raise ValueError(f"d_model must be > 0, got {self.d_model}")

        if self.vocab_size <= 0:
            raise ValueError(f"vocab_size must be > 0, got {self.vocab_size}")

        if self.mtp_depth <= 0:
            raise ValueError(f"mtp_depth must be > 0, got {self.mtp_depth}")

        if self.hidden_dim is not None and self.hidden_dim <= 0:
            raise ValueError(
                f"hidden_dim must be > 0 when provided, got {self.hidden_dim}"
            )

        valid_activations = {"silu", "gelu", "relu", "identity"}
        if self.activation not in valid_activations:
            raise ValueError(
                f"activation must be one of {valid_activations}, got {self.activation}"
            )

        if not (0.0 <= self.dropout < 1.0):
            raise ValueError(f"dropout must satisfy 0 <= dropout < 1, got {self.dropout}")

        if self.init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {self.init_std}")

        if self.mtp_loss_weight < 0:
            raise ValueError(
                f"mtp_loss_weight must be >= 0, got {self.mtp_loss_weight}"
            )

        if self.pad_token_id is not None:
            if not (0 <= self.pad_token_id < self.vocab_size):
                raise ValueError(
                    "pad_token_id must satisfy 0 <= pad_token_id < vocab_size. "
                    f"Got pad_token_id={self.pad_token_id}, vocab_size={self.vocab_size}"
                )

        if self.depth_loss_weights is not None:
            if len(self.depth_loss_weights) != self.mtp_depth:
                raise ValueError(
                    "depth_loss_weights must have length mtp_depth. "
                    f"Got len={len(self.depth_loss_weights)}, mtp_depth={self.mtp_depth}"
                )

            if any(w < 0 for w in self.depth_loss_weights):
                raise ValueError("depth_loss_weights must be non-negative.")

            if sum(self.depth_loss_weights) <= 0:
                raise ValueError("At least one depth_loss_weight must be > 0.")


# ============================================================
# ACTIVATION
# ============================================================

class MTPActivation(nn.Module):
    def __init__(self, activation: str):
        super().__init__()

        valid_activations = {"silu", "gelu", "relu", "identity"}
        if activation not in valid_activations:
            raise ValueError(
                f"activation must be one of {valid_activations}, got {activation}"
            )

        self.activation = activation

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.activation == "silu":
            return F.silu(x)

        if self.activation == "gelu":
            return F.gelu(x)

        if self.activation == "relu":
            return F.relu(x)

        if self.activation == "identity":
            return x

        raise RuntimeError(f"Unknown activation={self.activation}")


# ============================================================
# MTP TRANSFORM
# ============================================================

class MTPTransform(nn.Module):
    """
    Per-depth transformation before the MTP vocabulary head.

    If enabled:
        RMSNorm(d_model)
        Linear(d_model, hidden_dim)
        activation
        Linear(hidden_dim, d_model)
        Dropout

    Input:
        hidden_states: [B,T,D]

    Output:
        transformed: [B,T,D]
    """

    def __init__(
        self,
        d_model: int,
        hidden_dim: Optional[int] = None,
        activation: str = "silu",
        dropout: float = 0.0,
        use_bias: bool = False,
        init_std: float = 0.02,
    ):
        super().__init__()

        if d_model <= 0:
            raise ValueError(f"d_model must be > 0, got {d_model}")

        if hidden_dim is None:
            hidden_dim = d_model

        if hidden_dim <= 0:
            raise ValueError(f"hidden_dim must be > 0, got {hidden_dim}")

        if not (0.0 <= dropout < 1.0):
            raise ValueError(f"dropout must satisfy 0 <= dropout < 1, got {dropout}")

        if init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {init_std}")

        self.d_model = d_model
        self.hidden_dim = hidden_dim
        self.init_std = init_std

        self.norm = RMSNorm(dim=d_model)
        self.fc1 = nn.Linear(d_model, hidden_dim, bias=use_bias)
        self.act = MTPActivation(activation)
        self.fc2 = nn.Linear(hidden_dim, d_model, bias=use_bias)
        self.dropout = nn.Dropout(dropout)

        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in [self.fc1, self.fc2]:
            nn.init.normal_(module.weight, mean=0.0, std=self.init_std)

            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        if hidden_states.dim() != 3:
            raise ValueError(
                f"MTPTransform expects hidden_states [B,T,D], got {tuple(hidden_states.shape)}"
            )

        if hidden_states.shape[-1] != self.d_model:
            raise ValueError(
                f"Expected hidden size {self.d_model}, got {hidden_states.shape[-1]}"
            )

        h = self.norm(hidden_states)
        h = self.fc1(h)
        h = self.act(h)
        h = self.fc2(h)
        h = self.dropout(h)

        return h

In [73]:
# ============================================================
# MULTI-TOKEN PREDICTION HEAD
# ============================================================

class MultiTokenPredictionHead(nn.Module):
    """
    Multi-Token Prediction auxiliary heads.

    Main LM head predicts:
        x_{t+1}

    MTP heads predict:
        head 0 -> x_{t+2}
        head 1 -> x_{t+3}
        ...
        head k -> x_{t+k+2}

    Input:
        hidden_states: [B,T,d_model]

    Output dict:
        {
            "mtp_logits": [B,mtp_depth,T,vocab_size],
            "mtp_loss": weighted_mtp_loss or None,
            "aux": {...}
        }
    """

    def __init__(self, config: MTPConfig):
        super().__init__()

        config.validate()

        self.config = config

        self.d_model = config.d_model
        self.vocab_size = config.vocab_size
        self.mtp_depth = config.mtp_depth
        self.pad_token_id = config.pad_token_id
        self.ignore_index = config.ignore_index
        self.mtp_loss_weight = config.mtp_loss_weight
        self.tie_with_lm_head = config.tie_with_lm_head
        self.validate_label_range = config.validate_label_range

        if config.depth_loss_weights is None:
            depth_weights = torch.ones(config.mtp_depth, dtype=torch.float32)
        else:
            depth_weights = torch.tensor(config.depth_loss_weights, dtype=torch.float32)

        depth_weights = depth_weights / depth_weights.sum()
        self.register_buffer("depth_loss_weights", depth_weights, persistent=False)

        if config.use_mtp_transform:
            self.transforms = nn.ModuleList(
                [
                    MTPTransform(
                        d_model=config.d_model,
                        hidden_dim=config.hidden_dim,
                        activation=config.activation,
                        dropout=config.dropout,
                        use_bias=config.use_bias,
                        init_std=config.init_std,
                    )
                    for _ in range(config.mtp_depth)
                ]
            )
        else:
            self.transforms = nn.ModuleList(
                [nn.Identity() for _ in range(config.mtp_depth)]
            )

        self.heads = nn.ModuleList(
            [
                nn.Linear(
                    config.d_model,
                    config.vocab_size,
                    bias=False,
                )
                for _ in range(config.mtp_depth)
            ]
        )

        self.reset_head_parameters()

    def reset_head_parameters(self) -> None:
        for head in self.heads:
            nn.init.normal_(
                head.weight,
                mean=0.0,
                std=self.config.init_std,
            )

    def tie_weights(self, lm_head_weight: nn.Parameter) -> None:
        """
        Tie every MTP head weight to the main LM head weight.

        Args:
            lm_head_weight:
                nn.Parameter with shape [vocab_size, d_model]

        This shares the same Parameter object instead of copying values.
        """
        if lm_head_weight.shape != (self.vocab_size, self.d_model):
            raise ValueError(
                "lm_head_weight must have shape "
                f"{(self.vocab_size, self.d_model)}, got {tuple(lm_head_weight.shape)}"
            )

        for head in self.heads:
            head.weight = lm_head_weight

    def _validate_hidden_states(self, hidden_states: torch.Tensor) -> Tuple[int, int]:
        if hidden_states.dim() != 3:
            raise ValueError(
                f"hidden_states must have shape [B,T,d_model], got {tuple(hidden_states.shape)}"
            )

        B, T, D = hidden_states.shape

        if D != self.d_model:
            raise ValueError(
                f"Expected hidden_states.shape[-1] == d_model={self.d_model}, got {D}"
            )

        return B, T

    def _validate_mtp_labels(
        self,
        mtp_labels: torch.Tensor,
        batch_size: int,
        seq_len: int,
    ) -> None:
        expected_shape = (batch_size, self.mtp_depth, seq_len)

        if mtp_labels.shape != expected_shape:
            raise ValueError(
                "mtp_labels must have shape [B,mtp_depth,T]. "
                f"Expected {expected_shape}, got {tuple(mtp_labels.shape)}"
            )

        if torch.is_floating_point(mtp_labels):
            raise TypeError("mtp_labels must be integer token ids, not floating point.")

        if not self.validate_label_range:
            return

        labels = mtp_labels.long()

        valid_ignore = labels == int(self.ignore_index)
        valid_token = (labels >= 0) & (labels < self.vocab_size)

        valid = valid_ignore | valid_token

        if not valid.all():
            bad = labels[~valid]
            bad_min = bad.min().item()
            bad_max = bad.max().item()
            raise ValueError(
                "mtp_labels contain values outside [0, vocab_size) and not equal "
                f"to ignore_index={self.ignore_index}. "
                f"Bad value range: [{bad_min}, {bad_max}]"
            )

    def _compute_loss(
        self,
        mtp_logits: torch.Tensor,
        mtp_labels: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            mtp_logits:
                [B,K,T,V]

            mtp_labels:
                [B,K,T]

        Returns:
            weighted_mtp_loss:
                scalar

            raw_mtp_loss:
                scalar

            mtp_loss_per_depth:
                [K]
        """
        B, K, T, V = mtp_logits.shape

        labels = mtp_labels.long()

        losses = []

        for k in range(K):
            logits_k = mtp_logits[:, k, :, :]
            labels_k = labels[:, k, :]

            loss_k = F.cross_entropy(
                logits_k.reshape(B * T, V),
                labels_k.reshape(B * T),
                ignore_index=int(self.ignore_index),
            )

            losses.append(loss_k)

        mtp_loss_per_depth = torch.stack(losses, dim=0)  # [K]

        depth_weights = self.depth_loss_weights.to(
            device=mtp_loss_per_depth.device,
            dtype=mtp_loss_per_depth.dtype,
        )

        raw_mtp_loss = (depth_weights * mtp_loss_per_depth).sum()
        weighted_mtp_loss = self.mtp_loss_weight * raw_mtp_loss

        return weighted_mtp_loss, raw_mtp_loss, mtp_loss_per_depth

    def forward(
        self,
        hidden_states: torch.Tensor,
        mtp_labels: Optional[torch.Tensor] = None,
        return_aux: bool = False,
    ) -> Dict[str, Any]:

        B, T = self._validate_hidden_states(hidden_states)

        if mtp_labels is not None:
            self._validate_mtp_labels(
                mtp_labels=mtp_labels,
                batch_size=B,
                seq_len=T,
            )

        logits_list = []

        for k in range(self.mtp_depth):
            h_k = self.transforms[k](hidden_states)
            logits_k = self.heads[k](h_k)
            logits_list.append(logits_k)

        mtp_logits = torch.stack(logits_list, dim=1)
        # [B,mtp_depth,T,V]

        mtp_loss = None
        aux: Dict[str, Any] = {}

        if mtp_labels is not None:
            weighted_loss, raw_loss, loss_per_depth = self._compute_loss(
                mtp_logits=mtp_logits,
                mtp_labels=mtp_labels,
            )

            mtp_loss = weighted_loss

            aux["raw_mtp_loss"] = raw_loss
            aux["weighted_mtp_loss"] = weighted_loss
            aux["mtp_loss_per_depth"] = loss_per_depth
            aux["depth_loss_weights"] = self.depth_loss_weights.to(
                device=loss_per_depth.device,
                dtype=loss_per_depth.dtype,
            )

        elif return_aux:
            aux["raw_mtp_loss"] = None
            aux["weighted_mtp_loss"] = None
            aux["mtp_loss_per_depth"] = None
            aux["depth_loss_weights"] = self.depth_loss_weights

        return {
            "mtp_logits": mtp_logits,
            "mtp_loss": mtp_loss,
            "aux": aux if return_aux or mtp_labels is not None else {},
        }

In [74]:
# ============================================================
# MTP smoke test
# ============================================================

B, T, D, V = 2, 16, 64, 128
mtp_depth = 3

mtp_config = MTPConfig(
    d_model=D,
    vocab_size=V,
    mtp_depth=mtp_depth,

    hidden_dim=None,
    use_mtp_transform=True,
    activation="silu",

    dropout=0.0,
    use_bias=False,
    init_std=0.02,

    tie_with_lm_head=False,
    mtp_loss_weight=0.3,

    pad_token_id=0,
)

mtp_head = MultiTokenPredictionHead(mtp_config)

hidden_states = torch.randn(B, T, D)

mtp_labels = torch.randint(
    low=1,
    high=V,
    size=(B, mtp_depth, T),
    dtype=torch.long,
)

# Simulate ignored/padded positions
mtp_labels[:, :, -2:] = 0

outputs = mtp_head(
    hidden_states=hidden_states,
    mtp_labels=mtp_labels,
    return_aux=True,
)

print("mtp_logits:", outputs["mtp_logits"].shape)
print("mtp_loss:", outputs["mtp_loss"])
print("raw_mtp_loss:", outputs["aux"]["raw_mtp_loss"])
print("mtp_loss_per_depth:", outputs["aux"]["mtp_loss_per_depth"].shape)

assert outputs["mtp_logits"].shape == (B, mtp_depth, T, V)
assert outputs["mtp_loss"] is not None
assert outputs["mtp_loss"].dim() == 0
assert torch.isfinite(outputs["mtp_loss"])
assert outputs["aux"]["mtp_loss_per_depth"].shape == (mtp_depth,)
assert torch.isfinite(outputs["aux"]["mtp_loss_per_depth"]).all()

print("MTP smoke test OK.")

mtp_logits: torch.Size([2, 3, 16, 128])
mtp_loss: tensor(1.4556, grad_fn=<MulBackward0>)
raw_mtp_loss: tensor(4.8520, grad_fn=<SumBackward0>)
mtp_loss_per_depth: torch.Size([3])
MTP smoke test OK.


In [75]:
# ============================================================
# MTP tying smoke test
# ============================================================

lm_head = nn.Linear(D, V, bias=False)

mtp_config_tied = MTPConfig(
    d_model=D,
    vocab_size=V,
    mtp_depth=2,
    use_mtp_transform=False,
    tie_with_lm_head=True,
    mtp_loss_weight=0.3,
    pad_token_id=0,
)

mtp_head_tied = MultiTokenPredictionHead(mtp_config_tied)

mtp_head_tied.tie_weights(lm_head.weight)

for head in mtp_head_tied.heads:
    assert head.weight is lm_head.weight

hidden_states = torch.randn(B, T, D)

outputs = mtp_head_tied(hidden_states)

assert outputs["mtp_logits"].shape == (B, 2, T, V)

print("MTP weight tying OK.")

MTP weight tying OK.


In [76]:
# ============================================================
# MTP backward check
# ============================================================

mtp_head = MultiTokenPredictionHead(mtp_config)

hidden_states = torch.randn(B, T, D, requires_grad=True)

mtp_labels = torch.randint(
    low=1,
    high=V,
    size=(B, mtp_depth, T),
    dtype=torch.long,
)
mtp_labels[:, :, -2:] = 0

outputs = mtp_head(
    hidden_states=hidden_states,
    mtp_labels=mtp_labels,
    return_aux=True,
)

loss = outputs["mtp_loss"]
loss.backward()

assert hidden_states.grad is not None
assert torch.isfinite(hidden_states.grad).all()

for name, param in mtp_head.named_parameters():
    assert param.grad is not None, f"Missing grad for {name}"
    assert torch.isfinite(param.grad).all(), f"Non-finite grad for {name}"

print("MTP backward OK.")

MTP backward OK.


In [77]:
# @title
# ============================================================
# MultiTokenPredictionHead / MTP tests
# ============================================================

import pytest
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Helpers
# ============================================================

def make_mtp_config(**overrides):
    cfg = dict(
        d_model=64,
        vocab_size=128,
        mtp_depth=2,

        hidden_dim=64,
        use_mtp_transform=True,
        activation="silu",

        dropout=0.0,
        use_bias=True,
        init_std=0.02,

        tie_with_lm_head=False,
        mtp_loss_weight=0.3,

        ignore_index=-100,
        pad_token_id=0,

        depth_loss_weights=None,
        validate_label_range=True,
    )
    cfg.update(overrides)
    return MTPConfig(**cfg)


def make_mtp(**overrides):
    return MultiTokenPredictionHead(make_mtp_config(**overrides))


def make_hidden_states(B=2, T=8, D=64, dtype=torch.float32, device="cpu"):
    return torch.randn(B, T, D, dtype=dtype, device=device)


def make_mtp_labels(
    B=2,
    K=2,
    T=8,
    V=128,
    ignore_index=-100,
    include_ignore=False,
):
    labels = torch.randint(
        low=1,
        high=V,
        size=(B, K, T),
        dtype=torch.long,
    )

    if include_ignore:
        labels[:, :, -2:] = ignore_index

    return labels


# ============================================================
# A. Config tests
# ============================================================

def test_valid_mtp_config_builds():
    config = MTPConfig(
        d_model=256,
        vocab_size=3000,
        mtp_depth=2,
        use_mtp_transform=True,
        hidden_dim=256,
        activation="silu",
        dropout=0.0,
        use_bias=False,
        init_std=0.02,
        mtp_loss_weight=0.3,
        ignore_index=-100,
        pad_token_id=0,
        depth_loss_weights=None,
        validate_label_range=True,
    )

    mtp = MultiTokenPredictionHead(config)

    assert mtp.d_model == 256
    assert mtp.vocab_size == 3000
    assert mtp.mtp_depth == 2
    assert mtp.ignore_index == -100
    assert len(mtp.heads) == 2
    assert len(mtp.transforms) == 2


@pytest.mark.parametrize("d_model", [0, -1, -64])
def test_invalid_mtp_d_model_raises(d_model):
    with pytest.raises(ValueError):
        MultiTokenPredictionHead(make_mtp_config(d_model=d_model))


@pytest.mark.parametrize("vocab_size", [0, -1, -128])
def test_invalid_vocab_size_raises(vocab_size):
    with pytest.raises(ValueError):
        MultiTokenPredictionHead(make_mtp_config(vocab_size=vocab_size))


@pytest.mark.parametrize("mtp_depth", [0, -1, -2])
def test_invalid_mtp_depth_raises(mtp_depth):
    with pytest.raises(ValueError):
        MultiTokenPredictionHead(make_mtp_config(mtp_depth=mtp_depth))


@pytest.mark.parametrize("hidden_dim", [0, -1, -64])
def test_invalid_hidden_dim_raises(hidden_dim):
    with pytest.raises(ValueError):
        MultiTokenPredictionHead(make_mtp_config(hidden_dim=hidden_dim))


@pytest.mark.parametrize("activation", ["bad", "tanh", "swiglu"])
def test_invalid_activation_raises(activation):
    with pytest.raises(ValueError):
        MultiTokenPredictionHead(make_mtp_config(activation=activation))


@pytest.mark.parametrize(
    "kwargs",
    [
        {"dropout": -0.1},
        {"dropout": 1.0},
        {"dropout": 1.5},
        {"init_std": 0.0},
        {"init_std": -0.01},
        {"mtp_loss_weight": -0.1},
    ],
)
def test_invalid_dropout_init_loss_weight_raises(kwargs):
    with pytest.raises(ValueError):
        MultiTokenPredictionHead(make_mtp_config(**kwargs))


@pytest.mark.parametrize("pad_token_id", [-1, 128])
def test_invalid_pad_token_id_raises(pad_token_id):
    with pytest.raises(ValueError):
        MultiTokenPredictionHead(
            make_mtp_config(
                vocab_size=128,
                pad_token_id=pad_token_id,
            )
        )

@pytest.mark.parametrize(
    "depth_loss_weights",
    [
        (1.0,),              # wrong length for mtp_depth=2
        (1.0, -1.0),         # negative
        (0.0, 0.0),          # all zero
    ],
)
def test_invalid_depth_loss_weights_raise(depth_loss_weights):
    with pytest.raises(ValueError):
        MultiTokenPredictionHead(
            make_mtp_config(
                mtp_depth=2,
                depth_loss_weights=depth_loss_weights,
            )
        )


def test_valid_depth_loss_weights_builds_and_normalizes():
    mtp = make_mtp(
        mtp_depth=2,
        depth_loss_weights=(0.25, 0.75),
    )

    expected = torch.tensor([0.25, 0.75], dtype=mtp.depth_loss_weights.dtype)

    assert torch.allclose(mtp.depth_loss_weights.cpu(), expected)
    assert torch.allclose(
        mtp.depth_loss_weights.sum(),
        torch.tensor(1.0, dtype=mtp.depth_loss_weights.dtype),
    )


# ============================================================
# B. Internal structure tests
# ============================================================

def test_mtp_has_correct_number_of_heads():
    mtp = make_mtp(mtp_depth=3)

    assert len(mtp.heads) == 3
    assert len(mtp.transforms) == 3


def test_heads_have_expected_shape():
    D, V, K = 64, 128, 3

    mtp = make_mtp(d_model=D, vocab_size=V, mtp_depth=K)

    for head in mtp.heads:
        assert head.weight.shape == (V, D)


def test_transform_identity_when_disabled():
    mtp = make_mtp(use_mtp_transform=False, mtp_depth=3)

    for transform in mtp.transforms:
        assert isinstance(transform, nn.Identity)

    x = make_hidden_states()
    y = mtp.transforms[0](x)

    assert torch.equal(x, y)


def test_transform_exists_when_enabled():
    B, T, D = 2, 8, 64

    mtp = make_mtp(
        d_model=D,
        hidden_dim=128,
        use_mtp_transform=True,
        mtp_depth=2,
    )

    x = make_hidden_states(B=B, T=T, D=D)

    for transform in mtp.transforms:
        y = transform(x)
        assert y.shape == (B, T, D)
        assert torch.isfinite(y).all()

        has_params = any(p.requires_grad for p in transform.parameters())
        assert has_params


def test_build_mtp_labels_shape_and_shifts():
    input_ids = torch.tensor(
        [
            [10, 11, 12, 13, 14, 15],
            [20, 21, 22, 23, 24, 25],
        ],
        dtype=torch.long,
    )

    labels = build_mtp_labels(
        input_ids=input_ids,
        mtp_depth=2,
        ignore_index=-100,
    )

    expected = torch.tensor(
        [
            [
                [12, 13, 14, 15, -100, -100],  # x_{t+2}
                [13, 14, 15, -100, -100, -100],  # x_{t+3}
            ],
            [
                [22, 23, 24, 25, -100, -100],
                [23, 24, 25, -100, -100, -100],
            ],
        ],
        dtype=torch.long,
    )

    assert labels.shape == (2, 2, 6)
    assert torch.equal(labels, expected)


def test_build_mtp_labels_converts_future_pad_to_ignore_index():
    input_ids = torch.tensor(
        [[5, 6, 0, 7, 0, 8]],
        dtype=torch.long,
    )

    labels = build_mtp_labels(
        input_ids=input_ids,
        mtp_depth=1,
        ignore_index=-100,
        pad_token_id=0,
    )

    # depth 0 predicts x_{t+2}:
    # future targets: [0, 7, 0, 8, ignore, ignore]
    # pad targets become ignore_index.
    expected = torch.tensor(
        [[[-100, 7, -100, 8, -100, -100]]],
        dtype=torch.long,
    )

    assert torch.equal(labels, expected)


def test_build_mtp_labels_rejects_bad_input_rank():
    input_ids = torch.ones(2, 3, 4, dtype=torch.long)

    with pytest.raises(ValueError):
        build_mtp_labels(input_ids, mtp_depth=2)


def test_build_mtp_labels_rejects_float_input_ids():
    input_ids = torch.randn(2, 8)

    with pytest.raises(TypeError):
        build_mtp_labels(input_ids, mtp_depth=2)

def test_rejects_out_of_range_mtp_labels():
    mtp = make_mtp(
        vocab_size=128,
        mtp_depth=2,
        ignore_index=-100,
        validate_label_range=True,
    )

    hidden_states = make_hidden_states(B=2, T=8, D=64)
    mtp_labels = make_mtp_labels(B=2, K=2, T=8, V=128)

    mtp_labels[0, 0, 0] = 999  # invalid

    with pytest.raises(ValueError):
        mtp(hidden_states, mtp_labels=mtp_labels)


def test_allows_ignore_index_in_mtp_labels():
    mtp = make_mtp(
        vocab_size=128,
        mtp_depth=2,
        ignore_index=-100,
        validate_label_range=True,
    )

    hidden_states = make_hidden_states(B=2, T=8, D=64)
    mtp_labels = make_mtp_labels(
        B=2,
        K=2,
        T=8,
        V=128,
        ignore_index=-100,
        include_ignore=True,
    )

    outputs = mtp(
        hidden_states,
        mtp_labels=mtp_labels,
        return_aux=True,
    )

    assert outputs["mtp_loss"] is not None
    assert torch.isfinite(outputs["mtp_loss"])



# ============================================================
# C. Forward tests
# ============================================================

def test_mtp_logits_shape():
    B, T, D, V, K = 2, 8, 64, 128, 3

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
    )

    hidden_states = make_hidden_states(B=B, T=T, D=D)

    outputs = mtp(hidden_states)

    assert outputs["mtp_logits"].shape == (B, K, T, V)


def test_forward_without_labels_returns_no_loss():
    mtp = make_mtp()

    hidden_states = make_hidden_states()

    outputs = mtp(hidden_states, mtp_labels=None)

    assert outputs["mtp_loss"] is None


def test_forward_with_labels_returns_scalar_loss():
    B, T, D, V, K = 2, 8, 64, 128, 2

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
        pad_token_id=0,
    )

    hidden_states = make_hidden_states(B=B, T=T, D=D)
    mtp_labels = make_mtp_labels(B=B, K=K, T=T, V=V)

    outputs = mtp(
        hidden_states,
        mtp_labels=mtp_labels,
        return_aux=True,
    )

    assert outputs["mtp_loss"] is not None
    assert outputs["mtp_loss"].dim() == 0
    assert torch.isfinite(outputs["mtp_loss"])


@pytest.mark.parametrize(
    "bad_hidden_states",
    [
        torch.randn(8, 64),
        torch.randn(2, 8, 64, 1),
    ],
)
def test_rejects_wrong_hidden_states_rank(bad_hidden_states):
    mtp = make_mtp()

    with pytest.raises(ValueError):
        mtp(bad_hidden_states)


def test_rejects_wrong_hidden_size():
    mtp = make_mtp(d_model=64)

    hidden_states = torch.randn(2, 8, 32)

    with pytest.raises(ValueError):
        mtp(hidden_states)


@pytest.mark.parametrize(
    "bad_labels",
    [
        torch.ones(2, 2, 8, dtype=torch.long).permute(1, 0, 2),  # [K,B,T]
        torch.ones(2, 8, dtype=torch.long),                     # [B,T]
        torch.ones(2, 2, 8, 1, dtype=torch.long),                # [B,K,T,1]
        torch.ones(2, 2, 9, dtype=torch.long),                   # [B,K,T+1]
    ],
)
def test_rejects_wrong_mtp_labels_shape(bad_labels):
    mtp = make_mtp(
        d_model=64,
        vocab_size=128,
        mtp_depth=2,
    )

    hidden_states = make_hidden_states(B=2, T=8, D=64)

    with pytest.raises(ValueError):
        mtp(hidden_states, mtp_labels=bad_labels)


def test_rejects_float_mtp_labels():
    mtp = make_mtp()

    hidden_states = make_hidden_states()
    mtp_labels = torch.randn(2, 2, 8)

    with pytest.raises(TypeError):
        mtp(hidden_states, mtp_labels=mtp_labels)


# ============================================================
# D. Loss tests
# ============================================================

def test_mtp_loss_matches_manual_cross_entropy():
    B, T, D, V, K = 2, 8, 64, 128, 3
    ignore_index = -100
    loss_weight = 0.3

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
        dropout=0.0,
        mtp_loss_weight=loss_weight,
        ignore_index=ignore_index,
        pad_token_id=0,
    )
    mtp.eval()

    hidden_states = make_hidden_states(B=B, T=T, D=D)
    mtp_labels = make_mtp_labels(
        B=B,
        K=K,
        T=T,
        V=V,
        ignore_index=ignore_index,
        include_ignore=True,
    )

    outputs = mtp(
        hidden_states,
        mtp_labels=mtp_labels,
        return_aux=True,
    )

    mtp_logits = outputs["mtp_logits"]

    manual_losses = []

    for k in range(K):
        loss_k = F.cross_entropy(
            mtp_logits[:, k, :, :].reshape(B * T, V),
            mtp_labels[:, k, :].reshape(B * T),
            ignore_index=ignore_index,
        )
        manual_losses.append(loss_k)

    manual_per_depth = torch.stack(manual_losses)
    manual_raw = manual_per_depth.mean()
    manual_weighted = loss_weight * manual_raw

    assert torch.allclose(
        outputs["aux"]["raw_mtp_loss"],
        manual_raw,
        atol=1e-6,
        rtol=1e-6,
    )

    assert torch.allclose(
        outputs["mtp_loss"],
        manual_weighted,
        atol=1e-6,
        rtol=1e-6,
    )

    assert torch.allclose(
        outputs["aux"]["mtp_loss_per_depth"],
        manual_per_depth,
        atol=1e-6,
        rtol=1e-6,
    )

    assert "depth_loss_weights" in outputs["aux"]




def test_mtp_loss_per_depth_shape():
    B, T, D, V, K = 2, 8, 64, 128, 4

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
    )

    hidden_states = make_hidden_states(B=B, T=T, D=D)
    mtp_labels = make_mtp_labels(B=B, K=K, T=T, V=V)

    outputs = mtp(
        hidden_states,
        mtp_labels=mtp_labels,
        return_aux=True,
    )

    assert outputs["aux"]["mtp_loss_per_depth"].shape == (K,)


def test_mtp_loss_ignores_ignore_index():
    B, T, D, V, K = 2, 8, 64, 128, 2
    ignore_index = -100

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
        ignore_index=ignore_index,
        pad_token_id=0,
    )
    mtp.eval()

    hidden_states = make_hidden_states(B=B, T=T, D=D)

    mtp_labels = make_mtp_labels(
        B=B,
        K=K,
        T=T,
        V=V,
        ignore_index=ignore_index,
        include_ignore=True,
    )

    outputs = mtp(
        hidden_states,
        mtp_labels=mtp_labels,
        return_aux=True,
    )

    logits = outputs["mtp_logits"]

    manual_losses = []

    for k in range(K):
        manual_losses.append(
            F.cross_entropy(
                logits[:, k, :, :].reshape(B * T, V),
                mtp_labels[:, k, :].reshape(B * T),
                ignore_index=ignore_index,
            )
        )

    manual_raw = torch.stack(manual_losses).mean()

    assert torch.allclose(
        outputs["aux"]["raw_mtp_loss"],
        manual_raw,
        atol=1e-6,
        rtol=1e-6,
    )

def test_mtp_loss_uses_depth_loss_weights():
    B, T, D, V, K = 2, 8, 64, 128, 2
    ignore_index = -100
    depth_weights = (0.25, 0.75)

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
        mtp_loss_weight=1.0,
        ignore_index=ignore_index,
        depth_loss_weights=depth_weights,
    )
    mtp.eval()

    hidden_states = make_hidden_states(B=B, T=T, D=D)
    mtp_labels = make_mtp_labels(
        B=B,
        K=K,
        T=T,
        V=V,
        ignore_index=ignore_index,
        include_ignore=True,
    )

    outputs = mtp(
        hidden_states,
        mtp_labels=mtp_labels,
        return_aux=True,
    )

    logits = outputs["mtp_logits"]

    manual_losses = []

    for k in range(K):
        manual_losses.append(
            F.cross_entropy(
                logits[:, k, :, :].reshape(B * T, V),
                mtp_labels[:, k, :].reshape(B * T),
                ignore_index=ignore_index,
            )
        )

    manual_per_depth = torch.stack(manual_losses)
    weights = torch.tensor(depth_weights, dtype=manual_per_depth.dtype)
    weights = weights / weights.sum()

    manual_raw = (weights * manual_per_depth).sum()

    assert torch.allclose(
        outputs["aux"]["raw_mtp_loss"],
        manual_raw,
        atol=1e-6,
        rtol=1e-6,
    )


def test_zero_mtp_loss_weight_returns_zero_weighted_loss():
    B, T, D, V, K = 2, 8, 64, 128, 2

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
        mtp_loss_weight=0.0,
    )

    hidden_states = make_hidden_states(B=B, T=T, D=D)
    mtp_labels = make_mtp_labels(B=B, K=K, T=T, V=V)

    outputs = mtp(
        hidden_states,
        mtp_labels=mtp_labels,
        return_aux=True,
    )

    assert torch.allclose(
        outputs["mtp_loss"],
        torch.zeros_like(outputs["mtp_loss"]),
        atol=0.0,
        rtol=0.0,
    )

    assert torch.isfinite(outputs["aux"]["raw_mtp_loss"])


# ============================================================
# E. Weight tying tests
# ============================================================

def test_tie_weights_shares_parameter_across_heads():
    D, V, K = 64, 128, 3

    lm_head = nn.Linear(D, V, bias=False)

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
        use_mtp_transform=False,
    )

    mtp.tie_weights(lm_head.weight)

    for head in mtp.heads:
        assert head.weight is lm_head.weight


def test_tied_heads_change_when_lm_head_changes():
    D, V, K = 64, 128, 3

    lm_head = nn.Linear(D, V, bias=False)

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
        use_mtp_transform=False,
    )

    mtp.tie_weights(lm_head.weight)

    old_value = lm_head.weight[0, 0].detach().clone()

    with torch.no_grad():
        lm_head.weight[0, 0] += 1.0

    expected = old_value + 1.0

    for head in mtp.heads:
        assert head.weight is lm_head.weight
        assert torch.allclose(
            head.weight[0, 0],
            expected,
            atol=1e-6,
            rtol=1e-6,
        )


def test_tie_weights_rejects_wrong_shape():
    D, V = 64, 128

    bad_lm_head = nn.Linear(D + 1, V, bias=False)

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=2,
    )

    with pytest.raises(ValueError):
        mtp.tie_weights(bad_lm_head.weight)


# ============================================================
# F. Dropout / determinism tests
# ============================================================

def test_mtp_dropout_zero_is_deterministic():
    mtp = make_mtp(dropout=0.0)
    mtp.train()

    hidden_states = make_hidden_states()

    out1 = mtp(hidden_states)["mtp_logits"]
    out2 = mtp(hidden_states)["mtp_logits"]

    assert torch.equal(out1, out2)


def test_mtp_dropout_disabled_in_eval_mode():
    mtp = make_mtp(dropout=0.5)
    mtp.eval()

    hidden_states = make_hidden_states()

    out1 = mtp(hidden_states)["mtp_logits"]
    out2 = mtp(hidden_states)["mtp_logits"]

    assert torch.equal(out1, out2)


def test_mtp_dropout_active_in_train_mode():
    mtp = make_mtp(dropout=0.5, use_mtp_transform=True)
    mtp.train()

    hidden_states = make_hidden_states(B=4, T=16, D=64)

    out1 = mtp(hidden_states)["mtp_logits"]
    out2 = mtp(hidden_states)["mtp_logits"]

    assert not torch.equal(out1, out2)


# ============================================================
# G. Gradient tests
# ============================================================

def test_mtp_backward_computes_gradients():
    B, T, D, V, K = 2, 8, 64, 128, 2

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
        use_mtp_transform=True,
        dropout=0.0,
    )

    hidden_states = make_hidden_states(B=B, T=T, D=D)
    hidden_states.requires_grad_(True)

    mtp_labels = make_mtp_labels(B=B, K=K, T=T, V=V)

    outputs = mtp(
        hidden_states,
        mtp_labels=mtp_labels,
        return_aux=True,
    )

    loss = outputs["mtp_loss"]
    loss.backward()

    assert hidden_states.grad is not None
    assert torch.isfinite(hidden_states.grad).all()

    for name, param in mtp.named_parameters():
        assert param.grad is not None, f"Missing grad for {name}"
        assert torch.isfinite(param.grad).all(), f"Non-finite grad for {name}"


def test_backward_with_tied_lm_head():
    B, T, D, V, K = 2, 8, 64, 128, 2

    lm_head = nn.Linear(D, V, bias=False)

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
        use_mtp_transform=False,
    )

    mtp.tie_weights(lm_head.weight)

    hidden_states = make_hidden_states(B=B, T=T, D=D)
    hidden_states.requires_grad_(True)

    mtp_labels = make_mtp_labels(B=B, K=K, T=T, V=V)

    outputs = mtp(
        hidden_states,
        mtp_labels=mtp_labels,
        return_aux=True,
    )

    outputs["mtp_loss"].backward()

    assert lm_head.weight.grad is not None
    assert torch.isfinite(lm_head.weight.grad).all()


# ============================================================
# H. Integration tests
# ============================================================

def test_mtp_integrates_with_minicausallm_hidden_states():
    B, T, D, V, K = 2, 8, 64, 128, 2

    hidden_states = make_hidden_states(B=B, T=T, D=D)
    mtp_labels = make_mtp_labels(B=B, K=K, T=T, V=V)

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
        pad_token_id=0,
    )

    outputs = mtp(
        hidden_states,
        mtp_labels=mtp_labels,
        return_aux=True,
    )

    assert outputs["mtp_logits"].shape == (B, K, T, V)
    assert outputs["mtp_loss"] is not None
    assert torch.isfinite(outputs["mtp_loss"])


def test_mtp_labels_from_synthetic_dataset_format():
    """
    This test assumes your synthetic retrieval dataloader supports use_mtp=True
    and returns either:

        input_ids, labels, mtp_labels

    or a dict with key "mtp_labels".

    It is skipped if the synthetic MTP dataset functions are not in memory.
    """

    required_names = [
        "SyntheticRetrievalConfig",
        "create_synthetic_retrieval_dataloaders"]

    for name in required_names:
        if name not in globals():
            pytest.skip(f"{name} is not defined in this notebook/session.")

    data_cfg = SyntheticRetrievalConfig(
        num_train_examples=64,
        num_val_examples=16,
        block_size=32,
        min_filler_tokens=8,
        max_filler_tokens=16,
        num_keys_per_example=4,
        vocab_filler_size=100,
        num_key_types=32,
        num_value_types=64,
        batch_size=4,
        num_workers=0,
        seed=123)

    train_loader, _, tokenizer = create_synthetic_retrieval_dataloaders(
        cfg=data_cfg,
        use_mtp=True)

    batch = next(iter(train_loader))

    if isinstance(batch, dict):
        input_ids = batch["input_ids"]
        mtp_labels = batch["mtp_labels"]
    else:
        assert len(batch) == 3, (
            "Expected synthetic MTP batch to be either dict or "
            "(input_ids, labels, mtp_labels).")
        input_ids, _, mtp_labels = batch

    # Official MTP format must be [B,K,T].
    if mtp_labels.dim() == 3 and mtp_labels.shape[0] != input_ids.shape[0]:
        # Convert [K,B,T] -> [B,K,T] if dataset returns depth first.
        mtp_labels = mtp_labels.permute(1, 0, 2).contiguous()

    assert mtp_labels.dim() == 3
    assert mtp_labels.shape[0] == input_ids.shape[0]
    assert mtp_labels.shape[-1] == input_ids.shape[-1]

    B, K, T = mtp_labels.shape
    D = 64
    V = tokenizer.vocab_size

    hidden_states = torch.randn(B, T, D)

    mtp = make_mtp(
        d_model=D,
        vocab_size=V,
        mtp_depth=K,
        pad_token_id=tokenizer.pad_id)

    outputs = mtp(
        hidden_states,
        mtp_labels=mtp_labels,
        return_aux=True)

    assert outputs["mtp_logits"].shape == (B, K, T, V)
    assert outputs["mtp_loss"] is not None
    assert torch.isfinite(outputs["mtp_loss"])



In [78]:
# ============================================================
# Run ONLY MultiTokenPredictionHead / MTP tests in Jupyter
# Handles pytest-style parametrized tests manually
# ============================================================

mtp_plain_tests = [
    # A. Config
    "test_valid_mtp_config_builds",
    "test_valid_depth_loss_weights_builds_and_normalizes",

    # build_mtp_labels
    "test_build_mtp_labels_shape_and_shifts",
    "test_build_mtp_labels_converts_future_pad_to_ignore_index",
    "test_build_mtp_labels_rejects_bad_input_rank",
    "test_build_mtp_labels_rejects_float_input_ids",

    # B. Internal structure
    "test_mtp_has_correct_number_of_heads",
    "test_heads_have_expected_shape",
    "test_transform_identity_when_disabled",
    "test_transform_exists_when_enabled",

    # C. Forward
    "test_mtp_logits_shape",
    "test_forward_without_labels_returns_no_loss",
    "test_forward_with_labels_returns_scalar_loss",
    "test_rejects_wrong_hidden_size",
    "test_rejects_float_mtp_labels",
    "test_rejects_out_of_range_mtp_labels",
    "test_allows_ignore_index_in_mtp_labels",

    # D. Loss
    "test_mtp_loss_matches_manual_cross_entropy",
    "test_mtp_loss_per_depth_shape",
    "test_mtp_loss_ignores_ignore_index",
    "test_mtp_loss_uses_depth_loss_weights",
    "test_zero_mtp_loss_weight_returns_zero_weighted_loss",

    # E. Weight tying
    "test_tie_weights_shares_parameter_across_heads",
    "test_tied_heads_change_when_lm_head_changes",
    "test_tie_weights_rejects_wrong_shape",

    # F. Dropout / determinism
    "test_mtp_dropout_zero_is_deterministic",
    "test_mtp_dropout_disabled_in_eval_mode",
    "test_mtp_dropout_active_in_train_mode",

    # G. Gradient
    "test_mtp_backward_computes_gradients",
    "test_backward_with_tied_lm_head",

    # H. Integration
    "test_mtp_integrates_with_minicausallm_hidden_states",
    "test_mtp_labels_from_synthetic_dataset_format"]

mtp_param_tests = [
    ("test_invalid_mtp_d_model_raises", {"d_model": 0}),
    ("test_invalid_mtp_d_model_raises", {"d_model": -1}),
    ("test_invalid_mtp_d_model_raises", {"d_model": -64}),

    ("test_invalid_vocab_size_raises", {"vocab_size": 0}),
    ("test_invalid_vocab_size_raises", {"vocab_size": -1}),
    ("test_invalid_vocab_size_raises", {"vocab_size": -128}),

    ("test_invalid_mtp_depth_raises", {"mtp_depth": 0}),
    ("test_invalid_mtp_depth_raises", {"mtp_depth": -1}),
    ("test_invalid_mtp_depth_raises", {"mtp_depth": -2}),

    ("test_invalid_hidden_dim_raises", {"hidden_dim": 0}),
    ("test_invalid_hidden_dim_raises", {"hidden_dim": -1}),
    ("test_invalid_hidden_dim_raises", {"hidden_dim": -64}),

    ("test_invalid_activation_raises", {"activation": "bad"}),
    ("test_invalid_activation_raises", {"activation": "tanh"}),
    ("test_invalid_activation_raises", {"activation": "swiglu"}),

    ("test_invalid_dropout_init_loss_weight_raises", {"kwargs": {"dropout": -0.1}}),
    ("test_invalid_dropout_init_loss_weight_raises", {"kwargs": {"dropout": 1.0}}),
    ("test_invalid_dropout_init_loss_weight_raises", {"kwargs": {"dropout": 1.5}}),
    ("test_invalid_dropout_init_loss_weight_raises", {"kwargs": {"init_std": 0.0}}),
    ("test_invalid_dropout_init_loss_weight_raises", {"kwargs": {"init_std": -0.01}}),
    ("test_invalid_dropout_init_loss_weight_raises", {"kwargs": {"mtp_loss_weight": -0.1}}),

    ("test_invalid_pad_token_id_raises", {"pad_token_id": -1}),
    ("test_invalid_pad_token_id_raises", {"pad_token_id": 128}),

    ("test_invalid_depth_loss_weights_raise", {"depth_loss_weights": (1.0,)}),
    ("test_invalid_depth_loss_weights_raise", {"depth_loss_weights": (1.0, -1.0)}),
    ("test_invalid_depth_loss_weights_raise", {"depth_loss_weights": (0.0, 0.0)}),

    ("test_rejects_wrong_hidden_states_rank", {"bad_hidden_states": torch.randn(8, 64)}),
    ("test_rejects_wrong_hidden_states_rank", {"bad_hidden_states": torch.randn(2, 8, 64, 1)}),

    ("test_rejects_wrong_mtp_labels_shape", {"bad_labels": torch.ones(2, 3, 8, dtype=torch.long)}),
    ("test_rejects_wrong_mtp_labels_shape", {"bad_labels": torch.ones(3, 2, 8, dtype=torch.long)}),
    ("test_rejects_wrong_mtp_labels_shape", {"bad_labels": torch.ones(2, 2, 9, dtype=torch.long)}),
    ("test_rejects_wrong_mtp_labels_shape", {"bad_labels": torch.ones(2, 8, dtype=torch.long)}),
    ("test_rejects_wrong_mtp_labels_shape", {"bad_labels": torch.ones(2, 2, 8, 1, dtype=torch.long)})]

n_passed = 0

for name in mtp_plain_tests:
    globals()[name]()
    n_passed += 1

for name, kwargs in mtp_param_tests:
    globals()[name](**kwargs)
    n_passed += 1

print(f"OK: {n_passed} MTP tests/casos pasaron.")

Synthetic tokenizer vocab size: 216
OK: 65 MTP tests/casos pasaron.


---

---

# DeepSeekv4

In [187]:
# ============================================================
# Mini DeepSeek-V4 LM configurable
# Final full-sequence forward wrapper
# ============================================================

from dataclasses import dataclass
from typing import Optional, Dict, Any, Tuple, Union
import inspect

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# CONFIG
# ============================================================

@dataclass
class DeepSeekV4LMConfig:
    # Core
    vocab_size: int
    d_model: int
    n_layers: int
    max_seq_len: int = 1024
    pad_token_id: Optional[int] = None
    ignore_index: int = -100

    # Loss semantics
    # labels_are_shifted=True:
    #   input_ids: [x0, x1, ..., xT-1]
    #   labels:    [x1, x2, ..., xT]
    #   loss uses logits[:, :, :] vs labels[:, :]
    #
    # labels_are_shifted=False:
    #   input_ids: [x0, x1, ..., xT-1]
    #   labels:    [x0, x1, ..., xT-1], usually labels=input_ids
    #   loss uses logits[:, :-1, :] vs labels[:, 1:]
    labels_are_shifted: bool = True
    ignore_pad_token_in_loss: bool = True


    # Embedding
    embedding_dropout: float = 0.0
    scale_embeddings: bool = False
    tie_word_embeddings: bool = True

    # Norm
    rms_norm_eps: float = 1e-6

    # Attention selection
    # "mha"    -> all layers use standard MHA
    # "hca"    -> all layers use HCA
    # "csa"    -> all layers use CSA
    # "hybrid" -> interleaves attention modules by layer according to attention_pattern
    attention_type: str = "hybrid"  # "mha", "hca", "csa", "hybrid"
    attention_pattern: Tuple[str, ...] = ("csa", "hca")
    n_heads: int = 4
    head_dim: Optional[int] = None
    attention_dropout: float = 0.0
    residual_dropout: float = 0.0
    use_attention_bias: bool = False
    use_rope: bool = True
    rope_theta: float = 10000.0
    rotary_dim: Optional[int] = None

    # HCA / CSA shared
    compression_factor: int = 8
    hca_compression_factor: int = 16
    window_size: int = 32

    # CSA
    top_k_blocks: int = 8
    indexer_dim: int = 32
    n_indexer_heads: int = 2
    query_compression_dim: Optional[int] = None

    # Canonical CSA extras
    use_attention_sink: bool = True
    use_grouped_output_projection: bool = True
    output_projection_groups: Optional[int] = None
    use_indexer_score_bias: bool = False
    use_separate_local_kv: bool = True

    # FFN selection
    ffn_type: str = "moe"  # "dense", "moe"

    # Dense MLP
    mlp_hidden_dim: Optional[int] = None
    mlp_expansion_factor: float = 4.0
    mlp_multiple_of: int = 1
    mlp_dropout: float = 0.0
    use_mlp_bias: bool = False

    # MoE
    num_experts: int = 8
    top_k_experts: int = 2
    expert_hidden_dim: Optional[int] = None
    expert_expansion_factor: float = 4.0
    expert_multiple_of: int = 1
    shared_experts: int = 1
    shared_hidden_dim: Optional[int] = None
    shared_expansion_factor: float = 4.0

    router_type: str = "learned"  # "learned", "hash"
    router_score_fn: str = "sqrt_softplus"
    normalize_topk_weights: bool = True
    topk_weight_scale: float = 1.0
    router_jitter_noise: float = 0.0
    hash_routing_stride: int = 1

    routed_scale: float = 1.0
    shared_scale: float = 1.0

    balance_loss_weight: float = 0.0
    sequence_balance_loss_weight: float = 0.0

    # mHC
    use_mhc: bool = False
    n_hc: int = 4
    mhc_sinkhorn_iters: int = 20
    mhc_eps: float = 1e-6
    mhc_dynamic: bool = True
    mhc_expand_mode: str = "first"
    mhc_collapse_mode: str = "readout"  # "mean", "first", "sum", "readout"

    # Optional canonical mHC extras
    mhc_use_log_sinkhorn: bool = False
    mhc_sinkhorn_fp32: bool = True
    mhc_init_alpha: float = 1e-3
    mhc_alpha_max: float = 1.0
    mhc_bounded_alpha: bool = True

    # MTP
    use_mtp: bool = False
    mtp_depth: int = 1
    mtp_hidden_dim: Optional[int] = None
    use_mtp_transform: bool = True
    mtp_activation: str = "silu"
    mtp_dropout: float = 0.0
    mtp_loss_weight: float = 0.3
    mtp_tie_with_lm_head: bool = False
    mtp_depth_loss_weights: Optional[Tuple[float, ...]] = None
    mtp_validate_label_range: bool = True

    # Init
    init_std: float = 0.02

    def validate(self) -> None:
        if self.vocab_size <= 0:
            raise ValueError(f"vocab_size must be > 0, got {self.vocab_size}")

        if self.d_model <= 0:
            raise ValueError(f"d_model must be > 0, got {self.d_model}")

        if self.n_layers <= 0:
            raise ValueError(f"n_layers must be > 0, got {self.n_layers}")

        if self.max_seq_len <= 0:
            raise ValueError(f"max_seq_len must be > 0, got {self.max_seq_len}")

        if self.pad_token_id is not None:
            if not (0 <= self.pad_token_id < self.vocab_size):
                raise ValueError(
                    "pad_token_id must satisfy 0 <= pad_token_id < vocab_size. "
                    f"Got pad_token_id={self.pad_token_id}, vocab_size={self.vocab_size}"
                )

        if self.rms_norm_eps <= 0:
            raise ValueError(f"rms_norm_eps must be > 0, got {self.rms_norm_eps}")

        if self.attention_type not in {"mha", "hca", "csa", "hybrid"}:
            raise ValueError(
                f"attention_type must be one of {{'mha','hca','csa','hybrid'}}, "
                f"got {self.attention_type}"
            )

        if self.attention_type == "hybrid":
            if not isinstance(self.attention_pattern, tuple):
                raise ValueError(
                    f"attention_pattern must be a tuple, got {type(self.attention_pattern)}"
                )

            if len(self.attention_pattern) == 0:
                raise ValueError(
                    "attention_pattern must be non-empty when attention_type='hybrid'"
                )

            for attn_name in self.attention_pattern:
                if attn_name not in {"mha", "hca", "csa"}:
                    raise ValueError(
                        "Every element of attention_pattern must be one of "
                        f"{{'mha','hca','csa'}}, got {attn_name}"
                    )

        if self.ffn_type not in {"dense", "moe"}:
            raise ValueError(
                f"ffn_type must be one of {{'dense','moe'}}, got {self.ffn_type}"
            )

        if self.init_std <= 0:
            raise ValueError(f"init_std must be > 0, got {self.init_std}")

        if self.use_mhc:
            if self.n_hc < 2:
                raise ValueError(f"n_hc must be >= 2 when use_mhc=True, got {self.n_hc}")
            if self.mhc_collapse_mode not in {"mean", "first", "sum", "readout"}:
                raise ValueError(
                    "mhc_collapse_mode must be one of "
                    "{'mean','first','sum','readout'}, "
                    f"got {self.mhc_collapse_mode}"
                )

        if self.router_type not in {"learned", "hash"}:
            raise ValueError(
                f"router_type must be one of {'learned', 'hash'}, got {self.router_type}"
            )

        if self.hash_routing_stride <= 0:
            raise ValueError(
                f"hash_routing_stride must be > 0, got {self.hash_routing_stride}"
            )

        if self.n_heads <= 0:
            raise ValueError(f"n_heads must be > 0, got {self.n_heads}")

        if self.head_dim is None:
            if self.d_model % self.n_heads != 0:
                raise ValueError(
                    "If head_dim is None, d_model must be divisible by n_heads. "
                    f"Got d_model={self.d_model}, n_heads={self.n_heads}"
                )
        else:
            if self.head_dim <= 0:
                raise ValueError(f"head_dim must be > 0, got {self.head_dim}")

        if self.rotary_dim is not None:
            effective_head_dim = (
                self.head_dim if self.head_dim is not None else self.d_model // self.n_heads
            )
            if self.rotary_dim <= 0:
                raise ValueError(f"rotary_dim must be > 0, got {self.rotary_dim}")
            if self.rotary_dim > effective_head_dim:
                raise ValueError(
                    f"rotary_dim must be <= head_dim. "
                    f"Got rotary_dim={self.rotary_dim}, head_dim={effective_head_dim}"
                )
            if self.rotary_dim % 2 != 0:
                raise ValueError(f"rotary_dim must be even, got {self.rotary_dim}")

        if self.use_mtp:
            if self.mtp_depth <= 0:
                raise ValueError(f"mtp_depth must be > 0, got {self.mtp_depth}")
            if self.mtp_loss_weight < 0:
                raise ValueError(
                    f"mtp_loss_weight must be >= 0, got {self.mtp_loss_weight}"
                )

        if not isinstance(self.labels_are_shifted, bool):
            raise ValueError(
                f"labels_are_shifted must be bool, got {type(self.labels_are_shifted)}"
            )

        if not isinstance(self.ignore_pad_token_in_loss, bool):
            raise ValueError(
                "ignore_pad_token_in_loss must be bool, "
                f"got {type(self.ignore_pad_token_in_loss)}"
            )

# ============================================================
# SMALL COMPAT HELPERS
# ============================================================

def _supports_kwarg(module_or_fn, name: str) -> bool:
    try:
        sig = inspect.signature(module_or_fn.forward if isinstance(module_or_fn, nn.Module) else module_or_fn)
        return name in sig.parameters
    except Exception:
        return False


def _get_token_embedding_weight(embedding: nn.Module) -> nn.Parameter:
    """
    Supports both:
        TokenEmbedding.weight
    and common internal names:
        TokenEmbedding.embedding.weight
        TokenEmbedding.token_embedding.weight
    """
    if hasattr(embedding, "weight"):
        return embedding.weight

    if hasattr(embedding, "embedding") and hasattr(embedding.embedding, "weight"):
        return embedding.embedding.weight

    if hasattr(embedding, "token_embedding") and hasattr(embedding.token_embedding, "weight"):
        return embedding.token_embedding.weight

    raise AttributeError(
        "Could not find token embedding weight. Expected embedding.weight, "
        "embedding.embedding.weight, or embedding.token_embedding.weight."
    )


# ============================================================
# FACTORIES
# ============================================================

def build_deepseek_attention(
    config: DeepSeekV4LMConfig,
    attention_type: Optional[str] = None,
) -> nn.Module:
    """
    Build one concrete attention module.

    Important:
        attention_type="hybrid" is a model-level scheduling mode, not a
        concrete attention module. Resolve it per layer with
        get_layer_attention_type(config, layer_idx) before calling this factory.
    """

    if attention_type is None:
        attention_type = config.attention_type

    if attention_type == "hybrid":
        raise ValueError(
            "build_deepseek_attention received attention_type='hybrid'. "
            "Resolve the concrete layer attention first with "
            "get_layer_attention_type(config, layer_idx)."
        )

    if attention_type == "mha":
        return CausalMultiHeadAttention(
            CausalMHAConfig(
                d_model=config.d_model,
                n_heads=config.n_heads,
                head_dim=config.head_dim,
                attention_dropout=config.attention_dropout,
                residual_dropout=config.residual_dropout,
                use_bias=config.use_attention_bias,
                use_rope=config.use_rope,
                rope_theta=config.rope_theta,
                rotary_dim=config.rotary_dim,
                max_seq_len=config.max_seq_len,
                init_std=config.init_std,
            )
        )

    if attention_type == "hca":
        hca_kwargs = dict(
            d_model=config.d_model,
            n_heads=config.n_heads,
            head_dim=config.head_dim,
            compression_factor=config.hca_compression_factor,
            window_size=config.window_size,
            attention_dropout=config.attention_dropout,
            residual_dropout=config.residual_dropout,
            use_bias=config.use_attention_bias,
            use_rope=config.use_rope,
            rope_theta=config.rope_theta,
            rotary_dim=config.rotary_dim,
            max_seq_len=config.max_seq_len,
            init_std=config.init_std,
        )

        # Canonical HCA extras. Add only when the installed HCAConfig supports them.
        optional_hca_kwargs = {
            "use_attention_sink": config.use_attention_sink,
            "use_grouped_output_projection": config.use_grouped_output_projection,
            "output_projection_groups": config.output_projection_groups,
        }

        try:
            hca_sig = inspect.signature(HCAConfig)
            for key, value in optional_hca_kwargs.items():
                if key in hca_sig.parameters:
                    hca_kwargs[key] = value
        except Exception:
            # If signature introspection fails, fall back to the base kwargs.
            pass

        return HCAAttention(HCAConfig(**hca_kwargs))

    if attention_type == "csa":
        return CSAAttention(
            CSAConfig(
                d_model=config.d_model,
                n_heads=config.n_heads,
                head_dim=config.head_dim,
                compression_factor=config.compression_factor,
                top_k=config.top_k_blocks,
                window_size=config.window_size,
                indexer_dim=config.indexer_dim,
                n_indexer_heads=config.n_indexer_heads,
                query_compression_dim=config.query_compression_dim,
                attention_dropout=config.attention_dropout,
                residual_dropout=config.residual_dropout,
                use_bias=config.use_attention_bias,
                use_rope=config.use_rope,
                rope_theta=config.rope_theta,
                rotary_dim=config.rotary_dim,
                max_seq_len=config.max_seq_len,
                init_std=config.init_std,

                # Canonical CSA extras. These fields exist in your uploaded canonical CSA.
                use_attention_sink=config.use_attention_sink,
                use_grouped_output_projection=config.use_grouped_output_projection,
                output_projection_groups=config.output_projection_groups,
                use_indexer_score_bias=config.use_indexer_score_bias,
                use_separate_local_kv=config.use_separate_local_kv,
            )
        )

    raise RuntimeError(f"Unknown attention_type={attention_type}")


def get_layer_attention_type(config: DeepSeekV4LMConfig, layer_idx: int) -> str:
    """
    Resolve the concrete attention type for a given layer.

    If config.attention_type is:
        - "mha", "hca", or "csa": all layers use that attention.
        - "hybrid": layer attention is selected from attention_pattern cyclically.

    Example:
        attention_pattern=("csa", "hca")

        layer 0 -> csa
        layer 1 -> hca
        layer 2 -> csa
        layer 3 -> hca
    """

    if config.attention_type != "hybrid":
        return config.attention_type

    return config.attention_pattern[layer_idx % len(config.attention_pattern)]

def build_deepseek_ffn(config: DeepSeekV4LMConfig) -> nn.Module:
    if config.ffn_type == "dense":
        return SwiGLUMLP(
            SwiGLUMLPConfig(
                d_model=config.d_model,
                hidden_dim=config.mlp_hidden_dim,
                expansion_factor=config.mlp_expansion_factor,
                multiple_of=config.mlp_multiple_of,
                dropout=config.mlp_dropout,
                use_bias=config.use_mlp_bias,
                init_std=config.init_std,
            )
        )

    if config.ffn_type == "moe":
        return DeepSeekMoE(
            DeepSeekMoEConfig(
                d_model=config.d_model,
                num_experts=config.num_experts,
                top_k=config.top_k_experts,
                expert_hidden_dim=config.expert_hidden_dim,
                expert_expansion_factor=config.expert_expansion_factor,
                expert_multiple_of=config.expert_multiple_of,
                shared_experts=config.shared_experts,
                shared_hidden_dim=config.shared_hidden_dim,
                shared_expansion_factor=config.shared_expansion_factor,

                router_type=config.router_type,
                router_score_fn=config.router_score_fn,
                normalize_topk_weights=config.normalize_topk_weights,
                topk_weight_scale=config.topk_weight_scale,
                router_jitter_noise=config.router_jitter_noise,
                hash_routing_stride=config.hash_routing_stride,

                routed_scale=config.routed_scale,
                shared_scale=config.shared_scale,

                dropout=config.mlp_dropout,
                use_bias=config.use_mlp_bias,
                init_std=config.init_std,

                balance_loss_weight=config.balance_loss_weight,
                sequence_balance_loss_weight=config.sequence_balance_loss_weight,
            )
        )

    raise RuntimeError(f"Unknown ffn_type={config.ffn_type}")


def build_deepseek_mhc(config: DeepSeekV4LMConfig) -> "ManifoldHyperConnection":
    return ManifoldHyperConnection(
        ManifoldHyperConnectionConfig(
            d_model=config.d_model,
            n_hc=config.n_hc,
            sinkhorn_iters=config.mhc_sinkhorn_iters,
            eps=config.mhc_eps,
            use_log_sinkhorn=config.mhc_use_log_sinkhorn,
            sinkhorn_fp32=config.mhc_sinkhorn_fp32,
            dynamic=config.mhc_dynamic,
            init_alpha=config.mhc_init_alpha,
            alpha_max=config.mhc_alpha_max,
            bounded_alpha=config.mhc_bounded_alpha,
            init_std=config.init_std,
        )
    )


In [188]:
# ============================================================
# DEEPSEEK V4 BLOCK
# ============================================================

class DeepSeekV4Block(nn.Module):
    """
    Configurable DeepSeek-V4-style block.

    If use_mhc=False:
        x -> x + attention(norm1(x))
        x -> x + ffn(norm2(x))

    If use_mhc=True:
        X: [B,T,n_hc,D]
        X -> mHC attention update
        X -> mHC FFN update

    This block is intentionally compatibility-oriented:
        - MHA may not support need_weights.
        - HCA/CSA usually support need_weights.
        - Dense FFN has no aux.
        - MoE may need aux collection even when return_aux=False,
          because aux losses can contribute to the final LM loss.
        - Hash MoE needs input_ids.
    """

    def __init__(self, config: DeepSeekV4LMConfig, layer_idx: int):
        super().__init__()

        self.config = config
        self.layer_idx = layer_idx

        self.d_model = config.d_model
        self.use_mhc = config.use_mhc
        self.ffn_type = config.ffn_type
        self.attention_type = get_layer_attention_type(config, layer_idx)

        self.norm1 = RMSNorm(
            dim=config.d_model,
            eps=config.rms_norm_eps,
        )

        self.attention = build_deepseek_attention(
            config=config,
            attention_type=self.attention_type,
        )

        self.norm2 = RMSNorm(
            dim=config.d_model,
            eps=config.rms_norm_eps,
        )

        self.ffn = build_deepseek_ffn(config)

        if config.use_mhc:
            self.mhc_attn = build_deepseek_mhc(config)
            self.mhc_ffn = build_deepseek_mhc(config)
        else:
            self.mhc_attn = None
            self.mhc_ffn = None

    # --------------------------------------------------------
    # Attention call helper
    # --------------------------------------------------------

    def _call_attention(
        self,
        x_norm: torch.Tensor,
        attention_mask: Optional[torch.Tensor],
        position_ids: Optional[torch.Tensor],
        start_pos: int,
        need_weights: bool,
    ) -> Tuple[torch.Tensor, Optional[Dict[str, Any]]]:
        """
        Calls attention in a compatibility-safe way.

        Some modules, such as HCA/CSA, support:
            need_weights=True

        A simpler MHA module may not. Therefore this helper only passes
        arguments that the module actually supports.
        """

        kwargs = {}

        if _supports_kwarg(self.attention, "attention_mask"):
            kwargs["attention_mask"] = attention_mask

        if _supports_kwarg(self.attention, "position_ids"):
            kwargs["position_ids"] = position_ids

        if _supports_kwarg(self.attention, "start_pos"):
            kwargs["start_pos"] = start_pos

        if _supports_kwarg(self.attention, "need_weights"):
            kwargs["need_weights"] = need_weights

        result = self.attention(x_norm, **kwargs)

        # HCA/CSA with need_weights=True usually return:
        #   out, aux
        if isinstance(result, tuple):
            if len(result) != 2:
                raise RuntimeError(
                    "Attention module returned a tuple, but expected "
                    "(attention_output, attention_aux)."
                )

            attn_out, attn_aux = result
            return attn_out, attn_aux

        # MHA or attention modules without aux.
        return result, None

    # --------------------------------------------------------
    # FFN / MoE call helper
    # --------------------------------------------------------

    def _call_ffn(
        self,
        x_norm: torch.Tensor,
        input_ids: Optional[torch.Tensor],
        collect_aux: bool,
    ) -> Tuple[torch.Tensor, Optional[Dict[str, Any]]]:
        """
        Calls either dense SwiGLU or DeepSeekMoE.

        Dense:
            out = ffn(x)

        MoE:
            out, aux = ffn(x, input_ids=input_ids, return_aux=True/False)

        We may collect MoE aux even when the user did not ask for aux,
        because balance losses can be part of the total training loss.
        """

        if self.ffn_type == "dense":
            return self.ffn(x_norm), None

        if self.ffn_type != "moe":
            raise RuntimeError(f"Unknown ffn_type={self.ffn_type}")

        kwargs = {}

        if _supports_kwarg(self.ffn, "input_ids"):
            kwargs["input_ids"] = input_ids

        if _supports_kwarg(self.ffn, "return_aux"):
            kwargs["return_aux"] = collect_aux

        result = self.ffn(x_norm, **kwargs)

        if isinstance(result, tuple):
            if len(result) != 2:
                raise RuntimeError(
                    "MoE module returned a tuple, but expected "
                    "(ffn_output, moe_aux)."
                )

            ffn_out, ffn_aux = result
            return ffn_out, ffn_aux

        return result, None

    # --------------------------------------------------------
    # mHC helper
    # --------------------------------------------------------

    def _mhc_update(
        self,
        mhc: "ManifoldHyperConnection",
        X: torch.Tensor,
        sublayer_fn,
    ) -> Tuple[torch.Tensor, Dict[str, Any]]:
        """
        Uses the canonical modular mHC API when available:

            A, B, C = compute_ABC(X)
            x_sub = pre_mix(X, A=A)
            y_sub = sublayer_fn(x_sub)
            X_next = update(X, y_sub, B_mat=B, C=C)

        Falls back to the wrapper API if needed:

            X_next, aux = mhc(X, sublayer=sublayer_fn, return_aux=True)
        """

        if mhc is None:
            raise RuntimeError("mHC module is None but _mhc_update was called.")

        if all(hasattr(mhc, name) for name in ["compute_ABC", "pre_mix", "update"]):
            A, B_mat, C = mhc.compute_ABC(X)

            x_sub = mhc.pre_mix(X, A=A)

            y_sub = sublayer_fn(x_sub)

            X_next = mhc.update(
                X,
                y_sub,
                B_mat=B_mat,
                C=C,
            )

            aux = {
                "A": A,
                "B": B_mat,
                "C": C,
                "x_sub": x_sub,
                "y_sub": y_sub,
            }

            return X_next, aux

        # Fallback for older mHC wrapper.
        result = mhc(
            X,
            sublayer=sublayer_fn,
            return_aux=True,
        )

        if isinstance(result, tuple):
            if len(result) != 2:
                raise RuntimeError(
                    "mHC returned a tuple, but expected (X_next, aux)."
                )

            X_next, aux = result
            return X_next, aux

        return result, {}

    # --------------------------------------------------------
    # Forward
    # --------------------------------------------------------

    def forward(
        self,
        x_or_X: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        start_pos: int = 0,
        input_ids: Optional[torch.Tensor] = None,
        return_aux: bool = False,
        need_weights: bool = False,
        collect_moe_aux: Optional[bool] = None,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, Any]]]:

        aux: Dict[str, Any] = {}

        # If not explicitly provided, collect MoE aux whenever the block uses MoE.
        # This is useful because the LM wrapper may need balance losses even
        # when return_aux=False.
        if collect_moe_aux is None:
            collect_moe_aux = self.ffn_type == "moe"

        # ====================================================
        # Standard residual path
        # ====================================================
        if not self.use_mhc:
            x = x_or_X

            if x.dim() != 3:
                raise ValueError(
                    "DeepSeekV4Block without mHC expects x [B,T,D], "
                    f"got {tuple(x.shape)}"
                )

            if x.shape[-1] != self.d_model:
                raise ValueError(
                    f"Expected hidden size {self.d_model}, got {x.shape[-1]}"
                )

            # -------------------------
            # Attention residual update
            # -------------------------
            residual = x
            x_norm = self.norm1(x)

            attn_out, attn_aux = self._call_attention(
                x_norm=x_norm,
                attention_mask=attention_mask,
                position_ids=position_ids,
                start_pos=start_pos,
                need_weights=need_weights,
            )

            if attn_out.shape != residual.shape:
                raise ValueError(
                    "Attention output must have same shape as residual. "
                    f"Expected {tuple(residual.shape)}, got {tuple(attn_out.shape)}"
                )

            x = residual + attn_out

            if attn_aux is not None:
                aux["attention"] = attn_aux

            # -------------------------
            # FFN / MoE residual update
            # -------------------------
            residual = x
            x_norm = self.norm2(x)

            ffn_out, ffn_aux = self._call_ffn(
                x_norm=x_norm,
                input_ids=input_ids,
                collect_aux=collect_moe_aux or return_aux,
            )

            if ffn_out.shape != residual.shape:
                raise ValueError(
                    "FFN output must have same shape as residual. "
                    f"Expected {tuple(residual.shape)}, got {tuple(ffn_out.shape)}"
                )

            x = residual + ffn_out

            if ffn_aux is not None:
                aux["moe"] = ffn_aux

            if return_aux or need_weights or (collect_moe_aux and ffn_aux is not None):
                return x, aux

            return x

        # ====================================================
        # mHC residual path
        # ====================================================
        X = x_or_X

        if X.dim() != 4:
            raise ValueError(
                "DeepSeekV4Block with mHC expects X [B,T,n_hc,D], "
                f"got {tuple(X.shape)}"
            )

        if X.shape[2] != self.config.n_hc:
            raise ValueError(
                f"Expected n_hc={self.config.n_hc}, got {X.shape[2]}"
            )

        if X.shape[-1] != self.d_model:
            raise ValueError(
                f"Expected hidden size {self.d_model}, got {X.shape[-1]}"
            )

        # -------------------------
        # mHC attention update
        # -------------------------
        attn_aux_holder: Dict[str, Any] = {}

        def attn_sublayer(x_sub: torch.Tensor) -> torch.Tensor:
            x_norm = self.norm1(x_sub)

            attn_out, attn_aux = self._call_attention(
                x_norm=x_norm,
                attention_mask=attention_mask,
                position_ids=position_ids,
                start_pos=start_pos,
                need_weights=need_weights,
            )

            if attn_aux is not None:
                attn_aux_holder["attention"] = attn_aux

            return attn_out

        X, mhc_attn_aux = self._mhc_update(
            mhc=self.mhc_attn,
            X=X,
            sublayer_fn=attn_sublayer,
        )

        aux["mhc_attn"] = mhc_attn_aux

        if "attention" in attn_aux_holder:
            aux["attention"] = attn_aux_holder["attention"]

        # -------------------------
        # mHC FFN / MoE update
        # -------------------------
        ffn_aux_holder: Dict[str, Any] = {}

        def ffn_sublayer(x_sub: torch.Tensor) -> torch.Tensor:
            x_norm = self.norm2(x_sub)

            ffn_out, ffn_aux = self._call_ffn(
                x_norm=x_norm,
                input_ids=input_ids,
                collect_aux=collect_moe_aux or return_aux,
            )

            if ffn_aux is not None:
                ffn_aux_holder["moe"] = ffn_aux

            return ffn_out

        X, mhc_ffn_aux = self._mhc_update(
            mhc=self.mhc_ffn,
            X=X,
            sublayer_fn=ffn_sublayer,
        )

        aux["mhc_ffn"] = mhc_ffn_aux

        if "moe" in ffn_aux_holder:
            aux["moe"] = ffn_aux_holder["moe"]

        if return_aux or need_weights or (collect_moe_aux and "moe" in aux):
            return X, aux

        return X

In [189]:
# ============================================================
# DEEPSEEK V4 LM
# ============================================================

class DeepSeekV4LM(nn.Module):
    """
    Configurable DeepSeek-V4-style language model.

    Components:
        TokenEmbedding
        DeepSeekV4Block x n_layers
        final RMSNorm
        LM head
        optional MTP head

    Supported switches:
        attention_type: "mha", "hca", "csa", "hybrid"
        attention_pattern: cyclic per-layer schedule used when attention_type="hybrid"
        ffn_type: "dense", "moe"
        use_mhc: bool
        use_mtp: bool
    """

    def __init__(self, config: DeepSeekV4LMConfig):
        super().__init__()

        config.validate()

        self.config = config

        self.vocab_size = config.vocab_size
        self.d_model = config.d_model
        self.n_layers = config.n_layers
        self.max_seq_len = config.max_seq_len
        self.pad_token_id = config.pad_token_id
        self.ignore_index = config.ignore_index
        self.labels_are_shifted = config.labels_are_shifted
        self.ignore_pad_token_in_loss = config.ignore_pad_token_in_loss
        self.use_mhc = config.use_mhc
        self.use_mtp = config.use_mtp

        # ----------------------------------------------------
        # Embedding
        # ----------------------------------------------------
        self.embedding = TokenEmbedding(
            EmbeddingConfig(
                vocab_size=config.vocab_size,
                d_model=config.d_model,
                pad_token_id=config.pad_token_id,
                max_seq_len=config.max_seq_len,
                embedding_dropout=config.embedding_dropout,
                scale_embeddings=config.scale_embeddings,
                init_std=config.init_std,
                tie_word_embeddings=config.tie_word_embeddings,
            )
        )

        # ----------------------------------------------------
        # Blocks
        # ----------------------------------------------------
        self.blocks = nn.ModuleList(
            [
                DeepSeekV4Block(config=config, layer_idx=i)
                for i in range(config.n_layers)
            ]
        )

        # ----------------------------------------------------
        # mHC readout
        # ----------------------------------------------------
        if config.use_mhc and config.mhc_collapse_mode == "readout":
            self.mhc_readout = HyperConnectionReadout(
                n_hc=config.n_hc,
                init="mean",
            )
        else:
            self.mhc_readout = None

        # ----------------------------------------------------
        # Final norm + LM head
        # ----------------------------------------------------
        self.final_norm = RMSNorm(
            dim=config.d_model,
            eps=config.rms_norm_eps,
        )

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )

        self.reset_lm_head_parameters()

        if config.tie_word_embeddings:
            self.tie_lm_head_to_embeddings()

        # ----------------------------------------------------
        # Optional MTP
        # ----------------------------------------------------
        if config.use_mtp:
            self.mtp_head = MultiTokenPredictionHead(
                MTPConfig(
                    d_model=config.d_model,
                    vocab_size=config.vocab_size,
                    mtp_depth=config.mtp_depth,
                    hidden_dim=config.mtp_hidden_dim,
                    use_mtp_transform=config.use_mtp_transform,
                    activation=config.mtp_activation,
                    dropout=config.mtp_dropout,
                    use_bias=config.use_mlp_bias,
                    init_std=config.init_std,
                    tie_with_lm_head=config.mtp_tie_with_lm_head,
                    mtp_loss_weight=config.mtp_loss_weight,
                    ignore_index=config.ignore_index,
                    pad_token_id=config.pad_token_id,
                    depth_loss_weights=config.mtp_depth_loss_weights,
                    validate_label_range=config.mtp_validate_label_range,
                )
            )

            if config.mtp_tie_with_lm_head:
                self.mtp_head.tie_weights(self.lm_head.weight)
        else:
            self.mtp_head = None

    def reset_lm_head_parameters(self) -> None:
        nn.init.normal_(
            self.lm_head.weight,
            mean=0.0,
            std=self.config.init_std)

    def tie_lm_head_to_embeddings(self) -> None:
        self.lm_head.weight = _get_token_embedding_weight(self.embedding)

    def _validate_input_ids(self, input_ids: torch.Tensor) -> Tuple[int, int]:
        if input_ids.dim() != 2:
            raise ValueError(
                f"input_ids must have shape [B,T], got {tuple(input_ids.shape)}")

        if torch.is_floating_point(input_ids):
            raise TypeError("input_ids must be integer token ids, not floating point.")

        B, T = input_ids.shape

        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length T={T} exceeds max_seq_len={self.max_seq_len}")

        return B, T

    def _validate_labels(self, labels: torch.Tensor, input_ids: torch.Tensor) -> None:
        if labels.shape != input_ids.shape:
            raise ValueError(
                f"labels must have shape {tuple(input_ids.shape)}, "
                f"got {tuple(labels.shape)}")

        if torch.is_floating_point(labels):
            raise TypeError("labels must be integer token ids, not floating point.")

    def _validate_attention_mask(
        self,
        attention_mask: torch.Tensor,
        input_ids: torch.Tensor) -> torch.Tensor:

        if attention_mask.shape != input_ids.shape:
            raise ValueError(
                "attention_mask must have the same shape as input_ids. "
                f"Expected {tuple(input_ids.shape)}, got {tuple(attention_mask.shape)}")

        return attention_mask

    def _build_attention_mask(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor]) -> Optional[torch.Tensor]:

        if attention_mask is not None:
            return self._validate_attention_mask(attention_mask, input_ids)

        if self.pad_token_id is None:
            return None

        return (input_ids != self.pad_token_id).long()
    def _prepare_lm_loss_inputs(
        self,
        logits: torch.Tensor,
        labels: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Prepare logits and labels for causal LM loss.

        Supports two conventions:

        1. labels_are_shifted=True

            input_ids: [x_0, x_1, ..., x_{T-1}]
            labels:    [x_1, x_2, ..., x_T]

            The dataset already provides next-token labels.
            We compute CE over all positions:

                logits[:, :, :] vs labels[:, :]

        2. labels_are_shifted=False

            input_ids: [x_0, x_1, ..., x_{T-1}]
            labels:    [x_0, x_1, ..., x_{T-1}]

            HuggingFace-style labels=input_ids.
            We shift internally:

                logits[:, :-1, :] vs labels[:, 1:]

        Padding handling:
            If ignore_pad_token_in_loss=True and pad_token_id is not None,
            target labels equal to pad_token_id are converted to ignore_index.

            If attention_mask is provided, target positions with mask == 0 are
            also converted to ignore_index.
        """

        if logits.dim() != 3:
            raise ValueError(
                f"logits must have shape [B,T,V], got {tuple(logits.shape)}"
            )

        if labels.dim() != 2:
            raise ValueError(
                f"labels must have shape [B,T], got {tuple(labels.shape)}"
            )

        B, T, V = logits.shape

        if labels.shape != (B, T):
            raise ValueError(
                f"labels must have shape {(B, T)}, got {tuple(labels.shape)}"
            )

        if torch.is_floating_point(labels):
            raise TypeError("labels must be integer token ids, not floating point.")

        if attention_mask is not None and attention_mask.shape != labels.shape:
            raise ValueError(
                "attention_mask must have same shape as labels. "
                f"Expected {tuple(labels.shape)}, got {tuple(attention_mask.shape)}"
            )

        labels = labels.long()

        if self.labels_are_shifted:
            loss_logits = logits
            loss_labels = labels
            loss_attention_mask = attention_mask
        else:
            if T < 2:
                raise ValueError(
                    "Cannot compute internally shifted LM loss with sequence length < 2."
                )

            loss_logits = logits[:, :-1, :].contiguous()
            loss_labels = labels[:, 1:].contiguous()
            loss_attention_mask = (
                attention_mask[:, 1:].contiguous()
                if attention_mask is not None
                else None
            )

        loss_labels = loss_labels.clone()

        # Ignore pad-token targets.
        if self.ignore_pad_token_in_loss and self.pad_token_id is not None:
            loss_labels = torch.where(
                loss_labels == int(self.pad_token_id),
                torch.full_like(loss_labels, int(self.ignore_index)),
                loss_labels,
            )

        # Ignore masked target positions.
        if loss_attention_mask is not None:
            loss_labels = torch.where(
                loss_attention_mask.to(device=loss_labels.device, dtype=torch.bool),
                loss_labels,
                torch.full_like(loss_labels, int(self.ignore_index)),
            )

        return loss_logits, loss_labels

    def _compute_lm_loss(
        self,
        logits: torch.Tensor,
        labels: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Compute causal LM loss under either shifted-label or HF-style convention.
        """

        loss_logits, loss_labels = self._prepare_lm_loss_inputs(
            logits=logits,
            labels=labels,
            attention_mask=attention_mask,
        )

        B, T_loss, V = loss_logits.shape

        return F.cross_entropy(
            loss_logits.reshape(B * T_loss, V),
            loss_labels.reshape(B * T_loss),
            ignore_index=self.ignore_index,
        )

    def _collect_moe_aux_loss(
        self,
        block_aux_list: list,
        device: torch.device,
        dtype: torch.dtype) -> torch.Tensor:

        """
        Sum MoE auxiliary losses across blocks.

        Supports canonical keys:
            total_balance_loss
            balance_loss
            sequence_balance_loss

        If no MoE losses are present, returns scalar zero.
        """
        total = torch.zeros((), device=device, dtype=dtype)

        for block_aux in block_aux_list:
            if not isinstance(block_aux, dict):
                continue

            moe_aux = block_aux.get("moe", None)
            if not isinstance(moe_aux, dict):
                continue

            if "total_balance_loss" in moe_aux and moe_aux["total_balance_loss"] is not None:
                total = total + moe_aux["total_balance_loss"].to(device=device, dtype=dtype)
                continue

            if "balance_loss" in moe_aux and moe_aux["balance_loss"] is not None:
                total = total + moe_aux["balance_loss"].to(device=device, dtype=dtype)

            if "sequence_balance_loss" in moe_aux and moe_aux["sequence_balance_loss"] is not None:
                total = total + moe_aux["sequence_balance_loss"].to(device=device, dtype=dtype)

        return total

    def forward(
        self,
        input_ids: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        mtp_labels: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        start_pos: int = 0,
        return_aux: bool = False,
        need_weights: bool = False) -> Dict[str, Any]:


        B, T = self._validate_input_ids(input_ids)

        input_ids = input_ids.long()

        if labels is not None:
            self._validate_labels(labels, input_ids)
            labels = labels.long()

        attention_mask = self._build_attention_mask(
            input_ids=input_ids,
            attention_mask=attention_mask)

        # ----------------------------------------------------
        # Embedding
        # ----------------------------------------------------
        x = self.embedding(input_ids)
        # [B,T,D]

        block_aux_list = []

        # ----------------------------------------------------
        # Body
        # ----------------------------------------------------
        if self.use_mhc:
            X = expand_residual_stream(
                x,
                n_hc=self.config.n_hc,
                mode=self.config.mhc_expand_mode,
            )

            for block in self.blocks:
                if return_aux or need_weights or self.config.ffn_type == "moe":
                    X, block_aux = block(
                        X,
                        attention_mask=attention_mask,
                        position_ids=position_ids,
                        start_pos=start_pos,
                        input_ids=input_ids,
                        return_aux=return_aux,
                        need_weights=need_weights,
                        collect_moe_aux=self.config.ffn_type == "moe",
                    )
                    block_aux_list.append(block_aux)
                else:
                    X = block(
                        X,
                        attention_mask=attention_mask,
                        position_ids=position_ids,
                        start_pos=start_pos,
                        input_ids=input_ids,
                        return_aux=False,
                        need_weights=False,
                        collect_moe_aux=False,
                    )

            if self.config.mhc_collapse_mode == "readout":
                x = self.mhc_readout(X)
            else:
                x = collapse_residual_stream(
                    X,
                    mode=self.config.mhc_collapse_mode,
                )

        else:
            for block in self.blocks:
                if return_aux or need_weights or self.config.ffn_type == "moe":
                    x, block_aux = block(
                        x,
                        attention_mask=attention_mask,
                        position_ids=position_ids,
                        start_pos=start_pos,
                        input_ids=input_ids,
                        return_aux=return_aux,
                        need_weights=need_weights,
                        collect_moe_aux=self.config.ffn_type == "moe",
                    )
                    block_aux_list.append(block_aux)
                else:
                    x = block(
                        x,
                        attention_mask=attention_mask,
                        position_ids=position_ids,
                        start_pos=start_pos,
                        input_ids=input_ids,
                        return_aux=False,
                        need_weights=False,
                        collect_moe_aux=False,
                    )

        # ----------------------------------------------------
        # Final norm + logits
        # ----------------------------------------------------
        hidden_states = self.final_norm(x)
        logits = self.lm_head(hidden_states)

        # ----------------------------------------------------
        # LM loss
        # ----------------------------------------------------
        lm_loss = None

        if labels is not None:
            lm_loss = self._compute_lm_loss(
                logits=logits,
                labels=labels,
                attention_mask=attention_mask,
            )

        # ----------------------------------------------------
        # MTP
        # ----------------------------------------------------
        mtp_loss = None
        mtp_outputs = None

        if self.use_mtp:
            if mtp_labels is None and labels is not None:
                mtp_labels = build_mtp_labels(
                    input_ids=input_ids,
                    mtp_depth=self.config.mtp_depth,
                    ignore_index=self.ignore_index,
                    pad_token_id=self.pad_token_id,
                )

            mtp_outputs = self.mtp_head(
                hidden_states,
                mtp_labels=mtp_labels,
                return_aux=return_aux)

            mtp_loss = mtp_outputs["mtp_loss"]

        # ----------------------------------------------------
        # MoE aux loss
        # ----------------------------------------------------
        moe_aux_loss = self._collect_moe_aux_loss(
            block_aux_list=block_aux_list,
            device=logits.device,
            dtype=logits.dtype)

        has_moe_aux_loss = bool(
            self.config.ffn_type == "moe"
            and (
                self.config.balance_loss_weight > 0
                or self.config.sequence_balance_loss_weight > 0
            ))

        if not has_moe_aux_loss:
            # Keep output clean: no loss contribution unless explicitly weighted.
            moe_aux_loss = None

        # ----------------------------------------------------
        # Total loss
        # ----------------------------------------------------
        loss = None

        if lm_loss is not None:
            loss = lm_loss

            if mtp_loss is not None:
                loss = loss + mtp_loss.to(dtype=loss.dtype)

            if moe_aux_loss is not None:
                loss = loss + moe_aux_loss.to(dtype=loss.dtype)

        # ----------------------------------------------------
        # Aux
        # ----------------------------------------------------
        aux: Dict[str, Any] = {}

        if return_aux or need_weights:
          aux["blocks"] = block_aux_list
          aux["labels_are_shifted"] = self.labels_are_shifted
          aux["ignore_pad_token_in_loss"] = self.ignore_pad_token_in_loss

          if mtp_outputs is not None:
              aux["mtp"] = mtp_outputs.get("aux", {})

          if attention_mask is not None:
              aux["attention_mask"] = attention_mask

        return {
            "logits": logits,
            "loss": loss,
            "lm_loss": lm_loss,
            "mtp_loss": mtp_loss,
            "moe_aux_loss": moe_aux_loss,
            "hidden_states": hidden_states if return_aux else None,
            "aux": aux}

In [190]:
# ============================================================
# Smoke test 1: Dense + MHA, no mHC, no MTP
# ============================================================

cfg = DeepSeekV4LMConfig(
    vocab_size=216,
    d_model=64,
    n_layers=2,
    max_seq_len=64,
    pad_token_id=0,
    ignore_index=-100,

    attention_type="mha",
    n_heads=4,
    head_dim=16,
    rotary_dim=16,

    ffn_type="dense",
    mlp_hidden_dim=128,

    use_mhc=False,
    use_mtp=False,

    tie_word_embeddings=True,
    init_std=0.02,
)

model = DeepSeekV4LM(cfg)

B, T = 2, 32
input_ids = torch.randint(1, cfg.vocab_size, (B, T), dtype=torch.long)
labels = torch.randint(1, cfg.vocab_size, (B, T), dtype=torch.long)

out = model(
    input_ids=input_ids,
    labels=labels,
    return_aux=True,
    need_weights=True,
)

print("logits:", out["logits"].shape)
print("loss:", out["loss"])

assert out["logits"].shape == (B, T, cfg.vocab_size)
assert out["loss"] is not None
assert torch.isfinite(out["loss"])
assert len(out["aux"]["blocks"]) == cfg.n_layers

out["loss"].backward()

for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        assert torch.isfinite(param.grad).all(), f"Non-finite grad in {name}"

print("DeepSeekV4LM dense+mha smoke OK.")

logits: torch.Size([2, 32, 216])
loss: tensor(5.4033, grad_fn=<NllLossBackward0>)
DeepSeekV4LM dense+mha smoke OK.


In [191]:
# ============================================================
# Smoke test 2: CSA + MoE + MTP, no mHC
# ============================================================

cfg = DeepSeekV4LMConfig(
    vocab_size=216,
    d_model=64,
    n_layers=2,
    max_seq_len=64,
    pad_token_id=0,
    ignore_index=-100,

    attention_type="csa",
    n_heads=4,
    head_dim=16,
    rotary_dim=16,
    compression_factor=4,
    top_k_blocks=2,
    window_size=8,
    indexer_dim=8,
    n_indexer_heads=2,
    query_compression_dim=16,
    use_grouped_output_projection=True,
    output_projection_groups=2,
    use_attention_sink=True,
    use_indexer_score_bias=False,
    use_separate_local_kv=True,

    ffn_type="moe",
    num_experts=4,
    top_k_experts=2,
    expert_hidden_dim=128,
    shared_experts=1,
    shared_hidden_dim=128,
    router_type="learned",
    router_score_fn="sqrt_softplus",
    balance_loss_weight=0.01,
    sequence_balance_loss_weight=0.01,

    use_mhc=True,

    use_mtp=True,
    mtp_depth=2,
    mtp_hidden_dim=64,
    mtp_loss_weight=0.3,
    mtp_tie_with_lm_head=False,

    tie_word_embeddings=True,
    init_std=0.02,
)

model = DeepSeekV4LM(cfg)

B, T = 2, 32
input_ids = torch.randint(1, cfg.vocab_size, (B, T), dtype=torch.long)
labels = input_ids.clone()
labels[:, :-1] = input_ids[:, 1:]
labels[:, -1] = cfg.ignore_index

outputs = model(
    input_ids=input_ids,
    labels=labels,
    mtp_labels=None,       # auto build from input_ids
    return_aux=True,
    need_weights=True,
)

print("logits:", outputs["logits"].shape)
print("loss:", outputs["loss"])
print("lm_loss:", outputs["lm_loss"])
print("mtp_loss:", outputs["mtp_loss"])
print("moe_aux_loss:", outputs["moe_aux_loss"])

assert outputs["logits"].shape == (B, T, cfg.vocab_size)
assert outputs["loss"] is not None
assert outputs["lm_loss"] is not None
assert outputs["mtp_loss"] is not None
assert outputs["moe_aux_loss"] is not None
assert torch.isfinite(outputs["loss"])
assert len(outputs["aux"]["blocks"]) == cfg.n_layers
assert "mtp" in outputs["aux"]

outputs["loss"].backward()

assert any(
    p.grad is not None
    for p in model.parameters()
), "No gradients were produced."

print("DeepSeekV4LM CSA+MoE+MTP smoke OK.")

logits: torch.Size([2, 32, 216])
loss: tensor(6.9783, grad_fn=<AddBackward0>)
lm_loss: tensor(5.3656, grad_fn=<NllLossBackward0>)
mtp_loss: tensor(1.6126, grad_fn=<MulBackward0>)
moe_aux_loss: tensor(0.0001)
DeepSeekV4LM CSA+MoE+MTP smoke OK.


In [193]:
config = DeepSeekV4LMConfig(
    vocab_size=tokenizer.vocab_size,
    d_model=256,
    n_layers=4,
    max_seq_len=1024,
    pad_token_id=tokenizer.pad_id,

    attention_type="hybrid",
    attention_pattern=("csa", "hca"),

    n_heads=4,
    head_dim=64,
    rotary_dim=64,

    compression_factor=8,       # CSA compression
    hca_compression_factor=16,  # HCA heavier compression
    top_k_blocks=8,
    window_size=32,

    ffn_type="moe",
    use_mhc=True,
    use_mtp=True,

    balance_loss_weight=0.01,
    sequence_balance_loss_weight=0.01
)

model = DeepSeekV4LM(config)

B, T = 2, 32

input_ids = torch.randint(
    1,
    config.vocab_size,
    (B, T),
    dtype=torch.long,
)

labels = input_ids.clone()
labels[:, :-1] = input_ids[:, 1:]
labels[:, -1] = config.ignore_index

outputs = model(
    input_ids=input_ids,
    labels=labels,
    mtp_labels=None,
    return_aux=True,
    need_weights=True,
)

print("logits:", outputs["logits"].shape)
print("loss:", outputs["loss"])
print("lm_loss:", outputs["lm_loss"])
print("mtp_loss:", outputs["mtp_loss"])
print("moe_aux_loss:", outputs["moe_aux_loss"])

assert outputs["logits"].shape == (B, T, config.vocab_size)
assert outputs["loss"] is not None
assert outputs["lm_loss"] is not None
assert outputs["mtp_loss"] is not None

# Correcto: como balance_loss_weight=0 y sequence_balance_loss_weight=0,
# no debe entrar como componente de la loss total.
assert outputs["moe_aux_loss"] is None

assert torch.isfinite(outputs["loss"])
assert len(outputs["aux"]["blocks"]) == config.n_layers
assert "mtp" in outputs["aux"]

outputs["loss"].backward()

assert any(
    p.grad is not None
    for p in model.parameters()
), "No gradients were produced."

print("DeepSeekV4LM hybrid CSA/HCA + MoE + MTP smoke OK.")

logits: torch.Size([2, 32, 216])
loss: tensor(7.0553, grad_fn=<AddBackward0>)
lm_loss: tensor(5.4426, grad_fn=<NllLossBackward0>)
mtp_loss: tensor(1.6127, grad_fn=<MulBackward0>)
moe_aux_loss: None
DeepSeekV4LM hybrid CSA/HCA + MoE + MTP smoke OK.


In [194]:
# @title
# ============================================================
# DeepSeekV4LM tests
# Tiny configs only: CPU/Colab-safe
# ============================================================

import math
import pytest
import torch
import torch.nn.functional as F


# ============================================================
# Helpers
# ============================================================

def make_dsv4_config(**overrides):
    cfg = dict(
        vocab_size=128,
        d_model=32,
        n_layers=1,
        max_seq_len=32,
        pad_token_id=0,
        ignore_index=-100,

        embedding_dropout=0.0,
        scale_embeddings=False,
        tie_word_embeddings=True,

        rms_norm_eps=1e-6,

        attention_type="mha",
        n_heads=4,
        head_dim=8,
        attention_dropout=0.0,
        residual_dropout=0.0,
        use_attention_bias=False,
        use_rope=True,
        rope_theta=10000.0,
        rotary_dim=8,

        compression_factor=4,
        hca_compression_factor=4,
        window_size=4,

        top_k_blocks=2,
        indexer_dim=8,
        n_indexer_heads=2,
        query_compression_dim=8,

        use_attention_sink=True,
        use_grouped_output_projection=False,
        output_projection_groups=None,
        use_indexer_score_bias=True,   # useful for indexer gradients
        use_separate_local_kv=True,

        ffn_type="dense",

        mlp_hidden_dim=64,
        mlp_expansion_factor=2.0,
        mlp_multiple_of=1,
        mlp_dropout=0.0,
        use_mlp_bias=False,

        num_experts=4,
        top_k_experts=2,
        expert_hidden_dim=64,
        expert_expansion_factor=2.0,
        expert_multiple_of=1,
        shared_experts=1,
        shared_hidden_dim=64,
        shared_expansion_factor=2.0,

        router_type="learned",
        router_score_fn="sqrt_softplus",
        normalize_topk_weights=True,
        topk_weight_scale=1.0,
        router_jitter_noise=0.0,
        hash_routing_stride=1,

        routed_scale=1.0,
        shared_scale=1.0,

        balance_loss_weight=0.0,
        sequence_balance_loss_weight=0.0,

        use_mhc=False,
        n_hc=2,
        mhc_sinkhorn_iters=5,
        mhc_eps=1e-6,
        mhc_dynamic=True,
        mhc_expand_mode="first",
        mhc_collapse_mode="readout",
        mhc_use_log_sinkhorn=False,
        mhc_sinkhorn_fp32=True,
        mhc_init_alpha=1e-3,
        mhc_alpha_max=1.0,
        mhc_bounded_alpha=True,

        use_mtp=False,
        mtp_depth=2,
        mtp_hidden_dim=32,
        use_mtp_transform=True,
        mtp_activation="silu",
        mtp_dropout=0.0,
        mtp_loss_weight=0.3,
        mtp_tie_with_lm_head=False,
        mtp_depth_loss_weights=None,
        mtp_validate_label_range=True,

        init_std=0.02,
    )
    cfg.update(overrides)
    return DeepSeekV4LMConfig(**cfg)


def make_dsv4_model(**overrides):
    return DeepSeekV4LM(make_dsv4_config(**overrides))


def make_ids(B=2, T=12, V=128, pad_token_id=0, include_pad=False):
    input_ids = torch.randint(
        low=1,
        high=V,
        size=(B, T),
        dtype=torch.long,
    )

    if include_pad:
        input_ids[0, -2:] = pad_token_id

    return input_ids


def make_next_token_labels(input_ids, ignore_index=-100):
    labels = input_ids.clone()
    labels[:, :-1] = input_ids[:, 1:]
    labels[:, -1] = ignore_index
    return labels


def assert_finite_grads(model):
    any_grad = False

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if param.grad is None:
            continue

        any_grad = True
        assert torch.isfinite(param.grad).all(), f"Non-finite grad in {name}"

    assert any_grad, "No gradients were produced."


def get_embedding_weight_for_test(model):
    if hasattr(model.embedding, "weight"):
        return model.embedding.weight
    if hasattr(model.embedding, "embedding") and hasattr(model.embedding.embedding, "weight"):
        return model.embedding.embedding.weight
    if hasattr(model.embedding, "token_embedding") and hasattr(model.embedding.token_embedding, "weight"):
        return model.embedding.token_embedding.weight
    raise AssertionError("Could not find embedding weight.")


# ============================================================
# A. Config and construction
# ============================================================

def test_valid_deepseekv4lm_config_builds():
    cfg = make_dsv4_config(
        vocab_size=128,
        d_model=32,
        n_layers=1,
        n_heads=4,
        head_dim=8,
    )

    model = DeepSeekV4LM(cfg)

    assert model.vocab_size == 128
    assert model.d_model == 32
    assert model.n_layers == 1
    assert len(model.blocks) == 1


@pytest.mark.parametrize("attention_type", ["bad", "mla", "flash"])
def test_invalid_attention_type_raises(attention_type):
    with pytest.raises(ValueError):
        DeepSeekV4LM(make_dsv4_config(attention_type=attention_type))


@pytest.mark.parametrize("ffn_type", ["bad", "mlp", "expert"])
def test_invalid_ffn_type_raises(ffn_type):
    with pytest.raises(ValueError):
        DeepSeekV4LM(make_dsv4_config(ffn_type=ffn_type))


# ============================================================
# B. Forward basic
# ============================================================

@pytest.mark.parametrize("attention_type", ["mha", "hca", "csa"])
def test_logits_shape_all_attention_types(attention_type):
    B, T, V = 2, 12, 128

    model = make_dsv4_model(
        attention_type=attention_type,
        ffn_type="dense",
        vocab_size=V,
        max_seq_len=32,
    )
    model.eval()

    input_ids = make_ids(B=B, T=T, V=V)

    outputs = model(input_ids=input_ids)

    assert outputs["logits"].shape == (B, T, V)
    assert torch.isfinite(outputs["logits"]).all()


@pytest.mark.parametrize("ffn_type", ["dense", "moe"])
def test_forward_dense_and_moe_ffn(ffn_type):
    B, T, V = 2, 12, 128

    model = make_dsv4_model(
        attention_type="csa",
        ffn_type=ffn_type,
        vocab_size=V,
        max_seq_len=32,
    )
    model.eval()

    input_ids = make_ids(B=B, T=T, V=V)

    outputs = model(input_ids=input_ids)

    assert outputs["logits"].shape == (B, T, V)
    assert torch.isfinite(outputs["logits"]).all()


@pytest.mark.parametrize("use_mhc", [False, True])
def test_forward_with_and_without_mhc(use_mhc):
    B, T, V = 2, 12, 128

    model = make_dsv4_model(
        attention_type="hca",
        ffn_type="dense",
        use_mhc=use_mhc,
        vocab_size=V,
        max_seq_len=32,
        n_hc=2,
        mhc_sinkhorn_iters=5,
    )
    model.eval()

    input_ids = make_ids(B=B, T=T, V=V)

    outputs = model(input_ids=input_ids)

    assert outputs["logits"].shape == (B, T, V)
    assert torch.isfinite(outputs["logits"]).all()


@pytest.mark.parametrize("use_mtp", [False, True])
def test_forward_with_and_without_mtp(use_mtp):
    B, T, V = 2, 12, 128

    model = make_dsv4_model(
        attention_type="mha",
        ffn_type="dense",
        use_mtp=use_mtp,
        vocab_size=V,
        max_seq_len=32,
        mtp_depth=2,
    )

    input_ids = make_ids(B=B, T=T, V=V)
    labels = make_next_token_labels(input_ids)

    outputs = model(
        input_ids=input_ids,
        labels=labels,
        return_aux=True,
    )

    assert outputs["logits"].shape == (B, T, V)
    assert outputs["loss"] is not None
    assert outputs["lm_loss"] is not None
    assert torch.isfinite(outputs["loss"])

    if use_mtp:
        assert outputs["mtp_loss"] is not None
        assert torch.isfinite(outputs["mtp_loss"])
    else:
        assert outputs["mtp_loss"] is None


# ============================================================
# C. Loss
# ============================================================

def test_lm_loss_matches_manual_ce():
    B, T, V = 2, 12, 128

    model = make_dsv4_model(
        attention_type="mha",
        ffn_type="dense",
        use_mtp=False,
        vocab_size=V,
        max_seq_len=32,
    )
    model.eval()

    input_ids = make_ids(B=B, T=T, V=V)
    labels = make_next_token_labels(input_ids)

    outputs = model(
        input_ids=input_ids,
        labels=labels,
    )

    logits = outputs["logits"]

    manual = F.cross_entropy(
        logits.reshape(B * T, V),
        labels.reshape(B * T),
        ignore_index=model.ignore_index,
    )

    assert torch.allclose(outputs["lm_loss"], manual, atol=1e-6, rtol=1e-6)


def test_total_loss_includes_mtp_loss():
    B, T, V = 2, 12, 128

    model = make_dsv4_model(
        attention_type="mha",
        ffn_type="dense",
        use_mtp=True,
        mtp_depth=2,
        mtp_loss_weight=0.3,
        vocab_size=V,
        max_seq_len=32,
    )

    input_ids = make_ids(B=B, T=T, V=V)
    labels = make_next_token_labels(input_ids)

    outputs = model(
        input_ids=input_ids,
        labels=labels,
        mtp_labels=None,
    )

    assert outputs["lm_loss"] is not None
    assert outputs["mtp_loss"] is not None
    assert outputs["moe_aux_loss"] is None

    expected = outputs["lm_loss"] + outputs["mtp_loss"]

    assert torch.allclose(outputs["loss"], expected, atol=1e-6, rtol=1e-6)


def test_total_loss_includes_moe_aux_loss():
    B, T, V = 2, 12, 128

    model = make_dsv4_model(
        attention_type="csa",
        ffn_type="moe",
        use_mtp=False,
        vocab_size=V,
        max_seq_len=32,
        balance_loss_weight=0.01,
        sequence_balance_loss_weight=0.01,
    )

    input_ids = make_ids(B=B, T=T, V=V)
    labels = make_next_token_labels(input_ids)

    outputs = model(
        input_ids=input_ids,
        labels=labels,
        return_aux=True,
    )

    assert outputs["lm_loss"] is not None
    assert outputs["moe_aux_loss"] is not None
    assert outputs["moe_aux_loss"] >= 0

    expected = outputs["lm_loss"] + outputs["moe_aux_loss"]

    assert torch.allclose(outputs["loss"], expected, atol=1e-6, rtol=1e-6)


# ============================================================
# D. Weight tying
# ============================================================

def test_lm_head_tied_to_embedding():
    model = make_dsv4_model(
        attention_type="mha",
        ffn_type="dense",
        tie_word_embeddings=True,
    )

    embedding_weight = get_embedding_weight_for_test(model)

    assert model.lm_head.weight is embedding_weight


def test_mtp_heads_tied_to_lm_head_when_enabled():
    model = make_dsv4_model(
        attention_type="mha",
        ffn_type="dense",
        use_mtp=True,
        mtp_depth=2,
        tie_word_embeddings=True,
        mtp_tie_with_lm_head=True,
    )

    assert model.mtp_head is not None

    for head in model.mtp_head.heads:
        assert head.weight is model.lm_head.weight


# ============================================================
# E. Aux
# ============================================================

def test_return_aux_contains_blocks():
    model = make_dsv4_model(
        attention_type="csa",
        ffn_type="dense",
        n_layers=2,
    )
    model.eval()

    input_ids = make_ids(B=2, T=12, V=model.vocab_size)

    outputs = model(
        input_ids=input_ids,
        return_aux=True,
        need_weights=True,
    )

    assert "blocks" in outputs["aux"]
    assert len(outputs["aux"]["blocks"]) == 2


@pytest.mark.parametrize("attention_type", ["mha", "hca", "csa"])
def test_need_weights_attention_aux_shapes(attention_type):
    B, T, V = 2, 12, 128

    model = make_dsv4_model(
        attention_type=attention_type,
        ffn_type="dense",
        vocab_size=V,
        max_seq_len=32,
    )
    model.eval()

    input_ids = make_ids(B=B, T=T, V=V)

    outputs = model(
        input_ids=input_ids,
        return_aux=True,
        need_weights=True,
    )

    block_aux = outputs["aux"]["blocks"][0]

    if attention_type == "mha":
        if "attention" in block_aux:
            attn_aux = block_aux["attention"]

            if torch.is_tensor(attn_aux):
                assert attn_aux.shape == (B, model.config.n_heads, T, T)
                assert torch.isfinite(attn_aux).all()

            elif isinstance(attn_aux, dict):
                assert "attn_weights" in attn_aux
                assert attn_aux["attn_weights"].shape == (B, model.config.n_heads, T, T)
                assert torch.isfinite(attn_aux["attn_weights"]).all()

            else:
                raise TypeError(
                    "MHA attention aux must be either a tensor or a dict, "
                    f"got {type(attn_aux)}"
                )

        return

    assert "attention" in block_aux
    attn_aux = block_aux["attention"]

    assert isinstance(attn_aux, dict)
    assert "global_attn_weights" in attn_aux
    assert "local_attn_weights" in attn_aux

    assert attn_aux["local_attn_weights"].shape == (B, model.config.n_heads, T, T)

    if attention_type == "hca":
        S = math.ceil(T / model.config.hca_compression_factor)
        assert attn_aux["global_attn_weights"].shape == (B, model.config.n_heads, T, S)

    if attention_type == "csa":
        S = math.ceil(T / model.config.compression_factor)
        K = min(model.config.top_k_blocks, S)

        assert attn_aux["global_attn_weights"].shape == (B, model.config.n_heads, T, K)
        assert attn_aux["topk_indices"].shape == (B, T, K)


def test_moe_aux_present_when_ffn_type_moe():
    model = make_dsv4_model(
        attention_type="mha",
        ffn_type="moe",
        n_layers=2,
    )
    model.eval()

    input_ids = make_ids(B=2, T=12, V=model.vocab_size)

    outputs = model(
        input_ids=input_ids,
        return_aux=True,
    )

    assert len(outputs["aux"]["blocks"]) == 2

    for block_aux in outputs["aux"]["blocks"]:
        assert "moe" in block_aux
        assert isinstance(block_aux["moe"], dict)


def test_mhc_aux_present_when_use_mhc():
    model = make_dsv4_model(
        attention_type="hca",
        ffn_type="dense",
        use_mhc=True,
        n_hc=2,
        mhc_sinkhorn_iters=5,
    )
    model.eval()

    input_ids = make_ids(B=2, T=12, V=model.vocab_size)

    outputs = model(
        input_ids=input_ids,
        return_aux=True,
        need_weights=True,
    )

    block_aux = outputs["aux"]["blocks"][0]

    assert "mhc_attn" in block_aux
    assert "mhc_ffn" in block_aux

    for key in ["A", "B", "C"]:
        assert key in block_aux["mhc_attn"]
        assert key in block_aux["mhc_ffn"]


# ============================================================
# F. Causality
# ============================================================

@pytest.mark.parametrize("attention_type", ["mha", "hca", "csa"])
def test_changing_future_tokens_does_not_change_past_logits(attention_type):
    B, T, V = 2, 16, 128
    cut = 8

    model = make_dsv4_model(
        attention_type=attention_type,
        ffn_type="dense",
        vocab_size=V,
        max_seq_len=32,
        embedding_dropout=0.0,
        attention_dropout=0.0,
        residual_dropout=0.0,
        mlp_dropout=0.0,
        use_mtp=False,
    )
    model.eval()

    input_ids_1 = make_ids(B=B, T=T, V=V)
    input_ids_2 = input_ids_1.clone()

    future = torch.randint(1, V, input_ids_2[:, cut:].shape)
    input_ids_2[:, cut:] = future

    with torch.no_grad():
        logits_1 = model(input_ids=input_ids_1)["logits"]
        logits_2 = model(input_ids=input_ids_2)["logits"]

    assert torch.allclose(
        logits_1[:, :cut, :],
        logits_2[:, :cut, :],
        atol=1e-5,
        rtol=1e-5,
    )


# ============================================================
# G. Attention mask
# ============================================================

@pytest.mark.parametrize("attention_type", ["hca", "csa"])
def test_auto_attention_mask_from_pad_token(attention_type):
    B, T, V = 2, 12, 128
    pad = 0

    model = make_dsv4_model(
        attention_type=attention_type,
        ffn_type="dense",
        vocab_size=V,
        pad_token_id=pad,
    )
    model.eval()

    input_ids = make_ids(B=B, T=T, V=V, pad_token_id=pad, include_pad=True)

    outputs = model(
        input_ids=input_ids,
        attention_mask=None,
        return_aux=True,
        need_weights=True,
    )

    assert "attention_mask" in outputs["aux"]
    assert torch.equal(outputs["aux"]["attention_mask"], (input_ids != pad).long())

    attn_aux = outputs["aux"]["blocks"][0]["attention"]
    local_weights = attn_aux["local_attn_weights"]

    # pad keys at positions -2, -1 for batch 0 must receive zero local attention.
    assert torch.allclose(
        local_weights[0, :, :, -2:],
        torch.zeros_like(local_weights[0, :, :, -2:]),
        atol=0.0,
        rtol=0.0,
    )


@pytest.mark.parametrize("attention_type", ["hca", "csa"])
def test_explicit_attention_mask_overrides_auto_mask(attention_type):
    B, T, V = 2, 12, 128

    model = make_dsv4_model(
        attention_type=attention_type,
        ffn_type="dense",
        vocab_size=V,
        pad_token_id=0,
    )
    model.eval()

    input_ids = make_ids(B=B, T=T, V=V)

    attention_mask = torch.ones(B, T, dtype=torch.long)
    blocked_key = 5
    attention_mask[:, blocked_key] = 0

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        return_aux=True,
        need_weights=True,
    )

    assert torch.equal(outputs["aux"]["attention_mask"], attention_mask)

    local_weights = outputs["aux"]["blocks"][0]["attention"]["local_attn_weights"]

    assert torch.allclose(
        local_weights[:, :, :, blocked_key],
        torch.zeros_like(local_weights[:, :, :, blocked_key]),
        atol=0.0,
        rtol=0.0,
    )


# ============================================================
# H. MTP labels
# ============================================================

def test_auto_build_mtp_labels_when_missing():
    B, T, V = 2, 12, 128

    model = make_dsv4_model(
        attention_type="mha",
        ffn_type="dense",
        use_mtp=True,
        mtp_depth=2,
        vocab_size=V,
    )

    input_ids = make_ids(B=B, T=T, V=V)
    labels = make_next_token_labels(input_ids)

    outputs = model(
        input_ids=input_ids,
        labels=labels,
        mtp_labels=None,
        return_aux=True,
    )

    assert outputs["mtp_loss"] is not None
    assert torch.isfinite(outputs["mtp_loss"])
    assert "mtp" in outputs["aux"]


def test_explicit_mtp_labels_used():
    B, T, V, K = 2, 12, 128, 2

    model = make_dsv4_model(
        attention_type="mha",
        ffn_type="dense",
        use_mtp=True,
        mtp_depth=K,
        vocab_size=V,
    )

    input_ids = make_ids(B=B, T=T, V=V)
    labels = make_next_token_labels(input_ids)

    mtp_labels = build_mtp_labels(
        input_ids=input_ids,
        mtp_depth=K,
        ignore_index=model.ignore_index,
        pad_token_id=model.pad_token_id,
    )

    outputs = model(
        input_ids=input_ids,
        labels=labels,
        mtp_labels=mtp_labels,
        return_aux=True,
    )

    assert outputs["mtp_loss"] is not None
    assert torch.isfinite(outputs["mtp_loss"])


# ============================================================
# I. Backward
# ============================================================

@pytest.mark.parametrize(
    "overrides",
    [
        dict(attention_type="mha", ffn_type="dense", use_mhc=False, use_mtp=False),
        dict(attention_type="hca", ffn_type="dense", use_mhc=False, use_mtp=False),
        dict(attention_type="csa", ffn_type="dense", use_mhc=False, use_mtp=False),
        dict(attention_type="csa", ffn_type="moe", use_mhc=False, use_mtp=False, balance_loss_weight=0.01),
        dict(attention_type="csa", ffn_type="moe", use_mhc=False, use_mtp=True, balance_loss_weight=0.01),
        dict(attention_type="csa", ffn_type="moe", use_mhc=True, use_mtp=True, balance_loss_weight=0.01, n_hc=2),
    ],
)
def test_backward_all_major_configs(overrides):
    B, T, V = 2, 8, 128

    cfg = make_dsv4_config(
        vocab_size=V,
        max_seq_len=16,
        n_layers=1,
        d_model=32,
        n_heads=4,
        head_dim=8,
        rotary_dim=8,
        mlp_hidden_dim=64,
        expert_hidden_dim=64,
        shared_hidden_dim=64,
        num_experts=4,
        top_k_experts=2,
        compression_factor=4,
        top_k_blocks=2,
        window_size=4,
        indexer_dim=8,
        query_compression_dim=8,
        mhc_sinkhorn_iters=5,
        **overrides,
    )

    model = DeepSeekV4LM(cfg)

    input_ids = make_ids(B=B, T=T, V=V)
    labels = make_next_token_labels(input_ids)

    outputs = model(
        input_ids=input_ids,
        labels=labels,
        return_aux=True,
    )

    assert outputs["loss"] is not None
    assert torch.isfinite(outputs["loss"])

    outputs["loss"].backward()

    assert_finite_grads(model)


# ============================================================
# J. Synthetic dataset
# ============================================================

def test_synthetic_dataset_forward_loss():
    required = ["SyntheticRetrievalConfig", "create_synthetic_retrieval_dataloaders"]

    for name in required:
        if name not in globals():
            pytest.skip(f"{name} is not defined.")

    data_cfg = SyntheticRetrievalConfig(
        num_train_examples=32,
        num_val_examples=8,
        block_size=32,
        min_filler_tokens=8,
        max_filler_tokens=16,
        num_keys_per_example=4,
        vocab_filler_size=100,
        num_key_types=32,
        num_value_types=64,
        batch_size=4,
        num_workers=0,
        seed=123,
    )

    train_loader, _, tokenizer = create_synthetic_retrieval_dataloaders(
        cfg=data_cfg,
        use_mtp=False,
    )

    batch = next(iter(train_loader))

    if isinstance(batch, dict):
        input_ids = batch["input_ids"]
        labels = batch["labels"]
    else:
        input_ids, labels = batch[:2]

    model = make_dsv4_model(
        vocab_size=tokenizer.vocab_size,
        pad_token_id=tokenizer.pad_id,
        max_seq_len=data_cfg.block_size,
        attention_type="csa",
        ffn_type="dense",
        use_mtp=False,
    )

    outputs = model(
        input_ids=input_ids,
        labels=labels,
    )

    assert outputs["logits"].shape == (
        input_ids.shape[0],
        input_ids.shape[1],
        tokenizer.vocab_size,
    )
    assert outputs["loss"] is not None
    assert torch.isfinite(outputs["loss"])


def test_synthetic_dataset_backward():
    required = ["SyntheticRetrievalConfig", "create_synthetic_retrieval_dataloaders"]

    for name in required:
        if name not in globals():
            pytest.skip(f"{name} is not defined.")

    data_cfg = SyntheticRetrievalConfig(
        num_train_examples=32,
        num_val_examples=8,
        block_size=32,
        min_filler_tokens=8,
        max_filler_tokens=16,
        num_keys_per_example=4,
        vocab_filler_size=100,
        num_key_types=32,
        num_value_types=64,
        batch_size=4,
        num_workers=0,
        seed=123,
    )

    train_loader, _, tokenizer = create_synthetic_retrieval_dataloaders(
        cfg=data_cfg,
        use_mtp=False,
    )

    batch = next(iter(train_loader))

    if isinstance(batch, dict):
        input_ids = batch["input_ids"]
        labels = batch["labels"]
    else:
        input_ids, labels = batch[:2]

    model = make_dsv4_model(
        vocab_size=tokenizer.vocab_size,
        pad_token_id=tokenizer.pad_id,
        max_seq_len=data_cfg.block_size,
        attention_type="csa",
        ffn_type="moe",
        balance_loss_weight=0.01,
        use_mtp=False,
    )

    outputs = model(
        input_ids=input_ids,
        labels=labels,
    )

    assert outputs["loss"] is not None
    assert torch.isfinite(outputs["loss"])

    outputs["loss"].backward()

    assert_finite_grads(model)

In [195]:
# ============================================================
# Run ONLY DeepSeekV4LM tests in Jupyter
# ============================================================

dsv4_plain_tests = [
    "test_valid_deepseekv4lm_config_builds",

    "test_lm_loss_matches_manual_ce",
    "test_total_loss_includes_mtp_loss",
    "test_total_loss_includes_moe_aux_loss",

    "test_lm_head_tied_to_embedding",
    "test_mtp_heads_tied_to_lm_head_when_enabled",

    "test_return_aux_contains_blocks",
    "test_moe_aux_present_when_ffn_type_moe",
    "test_mhc_aux_present_when_use_mhc",

    "test_auto_build_mtp_labels_when_missing",
    "test_explicit_mtp_labels_used",

    "test_synthetic_dataset_forward_loss",
    "test_synthetic_dataset_backward",
]

dsv4_param_tests = [
    ("test_invalid_attention_type_raises", {"attention_type": "bad"}),
    ("test_invalid_attention_type_raises", {"attention_type": "mla"}),
    ("test_invalid_attention_type_raises", {"attention_type": "flash"}),

    ("test_invalid_ffn_type_raises", {"ffn_type": "bad"}),
    ("test_invalid_ffn_type_raises", {"ffn_type": "mlp"}),
    ("test_invalid_ffn_type_raises", {"ffn_type": "expert"}),

    ("test_logits_shape_all_attention_types", {"attention_type": "mha"}),
    ("test_logits_shape_all_attention_types", {"attention_type": "hca"}),
    ("test_logits_shape_all_attention_types", {"attention_type": "csa"}),

    ("test_forward_dense_and_moe_ffn", {"ffn_type": "dense"}),
    ("test_forward_dense_and_moe_ffn", {"ffn_type": "moe"}),

    ("test_forward_with_and_without_mhc", {"use_mhc": False}),
    ("test_forward_with_and_without_mhc", {"use_mhc": True}),

    ("test_forward_with_and_without_mtp", {"use_mtp": False}),
    ("test_forward_with_and_without_mtp", {"use_mtp": True}),

    ("test_need_weights_attention_aux_shapes", {"attention_type": "mha"}),
    ("test_need_weights_attention_aux_shapes", {"attention_type": "hca"}),
    ("test_need_weights_attention_aux_shapes", {"attention_type": "csa"}),

    ("test_changing_future_tokens_does_not_change_past_logits", {"attention_type": "mha"}),
    ("test_changing_future_tokens_does_not_change_past_logits", {"attention_type": "hca"}),
    ("test_changing_future_tokens_does_not_change_past_logits", {"attention_type": "csa"}),

    ("test_auto_attention_mask_from_pad_token", {"attention_type": "hca"}),
    ("test_auto_attention_mask_from_pad_token", {"attention_type": "csa"}),

    ("test_explicit_attention_mask_overrides_auto_mask", {"attention_type": "hca"}),
    ("test_explicit_attention_mask_overrides_auto_mask", {"attention_type": "csa"}),

    ("test_backward_all_major_configs", {
        "overrides": dict(
            attention_type="mha",
            ffn_type="dense",
            use_mhc=False,
            use_mtp=False,
        )
    }),
    ("test_backward_all_major_configs", {
        "overrides": dict(
            attention_type="hca",
            ffn_type="dense",
            use_mhc=False,
            use_mtp=False,
        )
    }),
    ("test_backward_all_major_configs", {
        "overrides": dict(
            attention_type="csa",
            ffn_type="dense",
            use_mhc=False,
            use_mtp=False,
        )
    }),
    ("test_backward_all_major_configs", {
        "overrides": dict(
            attention_type="csa",
            ffn_type="moe",
            use_mhc=False,
            use_mtp=False,
            balance_loss_weight=0.01,
        )
    }),
    ("test_backward_all_major_configs", {
        "overrides": dict(
            attention_type="csa",
            ffn_type="moe",
            use_mhc=False,
            use_mtp=True,
            balance_loss_weight=0.01,
        )
    }),
    ("test_backward_all_major_configs", {
        "overrides": dict(
            attention_type="csa",
            ffn_type="moe",
            use_mhc=True,
            use_mtp=True,
            balance_loss_weight=0.01,
            n_hc=2,
        )
    }),
]

n_passed = 0

for name in dsv4_plain_tests:
    globals()[name]()
    n_passed += 1

for name, kwargs in dsv4_param_tests:
    globals()[name](**kwargs)
    n_passed += 1

print(f"OK: {n_passed} DeepSeekV4LM tests/casos pasaron.")

Synthetic tokenizer vocab size: 216
Synthetic tokenizer vocab size: 216
OK: 44 DeepSeekV4LM tests/casos pasaron.


---

# Training

In [86]:
# ============================================================
# Seed
# ============================================================

def set_seed(seed: int, deterministic: bool = False) -> None:
    """
    Fix random seeds for Python, NumPy and PyTorch.

    Args:
        seed:
            Global random seed.
        deterministic:
            If True, enables deterministic CuDNN behavior.
            Useful for tests, but can slow down training.
    """
    if not isinstance(seed, int):
        raise TypeError(f"seed must be int, got {type(seed)}")

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        # Optional extra reproducibility.
        # Some operations may fail if no deterministic implementation exists.
        # Enable only if you truly need strict determinism.
        # torch.use_deterministic_algorithms(True)
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# Autocast

In [87]:
# ============================================================
# AMP / DEVICE / PRECISION HELPERS - FIXED VERSION
# ============================================================

from __future__ import annotations

import inspect
from contextlib import contextmanager, nullcontext
from typing import Any, Dict, Optional, Union

import torch


DTYPE_MAP = {
    "bf16": torch.bfloat16,
    "bfloat16": torch.bfloat16,
    "fp16": torch.float16,
    "float16": torch.float16,
    "fp32": torch.float32,
    "float32": torch.float32,
    "none": torch.float32,
}


def resolve_device(device: Union[str, torch.device] = "auto") -> torch.device:
    if isinstance(device, torch.device):
        requested = device
    elif device == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    else:
        requested = torch.device(device)

    if requested.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but CUDA is not available.")

    if requested.type == "mps":
        if not (hasattr(torch.backends, "mps") and torch.backends.mps.is_available()):
            raise RuntimeError("MPS was requested but MPS is not available.")

    return requested


def normalize_device_type(device: Union[str, torch.device] = "cuda") -> str:
    return torch.device(device).type


def resolve_amp_dtype(
    amp_dtype: str = "bf16",
    device: Union[str, torch.device] = "cuda",
) -> torch.dtype:
    amp_dtype = amp_dtype.lower()

    if amp_dtype not in DTYPE_MAP:
        raise ValueError(
            f"Unsupported amp_dtype={amp_dtype}. "
            f"Expected one of {sorted(DTYPE_MAP.keys())}."
        )

    return DTYPE_MAP[amp_dtype]


def cuda_supports_bf16() -> bool:
    if not torch.cuda.is_available():
        return False

    if hasattr(torch.cuda, "is_bf16_supported"):
        try:
            return bool(torch.cuda.is_bf16_supported())
        except Exception:
            pass

    try:
        major, _ = torch.cuda.get_device_capability()
        return major >= 8
    except Exception:
        return False


def get_effective_amp_dtype(
    amp_dtype: str = "bf16",
    device: Union[str, torch.device] = "cuda",
    fallback_bf16_to_fp16: bool = True,
) -> Optional[torch.dtype]:
    device_type = normalize_device_type(device)
    requested_dtype = resolve_amp_dtype(amp_dtype, device=device)

    if requested_dtype == torch.float32:
        return None

    if device_type == "cuda":
        if not torch.cuda.is_available():
            return None

        if requested_dtype == torch.bfloat16:
            if cuda_supports_bf16():
                return torch.bfloat16
            return torch.float16 if fallback_bf16_to_fp16 else None

        if requested_dtype == torch.float16:
            return torch.float16

        return None

    if device_type == "cpu":
        if requested_dtype == torch.bfloat16:
            return torch.bfloat16
        return None

    return None


def should_use_grad_scaler(
    device: Union[str, torch.device] = "cuda",
    amp_enabled: bool = True,
    amp_dtype: str = "bf16",
    fallback_bf16_to_fp16: bool = True,
) -> bool:
    if not amp_enabled:
        return False

    if normalize_device_type(device) != "cuda":
        return False

    effective_dtype = get_effective_amp_dtype(
        amp_dtype=amp_dtype,
        device=device,
        fallback_bf16_to_fp16=fallback_bf16_to_fp16,
    )

    return effective_dtype == torch.float16


def make_grad_scaler(
    device: Union[str, torch.device] = "cuda",
    amp_enabled: bool = True,
    amp_dtype: str = "bf16",
    fallback_bf16_to_fp16: bool = True,
):
    enabled = should_use_grad_scaler(
        device=device,
        amp_enabled=amp_enabled,
        amp_dtype=amp_dtype,
        fallback_bf16_to_fp16=fallback_bf16_to_fp16,
    )

    if not enabled:
        return None

    device_type = normalize_device_type(device)

    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            sig = inspect.signature(torch.amp.GradScaler)

            if "device" in sig.parameters:
                return torch.amp.GradScaler(device=device_type, enabled=True)

            return torch.amp.GradScaler(device_type, enabled=True)

        except Exception:
            pass

    if hasattr(torch.cuda, "amp") and hasattr(torch.cuda.amp, "GradScaler"):
        return torch.cuda.amp.GradScaler(enabled=True)

    return None


@contextmanager
def autocast_ctx(
    device: Union[str, torch.device] = "cuda",
    enabled: bool = True,
    amp_dtype: str = "bf16",
    cache_enabled: bool = True,
    fallback_bf16_to_fp16: bool = True,
):
    """
    Safe autocast context.

    Important:
    Do NOT wrap the yield itself in try/except.
    Otherwise, if the model forward fails, contextlib can raise:
        RuntimeError: generator didn't stop after throw()
    """
    if not enabled:
        with nullcontext():
            yield
        return

    device_type = normalize_device_type(device)

    effective_dtype = get_effective_amp_dtype(
        amp_dtype=amp_dtype,
        device=device,
        fallback_bf16_to_fp16=fallback_bf16_to_fp16,
    )

    if effective_dtype is None:
        with nullcontext():
            yield
        return

    if not hasattr(torch, "amp") or not hasattr(torch.amp, "autocast"):
        with nullcontext():
            yield
        return

    if device_type in {"cuda", "cpu"}:
        ctx = torch.amp.autocast(
            device_type=device_type,
            dtype=effective_dtype,
            cache_enabled=cache_enabled,
        )

        with ctx:
            yield

        return

    with nullcontext():
        yield


def setup_device_and_precision(
    device: Union[str, torch.device] = "auto",
    amp_enabled: bool = True,
    amp_dtype: str = "bf16",
    cache_enabled: bool = True,
    fallback_bf16_to_fp16: bool = True,
) -> Dict[str, Any]:
    resolved_device = resolve_device(device)
    device_type = normalize_device_type(resolved_device)

    effective_dtype = get_effective_amp_dtype(
        amp_dtype=amp_dtype,
        device=resolved_device,
        fallback_bf16_to_fp16=fallback_bf16_to_fp16,
    )

    final_amp_enabled = bool(amp_enabled and effective_dtype is not None)

    scaler = make_grad_scaler(
        device=resolved_device,
        amp_enabled=final_amp_enabled,
        amp_dtype=amp_dtype,
        fallback_bf16_to_fp16=fallback_bf16_to_fp16,
    )

    return {
        "device": resolved_device,
        "device_type": device_type,
        "amp_enabled": final_amp_enabled,
        "amp_dtype_requested": amp_dtype,
        "amp_dtype_effective": effective_dtype,
        "use_grad_scaler": scaler is not None,
        "scaler": scaler,
        "cache_enabled": cache_enabled,
        "fallback_bf16_to_fp16": fallback_bf16_to_fp16,
    }

In [88]:
def move_batch_to_device(
    batch: Union[Dict[str, Any], Tuple[Any, ...], torch.Tensor],
    device: torch.device,
    non_blocking: bool = True,
) -> Union[Dict[str, Any], Tuple[Any, ...], torch.Tensor]:
    """
    Move a batch to device.

    Designed for batches like:

        {
            "input_ids": Tensor[B, T],
            "labels": Tensor[B, T],
            "attention_mask": optional Tensor[B, T],
            "mtp_labels": optional Tensor[B, mtp_depth, T],
            ...
        }

    But it is recursive, so it also handles nested dicts/lists/tuples.

    Args:
        batch:
            Tensor, dict, tuple or list containing tensors.
        device:
            Destination device.
        non_blocking:
            Passed to tensor.to(...).

    Returns:
        Batch with all tensors moved to device.
    """
    if torch.is_tensor(batch):
        return batch.to(device=device, non_blocking=non_blocking)

    if isinstance(batch, dict):
        return {
            key: move_batch_to_device(value, device, non_blocking=non_blocking)
            for key, value in batch.items()
        }

    if isinstance(batch, tuple):
        return tuple(
            move_batch_to_device(value, device, non_blocking=non_blocking)
            for value in batch
        )

    if isinstance(batch, list):
        return [
            move_batch_to_device(value, device, non_blocking=non_blocking)
            for value in batch
        ]

    # Non-tensor metadata is left unchanged.
    return batch


# Optimizer

In [89]:
def _is_norm_parameter(name: str, module: Optional[nn.Module] = None) -> bool:
    """
    Conservative norm detector.

    Catches:
        - RMSNorm
        - LayerNorm
        - BatchNorm
        - parameters whose names include norm
    """
    lower_name = name.lower()

    if "norm" in lower_name:
        return True

    if module is not None:
        norm_classes = (
            nn.LayerNorm,
            nn.BatchNorm1d,
            nn.BatchNorm2d,
            nn.BatchNorm3d,
            nn.GroupNorm,
            nn.InstanceNorm1d,
            nn.InstanceNorm2d,
            nn.InstanceNorm3d,
            nn.LocalResponseNorm,
        )
        if isinstance(module, norm_classes):
            return True

    return False


def _is_embedding_parameter(name: str, module: Optional[nn.Module] = None) -> bool:
    lower_name = name.lower()

    if "embedding" in lower_name or "embed" in lower_name or "wte" in lower_name:
        return True

    if module is not None and isinstance(module, nn.Embedding):
        return True

    return False


def _get_module_by_parameter_name(model: nn.Module) -> Dict[str, nn.Module]:
    """
    Build mapping from full parameter name to the module that owns it.

    Example:
        blocks.0.attn.q_proj.weight -> q_proj module
    """
    param_to_module: Dict[str, nn.Module] = {}

    for module_name, module in model.named_modules():
        for param_name, _ in module.named_parameters(recurse=False):
            full_name = f"{module_name}.{param_name}" if module_name else param_name
            param_to_module[full_name] = module

    return param_to_module


def build_adamw_parameter_groups(
    model: nn.Module,
    weight_decay: float = 0.1,
    no_decay_weight_decay: float = 0.0,
    verbose: bool = False,
) -> Tuple[list, Dict[str, Any]]:
    """
    Build AdamW parameter groups.

    Policy:
        decay:
            Parameters with ndim >= 2 that are not embeddings and not lm_head.

        no_decay:
            biases
            norm weights
            embeddings
            lm_head
            scalar/vector params
            mHC small/static params
            router/scalar gates if they are not matrices

    This is intentionally conservative for DeepSeek-V4 mini because mHC,
    routing scalars, normalization and embeddings should not receive decay.

    Args:
        model:
            DeepSeekV4LM or any nn.Module.
        weight_decay:
            Weight decay for decay group.
        no_decay_weight_decay:
            Usually 0.0.
        verbose:
            If True, returns names and prints group sizes.

    Returns:
        optimizer_groups, info
    """
    decay_params = []
    no_decay_params = []

    decay_names = []
    no_decay_names = []

    param_to_module = _get_module_by_parameter_name(model)

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        module = param_to_module.get(name, None)
        lower_name = name.lower()

        is_bias = name.endswith(".bias")
        is_norm = _is_norm_parameter(name, module)
        is_embedding = _is_embedding_parameter(name, module)
        is_lm_head = "lm_head" in lower_name or "prediction_head" in lower_name
        is_mtp_vocab_head = "mtp" in lower_name and "head" in lower_name
        is_scalar_or_vector = param.ndim < 2

        # mHC has small dynamic/static/gating parameters where decay is risky.
        is_mhc_small_param = (
            "mhc" in lower_name
            or "hyper" in lower_name
            or "sinkhorn" in lower_name
            or "alpha" in lower_name
            or "static_" in lower_name
        )

        # Compression/indexer biases or small controls should be no_decay.
        is_special_small_control = (
            "bias_a" in lower_name
            or "bias_b" in lower_name
            or "temperature" in lower_name
            or "scale" in lower_name
            or "gate" in lower_name and param.ndim < 2
        )

        use_no_decay = (
            is_bias
            or is_norm
            or is_embedding
            or is_lm_head
            or is_mtp_vocab_head
            or is_scalar_or_vector
            or is_mhc_small_param
            or is_special_small_control
        )

        if use_no_decay:
            no_decay_params.append(param)
            no_decay_names.append(name)
        else:
            decay_params.append(param)
            decay_names.append(name)

    optimizer_groups = [
        {
            "params": decay_params,
            "weight_decay": weight_decay,
            "group_name": "decay",
        },
        {
            "params": no_decay_params,
            "weight_decay": no_decay_weight_decay,
            "group_name": "no_decay",
        },
    ]

    info = {
        "num_decay_params": sum(p.numel() for p in decay_params),
        "num_no_decay_params": sum(p.numel() for p in no_decay_params),
        "num_decay_tensors": len(decay_params),
        "num_no_decay_tensors": len(no_decay_params),
        "decay_names": decay_names,
        "no_decay_names": no_decay_names,
    }

    if verbose:
        total = info["num_decay_params"] + info["num_no_decay_params"]
        print("=" * 80)
        print("AdamW parameter groups")
        print("=" * 80)
        print(f"Decay tensors     : {info['num_decay_tensors']}")
        print(f"No-decay tensors  : {info['num_no_decay_tensors']}")
        print(f"Decay params      : {info['num_decay_params']:,}")
        print(f"No-decay params   : {info['num_no_decay_params']:,}")
        print(f"Total params      : {total:,}")
        print("=" * 80)

    return optimizer_groups, info

In [90]:
def build_adamw_optimizer(
    model: nn.Module,
    learning_rate: float = 3e-4,
    weight_decay: float = 0.1,
    betas: Tuple[float, float] = (0.9, 0.95),
    eps: float = 1e-8,
    verbose: bool = False,
) -> Tuple[torch.optim.AdamW, Dict[str, Any]]:
    """
    Build AdamW optimizer with DeepSeek-V4-mini-safe parameter grouping.

    Args:
        model:
            Model to optimize.
        learning_rate:
            AdamW learning rate.
        weight_decay:
            Weight decay for matrix-like trainable weights.
        betas:
            AdamW betas.
        eps:
            AdamW epsilon.
        verbose:
            Whether to print parameter group summary.

    Returns:
        optimizer, group_info
    """
    if learning_rate <= 0:
        raise ValueError(f"learning_rate must be > 0, got {learning_rate}")

    if weight_decay < 0:
        raise ValueError(f"weight_decay must be >= 0, got {weight_decay}")

    if len(betas) != 2:
        raise ValueError(f"betas must have length 2, got {betas}")

    parameter_groups, group_info = build_adamw_parameter_groups(
        model=model,
        weight_decay=weight_decay,
        no_decay_weight_decay=0.0,
        verbose=verbose,
    )

    optimizer = torch.optim.AdamW(
        parameter_groups,
        lr=learning_rate,
        betas=betas,
        eps=eps,
    )

    return optimizer, group_info

In [196]:
import numpy as np

CFG = SyntheticRetrievalConfig(
    # Dataset size
    num_train_examples=50_000,
    num_val_examples=5_000,

    # Sequence length
    block_size=64,              # debe ser <= model.config.max_seq_len
    min_filler_tokens=8,
    max_filler_tokens=32,

    # Task structure
    num_keys_per_example=4,
    vocab_filler_size=68,       # 20 + 64 + 64 + 68 = 216
    num_key_types=64,
    num_value_types=64,

    # DataLoader
    batch_size=32,
    num_workers=0,

    # Reproducibility
    seed=42)

train_loader, val_loader, tokenizer = create_synthetic_retrieval_dataloaders(
    cfg=CFG,
    use_mtp=False,)

print("Tokenizer vocab:", tokenizer.vocab_size)
print("Model vocab:", model.config.vocab_size)

precision = setup_device_and_precision(
    device="auto",
    amp_enabled=True,
    amp_dtype="bf16",
)

device = precision["device"]
model = model.to(device)

optimizer, opt_info = build_adamw_optimizer(
    model=model,
    learning_rate=3e-4,
    weight_decay=0.1,
    betas=(0.9, 0.95),
    eps=1e-8,
    verbose=True,
)


batch = next(iter(train_loader))

batch = normalize_lm_batch(batch)
batch = move_batch_to_device(batch, device)

with autocast_ctx(
    device=device,
    enabled=precision["amp_enabled"],
    amp_dtype=precision["amp_dtype_requested"],
    cache_enabled=precision["cache_enabled"],
    fallback_bf16_to_fp16=precision["fallback_bf16_to_fp16"],
):
    outputs = model(**batch)
    loss = outputs["loss"]

print(loss)

scaler = precision["scaler"]

if scaler is not None:
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
else:
    loss.backward()
    optimizer.step()

Synthetic tokenizer vocab size: 216
Tokenizer vocab: 216
Model vocab: 216
AdamW parameter groups
Decay tensors     : 170
No-decay tensors  : 127
Decay params      : 28,851,712
No-decay params   : 452,508
Total params      : 29,304,220
tensor(7.0348, device='cuda:0', grad_fn=<AddBackward0>)


---

# Chekpoint

In [95]:
# ============================================================
# CHECKPOINT UTILITIES
# DeepSeek-V4 Mini Training Stack
# ============================================================

from __future__ import annotations

import os
import re
import json
import random
import shutil
from pathlib import Path
from typing import Any, Dict, Optional, Union

import numpy as np
import torch
import torch.nn as nn


# ============================================================
# Small helpers
# ============================================================

def _safe_to_serializable(obj: Any) -> Any:
    """
    Best-effort conversion for configs/metadata into JSON-safe objects.

    Useful for saving config snapshots next to the .pt checkpoint.
    """
    if obj is None:
        return None

    if isinstance(obj, (str, int, float, bool)):
        return obj

    if isinstance(obj, (list, tuple)):
        return [_safe_to_serializable(x) for x in obj]

    if isinstance(obj, dict):
        return {
            str(k): _safe_to_serializable(v)
            for k, v in obj.items()
        }

    if hasattr(obj, "__dict__"):
        return {
            str(k): _safe_to_serializable(v)
            for k, v in vars(obj).items()
            if not k.startswith("_")
        }

    return str(obj)


def _get_rng_state() -> Dict[str, Any]:
    """
    Capture random states so training can be resumed more faithfully.
    """
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": None,
    }

    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()

    return state


def _set_rng_state(state):
    """
    Restore Python, NumPy, PyTorch CPU and CUDA RNG states safely.

    Important:
        torch.set_rng_state expects a CPU ByteTensor.
        torch.cuda.set_rng_state_all expects CUDA-compatible RNG states,
        but CPU ByteTensors are also accepted in most PyTorch versions.
    """
    if not state:
        return

    # Python RNG
    if "python" in state and state["python"] is not None:
        random.setstate(state["python"])

    # NumPy RNG
    if "numpy" in state and state["numpy"] is not None:
        np.random.set_state(state["numpy"])

    # Torch CPU RNG
    if "torch" in state and state["torch"] is not None:
        torch_state = state["torch"]

        if not torch.is_tensor(torch_state):
            torch_state = torch.tensor(torch_state, dtype=torch.uint8)

        torch_state = torch_state.detach().cpu().to(dtype=torch.uint8)
        torch.set_rng_state(torch_state)

    # CUDA RNG
    if torch.cuda.is_available() and state.get("cuda") is not None:
        cuda_states = state["cuda"]

        fixed_cuda_states = []
        for s in cuda_states:
            if not torch.is_tensor(s):
                s = torch.tensor(s, dtype=torch.uint8)

            # Keep as CPU ByteTensor; PyTorch usually accepts this.
            # If your version requires CUDA tensors, change `.cpu()` to `.cuda(i)`.
            s = s.detach().cpu().to(dtype=torch.uint8)
            fixed_cuda_states.append(s)

        torch.cuda.set_rng_state_all(fixed_cuda_states)


def _unwrap_model(model: nn.Module) -> nn.Module:
    """
    Handles DataParallel/DDP-style wrappers.
    """
    return model.module if hasattr(model, "module") else model


def _atomic_torch_save(obj: Dict[str, Any], path: Union[str, Path]) -> None:
    """
    Atomic save:
        1. save to temporary file
        2. rename to final path

    This avoids corrupting the last checkpoint if the process crashes mid-save.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = path.with_suffix(path.suffix + ".tmp")

    torch.save(obj, tmp_path)
    os.replace(tmp_path, path)


def _extract_step_from_name(path: Union[str, Path]) -> int:
    """
    Extract step number from filenames like:
        checkpoint_step_000001.pt
        ckpt_step_100.pt
    """
    name = Path(path).name
    matches = re.findall(r"step[_-](\d+)", name)

    if not matches:
        return -1

    return int(matches[-1])


def cleanup_old_checkpoints(
    checkpoint_dir: Union[str, Path],
    keep_last_n: int = 3,
) -> None:
    """
    Keep only the last N step checkpoints.

    Does not delete:
        latest.pt
        checkpoint_best_*.pt
    """
    checkpoint_dir = Path(checkpoint_dir)

    if keep_last_n <= 0:
        return

    candidates = list(checkpoint_dir.glob("checkpoint_step_*.pt"))

    if len(candidates) <= keep_last_n:
        return

    candidates = sorted(
        candidates,
        key=lambda p: (_extract_step_from_name(p), p.stat().st_mtime),
    )

    to_delete = candidates[:-keep_last_n]

    for path in to_delete:
        try:
            path.unlink()
        except Exception:
            pass

        sidecar = path.with_suffix(".json")
        if sidecar.exists():
            try:
                sidecar.unlink()
            except Exception:
                pass


# ============================================================
# Save checkpoint
# ============================================================

def save_checkpoint(
    checkpoint_dir: Union[str, Path],
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[Any] = None,
    scaler: Optional[Any] = None,
    ema: Optional[Any] = None,
    epoch: int = 0,
    step: int = 0,
    best_metric: Optional[float] = None,
    config: Optional[Any] = None,
    extra_state: Optional[Dict[str, Any]] = None,
    filename: Optional[str] = None,
    save_rng_state: bool = True,
    keep_last_n: Optional[int] = None,
    tag: Optional[str] = None,
) -> Path:
    """
    Save a full training checkpoint.

    Saves:
        - model
        - optimizer, if provided
        - scheduler, if provided
        - scaler, if provided
        - EMA, if provided
        - epoch / step / best_metric
        - config / extra_state
        - RNG state, if save_rng_state=True
    """
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    if filename is None:
        if tag is not None:
            filename = f"checkpoint_{tag}_step_{step:08d}.pt"
        else:
            filename = f"checkpoint_step_{step:08d}.pt"

    ckpt_path = checkpoint_dir / filename

    raw_model = _unwrap_model(model)

    checkpoint = {
        "model_state_dict": raw_model.state_dict(),
        "epoch": int(epoch),
        "step": int(step),
        "best_metric": best_metric,
        "config": _safe_to_serializable(config),
        "extra_state": extra_state or {},
        "rng_state": _get_rng_state() if save_rng_state else None,
        "has_ema": ema is not None,
    }

    if optimizer is not None:
        checkpoint["optimizer_state_dict"] = optimizer.state_dict()

    if scheduler is not None:
        if hasattr(scheduler, "state_dict"):
            checkpoint["scheduler_state_dict"] = scheduler.state_dict()
        else:
            checkpoint["scheduler_state_dict"] = None

    if scaler is not None:
        if hasattr(scaler, "state_dict"):
            checkpoint["scaler_state_dict"] = scaler.state_dict()
        else:
            checkpoint["scaler_state_dict"] = None

    if ema is not None:
        if hasattr(ema, "state_dict"):
            checkpoint["ema_state_dict"] = ema.state_dict()
        else:
            raise TypeError(
                "ema was provided but does not implement state_dict()."
            )

    _atomic_torch_save(checkpoint, ckpt_path)

    # Save a lightweight JSON sidecar for quick inspection.
    metadata_path = ckpt_path.with_suffix(".json")
    metadata = {
        "checkpoint": ckpt_path.name,
        "epoch": int(epoch),
        "step": int(step),
        "best_metric": best_metric,
        "tag": tag,
        "has_optimizer": optimizer is not None,
        "has_scheduler": scheduler is not None,
        "has_scaler": scaler is not None,
        "has_ema": ema is not None,
        "config": _safe_to_serializable(config),
        "extra_state": _safe_to_serializable(extra_state or {}),
    }

    try:
        with open(metadata_path, "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)
    except Exception:
        pass

    # Update latest pointer.
    latest_path = checkpoint_dir / "latest.pt"
    try:
        shutil.copyfile(ckpt_path, latest_path)
    except Exception:
        pass

    # Optional retention policy.
    if keep_last_n is not None and keep_last_n > 0:
        cleanup_old_checkpoints(
            checkpoint_dir=checkpoint_dir,
            keep_last_n=keep_last_n,
        )

    return ckpt_path


# ============================================================
# Load checkpoint
# ============================================================

def load_checkpoint(
    checkpoint_path: Union[str, Path],
    model: Optional[nn.Module] = None,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[Any] = None,
    scaler: Optional[Any] = None,
    ema: Optional[Any] = None,
    map_location: Union[str, torch.device] = "cpu",
    strict: bool = True,
    load_optimizer: bool = True,
    load_scheduler: bool = True,
    load_scaler: bool = True,
    load_ema: bool = True,
    load_rng_state: bool = True,
) -> Dict[str, Any]:
    """
    Load checkpoint into model/optimizer/scheduler/scaler/EMA if provided.

    Returns:
        state dict with epoch, step, best_metric, config, extra_state, etc.
    """
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(
        checkpoint_path,
        map_location=map_location,
        weights_only=False,
    )

    if model is not None:
        raw_model = _unwrap_model(model)

        missing, unexpected = raw_model.load_state_dict(
            checkpoint["model_state_dict"],
            strict=strict,
        )

        checkpoint["missing_keys"] = missing
        checkpoint["unexpected_keys"] = unexpected

    if (
        optimizer is not None
        and load_optimizer
        and "optimizer_state_dict" in checkpoint
    ):
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if (
        scheduler is not None
        and load_scheduler
        and "scheduler_state_dict" in checkpoint
        and checkpoint["scheduler_state_dict"] is not None
    ):
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if (
        scaler is not None
        and load_scaler
        and "scaler_state_dict" in checkpoint
        and checkpoint["scaler_state_dict"] is not None
    ):
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    if (
        ema is not None
        and load_ema
        and "ema_state_dict" in checkpoint
        and checkpoint["ema_state_dict"] is not None
    ):
        # Tu EMA acepta strict=False si usas la versión adaptada.
        # Si usas la EMA original, esta llamada también funciona sin strict.
        try:
            ema.load_state_dict(checkpoint["ema_state_dict"], strict=False)
        except TypeError:
            ema.load_state_dict(checkpoint["ema_state_dict"])

    if load_rng_state:
        _set_rng_state(checkpoint.get("rng_state", None))

    return {
        "epoch": checkpoint.get("epoch", 0),
        "step": checkpoint.get("step", 0),
        "best_metric": checkpoint.get("best_metric", None),
        "config": checkpoint.get("config", None),
        "extra_state": checkpoint.get("extra_state", {}),
        "checkpoint_path": str(checkpoint_path),
        "missing_keys": checkpoint.get("missing_keys", []),
        "unexpected_keys": checkpoint.get("unexpected_keys", []),
        "has_ema": bool(checkpoint.get("has_ema", "ema_state_dict" in checkpoint)),
        "loaded_ema": bool(
            ema is not None
            and load_ema
            and "ema_state_dict" in checkpoint
            and checkpoint["ema_state_dict"] is not None
        ),
    }

In [96]:
ckpt_path = save_checkpoint(
    checkpoint_dir="checkpoints/deepseekv4_mini",
    model=model,
    optimizer=optimizer,
    scheduler=None,
    scaler=precision["scaler"],
    epoch=0,
    step=1,
    best_metric=None,
    config=getattr(model, "config", None),
    extra_state={
        "loss": float(loss.detach().cpu().item()),
        "tokenizer_vocab_size": tokenizer.vocab_size,
    },
    keep_last_n=3,
)

print("Saved:", ckpt_path)

Saved: checkpoints/deepseekv4_mini/checkpoint_step_00000001.pt


In [97]:
state = load_checkpoint(
    checkpoint_path=ckpt_path,
    model=model,
    optimizer=optimizer,
    scheduler=None,
    scaler=precision["scaler"],
    map_location=device,
    strict=True,
)

print(state)

{'epoch': 0, 'step': 1, 'best_metric': None, 'config': {'vocab_size': 216, 'd_model': 64, 'n_layers': 2, 'max_seq_len': 64, 'pad_token_id': 0, 'ignore_index': -100, 'labels_are_shifted': True, 'ignore_pad_token_in_loss': True, 'embedding_dropout': 0.0, 'scale_embeddings': False, 'tie_word_embeddings': True, 'rms_norm_eps': 1e-06, 'attention_type': 'csa', 'n_heads': 4, 'head_dim': 16, 'attention_dropout': 0.0, 'residual_dropout': 0.0, 'use_attention_bias': False, 'use_rope': True, 'rope_theta': 10000.0, 'rotary_dim': 16, 'compression_factor': 4, 'hca_compression_factor': 16, 'window_size': 8, 'top_k_blocks': 2, 'indexer_dim': 8, 'n_indexer_heads': 2, 'query_compression_dim': 16, 'use_attention_sink': True, 'use_grouped_output_projection': True, 'output_projection_groups': 2, 'use_indexer_score_bias': False, 'use_separate_local_kv': True, 'ffn_type': 'moe', 'mlp_hidden_dim': None, 'mlp_expansion_factor': 4.0, 'mlp_multiple_of': 1, 'mlp_dropout': 0.0, 'use_mlp_bias': False, 'num_experts':

---

# EMA

In [98]:
# ============================================================
# EMA UTILITIES
# DeepSeek-V4 Mini Training Stack
# ============================================================

from __future__ import annotations

from contextlib import contextmanager
from typing import Dict, Optional, Union, Iterable

import torch
import torch.nn as nn


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if hasattr(model, "module") else model


class EMA:
    """
    Exponential Moving Average over trainable model parameters.

    Designed for Mini DeepSeek-V4 training.

    Features:
        - Shadow weights stored in fp32.
        - Optional CPU offload.
        - Name-based parameter tracking.
        - Safe temporary swap for EMA evaluation.
        - Compatible with checkpoint state_dict.
        - Supports update_after_step and update_every.
    """

    def __init__(
        self,
        model: nn.Module,
        decay: float = 0.999,
        device: Optional[Union[str, torch.device]] = None,
        use_num_updates: bool = True,
        update_after_step: int = 0,
        update_every: int = 1,
        exclude_names: Optional[Iterable[str]] = None,
    ):
        if not (0.0 <= decay < 1.0):
            raise ValueError(f"EMA decay must satisfy 0 <= decay < 1. Got {decay}.")

        if update_after_step < 0:
            raise ValueError(f"update_after_step must be >= 0. Got {update_after_step}.")

        if update_every <= 0:
            raise ValueError(f"update_every must be > 0. Got {update_every}.")

        self.decay = float(decay)
        self.device = torch.device(device) if device is not None else None
        self.use_num_updates = bool(use_num_updates)
        self.update_after_step = int(update_after_step)
        self.update_every = int(update_every)

        self.num_updates = 0
        self.total_steps_seen = 0

        self.exclude_names = tuple(exclude_names or ())

        model = unwrap_model(model)

        self.shadow: Dict[str, torch.Tensor] = {}
        self.backup: Dict[str, torch.Tensor] = {}

        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue

            if self._is_excluded(name):
                continue

            shadow = p.detach().to(dtype=torch.float32).clone()

            if self.device is not None:
                shadow = shadow.to(self.device)

            self.shadow[name] = shadow

    def _is_excluded(self, name: str) -> bool:
        return any(pattern in name for pattern in self.exclude_names)

    def _compute_decay(self) -> float:
        """
        Compute effective decay.

        Warmup formula:
            starts lower and approaches target decay.
        """
        if not self.use_num_updates:
            return self.decay

        d = min(self.decay, (1.0 + self.num_updates) / (10.0 + self.num_updates))
        return float(d)

    @torch.no_grad()
    def update(self, model: nn.Module, step: Optional[int] = None) -> bool:
        """
        Update EMA shadow from current model parameters.

        Args:
            model:
                Live model.
            step:
                Optional global step. If not provided, internal counter is used.

        Returns:
            True if EMA was updated, False otherwise.
        """
        if step is None:
            self.total_steps_seen += 1
            current_step = self.total_steps_seen
        else:
            current_step = int(step)
            self.total_steps_seen = max(self.total_steps_seen, current_step)

        if current_step < self.update_after_step:
            return False

        if (current_step - self.update_after_step) % self.update_every != 0:
            return False

        model = unwrap_model(model)

        self.num_updates += 1
        decay = self._compute_decay()

        for name, p in model.named_parameters():
            if name not in self.shadow:
                continue

            if not p.requires_grad:
                continue

            shadow = self.shadow[name]
            param_fp32 = p.detach().to(dtype=torch.float32)

            if shadow.device != param_fp32.device:
                param_fp32 = param_fp32.to(shadow.device)

            shadow.mul_(decay).add_(param_fp32, alpha=1.0 - decay)

        return True

    @torch.no_grad()
    def copy_to(self, model: nn.Module) -> None:
        """
        Copy EMA weights into the live model.
        """
        model = unwrap_model(model)

        for name, p in model.named_parameters():
            if name in self.shadow:
                p.data.copy_(self.shadow[name].to(device=p.device, dtype=p.dtype))

    @torch.no_grad()
    def store(self, model: nn.Module) -> None:
        """
        Store current live model parameters before EMA evaluation.
        """
        model = unwrap_model(model)

        self.backup = {}

        for name, p in model.named_parameters():
            if name in self.shadow:
                self.backup[name] = p.detach().clone()

    @torch.no_grad()
    def restore(self, model: nn.Module) -> None:
        """
        Restore live model parameters after EMA evaluation.
        """
        model = unwrap_model(model)

        for name, p in model.named_parameters():
            if name in self.backup:
                p.data.copy_(self.backup[name].to(device=p.device, dtype=p.dtype))

        self.backup = {}

    @contextmanager
    def average_parameters(self, model: nn.Module):
        """
        Temporarily evaluate with EMA weights.

        Usage:
            with ema.average_parameters(model):
                val_loss = evaluate(...)
        """
        self.store(model)
        self.copy_to(model)

        try:
            yield
        finally:
            self.restore(model)

    @torch.no_grad()
    def to(self, device: Union[str, torch.device]) -> None:
        """
        Move EMA shadow weights to a device.
        """
        self.device = torch.device(device)

        for name in self.shadow:
            self.shadow[name] = self.shadow[name].to(self.device)

    @torch.no_grad()
    def reinit_from_model(self, model: nn.Module) -> None:
        """
        Hard reset EMA weights from current model weights.
        """
        model = unwrap_model(model)

        for name, p in model.named_parameters():
            if name in self.shadow:
                self.shadow[name].data.copy_(
                    p.detach().to(device=self.shadow[name].device, dtype=torch.float32)
                )

    def state_dict(self) -> Dict:
        """
        Checkpoint-safe EMA state.
        """
        return {
            "decay": self.decay,
            "device": str(self.device) if self.device is not None else None,
            "use_num_updates": self.use_num_updates,
            "update_after_step": self.update_after_step,
            "update_every": self.update_every,
            "num_updates": self.num_updates,
            "total_steps_seen": self.total_steps_seen,
            "exclude_names": self.exclude_names,
            "shadow": {
                name: tensor.detach().cpu()
                for name, tensor in self.shadow.items()
            },
        }

    @torch.no_grad()
    def load_state_dict(self, state_dict: Dict, strict: bool = False) -> None:
        """
        Restore EMA state from checkpoint.
        """
        self.decay = float(state_dict.get("decay", self.decay))
        self.use_num_updates = bool(state_dict.get("use_num_updates", self.use_num_updates))
        self.update_after_step = int(state_dict.get("update_after_step", self.update_after_step))
        self.update_every = int(state_dict.get("update_every", self.update_every))
        self.num_updates = int(state_dict.get("num_updates", self.num_updates))
        self.total_steps_seen = int(state_dict.get("total_steps_seen", self.total_steps_seen))
        self.exclude_names = tuple(state_dict.get("exclude_names", self.exclude_names))

        loaded_shadow = state_dict.get("shadow", {})

        missing = []
        unexpected = []

        for name, shadow in self.shadow.items():
            if name in loaded_shadow:
                shadow.data.copy_(
                    loaded_shadow[name].to(device=shadow.device, dtype=shadow.dtype)
                )
            else:
                missing.append(name)

        for name in loaded_shadow:
            if name not in self.shadow:
                unexpected.append(name)

        if strict and (missing or unexpected):
            raise RuntimeError(
                f"EMA state mismatch. Missing={missing[:10]}, Unexpected={unexpected[:10]}"
            )

        if missing:
            print(f"[EMA] Warning: {len(missing)} missing EMA parameters.")

        if unexpected:
            print(f"[EMA] Warning: {len(unexpected)} unexpected EMA parameters.")

    def __len__(self) -> int:
        return len(self.shadow)


@torch.no_grad()
def ema_health(
    ema: EMA,
    model: nn.Module,
    rel_tol: float = 5.0,
):
    """
    Basic sanity check comparing EMA weights against live model weights.

    Returns:
        ok, status, relative_difference
    """
    model = unwrap_model(model)

    model_params = []
    ema_params = []

    for name, p in model.named_parameters():
        if name in ema.shadow:
            model_params.append(p.detach().float().cpu().reshape(-1))
            ema_params.append(ema.shadow[name].detach().float().cpu().reshape(-1))

    if not model_params:
        return False, "empty_ema", float("inf")

    model_flat = torch.cat(model_params, dim=0)
    ema_flat = torch.cat(ema_params, dim=0)

    if not torch.isfinite(ema_flat).all():
        return False, "nan_or_inf_in_ema", float("inf")

    model_norm = model_flat.norm().item()
    ema_norm = ema_flat.norm().item()

    if model_norm < 1e-12:
        return False, "model_zero_norm", float("inf")

    if ema_norm < 1e-12:
        return False, "ema_zero_norm", float("inf")

    rel_diff = (model_flat - ema_flat).norm().item() / (model_norm + 1e-8)

    if rel_diff > rel_tol:
        return False, "large_rel_diff", rel_diff

    return True, "ok", rel_diff

In [99]:
ema = EMA(
    model=model,
    decay=0.999,
    device="cpu",
    use_num_updates=True,
    update_after_step=1,
    update_every=1,
)

optimizer.zero_grad(set_to_none=True)

with autocast_ctx(
    device=device,
    enabled=precision["amp_enabled"],
    amp_dtype=precision["amp_dtype_requested"],
    cache_enabled=precision["cache_enabled"],
    fallback_bf16_to_fp16=precision["fallback_bf16_to_fp16"],
):
    outputs = model(**batch)
    loss = outputs["loss"]

scaler = precision["scaler"]

if scaler is not None:
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
else:
    loss.backward()
    optimizer.step()

ema.update(model, step=1)

ok, status, rel = ema_health(ema, model)
print("EMA:", ok, status, rel)

EMA: True ok 0.0009718340313424267


---
# Metrics

In [100]:
# ============================================================
# EVALUATION METRICS
# DeepSeek-V4 Mini Training Stack
# ============================================================

from __future__ import annotations

import math
from typing import Any, Dict, Optional, Sequence, Union

import torch
import torch.nn.functional as F
import torch.nn as nn


# ============================================================
# Small helpers
# ============================================================

def _get_output_value(outputs: Any, key: str, default: Any = None) -> Any:
    """
    Robustly get value from model outputs.

    Supports:
        - dict outputs
        - objects with attributes
    """
    if isinstance(outputs, dict):
        return outputs.get(key, default)

    return getattr(outputs, key, default)


def _get_logits_from_outputs(outputs: Any) -> torch.Tensor:
    """
    Extract logits from model outputs.
    """
    logits = _get_output_value(outputs, "logits", None)

    if logits is None:
        logits = _get_output_value(outputs, "lm_logits", None)

    if logits is None:
        logits = _get_output_value(outputs, "prediction_logits", None)

    if logits is None:
        raise KeyError(
            "Could not find logits in model outputs. Expected one of: "
            "'logits', 'lm_logits', 'prediction_logits'."
        )

    return logits


def _get_loss_from_outputs(outputs: Any) -> Optional[torch.Tensor]:
    """
    Extract loss from model outputs if available.
    """
    loss = _get_output_value(outputs, "loss", None)
    return loss


def _get_model_config(model: nn.Module) -> Any:
    """
    Get model.config safely, handling DDP/DataParallel wrappers.
    """
    raw_model = model.module if hasattr(model, "module") else model
    return getattr(raw_model, "config", None)


def _prepare_logits_and_labels_for_metrics(
    model: nn.Module,
    logits: torch.Tensor,
    labels: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Align logits and labels according to model.config.labels_are_shifted.

    Case 1:
        labels_are_shifted=True

        input_ids: [x0, x1, ..., xT-1]
        labels:    [x1, x2, ..., xT]

        Metrics use:
            logits[:, :, :] vs labels[:, :]

    Case 2:
        labels_are_shifted=False

        input_ids: [x0, x1, ..., xT-1]
        labels:    [x0, x1, ..., xT-1]

        Metrics use:
            logits[:, :-1, :] vs labels[:, 1:]
    """
    cfg = _get_model_config(model)
    labels_are_shifted = True

    if cfg is not None and hasattr(cfg, "labels_are_shifted"):
        labels_are_shifted = bool(cfg.labels_are_shifted)

    if labels_are_shifted:
        T = min(logits.size(1), labels.size(1))
        return logits[:, :T, :], labels[:, :T]

    T = min(logits.size(1) - 1, labels.size(1) - 1)

    if T <= 0:
        raise ValueError(
            "Cannot compute shifted metrics because sequence length is too short."
        )

    return logits[:, :T, :], labels[:, 1:T + 1]


def _build_valid_token_mask(
    model: nn.Module,
    labels: torch.Tensor,
    ignore_index: Optional[int] = None,
    pad_token_id: Optional[int] = None,
) -> torch.Tensor:
    """
    Build mask for valid labels.

    Excludes:
        - ignore_index
        - pad_token_id, if model.config.ignore_pad_token_in_loss=True
    """
    cfg = _get_model_config(model)

    if ignore_index is None:
        ignore_index = getattr(cfg, "ignore_index", -100) if cfg is not None else -100

    if pad_token_id is None:
        pad_token_id = getattr(cfg, "pad_token_id", None) if cfg is not None else None

    ignore_pad = True
    if cfg is not None and hasattr(cfg, "ignore_pad_token_in_loss"):
        ignore_pad = bool(cfg.ignore_pad_token_in_loss)

    mask = labels != int(ignore_index)

    if pad_token_id is not None and ignore_pad:
        mask = mask & (labels != int(pad_token_id))

    return mask


# ============================================================
# Core LM metrics from logits
# ============================================================

@torch.no_grad()
def compute_lm_metrics_from_logits(
    model: nn.Module,
    logits: torch.Tensor,
    labels: torch.Tensor,
    topk: Sequence[int] = (1, 5),
    ignore_index: Optional[int] = None,
    pad_token_id: Optional[int] = None,
) -> Dict[str, float]:
    """
    Compute autoregressive LM metrics from logits and labels.

    Returns:
        {
            "ce_sum",
            "valid_tokens",
            "loss",
            "perplexity",
            "token_accuracy",
            "top_5_accuracy",
            "sequence_accuracy",
            "mean_confidence",
            "mean_true_token_prob",
            "mean_entropy",
        }
    """
    logits, labels = _prepare_logits_and_labels_for_metrics(
        model=model,
        logits=logits,
        labels=labels,
    )

    valid_mask = _build_valid_token_mask(
        model=model,
        labels=labels,
        ignore_index=ignore_index,
        pad_token_id=pad_token_id,
    )

    valid_tokens = int(valid_mask.sum().item())

    if valid_tokens == 0:
        return {
            "ce_sum": 0.0,
            "valid_tokens": 0,
            "loss": float("nan"),
            "perplexity": float("nan"),
            "token_accuracy": float("nan"),
            "sequence_accuracy": float("nan"),
            "mean_confidence": float("nan"),
            "mean_true_token_prob": float("nan"),
            "mean_entropy": float("nan"),
        }

    V = logits.size(-1)

    logits_flat = logits.reshape(-1, V)
    labels_flat = labels.reshape(-1)
    mask_flat = valid_mask.reshape(-1)

    valid_logits = logits_flat[mask_flat]
    valid_labels = labels_flat[mask_flat]

    # CE sum in fp32 for stable aggregation.
    ce_sum = F.cross_entropy(
        valid_logits.float(),
        valid_labels,
        reduction="sum",
    )

    loss = ce_sum / max(valid_tokens, 1)

    # Avoid overflow in exp.
    ppl = math.exp(min(float(loss.detach().cpu().item()), 50.0))

    probs = torch.softmax(valid_logits.float(), dim=-1)
    pred = torch.argmax(probs, dim=-1)

    token_acc = (pred == valid_labels).float().mean()

    metrics = {
        "ce_sum": float(ce_sum.detach().cpu().item()),
        "valid_tokens": float(valid_tokens),
        "loss": float(loss.detach().cpu().item()),
        "perplexity": float(ppl),
        "token_accuracy": float(token_acc.detach().cpu().item()),
    }

    # Top-k accuracies.
    max_k = max(topk) if len(topk) > 0 else 1
    max_k = min(max_k, V)

    topk_indices = torch.topk(valid_logits.float(), k=max_k, dim=-1).indices

    for k in topk:
        k_eff = min(int(k), V)
        correct_k = (topk_indices[:, :k_eff] == valid_labels[:, None]).any(dim=-1)
        metrics[f"top_{k_eff}_accuracy"] = float(correct_k.float().mean().detach().cpu().item())

    # Sequence accuracy: sequence counts as correct if all valid positions are correct.
    # This is harsh for language modeling, but useful for tiny smoke tests.
    with torch.no_grad():
        full_pred = torch.argmax(logits.float(), dim=-1)
        token_correct_full = (full_pred == labels) | (~valid_mask)
        seq_has_any_valid = valid_mask.any(dim=1)
        seq_correct = token_correct_full.all(dim=1) & seq_has_any_valid

        if seq_has_any_valid.any():
            sequence_accuracy = seq_correct.float().sum() / seq_has_any_valid.float().sum()
            metrics["sequence_accuracy"] = float(sequence_accuracy.detach().cpu().item())
        else:
            metrics["sequence_accuracy"] = float("nan")

    # Confidence and entropy.
    confidence = probs.max(dim=-1).values.mean()
    true_token_prob = probs.gather(1, valid_labels[:, None]).squeeze(1).mean()
    entropy = -(probs * torch.log(probs.clamp_min(1e-12))).sum(dim=-1).mean()

    metrics["mean_confidence"] = float(confidence.detach().cpu().item())
    metrics["mean_true_token_prob"] = float(true_token_prob.detach().cpu().item())
    metrics["mean_entropy"] = float(entropy.detach().cpu().item())

    return metrics


# ============================================================
# Aggregator
# ============================================================

class MetricAverager:
    """
    Weighted metric accumulator.

    Loss is aggregated by token count using ce_sum / valid_tokens.
    Other metrics are weighted by valid_tokens when possible.
    """

    def __init__(self):
        self.total_ce_sum = 0.0
        self.total_valid_tokens = 0.0
        self.weighted = {}
        self.weights = {}

    def update(self, metrics: Dict[str, float]) -> None:
        valid_tokens = float(metrics.get("valid_tokens", 0.0))

        if "ce_sum" in metrics:
            self.total_ce_sum += float(metrics["ce_sum"])

        self.total_valid_tokens += valid_tokens

        for key, value in metrics.items():
            if key in {"ce_sum", "valid_tokens", "loss", "perplexity"}:
                continue

            if value is None:
                continue

            try:
                value = float(value)
            except Exception:
                continue

            if not math.isfinite(value):
                continue

            w = max(valid_tokens, 1.0)

            self.weighted[key] = self.weighted.get(key, 0.0) + value * w
            self.weights[key] = self.weights.get(key, 0.0) + w

    def compute(self) -> Dict[str, float]:
        output = {}

        if self.total_valid_tokens > 0:
            loss = self.total_ce_sum / self.total_valid_tokens
            output["loss"] = float(loss)
            output["perplexity"] = float(math.exp(min(loss, 50.0)))
            output["valid_tokens"] = float(self.total_valid_tokens)
        else:
            output["loss"] = float("nan")
            output["perplexity"] = float("nan")
            output["valid_tokens"] = 0.0

        for key, total in self.weighted.items():
            denom = self.weights.get(key, 0.0)
            output[key] = float(total / denom) if denom > 0 else float("nan")

        return output


# ============================================================
# Evaluate one dataloader
# ============================================================

@torch.no_grad()
def evaluate_lm(
    model: nn.Module,
    dataloader,
    device: torch.device,
    precision: Optional[Dict[str, Any]] = None,
    max_batches: Optional[int] = None,
    topk: Sequence[int] = (1, 5),
    ema: Optional[Any] = None,
    use_ema: bool = False,
    return_aux_loss_items: bool = True,
) -> Dict[str, float]:
    """
    Evaluate LM model over a dataloader.

    Intended use:
        - call between epochs
        - optionally evaluate with EMA weights

    Args:
        model:
            DeepSeekV4LM.
        dataloader:
            Validation dataloader.
        device:
            Training/eval device.
        precision:
            Dict returned by setup_device_and_precision.
        max_batches:
            Optional cap for faster validation.
        topk:
            Top-k accuracies to compute.
        ema:
            Optional EMA object.
        use_ema:
            If True, temporarily swaps EMA weights into model.
        return_aux_loss_items:
            If True, averages scalar loss items from model outputs when present.

    Returns:
        Dict of metrics.
    """
    was_training = model.training
    model.eval()

    if precision is None:
        precision = {
            "amp_enabled": False,
            "amp_dtype_requested": "fp32",
            "cache_enabled": True,
            "fallback_bf16_to_fp16": True,
        }

    averager = MetricAverager()
    aux_sums = {}
    aux_counts = {}

    def _eval_loop():
        for batch_idx, batch in enumerate(dataloader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            batch = normalize_lm_batch(batch)
            batch = move_batch_to_device(batch, device)

            with autocast_ctx(
                device=device,
                enabled=precision["amp_enabled"],
                amp_dtype=precision["amp_dtype_requested"],
                cache_enabled=precision["cache_enabled"],
                fallback_bf16_to_fp16=precision["fallback_bf16_to_fp16"],
            ):
                outputs = model(**batch)

            logits = _get_logits_from_outputs(outputs)
            labels = batch["labels"]

            metrics = compute_lm_metrics_from_logits(
                model=model,
                logits=logits,
                labels=labels,
                topk=topk,
            )

            averager.update(metrics)

            if return_aux_loss_items:
                # Collect scalar diagnostics if present.
                # Common output keys: lm_loss, mtp_loss, moe_aux_loss, loss.
                for key in [
                    "loss",
                    "lm_loss",
                    "mtp_loss",
                    "moe_aux_loss",
                    "balance_loss",
                    "sequence_balance_loss",
                ]:
                    value = _get_output_value(outputs, key, None)

                    if value is None:
                        continue

                    if torch.is_tensor(value):
                        if value.numel() != 1:
                            continue
                        value = float(value.detach().float().cpu().item())
                    else:
                        try:
                            value = float(value)
                        except Exception:
                            continue

                    if math.isfinite(value):
                        aux_sums[key] = aux_sums.get(key, 0.0) + value
                        aux_counts[key] = aux_counts.get(key, 0) + 1

    if use_ema:
        if ema is None:
            raise ValueError("use_ema=True but ema=None.")

        with ema.average_parameters(model):
            _eval_loop()
    else:
        _eval_loop()

    metrics = averager.compute()

    for key, total in aux_sums.items():
        count = aux_counts.get(key, 0)
        if count > 0:
            metrics[f"model_{key}"] = total / count

    metrics["num_batches"] = float(
        max_batches if max_batches is not None else len(dataloader)
    )

    metrics["used_ema"] = bool(use_ema)

    if was_training:
        model.train()

    return metrics


# ============================================================
# Pretty print
# ============================================================

def format_metrics(metrics: Dict[str, float], prefix: str = "val") -> str:
    """
    Compact formatting for logs.
    """
    keys = [
        "loss",
        "perplexity",
        "token_accuracy",
        "top_5_accuracy",
        "sequence_accuracy",
        "mean_confidence",
        "mean_true_token_prob",
        "mean_entropy",
        "valid_tokens",
    ]

    parts = []

    for key in keys:
        if key not in metrics:
            continue

        value = metrics[key]

        if isinstance(value, bool):
            parts.append(f"{prefix}/{key}={value}")
        elif isinstance(value, (int, float)):
            if key == "valid_tokens":
                parts.append(f"{prefix}/{key}={int(value)}")
            else:
                parts.append(f"{prefix}/{key}={value:.4f}")
        else:
            parts.append(f"{prefix}/{key}={value}")

    return " | ".join(parts)

In [101]:
val_metrics = evaluate_lm(
    model=model,
    dataloader=val_loader,
    device=device,
    precision=precision,
    max_batches=50,
    topk=(1, 5),
    ema=ema,
    use_ema=False,
)

print(format_metrics(val_metrics, prefix="val"))

val/loss=5.3535 | val/perplexity=211.3408 | val/token_accuracy=0.0074 | val/top_5_accuracy=0.0665 | val/sequence_accuracy=0.0000 | val/mean_confidence=0.0145 | val/mean_true_token_prob=0.0048 | val/mean_entropy=5.3563 | val/valid_tokens=86117


# Special Metrics for DeepSeekv4 modules

In [106]:
# ============================================================
# DEEPSEEK-V4 MODULE DIAGNOSTIC METRICS
# Loss components / MoE / MTP / mHC
# ============================================================

from __future__ import annotations

import math
from typing import Any, Dict, Iterable, List, Optional, Tuple

import torch
import torch.nn as nn


# ============================================================
# Generic helpers
# ============================================================

def _unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if hasattr(model, "module") else model


def _is_scalar_tensor(x: Any) -> bool:
    return torch.is_tensor(x) and x.numel() == 1


def _to_float(x: Any) -> Optional[float]:
    """
    Convert scalar-like objects to Python float.
    Returns None if conversion is not safe.
    """
    if x is None:
        return None

    if torch.is_tensor(x):
        if x.numel() != 1:
            return None
        value = float(x.detach().float().cpu().item())
        return value if math.isfinite(value) else None

    try:
        value = float(x)
        return value if math.isfinite(value) else None
    except Exception:
        return None


def _tensor_float(x: Any) -> Optional[torch.Tensor]:
    """
    Return detached fp32 tensor on CPU for diagnostics.
    """
    if not torch.is_tensor(x):
        return None

    if x.numel() == 0:
        return None

    return x.detach().float().cpu()


def _get_from_output(outputs: Any, key: str, default: Any = None) -> Any:
    if isinstance(outputs, dict):
        return outputs.get(key, default)
    return getattr(outputs, key, default)


def _iter_nested_dicts(obj: Any):
    """
    Recursively yield all dictionaries inside outputs/aux structures.
    """
    if isinstance(obj, dict):
        yield obj
        for v in obj.values():
            yield from _iter_nested_dicts(v)

    elif isinstance(obj, (list, tuple)):
        for item in obj:
            yield from _iter_nested_dicts(item)


def _collect_values_by_key(obj: Any, key: str) -> List[Any]:
    values = []

    for d in _iter_nested_dicts(obj):
        if key in d:
            values.append(d[key])

    return values


def _collect_values_by_any_key(obj: Any, keys: Iterable[str]) -> List[Any]:
    values = []
    key_set = set(keys)

    for d in _iter_nested_dicts(obj):
        for k in key_set:
            if k in d:
                values.append(d[k])

    return values


def _mean_of_scalar_values(values: List[Any]) -> Optional[float]:
    floats = []

    for v in values:
        fv = _to_float(v)
        if fv is not None:
            floats.append(fv)

    if not floats:
        return None

    return float(sum(floats) / len(floats))


def _cat_flat_tensors(values: List[Any]) -> Optional[torch.Tensor]:
    tensors = []

    for v in values:
        t = _tensor_float(v)
        if t is not None:
            tensors.append(t.reshape(-1))

    if not tensors:
        return None

    return torch.cat(tensors, dim=0)


def _safe_stat_tensor(t: torch.Tensor, prefix: str) -> Dict[str, float]:
    """
    Basic stats for a tensor.
    """
    t = t.detach().float().cpu()

    if t.numel() == 0:
        return {}

    return {
        f"{prefix}_mean": float(t.mean().item()),
        f"{prefix}_std": float(t.std(unbiased=False).item()) if t.numel() > 1 else 0.0,
        f"{prefix}_min": float(t.min().item()),
        f"{prefix}_max": float(t.max().item()),
    }


def _maybe_add_metric(metrics: Dict[str, float], key: str, value: Any) -> None:
    fv = _to_float(value)
    if fv is not None:
        metrics[key] = fv

### Loss components: LM / MTP / MoE aux

In [107]:
# ============================================================
# 1. Loss component metrics
# ============================================================

def compute_deepseek_loss_metrics(
    outputs: Any,
    model: Optional[nn.Module] = None,
    prefix: str = "train",
) -> Dict[str, float]:
    """
    Extract main loss components from DeepSeekV4LM outputs.

    Expected / supported keys:
        loss
        lm_loss
        mtp_loss
        moe_aux_loss
        raw_mtp_loss
        weighted_mtp_loss
        balance_loss
        sequence_balance_loss

    Notes:
        - perplexity should use lm_loss, not total loss.
        - if lm_loss is missing, we do not compute perplexity from total loss.
    """
    metrics: Dict[str, float] = {}

    # Main scalar losses.
    for key in [
        "loss",
        "lm_loss",
        "mtp_loss",
        "moe_aux_loss",
        "raw_mtp_loss",
        "weighted_mtp_loss",
        "balance_loss",
        "sequence_balance_loss",
    ]:
        value = _get_from_output(outputs, key, None)

        # If not top-level, search nested aux dicts.
        if value is None:
            nested = _collect_values_by_key(outputs, key)
            value = _mean_of_scalar_values(nested)

        _maybe_add_metric(metrics, f"{prefix}/{key}", value)

    # Perplexity from lm_loss only.
    lm_loss = metrics.get(f"{prefix}/lm_loss", None)

    if lm_loss is not None and math.isfinite(lm_loss):
        metrics[f"{prefix}/perplexity_from_lm_loss"] = float(math.exp(min(lm_loss, 50.0)))

    # Useful sanity: total - components.
    total = metrics.get(f"{prefix}/loss", None)
    lm = metrics.get(f"{prefix}/lm_loss", 0.0)
    mtp = metrics.get(f"{prefix}/mtp_loss", 0.0)
    moe = metrics.get(f"{prefix}/moe_aux_loss", 0.0)

    if total is not None:
        metrics[f"{prefix}/loss_minus_components"] = float(total - lm - mtp - moe)

    return metrics

## 2. MoE diagnostics: expert usage, router entropy, top-k weights

In [157]:
# ============================================================
# 2. MoE diagnostics
# ============================================================

def _compute_entropy_from_probs(probs: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    probs = probs.float()
    probs = probs / probs.sum(dim=-1, keepdim=True).clamp_min(eps)
    return -(probs * torch.log(probs.clamp_min(eps))).sum(dim=-1)


def compute_moe_diagnostics(
    outputs: Any,
    model: Optional[nn.Module] = None,
    prefix: str = "moe",
    eps: float = 1e-12,
) -> Dict[str, float]:
    """
    Compute MoE diagnostics from returned aux dictionaries.

    Looks for keys:
        expert_fraction
        sequence_expert_fraction
        router_entropy
        router_probs / router_scores
        topk_weights
        topk_indices
        balance_loss
        sequence_balance_loss
        moe_aux_loss

    Returns:
        moe/router_entropy_mean
        moe/expert_fraction_min/max/std
        moe/sequence_expert_fraction_min/max/std
        moe/topk_weights_mean/std/min/max
        moe/n_experts_used_per_batch
        moe/dead_experts
        moe/balance_loss_mean
        moe/sequence_balance_loss_mean
    """
    metrics: Dict[str, float] = {}

    # -----------------------------
    # Expert fraction
    # -----------------------------
    expert_fraction_values = _collect_values_by_key(outputs, "expert_fraction")
    expert_fraction = _cat_flat_tensors(expert_fraction_values)

    if expert_fraction is not None:
        metrics.update(_safe_stat_tensor(expert_fraction, f"{prefix}/expert_fraction"))

        # Counts across all collected MoE layers.
        # Example: n_layers=2 and num_experts=4 => max possible = 8.
        metrics[f"{prefix}/dead_experts_across_layers"] = float(
            (expert_fraction <= 0).sum().item()
        )
        metrics[f"{prefix}/active_experts_across_layers"] = float(
            (expert_fraction > 0).sum().item()
        )

    # -----------------------------
    # Sequence expert fraction
    # -----------------------------
    seq_expert_values = _collect_values_by_key(outputs, "sequence_expert_fraction")

    seq_tensors = []
    for v in seq_expert_values:
        t = _tensor_float(v)
        if t is not None:
            seq_tensors.append(t)

    if seq_tensors:
        seq_flat = torch.cat([t.reshape(-1) for t in seq_tensors], dim=0)
        metrics.update(_safe_stat_tensor(seq_flat, f"{prefix}/sequence_expert_fraction"))

        # If shape is [..., E], count experts used per batch/sequence item.
        used_counts = []
        for t in seq_tensors:
            if t.dim() >= 1:
                used = (t > 0).sum(dim=-1).float()
                used_counts.append(used.reshape(-1))

        if used_counts:
            used_counts = torch.cat(used_counts, dim=0)
            metrics[f"{prefix}/n_experts_used_per_batch_mean"] = float(used_counts.mean().item())
            metrics[f"{prefix}/n_experts_used_per_batch_min"] = float(used_counts.min().item())
            metrics[f"{prefix}/n_experts_used_per_batch_max"] = float(used_counts.max().item())

    # Fallback: if no sequence fraction, use global expert fraction.
    elif expert_fraction is not None:
        metrics[f"{prefix}/n_experts_used_per_batch_mean"] = float((expert_fraction > 0).sum().item())

    # -----------------------------
    # Router entropy
    # -----------------------------
    router_entropy_values = _collect_values_by_key(outputs, "router_entropy")
    router_entropy = _cat_flat_tensors(router_entropy_values)

    if router_entropy is not None:
        metrics[f"{prefix}/router_entropy_mean"] = float(router_entropy.mean().item())
        metrics[f"{prefix}/router_entropy_std"] = (
            float(router_entropy.std(unbiased=False).item())
            if router_entropy.numel() > 1 else 0.0
        )
    else:
        # Fallback: compute from router_probs or router_scores if available.
        prob_values = _collect_values_by_any_key(outputs, ["router_probs", "router_probabilities"])
        score_values = _collect_values_by_key(outputs, "router_scores")

        entropy_chunks = []

        for v in prob_values:
            t = _tensor_float(v)
            if t is not None and t.dim() >= 1:
                entropy_chunks.append(_compute_entropy_from_probs(t, eps=eps).reshape(-1))

        for v in score_values:
            t = _tensor_float(v)
            if t is not None and t.dim() >= 1:
                probs = t / t.sum(dim=-1, keepdim=True).clamp_min(eps)
                entropy_chunks.append(_compute_entropy_from_probs(probs, eps=eps).reshape(-1))

        if entropy_chunks:
            ent = torch.cat(entropy_chunks, dim=0)
            metrics[f"{prefix}/router_entropy_mean"] = float(ent.mean().item())
            metrics[f"{prefix}/router_entropy_std"] = (
                float(ent.std(unbiased=False).item()) if ent.numel() > 1 else 0.0
            )

    # -----------------------------
    # Top-k weights
    # -----------------------------
    topk_weight_values = _collect_values_by_key(outputs, "topk_weights")
    topk_weights = _cat_flat_tensors(topk_weight_values)

    if topk_weights is not None:
        metrics.update(_safe_stat_tensor(topk_weights, f"{prefix}/topk_weights"))

    # Optional top-k indices: useful for number of actually selected experts.
    topk_indices_values = _collect_values_by_key(outputs, "topk_indices")
    topk_indices_flat = []

    for v in topk_indices_values:
        if torch.is_tensor(v) and v.numel() > 0:
            topk_indices_flat.append(v.detach().cpu().reshape(-1))

    if topk_indices_flat:
        idx = torch.cat(topk_indices_flat, dim=0)
        metrics[f"{prefix}/n_unique_selected_experts"] = float(torch.unique(idx).numel())

    # -----------------------------
    # Aux losses
    # -----------------------------
    for key in ["moe_aux_loss", "balance_loss", "sequence_balance_loss"]:
        values = _collect_values_by_key(outputs, key)
        mean_value = _mean_of_scalar_values(values)

        if mean_value is not None:
            metrics[f"{prefix}/{key}_mean"] = mean_value

    return metrics

## 3. MTP diagnostics: raw, weighted, per-depth

In [109]:
# ============================================================
# 3. MTP diagnostics
# ============================================================

def compute_mtp_diagnostics(
    outputs: Any,
    model: Optional[nn.Module] = None,
    prefix: str = "mtp",
) -> Dict[str, float]:
    """
    Compute MTP diagnostics.

    Looks for:
        raw_mtp_loss
        weighted_mtp_loss
        mtp_loss
        mtp_loss_per_depth
        depth_losses
        mtp_depth_losses

    If only mtp_loss exists, logs it as weighted_mtp_loss fallback.
    """
    metrics: Dict[str, float] = {}

    raw_keys = [
        "raw_mtp_loss",
        "mtp_raw_loss",
        "unweighted_mtp_loss",
    ]

    weighted_keys = [
        "weighted_mtp_loss",
        "mtp_weighted_loss",
        "mtp_loss",
    ]

    per_depth_keys = [
        "mtp_loss_per_depth",
        "mtp_losses_per_depth",
        "mtp_depth_losses",
        "depth_losses",
    ]

    raw_values = _collect_values_by_any_key(outputs, raw_keys)
    weighted_values = _collect_values_by_any_key(outputs, weighted_keys)
    per_depth_values = _collect_values_by_any_key(outputs, per_depth_keys)

    raw_mean = _mean_of_scalar_values(raw_values)
    weighted_mean = _mean_of_scalar_values(weighted_values)

    if raw_mean is not None:
        metrics[f"{prefix}/raw_mtp_loss"] = raw_mean

    if weighted_mean is not None:
        metrics[f"{prefix}/weighted_mtp_loss"] = weighted_mean

    # Fallback derivation if raw missing but weighted exists and config has mtp_loss_weight.
    if raw_mean is None and weighted_mean is not None and model is not None:
        raw_model = _unwrap_model(model)
        cfg = getattr(raw_model, "config", None)
        weight = getattr(cfg, "mtp_loss_weight", None) if cfg is not None else None

        if weight is not None and float(weight) > 0:
            metrics[f"{prefix}/raw_mtp_loss_derived"] = float(weighted_mean / float(weight))

    # Loss weight.
    if model is not None:
        raw_model = _unwrap_model(model)
        cfg = getattr(raw_model, "config", None)

        if cfg is not None and hasattr(cfg, "mtp_loss_weight"):
            metrics[f"{prefix}/loss_weight"] = float(cfg.mtp_loss_weight)

        if cfg is not None and hasattr(cfg, "mtp_depth"):
            metrics[f"{prefix}/depth"] = float(cfg.mtp_depth)

    # Per-depth losses.
    per_depth_tensors = []

    for v in per_depth_values:
        t = _tensor_float(v)
        if t is not None:
            per_depth_tensors.append(t.reshape(-1))

    if per_depth_tensors:
        # Stack with possible different sources/layers. We average by depth index when possible.
        max_depth = max(t.numel() for t in per_depth_tensors)

        padded = []
        masks = []

        for t in per_depth_tensors:
            pad_len = max_depth - t.numel()
            if pad_len > 0:
                padded.append(torch.cat([t, torch.zeros(pad_len)], dim=0))
                masks.append(torch.cat([torch.ones_like(t), torch.zeros(pad_len)], dim=0))
            else:
                padded.append(t)
                masks.append(torch.ones_like(t))

        values = torch.stack(padded, dim=0)
        mask = torch.stack(masks, dim=0)

        depth_mean = (values * mask).sum(dim=0) / mask.sum(dim=0).clamp_min(1.0)

        for i, value in enumerate(depth_mean):
            metrics[f"{prefix}/loss_depth_{i + 1}"] = float(value.item())

        metrics[f"{prefix}/loss_per_depth_mean"] = float(depth_mean.mean().item())
        metrics[f"{prefix}/loss_per_depth_std"] = (
            float(depth_mean.std(unbiased=False).item())
            if depth_mean.numel() > 1 else 0.0
        )

    return metrics

## 4. mHC diagnostics: B stochasticity, A/C stats, alphas

In [110]:
# ============================================================
# 4. mHC diagnostics
# ============================================================

def _class_name_contains(module: nn.Module, text: str) -> bool:
    return text.lower() in module.__class__.__name__.lower()


def _collect_mhc_abc_dicts(outputs: Any) -> List[Dict[str, Any]]:
    """
    Collect nested aux dictionaries that contain A, B, C.
    """
    abc_dicts = []

    for d in _iter_nested_dicts(outputs):
        if all(k in d for k in ["A", "B", "C"]):
            abc_dicts.append(d)

    return abc_dicts


def _collect_mhc_alpha_metrics_from_model(
    model: Optional[nn.Module],
    prefix: str = "mhc",
) -> Dict[str, float]:
    """
    Collect alpha parameters from ManifoldHyperConnection modules.

    Supports possible names:
        alpha_A / alpha_pre
        alpha_B / alpha_res
        alpha_C / alpha_post

    Also logs any generic parameter containing 'alpha'.
    """
    metrics: Dict[str, float] = {}

    if model is None:
        return metrics

    raw_model = _unwrap_model(model)

    alpha_A_values = []
    alpha_B_values = []
    alpha_C_values = []
    alpha_all_values = []

    for module_name, module in raw_model.named_modules():
        is_mhc_like = (
            _class_name_contains(module, "ManifoldHyperConnection")
            or (
                hasattr(module, "compute_ABC")
                and hasattr(module, "pre_mix")
                and hasattr(module, "update")
            )
        )

        if not is_mhc_like:
            continue

        for pname, p in module.named_parameters(recurse=False):
            lower = pname.lower()

            if "alpha" not in lower:
                continue

            t = p.detach().float().cpu().reshape(-1)
            alpha_all_values.append(t)

            # Flexible mapping.
            if "alpha_a" in lower or "pre" in lower:
                alpha_A_values.append(t)
            elif "alpha_b" in lower or "res" in lower:
                alpha_B_values.append(t)
            elif "alpha_c" in lower or "post" in lower:
                alpha_C_values.append(t)

    def _add_alpha_stats(name: str, tensors: List[torch.Tensor]):
        if tensors:
            x = torch.cat(tensors, dim=0)
            metrics[f"{prefix}/{name}_mean"] = float(x.mean().item())
            metrics[f"{prefix}/{name}_std"] = (
                float(x.std(unbiased=False).item()) if x.numel() > 1 else 0.0
            )
            metrics[f"{prefix}/{name}_min"] = float(x.min().item())
            metrics[f"{prefix}/{name}_max"] = float(x.max().item())

    _add_alpha_stats("alpha_A", alpha_A_values)
    _add_alpha_stats("alpha_B", alpha_B_values)
    _add_alpha_stats("alpha_C", alpha_C_values)
    _add_alpha_stats("alpha_all", alpha_all_values)

    return metrics


def compute_mhc_diagnostics(
    outputs: Any,
    model: Optional[nn.Module] = None,
    prefix: str = "mhc",
) -> Dict[str, float]:
    """
    Compute mHC diagnostics.

    From aux A/B/C:
        B_row_sum_error
        B_column_sum_error
        B_min/max
        A mean/std
        C mean/std

    From model params:
        alpha_A / alpha_B / alpha_C
    """
    metrics: Dict[str, float] = {}

    abc_dicts = _collect_mhc_abc_dicts(outputs)

    A_values = []
    B_values = []
    C_values = []

    row_errors = []
    col_errors = []
    row_errors_max = []
    col_errors_max = []

    for d in abc_dicts:
        A = _tensor_float(d.get("A", None))
        B = _tensor_float(d.get("B", None))
        C = _tensor_float(d.get("C", None))

        if A is not None:
            A_values.append(A.reshape(-1))

        if C is not None:
            C_values.append(C.reshape(-1))

        if B is not None:
            B_values.append(B.reshape(-1))

            # B shape can be [n_hc, n_hc] or [B,T,n_hc,n_hc].
            if B.dim() >= 2:
                row_sum = B.sum(dim=-1)
                col_sum = B.sum(dim=-2)

                row_err = (row_sum - 1.0).abs()
                col_err = (col_sum - 1.0).abs()

                row_errors.append(row_err.reshape(-1))
                col_errors.append(col_err.reshape(-1))
                row_errors_max.append(row_err.max().reshape(1))
                col_errors_max.append(col_err.max().reshape(1))

    if A_values:
        A_cat = torch.cat(A_values, dim=0)
        metrics.update(_safe_stat_tensor(A_cat, f"{prefix}/A"))

    if C_values:
        C_cat = torch.cat(C_values, dim=0)
        metrics.update(_safe_stat_tensor(C_cat, f"{prefix}/C"))

    if B_values:
        B_cat = torch.cat(B_values, dim=0)
        metrics.update(_safe_stat_tensor(B_cat, f"{prefix}/B"))

    if row_errors:
        row_cat = torch.cat(row_errors, dim=0)
        metrics[f"{prefix}/B_row_sum_error_mean"] = float(row_cat.mean().item())
        metrics[f"{prefix}/B_row_sum_error_max"] = float(torch.cat(row_errors_max).max().item())

    if col_errors:
        col_cat = torch.cat(col_errors, dim=0)
        metrics[f"{prefix}/B_column_sum_error_mean"] = float(col_cat.mean().item())
        metrics[f"{prefix}/B_column_sum_error_max"] = float(torch.cat(col_errors_max).max().item())

    # Add alpha diagnostics from model.
    metrics.update(_collect_mhc_alpha_metrics_from_model(model, prefix=prefix))

    # If no mHC active, make it explicit but non-invasive.
    if not abc_dicts:
        if model is not None:
            raw_model = _unwrap_model(model)
            use_mhc = bool(getattr(getattr(raw_model, "config", None), "use_mhc", False))
            metrics[f"{prefix}/active"] = float(use_mhc)
        else:
            metrics[f"{prefix}/active"] = 0.0
    else:
        metrics[f"{prefix}/active"] = 1.0
        metrics[f"{prefix}/num_abc_aux_dicts"] = float(len(abc_dicts))

    return metrics

# All metrics

In [161]:
# ============================================================
# 5. Combined DeepSeek module metrics
# ============================================================
import math

def compute_deepseek_module_metrics(
    outputs: Any,
    model: Optional[nn.Module] = None,
    include_loss: bool = True,
    include_moe: bool = True,
    include_mtp: bool = True,
    include_mhc: bool = True,
    prefix: str = "train",
) -> Dict[str, float]:
    """
    Combined module-level diagnostics for DeepSeekV4LM training.

    This is intended to be called inside train_step after forward.

    Example:
        outputs = model(..., return_aux=True, need_weights=False)
        metrics = compute_deepseek_module_metrics(outputs, model=model)
    """
    metrics: Dict[str, float] = {}

    if include_loss:
        metrics.update(
            compute_deepseek_loss_metrics(
                outputs=outputs,
                model=model,
                prefix=prefix,
            )
        )

    if include_moe:
        moe_metrics = compute_moe_diagnostics(
            outputs=outputs,
            model=model,
            prefix=f"{prefix}/moe",
        )
        metrics.update(moe_metrics)

    if include_mtp:
        mtp_metrics = compute_mtp_diagnostics(
            outputs=outputs,
            model=model,
            prefix=f"{prefix}/mtp",
        )
        metrics.update(mtp_metrics)

    if include_mhc:
        mhc_metrics = compute_mhc_diagnostics(
            outputs=outputs,
            model=model,
            prefix=f"{prefix}/mhc",
        )
        metrics.update(mhc_metrics)

    return metrics

# ============================================================
# Pretty formatter for DeepSeek module metrics
# ============================================================

def _fmt_metric_value(value, precision: int = 4) -> str:
    if value is None:
        return "n/a"

    if isinstance(value, bool):
        return str(value)

    try:
        value = float(value)
    except Exception:
        return str(value)

    if not math.isfinite(value):
        return str(value)

    abs_v = abs(value)

    if abs_v == 0:
        return "0"

    if abs_v >= 1e4 or abs_v < 1e-3:
        return f"{value:.{precision}e}"

    return f"{value:.{precision}f}"


def _print_metric_block(
    title: str,
    metrics: dict,
    rows: list[tuple[str, str]],
    precision: int = 4,
    indent: int = 2,
) -> None:
    """
    rows:
        [
            (metric_key, reference_note),
            ...
        ]
    """
    available = [(k, note) for k, note in rows if k in metrics]

    if not available:
        return

    pad = " " * indent
    print(f"\n{title}")
    print("-" * len(title))

    names = [k.split("/")[-1] for k, _ in available]
    max_name_len = max(len(name) for name in names)
    max_value_len = max(len(_fmt_metric_value(metrics[k], precision)) for k, _ in available)

    for key, note in available:
        short_name = key.split("/")[-1]
        value_str = _fmt_metric_value(metrics[key], precision=precision)

        if note:
            print(f"{pad}{short_name:<{max_name_len}} : {value_str:<{max_value_len}}   # {note}")
        else:
            print(f"{pad}{short_name:<{max_name_len}} : {value_str}")


def _router_entropy_reference(num_experts: int | None) -> str:
    if num_experts is None or num_experts <= 1:
        return "higher = more uniform routing"
    return f"max ≈ log(E)={math.log(num_experts):.3f}; very low => routing collapse"


def _expert_fraction_reference(num_experts: int | None) -> str:
    if num_experts is None or num_experts <= 0:
        return "watch min/max; extreme imbalance is bad"
    return f"ideal mean ≈ 1/E={1.0 / num_experts:.3f}; min near 0 => dead expert"


def _active_experts_reference(num_experts: int | None, n_layers: int | None) -> str:
    if num_experts is None or n_layers is None:
        return "should stay high; falling means dead experts"
    return f"max = n_layers*E = {n_layers * num_experts}; lower => unused experts"


def print_deepseek_module_metrics(
    metrics: dict,
    prefix: str = "train",
    precision: int = 4,
    title: str | None = None,
    num_experts: int | None = None,
    top_k_experts: int | None = None,
    n_layers: int | None = None,
) -> None:
    """
    Pretty block-style printer for DeepSeek-V4 module diagnostics.

    Ordered by importance for training monitoring.

    Example:
        print_deepseek_module_metrics(
            metrics,
            prefix="train",
            title="DeepSeek-V4 module diagnostics",
            num_experts=model.config.num_experts,
            top_k_experts=model.config.top_k_experts,
            n_layers=model.config.n_layers,
        )
    """

    if title is not None:
        print("\n" + "=" * 96)
        print(title)
        print("=" * 96)

    # ========================================================
    # 1. Core convergence metrics
    # ========================================================

    loss_rows = [
        (
            f"{prefix}/loss",
            "lower is better; total optimized objective",
        ),
        (
            f"{prefix}/lm_loss",
            "lower is better; main next-token CE",
        ),
        (
            f"{prefix}/perplexity_from_lm_loss",
            "lower is better; exp(lm_loss)",
        ),
        (
            f"{prefix}/mtp_loss",
            "auxiliary; should not dominate total loss",
        ),
        (
            f"{prefix}/moe_aux_loss",
            "should be small; high values imply routing imbalance penalty",
        ),
        (
            f"{prefix}/loss_minus_components",
            "should be ≈ 0; nonzero means loss accounting bug",
        ),
        (
            f"{prefix}/raw_mtp_loss",
            "unweighted MTP CE; compare with lm_loss",
        ),
        (
            f"{prefix}/weighted_mtp_loss",
            "raw_mtp_loss * mtp_loss_weight",
        ),
    ]

    # ========================================================
    # 2. MoE health: most important after loss
    # ========================================================

    moe_rows_critical = [
        (
            f"{prefix}/moe/router_entropy_mean",
            _router_entropy_reference(num_experts),
        ),
        (
            f"{prefix}/moe/expert_fraction_min",
            "too close to 0 => dead/underused expert",
        ),
        (
            f"{prefix}/moe/expert_fraction_max",
            "too high => expert collapse / overload",
        ),
        (
            f"{prefix}/moe/expert_fraction_mean",
            _expert_fraction_reference(num_experts),
        ),
        (
            f"{prefix}/moe/dead_experts_across_layers",
            "should be 0 after warmup",
        ),
        (
            f"{prefix}/moe/active_experts_across_layers",
            _active_experts_reference(num_experts, n_layers),
        ),
        (
            f"{prefix}/moe/n_unique_selected_experts",
            "should approach E within a batch",
        ),
        (
            f"{prefix}/moe/n_experts_used_per_batch_mean",
            "higher is better; ideally all experts used per layer/batch",
        ),
        (
            f"{prefix}/moe/topk_weights_mean",
            "with normalized top-k, ideal ≈ 1/top_k",
        ),
        (
            f"{prefix}/moe/topk_weights_std",
            "very high => routing confidence collapse to few experts",
        ),
    ]

    moe_rows_secondary = [
        (
            f"{prefix}/moe/router_entropy_std",
            "large std => unstable routing across tokens/layers",
        ),
        (
            f"{prefix}/moe/expert_fraction_std",
            "lower is more balanced; rising can signal specialization or collapse",
        ),
        (
            f"{prefix}/moe/sequence_expert_fraction_min",
            "per-sequence min; near 0 means sequence-level imbalance",
        ),
        (
            f"{prefix}/moe/sequence_expert_fraction_max",
            "per-sequence max; high means sequence overuses one expert",
        ),
        (
            f"{prefix}/moe/sequence_expert_fraction_mean",
            "should be near 1/E",
        ),
        (
            f"{prefix}/moe/sequence_expert_fraction_std",
            "sequence-level load dispersion",
        ),
        (
            f"{prefix}/moe/topk_weights_min",
            "",
        ),
        (
            f"{prefix}/moe/topk_weights_max",
            "",
        ),
        (
            f"{prefix}/moe/n_experts_used_per_batch_min",
            "",
        ),
        (
            f"{prefix}/moe/n_experts_used_per_batch_max",
            "",
        ),
        (
            f"{prefix}/moe/moe_aux_loss_mean",
            "small is expected",
        ),
        (
            f"{prefix}/moe/balance_loss_mean",
            "small is expected",
        ),
        (
            f"{prefix}/moe/sequence_balance_loss_mean",
            "small is expected",
        ),
    ]

    # ========================================================
    # 3. MTP health
    # ========================================================

    mtp_rows = [
        (
            f"{prefix}/mtp/weighted_mtp_loss",
            "contributes to total loss; should not dominate lm_loss",
        ),
        (
            f"{prefix}/mtp/raw_mtp_loss",
            "compare to lm_loss; should fall with training",
        ),
        (
            f"{prefix}/mtp/loss_weight",
            "weighted_mtp_loss = raw * this weight",
        ),
        (
            f"{prefix}/mtp/depth",
            "number of future-token objectives",
        ),
        (
            f"{prefix}/mtp/loss_depth_1",
            "near-future target; should improve fastest",
        ),
        (
            f"{prefix}/mtp/loss_depth_2",
            "farther target; usually harder",
        ),
        (
            f"{prefix}/mtp/loss_depth_3",
            "",
        ),
        (
            f"{prefix}/mtp/loss_depth_4",
            "",
        ),
        (
            f"{prefix}/mtp/loss_per_depth_mean",
            "",
        ),
        (
            f"{prefix}/mtp/loss_per_depth_std",
            "large std => uneven MTP depth learning",
        ),
        (
            f"{prefix}/mtp/raw_mtp_loss_derived",
            "",
        ),
    ]

    # ========================================================
    # 4. mHC numerical stability
    # ========================================================

    mhc_rows_critical = [
        (
            f"{prefix}/mhc/active",
            "1 if mHC active",
        ),
        (
            f"{prefix}/mhc/B_row_sum_error_mean",
            "should be near 0; high => Sinkhorn/manifold failure",
        ),
        (
            f"{prefix}/mhc/B_column_sum_error_mean",
            "should be near 0; high => not doubly stochastic",
        ),
        (
            f"{prefix}/mhc/B_row_sum_error_max",
            "large spikes can signal numerical instability",
        ),
        (
            f"{prefix}/mhc/B_column_sum_error_max",
            "large spikes can signal numerical instability",
        ),
        (
            f"{prefix}/mhc/alpha_A_mean",
            "small initially; rapid growth may destabilize pre-mixing",
        ),
        (
            f"{prefix}/mhc/alpha_B_mean",
            "small initially; rapid growth may destabilize residual mixing",
        ),
        (
            f"{prefix}/mhc/alpha_C_mean",
            "small initially; rapid growth may destabilize post-mixing",
        ),
    ]

    mhc_rows_secondary = [
        (
            f"{prefix}/mhc/num_abc_aux_dicts",
            "",
        ),
        (
            f"{prefix}/mhc/A_mean",
            "A in [0,1] after sigmoid; monitors pre-mixing scale",
        ),
        (
            f"{prefix}/mhc/A_std",
            "",
        ),
        (
            f"{prefix}/mhc/A_min",
            "",
        ),
        (
            f"{prefix}/mhc/A_max",
            "",
        ),
        (
            f"{prefix}/mhc/B_mean",
            "for n_hc=4, expected mean ≈ 0.25",
        ),
        (
            f"{prefix}/mhc/B_std",
            "",
        ),
        (
            f"{prefix}/mhc/B_min",
            "",
        ),
        (
            f"{prefix}/mhc/B_max",
            "",
        ),
        (
            f"{prefix}/mhc/C_mean",
            "C constrained; monitors post-block injection scale",
        ),
        (
            f"{prefix}/mhc/C_std",
            "",
        ),
        (
            f"{prefix}/mhc/C_min",
            "",
        ),
        (
            f"{prefix}/mhc/C_max",
            "",
        ),
        (
            f"{prefix}/mhc/alpha_A_std",
            "",
        ),
        (
            f"{prefix}/mhc/alpha_B_std",
            "",
        ),
        (
            f"{prefix}/mhc/alpha_C_std",
            "",
        ),
        (
            f"{prefix}/mhc/alpha_all_mean",
            "",
        ),
        (
            f"{prefix}/mhc/alpha_all_std",
            "",
        ),
    ]

    _print_metric_block(
        title="1. Loss / convergence metrics",
        metrics=metrics,
        rows=loss_rows,
        precision=precision,
    )

    _print_metric_block(
        title="2. MoE routing health — critical",
        metrics=metrics,
        rows=moe_rows_critical,
        precision=precision,
    )

    _print_metric_block(
        title="3. MTP auxiliary objective",
        metrics=metrics,
        rows=mtp_rows,
        precision=precision,
    )

    _print_metric_block(
        title="4. mHC numerical stability — critical",
        metrics=metrics,
        rows=mhc_rows_critical,
        precision=precision,
    )

    _print_metric_block(
        title="5. MoE detailed diagnostics",
        metrics=metrics,
        rows=moe_rows_secondary,
        precision=precision,
    )

    _print_metric_block(
        title="6. mHC detailed diagnostics",
        metrics=metrics,
        rows=mhc_rows_secondary,
        precision=precision,
    )

    if title is not None:
        print("\n" + "=" * 96)

In [162]:
outputs = model(
    input_ids=batch["input_ids"],
    labels=batch.get("labels", None),
    mtp_labels=batch.get("mtp_labels", None),
    attention_mask=batch.get("attention_mask", None),
    position_ids=batch.get("position_ids", None),
    return_aux=True,
    need_weights=False,
)

loss = outputs["loss"]

deepseek_metrics = compute_deepseek_module_metrics(
    outputs=outputs,
    model=model,
    prefix="train",
)

print(deepseek_metrics)

{'train/loss': 6.985154151916504, 'train/lm_loss': 5.372533321380615, 'train/mtp_loss': 1.612534999847412, 'train/moe_aux_loss': 8.572459046263248e-05, 'train/raw_mtp_loss': 5.375116348266602, 'train/weighted_mtp_loss': 1.612534999847412, 'train/balance_loss': 8.863806669978658e-06, 'train/sequence_balance_loss': 3.399848901608493e-05, 'train/perplexity_from_lm_loss': 215.40787444813452, 'train/loss_minus_components': 1.0609801393002272e-07, 'train/moe/expert_fraction_mean': 0.25, 'train/moe/expert_fraction_std': 0.029772145673632622, 'train/moe/expert_fraction_min': 0.195068359375, 'train/moe/expert_fraction_max': 0.294189453125, 'train/moe/dead_experts_across_layers': 0.0, 'train/moe/active_experts_across_layers': 8.0, 'train/moe/sequence_expert_fraction_mean': 0.25, 'train/moe/sequence_expert_fraction_std': 0.05830822512507439, 'train/moe/sequence_expert_fraction_min': 0.0859375, 'train/moe/sequence_expert_fraction_max': 0.3828125, 'train/moe/n_experts_used_per_batch_mean': 4.0, 'tr

In [163]:
print_deepseek_module_metrics(
    deepseek_metrics,
    prefix="train",
    title="DeepSeek-V4 module diagnostics",
)


DeepSeek-V4 module diagnostics

1. Loss / convergence metrics
-----------------------------
  loss                    : 6.9852       # lower is better; total optimized objective
  lm_loss                 : 5.3725       # lower is better; main next-token CE
  perplexity_from_lm_loss : 215.4079     # lower is better; exp(lm_loss)
  mtp_loss                : 1.6125       # auxiliary; should not dominate total loss
  moe_aux_loss            : 8.5725e-05   # should be small; high values imply routing imbalance penalty
  loss_minus_components   : 1.0610e-07   # should be ≈ 0; nonzero means loss accounting bug
  raw_mtp_loss            : 5.3751       # unweighted MTP CE; compare with lm_loss
  weighted_mtp_loss       : 1.6125       # raw_mtp_loss * mtp_loss_weight

2. MoE routing health — critical
--------------------------------
  router_entropy_mean           : 1.3850   # higher = more uniform routing
  expert_fraction_min           : 0.1951   # too close to 0 => dead/underused expert
  ex

---

# MUON + HYBRID MUON-ADAMW OPTIMIZER

In [102]:
# ============================================================
# MUON + HYBRID MUON-ADAMW OPTIMIZER
# DeepSeek-V4 Mini Training Stack
# ============================================================

from __future__ import annotations

import math
from typing import Any, Dict, Iterable, List, Optional, Tuple, Union

import torch
import torch.nn as nn


# ============================================================
# Newton-Schulz orthogonalization
# ============================================================

@torch.no_grad()
def zeropower_via_newtonschulz5(
    G: torch.Tensor,
    steps: int = 5,
    eps: float = 1e-7,
) -> torch.Tensor:
    """
    Approximate zeroth power / orthogonalized update using Newton-Schulz iterations.

    Args:
        G:
            2D gradient/update matrix.
        steps:
            Number of Newton-Schulz iterations.
        eps:
            Numerical stability epsilon.

    Returns:
        Tensor with same shape/device/dtype as G.
    """
    if G.dim() != 2:
        raise ValueError(f"zeropower_via_newtonschulz5 expects a 2D tensor, got shape={tuple(G.shape)}.")

    if steps <= 0:
        raise ValueError(f"steps must be > 0, got {steps}.")

    if eps <= 0:
        raise ValueError(f"eps must be > 0, got {eps}.")

    original_dtype = G.dtype
    original_device = G.device

    X = G.detach().float()

    if not torch.isfinite(X).all():
        raise FloatingPointError("Input to zeropower_via_newtonschulz5 contains NaN or Inf.")

    norm = X.norm()

    if norm <= eps:
        return torch.zeros_like(G)

    # For rectangular matrices, operate on the cheaper orientation.
    transposed = False
    if X.shape[0] > X.shape[1]:
        X = X.T
        transposed = True

    X = X / (X.norm() + eps)

    # Quintic coefficients commonly used in modern Muon-style implementations.
    a = 3.4445
    b = -4.7750
    c = 2.0315

    for _ in range(steps):
        A = X @ X.T
        B = A @ A
        X = a * X + b * (A @ X) + c * (B @ X)

    if transposed:
        X = X.T

    return X.to(device=original_device, dtype=original_dtype)


# ============================================================
# Muon optimizer
# ============================================================

class Muon(torch.optim.Optimizer):
    """
    Muon optimizer for 2D matrix parameters.

    Intended use:
        - hidden Linear weights
        - attention projections
        - MLP/MoE projection matrices
        - selected internal 2D transformations

    Not intended for:
        - embeddings
        - LM heads
        - norms
        - biases
        - scalar/vector parameters
        - mHC static/gating parameters
    """

    def __init__(
        self,
        params,
        lr: float = 1e-3,
        momentum: float = 0.95,
        weight_decay: float = 0.0,
        nesterov: bool = True,
        ns_steps: int = 5,
        eps: float = 1e-7,
    ):
        if lr <= 0:
            raise ValueError(f"lr must be > 0, got {lr}.")

        if not (0.0 <= momentum < 1.0):
            raise ValueError(f"momentum must satisfy 0 <= momentum < 1, got {momentum}.")

        if weight_decay < 0:
            raise ValueError(f"weight_decay must be >= 0, got {weight_decay}.")

        if ns_steps <= 0:
            raise ValueError(f"ns_steps must be > 0, got {ns_steps}.")

        if eps <= 0:
            raise ValueError(f"eps must be > 0, got {eps}.")

        defaults = dict(
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay,
            nesterov=nesterov,
            ns_steps=ns_steps,
            eps=eps,
        )

        super().__init__(params, defaults)

        # Validate all parameters upfront.
        for group in self.param_groups:
            for p in group["params"]:
                if p.ndim != 2:
                    raise ValueError(
                        "Muon only supports 2D parameters. "
                        f"Got parameter with shape={tuple(p.shape)}."
                    )

    @torch.no_grad()
    def step(self, closure=None):
        """
        Perform one Muon optimization step.
        """
        loss = None

        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group["lr"]
            momentum = group["momentum"]
            weight_decay = group["weight_decay"]
            nesterov = group["nesterov"]
            ns_steps = group["ns_steps"]
            eps = group["eps"]

            for p in group["params"]:
                if p.grad is None:
                    continue

                if p.ndim != 2:
                    raise ValueError(
                        "Muon only supports 2D parameters during step. "
                        f"Got shape={tuple(p.shape)}."
                    )

                grad = p.grad.detach()

                if not torch.isfinite(grad).all():
                    raise FloatingPointError(
                        f"Non-finite gradient found in Muon parameter with shape={tuple(p.shape)}."
                    )

                state = self.state[p]

                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(grad)

                buf = state["momentum_buffer"]

                # Momentum buffer tracks raw gradient/update direction.
                buf.mul_(momentum).add_(grad)

                if nesterov:
                    update = grad.add(buf, alpha=momentum)
                else:
                    update = buf

                update = zeropower_via_newtonschulz5(
                    update,
                    steps=ns_steps,
                    eps=eps,
                )

                if weight_decay != 0.0:
                    # Decoupled weight decay.
                    p.data.mul_(1.0 - lr * weight_decay)

                p.data.add_(update, alpha=-lr)

        return loss


# ============================================================
# Hybrid optimizer wrapper
# ============================================================

class HybridMuonAdamW:
    """
    Thin wrapper combining:
        - Muon for selected 2D hidden matrices
        - AdamW for all remaining parameters

    It exposes:
        - zero_grad
        - step
        - state_dict/load_state_dict
        - set_lr
        - param_groups

    Compatible with our checkpointing code.
    """

    def __init__(
        self,
        muon_optimizer: Muon,
        adamw_optimizer: torch.optim.AdamW,
        metadata: Optional[Dict[str, Any]] = None,
    ):
        self.muon = muon_optimizer
        self.adamw = adamw_optimizer
        self.metadata = metadata or {}

    @property
    def param_groups(self):
        # Returned list is for inspection/logging.
        # For changing lr, use set_lr().
        return self.muon.param_groups + self.adamw.param_groups

    @property
    def defaults(self):
        return {
            "muon": getattr(self.muon, "defaults", {}),
            "adamw": getattr(self.adamw, "defaults", {}),
        }

    def zero_grad(self, set_to_none: bool = True):
        self.muon.zero_grad(set_to_none=set_to_none)
        self.adamw.zero_grad(set_to_none=set_to_none)

    def step(self, closure=None):
        loss = None

        if closure is not None:
            loss = closure()

        self.muon.step()
        self.adamw.step()

        return loss

    def set_lr(self, lr: float, muon_lr: Optional[float] = None):
        """
        Scheduler-compatible LR setter.

        Args:
            lr:
                AdamW LR.
            muon_lr:
                Optional Muon LR. If None, uses lr.
        """
        if lr <= 0:
            raise ValueError(f"lr must be > 0, got {lr}.")

        if muon_lr is not None and muon_lr <= 0:
            raise ValueError(f"muon_lr must be > 0, got {muon_lr}.")

        for group in self.adamw.param_groups:
            group["lr"] = lr

        effective_muon_lr = lr if muon_lr is None else muon_lr

        for group in self.muon.param_groups:
            group["lr"] = effective_muon_lr

    def state_dict(self):
        return {
            "muon": self.muon.state_dict(),
            "adamw": self.adamw.state_dict(),
            "metadata": self.metadata,
        }

    def load_state_dict(self, state_dict):
        self.muon.load_state_dict(state_dict["muon"])
        self.adamw.load_state_dict(state_dict["adamw"])
        self.metadata = state_dict.get("metadata", self.metadata)


# ============================================================
# Parameter grouping helpers
# ============================================================

def _get_module_by_parameter_name(model: nn.Module) -> Dict[str, nn.Module]:
    """
    Build mapping from full parameter name to the owning module.
    """
    param_to_module: Dict[str, nn.Module] = {}

    for module_name, module in model.named_modules():
        for param_name, _ in module.named_parameters(recurse=False):
            full_name = f"{module_name}.{param_name}" if module_name else param_name
            param_to_module[full_name] = module

    return param_to_module


def _is_embedding_parameter(name: str, module: Optional[nn.Module] = None) -> bool:
    lower = name.lower()

    if "embedding" in lower or "embed" in lower or "wte" in lower:
        return True

    if module is not None and isinstance(module, nn.Embedding):
        return True

    return False


def _is_norm_parameter(name: str, module: Optional[nn.Module] = None) -> bool:
    lower = name.lower()

    if "norm" in lower:
        return True

    if module is not None:
        norm_classes = (
            nn.LayerNorm,
            nn.BatchNorm1d,
            nn.BatchNorm2d,
            nn.BatchNorm3d,
            nn.GroupNorm,
            nn.InstanceNorm1d,
            nn.InstanceNorm2d,
            nn.InstanceNorm3d,
            nn.LocalResponseNorm,
        )

        if isinstance(module, norm_classes):
            return True

    return False


def should_use_muon(
    name: str,
    param: nn.Parameter,
    module: Optional[nn.Module] = None,
) -> bool:
    """
    Conservative rule for assigning a parameter to Muon.

    Muon gets only:
        - trainable
        - 2D
        - .weight
        - not excluded by known fragile/special modules
    """
    if not param.requires_grad:
        return False

    if param.ndim != 2:
        return False

    if not name.endswith(".weight"):
        return False

    lower = name.lower()

    exclude_keywords = [
        "embedding",
        "embed",
        "token_embedding",
        "lm_head",
        "prediction_head",

        # MTP vocab heads should stay AdamW.
        "mtp_head.heads",

        # Norms and small controls.
        "norm",
        "bias",
        "static_",
        "alpha_",
        "bias_a",
        "bias_b",
        "temperature",
        "scale",

        # mHC / hyper-connection delicate params.
        "mhc",
        "hyper",
        "sinkhorn",
        "readout",
        "mhc_readout",

        # Explicit loss/gating weights.
        "depth_loss_weights",
    ]

    if any(k in lower for k in exclude_keywords):
        return False

    if _is_embedding_parameter(name, module):
        return False

    if _is_norm_parameter(name, module):
        return False

    return True


def should_use_adamw_no_decay(
    name: str,
    param: nn.Parameter,
    module: Optional[nn.Module] = None,
) -> bool:
    """
    Conservative AdamW no-decay rule.
    """
    lower = name.lower()

    if not param.requires_grad:
        return False

    if param.ndim < 2:
        return True

    if name.endswith(".bias"):
        return True

    if _is_norm_parameter(name, module):
        return True

    if _is_embedding_parameter(name, module):
        return True

    no_decay_keywords = [
        "lm_head",
        "prediction_head",
        "mtp_head.heads",
        "bias_a",
        "bias_b",
        "alpha_",
        "static_",
        "temperature",
        "scale",
        "depth_loss_weights",
        "sinkhorn",
        "readout",
        "mhc",
        "hyper",
    ]

    if any(k in lower for k in no_decay_keywords):
        return True

    return False


def build_muon_adamw_parameter_groups(
    model: nn.Module,
    adamw_weight_decay: float = 0.1,
    muon_weight_decay: float = 0.0,
    verbose: bool = False,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], Dict[str, Any]]:
    """
    Build parameter groups for HybridMuonAdamW.

    Returns:
        muon_groups:
            list of groups for Muon.
        adamw_groups:
            list of groups for AdamW.
        metadata:
            diagnostics and parameter names.
    """
    if adamw_weight_decay < 0:
        raise ValueError(f"adamw_weight_decay must be >= 0, got {adamw_weight_decay}.")

    if muon_weight_decay < 0:
        raise ValueError(f"muon_weight_decay must be >= 0, got {muon_weight_decay}.")

    param_to_module = _get_module_by_parameter_name(model)

    muon_params = []
    adamw_decay_params = []
    adamw_no_decay_params = []

    muon_names = []
    adamw_decay_names = []
    adamw_no_decay_names = []

    seen_param_ids = set()

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        module = param_to_module.get(name, None)

        if should_use_muon(name, param, module):
            target_list = muon_params
            target_names = muon_names
        else:
            if should_use_adamw_no_decay(name, param, module):
                target_list = adamw_no_decay_params
                target_names = adamw_no_decay_names
            else:
                target_list = adamw_decay_params
                target_names = adamw_decay_names

        pid = id(param)

        if pid in seen_param_ids:
            raise RuntimeError(f"Duplicate parameter assignment detected for parameter: {name}")

        seen_param_ids.add(pid)

        target_list.append(param)
        target_names.append(name)

    # Validate no trainable params missing.
    trainable_params = [
        (name, p)
        for name, p in model.named_parameters()
        if p.requires_grad
    ]

    trainable_ids = {id(p) for _, p in trainable_params}
    assigned_ids = {id(p) for p in muon_params + adamw_decay_params + adamw_no_decay_params}

    missing = [
        name
        for name, p in trainable_params
        if id(p) not in assigned_ids
    ]

    extra = assigned_ids - trainable_ids

    if missing:
        raise RuntimeError(f"Some trainable parameters were not assigned to any optimizer group: {missing[:20]}")

    if extra:
        raise RuntimeError("Some assigned parameters are not trainable model parameters.")

    # Muon must only receive 2D tensors.
    bad_muon = [
        name
        for name, p in zip(muon_names, muon_params)
        if p.ndim != 2
    ]

    if bad_muon:
        raise RuntimeError(f"Non-2D parameters assigned to Muon: {bad_muon[:20]}")

    muon_groups = [
        {
            "params": muon_params,
            "weight_decay": muon_weight_decay,
            "group_name": "muon",
        }
    ]

    adamw_groups = [
        {
            "params": adamw_decay_params,
            "weight_decay": adamw_weight_decay,
            "group_name": "adamw_decay",
        },
        {
            "params": adamw_no_decay_params,
            "weight_decay": 0.0,
            "group_name": "adamw_no_decay",
        },
    ]

    metadata = {
        "num_muon_tensors": len(muon_params),
        "num_adamw_decay_tensors": len(adamw_decay_params),
        "num_adamw_no_decay_tensors": len(adamw_no_decay_params),
        "num_adamw_tensors": len(adamw_decay_params) + len(adamw_no_decay_params),

        "num_muon_params": int(sum(p.numel() for p in muon_params)),
        "num_adamw_decay_params": int(sum(p.numel() for p in adamw_decay_params)),
        "num_adamw_no_decay_params": int(sum(p.numel() for p in adamw_no_decay_params)),
        "num_adamw_params": int(
            sum(p.numel() for p in adamw_decay_params)
            + sum(p.numel() for p in adamw_no_decay_params)
        ),

        "muon_names": muon_names,
        "adamw_decay_names": adamw_decay_names,
        "adamw_no_decay_names": adamw_no_decay_names,
    }

    metadata["total_trainable_params"] = (
        metadata["num_muon_params"]
        + metadata["num_adamw_params"]
    )

    if verbose:
        print("=" * 80)
        print("Hybrid Muon + AdamW parameter groups")
        print("=" * 80)
        print(f"Muon tensors          : {metadata['num_muon_tensors']}")
        print(f"AdamW decay tensors   : {metadata['num_adamw_decay_tensors']}")
        print(f"AdamW no-decay tensors: {metadata['num_adamw_no_decay_tensors']}")
        print("-" * 80)
        print(f"Muon params           : {metadata['num_muon_params']:,}")
        print(f"AdamW params          : {metadata['num_adamw_params']:,}")
        print(f"Total trainable params: {metadata['total_trainable_params']:,}")
        print("=" * 80)

    return muon_groups, adamw_groups, metadata


# ============================================================
# Optimizer builder
# ============================================================

def build_muon_adamw_optimizer(
    model: nn.Module,
    learning_rate: float = 3e-4,
    weight_decay: float = 0.1,
    betas: Tuple[float, float] = (0.9, 0.95),
    eps: float = 1e-8,
    muon_lr: Optional[float] = None,
    muon_momentum: float = 0.95,
    muon_nesterov: bool = True,
    muon_ns_steps: int = 5,
    muon_eps: float = 1e-7,
    muon_weight_decay: float = 0.0,
    verbose: bool = False,
) -> Tuple[HybridMuonAdamW, Dict[str, Any]]:
    """
    Build hybrid Muon + AdamW optimizer.

    Usage:
        optimizer, opt_info = build_muon_adamw_optimizer(
            model=model,
            learning_rate=3e-4,
            weight_decay=0.1,
            betas=(0.9, 0.95),
            eps=1e-8,
            muon_lr=None,
            muon_momentum=0.95,
            muon_nesterov=True,
            muon_ns_steps=5,
            muon_weight_decay=0.0,
            verbose=True,
        )
    """
    if learning_rate <= 0:
        raise ValueError(f"learning_rate must be > 0, got {learning_rate}.")

    if muon_lr is not None and muon_lr <= 0:
        raise ValueError(f"muon_lr must be > 0 if provided, got {muon_lr}.")

    if weight_decay < 0:
        raise ValueError(f"weight_decay must be >= 0, got {weight_decay}.")

    if len(betas) != 2:
        raise ValueError(f"betas must have length 2, got {betas}.")

    effective_muon_lr = learning_rate if muon_lr is None else muon_lr

    muon_groups, adamw_groups, metadata = build_muon_adamw_parameter_groups(
        model=model,
        adamw_weight_decay=weight_decay,
        muon_weight_decay=muon_weight_decay,
        verbose=verbose,
    )

    # If no Muon params exist, this is likely a config/detection issue.
    if metadata["num_muon_tensors"] == 0:
        raise RuntimeError(
            "No parameters were assigned to Muon. "
            "Check should_use_muon exclusions or model architecture."
        )

    muon_optimizer = Muon(
        muon_groups,
        lr=effective_muon_lr,
        momentum=muon_momentum,
        weight_decay=muon_weight_decay,
        nesterov=muon_nesterov,
        ns_steps=muon_ns_steps,
        eps=muon_eps,
    )

    adamw_optimizer = torch.optim.AdamW(
        adamw_groups,
        lr=learning_rate,
        betas=betas,
        eps=eps,
    )

    metadata["optimizer_type"] = "muon_adamw"
    metadata["learning_rate"] = learning_rate
    metadata["muon_lr"] = effective_muon_lr
    metadata["adamw_lr"] = learning_rate
    metadata["weight_decay"] = weight_decay
    metadata["muon_weight_decay"] = muon_weight_decay
    metadata["muon_momentum"] = muon_momentum
    metadata["muon_nesterov"] = muon_nesterov
    metadata["muon_ns_steps"] = muon_ns_steps
    metadata["muon_eps"] = muon_eps

    optimizer = HybridMuonAdamW(
        muon_optimizer=muon_optimizer,
        adamw_optimizer=adamw_optimizer,
        metadata=metadata,
    )

    return optimizer, metadata

In [103]:
optimizer1, opt_info1 = build_muon_adamw_optimizer(
    model=model,
    learning_rate=3e-4,
    weight_decay=0.1,
    betas=(0.9, 0.95),
    eps=1e-8,
    muon_lr=None,              # None => usa learning_rate
    muon_momentum=0.95,
    muon_nesterov=True,
    muon_ns_steps=5,
    muon_eps=1e-7,
    muon_weight_decay=0.0,     # recomendado inicial
    verbose=True,
)

Hybrid Muon + AdamW parameter groups
Muon tensors          : 66
AdamW decay tensors   : 4
AdamW no-decay tensors: 18
--------------------------------------------------------------------------------
Muon params           : 285,952
AdamW params          : 42,368
Total trainable params: 328,320


In [104]:
def audit_muon_assignment(model, opt_info):
    muon = set(opt_info["muon_names"])
    adamw_decay = set(opt_info["adamw_decay_names"])
    adamw_no_decay = set(opt_info["adamw_no_decay_names"])
    adamw = adamw_decay | adamw_no_decay

    # Detect tied lm_head.
    lm_head_tied = False
    try:
        emb_weight = model.embedding.token_embedding.weight
        lm_weight = model.lm_head.weight
        lm_head_tied = emb_weight.data_ptr() == lm_weight.data_ptr()
    except Exception:
        pass

    embedding_in_adamw = any(
        "embedding" in n or "token_embedding" in n
        for n in adamw
    )

    lm_head_in_adamw_by_name = any("lm_head" in n for n in adamw)
    lm_head_in_muon_by_name = any("lm_head" in n for n in muon)

    # If tied, it is okay that lm_head does not appear by name,
    # as long as the shared embedding parameter is in AdamW.
    lm_head_effectively_in_adamw = (
        lm_head_in_adamw_by_name
        or (lm_head_tied and embedding_in_adamw)
    )

    checks = {
        "embedding_in_adamw": embedding_in_adamw,
        "lm_head_tied_to_embedding": lm_head_tied,
        "lm_head_in_adamw_by_name": lm_head_in_adamw_by_name,
        "lm_head_effectively_in_adamw": lm_head_effectively_in_adamw,
        "lm_head_not_in_muon": not lm_head_in_muon_by_name,
        "mtp_vocab_heads_in_adamw": any("mtp_head.heads" in n for n in adamw),
        "norms_in_adamw": any("norm" in n.lower() for n in adamw),
        "attention_linears_in_muon": any(".attention." in n and n.endswith(".weight") for n in muon),
        "moe_experts_in_muon": any(".ffn.experts." in n and n.endswith(".weight") for n in muon),
        "shared_experts_in_muon": any(".ffn.shared_experts." in n and n.endswith(".weight") for n in muon),
        "router_in_muon": any(".ffn.router.weight" in n for n in muon),
        "mtp_transforms_in_muon": any("mtp_head.transforms" in n and n.endswith(".weight") for n in muon),
    }

    for k, v in checks.items():
        print(f"{k}: {v}")

    return checks

audit_muon_assignment(model , opt_info1)

embedding_in_adamw: True
lm_head_tied_to_embedding: True
lm_head_in_adamw_by_name: False
lm_head_effectively_in_adamw: True
lm_head_not_in_muon: True
mtp_vocab_heads_in_adamw: True
norms_in_adamw: True
attention_linears_in_muon: True
moe_experts_in_muon: True
shared_experts_in_muon: True
router_in_muon: True
mtp_transforms_in_muon: True


{'embedding_in_adamw': True,
 'lm_head_tied_to_embedding': True,
 'lm_head_in_adamw_by_name': False,
 'lm_head_effectively_in_adamw': True,
 'lm_head_not_in_muon': True,
 'mtp_vocab_heads_in_adamw': True,
 'norms_in_adamw': True,
 'attention_linears_in_muon': True,
 'moe_experts_in_muon': True,
 'shared_experts_in_muon': True,
 'router_in_muon': True,
 'mtp_transforms_in_muon': True}

In [145]:
batch = next(iter(train_loader))

batch = normalize_lm_batch(batch)
batch = move_batch_to_device(batch, device)

optimizer1.zero_grad(set_to_none=True)

with autocast_ctx(
    device=device,
    enabled=precision["amp_enabled"],
    amp_dtype=precision["amp_dtype_requested"],
    cache_enabled=precision["cache_enabled"],
    fallback_bf16_to_fp16=precision["fallback_bf16_to_fp16"],
):
    outputs = model(**batch)
    loss = outputs["loss"]

print(loss)

scaler = precision["scaler"]

if scaler is not None:
    # Para fp16 + HybridMuonAdamW habría que hacer step separado.
    # Para nuestro caso recomendado: bf16 => scaler=None.
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer1.muon)
    scaler.unscale_(optimizer1.adamw)
    scaler.step(optimizer1.muon)
    scaler.step(optimizer1.adamw)
    scaler.update()
else:
    loss.backward()
    optimizer1.step()

print("muon-adamw step ok")

tensor(9.1323, device='cuda:0', grad_fn=<NllLossBackward0>)
muon-adamw step ok


---

# Scheduler

In [165]:
# ============================================================
# WARMUP + COSINE SCHEDULER
# Supports AdamW and HybridMuonAdamW
# DeepSeek-V4 Mini Training Stack
# ============================================================

from __future__ import annotations

import math
from typing import Any, Dict, List, Optional, Union


class WarmupCosineLR:
    """
    Step-based linear warmup + cosine decay scheduler.

    Supports:
        1. Standard torch optimizer, e.g. AdamW
        2. HybridMuonAdamW with:
            optimizer.muon
            optimizer.adamw
            optimizer.set_lr(lr, muon_lr=None)

    Behavior:
        - steps 1..warmup_steps:
            lr increases linearly from 0 to base_lr
        - after warmup:
            cosine decay from base_lr to min_lr
        - resume-safe with state_dict/load_state_dict

    Usage:
        scheduler = WarmupCosineLR(
            optimizer=optimizer,
            total_steps=10000,
            warmup_steps=500,
            min_lr=3e-5,
        )

        optimizer.step()
        scheduler.step()
    """

    def __init__(
        self,
        optimizer,
        total_steps: int,
        warmup_steps: int,
        min_lr: float = 0.0,
        min_muon_lr: Optional[float] = None,
    ):
        if total_steps <= 0:
            raise ValueError(f"total_steps must be > 0, got {total_steps}")

        if warmup_steps < 0:
            raise ValueError(f"warmup_steps must be >= 0, got {warmup_steps}")

        if min_lr < 0:
            raise ValueError(f"min_lr must be >= 0, got {min_lr}")

        if min_muon_lr is not None and min_muon_lr < 0:
            raise ValueError(f"min_muon_lr must be >= 0, got {min_muon_lr}")

        self.optimizer = optimizer
        self.total_steps = int(total_steps)
        self.warmup_steps = int(warmup_steps)
        self.min_lr = float(min_lr)
        self.min_muon_lr = float(min_muon_lr) if min_muon_lr is not None else None

        self.step_num = 0

        self.is_hybrid = (
            hasattr(optimizer, "muon")
            and hasattr(optimizer, "adamw")
        )

        if self.is_hybrid:
            self.base_adamw_lrs = [
                float(group["lr"])
                for group in optimizer.adamw.param_groups
            ]

            self.base_muon_lrs = [
                float(group["lr"])
                for group in optimizer.muon.param_groups
            ]

            # For compatibility with logging/checkpoint inspection.
            self.base_lrs = self.base_muon_lrs + self.base_adamw_lrs

        else:
            self.base_lrs = [
                float(group["lr"])
                for group in optimizer.param_groups
            ]

            self.base_adamw_lrs = None
            self.base_muon_lrs = None

    def _compute_lr(
        self,
        base_lr: float,
        min_lr: float,
        step: int,
    ) -> float:
        """
        Compute LR for a given base_lr and current step.
        """
        if self.warmup_steps > 0 and step <= self.warmup_steps:
            return base_lr * (step / max(1, self.warmup_steps))

        if self.total_steps <= self.warmup_steps:
            return min_lr

        t = min(max(step, self.warmup_steps), self.total_steps)

        denom = max(1, self.total_steps - self.warmup_steps)
        progress = (t - self.warmup_steps) / denom
        progress = min(1.0, max(0.0, progress))

        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        lr = min_lr + (base_lr - min_lr) * cosine

        return float(lr)

    def _set_lr_standard(self, step: int) -> None:
        """
        Set LR for standard torch optimizers.
        """
        for i, group in enumerate(self.optimizer.param_groups):
            base_lr = self.base_lrs[i]
            group["lr"] = self._compute_lr(
                base_lr=base_lr,
                min_lr=self.min_lr,
                step=step,
            )

    def _set_lr_hybrid(self, step: int) -> None:
        """
        Set LR for HybridMuonAdamW.

        AdamW and Muon can have different base LRs and different min LRs.
        """
        muon_min_lr = self.min_lr if self.min_muon_lr is None else self.min_muon_lr

        for i, group in enumerate(self.optimizer.adamw.param_groups):
            base_lr = self.base_adamw_lrs[i]
            group["lr"] = self._compute_lr(
                base_lr=base_lr,
                min_lr=self.min_lr,
                step=step,
            )

        for i, group in enumerate(self.optimizer.muon.param_groups):
            base_lr = self.base_muon_lrs[i]
            group["lr"] = self._compute_lr(
                base_lr=base_lr,
                min_lr=muon_min_lr,
                step=step,
            )

    def _set_lr(self, step: int) -> None:
        if self.is_hybrid:
            self._set_lr_hybrid(step)
        else:
            self._set_lr_standard(step)

    def step(self) -> None:
        """
        Advance scheduler by one step.

        Recommended order:
            optimizer.step()
            scheduler.step()
        """
        self.step_num += 1
        self._set_lr(self.step_num)

    def set_step(self, step: int) -> None:
        """
        Explicitly set scheduler step.

        Useful when resuming or manually syncing global_step.
        """
        if step < 0:
            raise ValueError(f"step must be >= 0, got {step}")

        self.step_num = int(step)
        self._set_lr(self.step_num)

    def get_last_lr(self) -> List[float]:
        """
        Return current learning rates.
        """
        if self.is_hybrid:
            return (
                [group["lr"] for group in self.optimizer.muon.param_groups]
                + [group["lr"] for group in self.optimizer.adamw.param_groups]
            )

        return [group["lr"] for group in self.optimizer.param_groups]

    def get_lr_dict(self) -> Dict[str, Any]:
        """
        Logging-friendly LR dictionary.
        """
        if self.is_hybrid:
            muon_lrs = [group["lr"] for group in self.optimizer.muon.param_groups]
            adamw_lrs = [group["lr"] for group in self.optimizer.adamw.param_groups]

            return {
                "step": int(self.step_num),
                "muon_lr": float(muon_lrs[0]) if muon_lrs else None,
                "adamw_lr": float(adamw_lrs[0]) if adamw_lrs else None,
                "muon_lrs": [float(x) for x in muon_lrs],
                "adamw_lrs": [float(x) for x in adamw_lrs],
            }

        lrs = [group["lr"] for group in self.optimizer.param_groups]

        return {
            "step": int(self.step_num),
            "lr": float(lrs[0]) if lrs else None,
            "lrs": [float(x) for x in lrs],
        }

    def state_dict(self) -> Dict[str, Any]:
        state = {
            "step_num": int(self.step_num),
            "total_steps": int(self.total_steps),
            "warmup_steps": int(self.warmup_steps),
            "min_lr": float(self.min_lr),
            "min_muon_lr": self.min_muon_lr,
            "is_hybrid": bool(self.is_hybrid),
            "base_lrs": list(self.base_lrs),
        }

        if self.is_hybrid:
            state["base_adamw_lrs"] = list(self.base_adamw_lrs)
            state["base_muon_lrs"] = list(self.base_muon_lrs)

        return state

    def load_state_dict(self, state_dict: Dict[str, Any]) -> None:
        if not isinstance(state_dict, dict):
            return

        self.step_num = int(state_dict.get("step_num", 0))
        self.total_steps = int(state_dict.get("total_steps", self.total_steps))
        self.warmup_steps = int(state_dict.get("warmup_steps", self.warmup_steps))
        self.min_lr = float(state_dict.get("min_lr", self.min_lr))

        loaded_min_muon_lr = state_dict.get("min_muon_lr", self.min_muon_lr)
        self.min_muon_lr = (
            float(loaded_min_muon_lr)
            if loaded_min_muon_lr is not None
            else None
        )

        if self.is_hybrid:
            loaded_adamw = state_dict.get("base_adamw_lrs", None)
            loaded_muon = state_dict.get("base_muon_lrs", None)

            if (
                isinstance(loaded_adamw, (list, tuple))
                and len(loaded_adamw) == len(self.optimizer.adamw.param_groups)
            ):
                self.base_adamw_lrs = [float(x) for x in loaded_adamw]

            if (
                isinstance(loaded_muon, (list, tuple))
                and len(loaded_muon) == len(self.optimizer.muon.param_groups)
            ):
                self.base_muon_lrs = [float(x) for x in loaded_muon]

            self.base_lrs = self.base_muon_lrs + self.base_adamw_lrs

        else:
            loaded_base_lrs = state_dict.get("base_lrs", None)

            if (
                isinstance(loaded_base_lrs, (list, tuple))
                and len(loaded_base_lrs) == len(self.optimizer.param_groups)
            ):
                self.base_lrs = [float(x) for x in loaded_base_lrs]

        # Restore LR exactly to resumed step.
        self._set_lr(self.step_num)

def build_warmup_cosine_scheduler(
    optimizer,
    total_steps: int,
    warmup_steps: int,
    min_lr: float = 3e-5,
    min_muon_lr: Optional[float] = None,
) -> WarmupCosineLR:
    """
    Build scheduler compatible with AdamW or HybridMuonAdamW.
    """
    return WarmupCosineLR(
        optimizer=optimizer,
        total_steps=total_steps,
        warmup_steps=warmup_steps,
        min_lr=min_lr,
        min_muon_lr=min_muon_lr,
    )

In [166]:
optimizer, opt_info = build_adamw_optimizer(
    model=model,
    learning_rate=3e-4,
    weight_decay=0.1,
    betas=(0.9, 0.95),
    eps=1e-8,
    verbose=True,
)

scheduler = build_warmup_cosine_scheduler(
    optimizer=optimizer,
    total_steps=10_000,
    warmup_steps=500,
    min_lr=3e-5,
)

AdamW parameter groups
Decay tensors     : 66
No-decay tensors  : 75
Decay params      : 269,632
No-decay params   : 84,496
Total params      : 354,128


In [167]:
optimizer, opt_info = build_muon_adamw_optimizer(
    model=model,
    learning_rate=3e-4,
    weight_decay=0.1,
    betas=(0.9, 0.95),
    eps=1e-8,
    muon_lr=None,
    muon_momentum=0.95,
    muon_nesterov=True,
    muon_ns_steps=5,
    muon_eps=1e-7,
    muon_weight_decay=0.0,
    verbose=True,
)

scheduler = build_warmup_cosine_scheduler(
    optimizer=optimizer,
    total_steps=10_000,
    warmup_steps=500,
    min_lr=3e-5,
    min_muon_lr=None,  # None => usa min_lr también para Muon
)

Hybrid Muon + AdamW parameter groups
Muon tensors          : 66
AdamW decay tensors   : 4
AdamW no-decay tensors: 71
--------------------------------------------------------------------------------
Muon params           : 285,952
AdamW params          : 68,176
Total trainable params: 354,128


In [168]:
print("Initial LR:", scheduler.get_lr_dict())

for _ in range(5):
    scheduler.step()
    print(scheduler.get_lr_dict())

Initial LR: {'step': 0, 'muon_lr': 0.0003, 'adamw_lr': 0.0003, 'muon_lrs': [0.0003], 'adamw_lrs': [0.0003, 0.0003]}
{'step': 1, 'muon_lr': 6e-07, 'adamw_lr': 6e-07, 'muon_lrs': [6e-07], 'adamw_lrs': [6e-07, 6e-07]}
{'step': 2, 'muon_lr': 1.2e-06, 'adamw_lr': 1.2e-06, 'muon_lrs': [1.2e-06], 'adamw_lrs': [1.2e-06, 1.2e-06]}
{'step': 3, 'muon_lr': 1.8e-06, 'adamw_lr': 1.8e-06, 'muon_lrs': [1.8e-06], 'adamw_lrs': [1.8e-06, 1.8e-06]}
{'step': 4, 'muon_lr': 2.4e-06, 'adamw_lr': 2.4e-06, 'muon_lrs': [2.4e-06], 'adamw_lrs': [2.4e-06, 2.4e-06]}
{'step': 5, 'muon_lr': 2.9999999999999997e-06, 'adamw_lr': 2.9999999999999997e-06, 'muon_lrs': [2.9999999999999997e-06], 'adamw_lrs': [2.9999999999999997e-06, 2.9999999999999997e-06]}


---

# Train One Epoch

In [172]:
# ============================================================
# Forward helpers
# ============================================================

def filter_forward_kwargs(model: nn.Module, kwargs: Dict[str, Any]) -> Dict[str, Any]:
    """
    Keep only kwargs accepted by model.forward.

    This makes the trainer robust to:
        - DeepSeekV4LM
        - simpler LM wrappers
    """
    raw_model = model.module if hasattr(model, "module") else model
    sig = inspect.signature(raw_model.forward)
    allowed = set(sig.parameters.keys())

    return {
        k: v
        for k, v in kwargs.items()
        if k in allowed and v is not None
    }


def call_model_for_training(
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    return_aux: bool = False,
    need_weights: bool = False,
):
    """
    Safe model call for training.

    For DeepSeekV4LM, supports:
        input_ids, labels, mtp_labels, attention_mask,
        position_ids, start_pos, return_aux, need_weights.

    For simpler models, unsupported kwargs are dropped.
    """
    kwargs = {
        "input_ids": batch["input_ids"],
        "labels": batch.get("labels", None),
        "mtp_labels": batch.get("mtp_labels", None),
        "attention_mask": batch.get("attention_mask", None),
        "position_ids": batch.get("position_ids", None),
        "start_pos": batch.get("start_pos", None),
        "return_aux": return_aux,
        "need_weights": need_weights,
    }

    kwargs = filter_forward_kwargs(model, kwargs)
    return model(**kwargs)


def get_loss_from_outputs(outputs: Any) -> torch.Tensor:
    if isinstance(outputs, dict):
        if "loss" not in outputs:
            raise KeyError("Model outputs dict does not contain key 'loss'.")
        return outputs["loss"]

    if hasattr(outputs, "loss"):
        return outputs.loss

    raise KeyError("Could not extract loss from model outputs.")


# ============================================================
# Optimizer / scaler helpers
# ============================================================

def is_hybrid_muon_adamw_optimizer(optimizer: Any) -> bool:
    return hasattr(optimizer, "muon") and hasattr(optimizer, "adamw")


def unscale_optimizer_grads(scaler: Any, optimizer: Any) -> None:
    """
    GradScaler.unscale_ for AdamW or HybridMuonAdamW.
    """
    if scaler is None:
        return

    if is_hybrid_muon_adamw_optimizer(optimizer):
        scaler.unscale_(optimizer.muon)
        scaler.unscale_(optimizer.adamw)
    else:
        scaler.unscale_(optimizer)


def scaler_step_optimizer(scaler: Any, optimizer: Any) -> None:
    """
    scaler.step for AdamW or HybridMuonAdamW.
    """
    if is_hybrid_muon_adamw_optimizer(optimizer):
        scaler.step(optimizer.muon)
        scaler.step(optimizer.adamw)
    else:
        scaler.step(optimizer)


def get_current_lrs(optimizer: Any, scheduler: Optional[Any] = None) -> Dict[str, float]:
    """
    Logging-friendly LR extraction.
    """
    if scheduler is not None and hasattr(scheduler, "get_lr_dict"):
        lr_dict = scheduler.get_lr_dict()

        out = {}

        for k, v in lr_dict.items():
            if isinstance(v, (int, float)) and v is not None:
                out[f"lr/{k}"] = float(v)

        return out

    if is_hybrid_muon_adamw_optimizer(optimizer):
        out = {}

        if optimizer.muon.param_groups:
            out["lr/muon_lr"] = float(optimizer.muon.param_groups[0]["lr"])

        if optimizer.adamw.param_groups:
            out["lr/adamw_lr"] = float(optimizer.adamw.param_groups[0]["lr"])

        return out

    if hasattr(optimizer, "param_groups") and optimizer.param_groups:
        return {"lr/lr": float(optimizer.param_groups[0]["lr"])}

    return {}

In [184]:
# ============================================================
# DEEPSEEK-V4 TRAINING ORCHESTRATION
# train_one_epoch + train_deepseekv4
# ============================================================

from __future__ import annotations

import os
import sys
import json
import time
import math
import shutil
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

import torch
import torch.nn as nn


# ============================================================
# COLAB / DRIVE UTILS
# ============================================================

def fmt_hms(sec: float) -> str:
    m, s = divmod(int(sec), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}"


def rule(w: int = 110, ch: str = "─") -> str:
    return ch * w


def is_colab() -> bool:
    return "google.colab" in sys.modules


def ensure_drive_mounted() -> None:
    if is_colab():
        drive_root = "/content/drive"
        if not os.path.isdir(drive_root):
            try:
                from google.colab import drive
                drive.mount(drive_root, force_remount=False)
            except Exception as e:
                print(f"[DRIVE] No se pudo montar automáticamente: {e}")


def copy_ckpt_to_drive_fixed(
    src_path: str | Path,
    drive_dir: str | Path,
    fixed_name: str = "latest_deepseekv4.pt",
) -> None:
    try:
        if not drive_dir:
            return

        src_path = Path(src_path)
        drive_dir = Path(drive_dir)

        if str(drive_dir).startswith("/content/drive"):
            ensure_drive_mounted()

        drive_dir.mkdir(parents=True, exist_ok=True)

        dst_path = drive_dir / fixed_name

        if dst_path.exists():
            dst_path.unlink()

        shutil.copy2(src_path, dst_path)
        print(f"└─ [DRIVE] copiado → {dst_path}")

    except Exception as e:
        print(f"└─ [DRIVE] ERROR al copiar a Drive: {e}")


def append_jsonl(path: str | Path, record: Dict[str, Any]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    safe_record = {}
    for k, v in record.items():
        if isinstance(v, torch.Tensor):
            if v.numel() == 1:
                safe_record[k] = float(v.detach().cpu().item())
            else:
                safe_record[k] = str(tuple(v.shape))
        elif isinstance(v, (int, float, str, bool)) or v is None:
            safe_record[k] = v
        else:
            try:
                json.dumps(v)
                safe_record[k] = v
            except Exception:
                safe_record[k] = str(v)

    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(safe_record, ensure_ascii=False) + "\n")


def copy_metrics_to_drive(
    src_path: str | Path,
    drive_dir: str | Path,
    fixed_name: str = "metrics_deepseekv4.jsonl",
) -> None:
    try:
        if not drive_dir:
            return

        src_path = Path(src_path)
        drive_dir = Path(drive_dir)

        if not src_path.exists():
            return

        if str(drive_dir).startswith("/content/drive"):
            ensure_drive_mounted()

        drive_dir.mkdir(parents=True, exist_ok=True)

        dst_path = drive_dir / fixed_name
        shutil.copy2(src_path, dst_path)

    except Exception as e:
        print(f"└─ [DRIVE] ERROR al copiar métricas a Drive: {e}")


def gpu_mem_mb(device="cuda") -> Tuple[float, float]:
    device_type = torch.device(device).type

    if torch.cuda.is_available() and device_type == "cuda":
        device_obj = torch.device(device)
        alloc = torch.cuda.memory_allocated(device=device_obj) / (1024 ** 2)
        reserv = torch.cuda.memory_reserved(device=device_obj) / (1024 ** 2)
        return float(alloc), float(reserv)

    return 0.0, 0.0


# ============================================================
# MONITORING HELPERS
# ============================================================

VALID_MONITOR_NAMES = {
    "loss",
    "lm_loss",
    "mtp_loss",
    "moe_aux_loss",
    "perplexity",
    "token_accuracy",
    "top_5_accuracy",
    "sequence_accuracy",
    "mean_confidence",
    "mean_true_token_prob",
    "mean_entropy",
    "train_loss",
    "train_lm_loss",
    "train_mtp_loss",
    "train_moe_aux_loss",
    "eval_loss",
    "eval_perplexity",
    "eval_token_accuracy",
    "eval_top_5_accuracy",
    "eval_sequence_accuracy",
}


def _is_better_metric(
    current: float,
    best: Optional[float],
    mode: str = "min",
) -> bool:
    if best is None:
        return True

    if mode == "min":
        return current < best

    if mode == "max":
        return current > best

    raise ValueError(f"monitor_mode must be 'min' or 'max', got {mode}")


def _prefixed_stats(prefix: str, stats: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    if stats is None:
        return {}

    return {f"{prefix}_{k}": v for k, v in stats.items()}


def _resolve_monitor_value(
    monitor_name: str,
    train_stats: Dict[str, Any],
    eval_stats: Optional[Dict[str, Any]],
) -> float:
    """
    Supports:
        loss
        train_loss
        eval_loss
        eval_token_accuracy
        etc.
    """
    if monitor_name.startswith("train_"):
        key = monitor_name[len("train_"):]
        source = train_stats
    elif monitor_name.startswith("eval_"):
        if eval_stats is None:
            raise ValueError(
                f"monitor_name='{monitor_name}' requiere eval_stats, pero eval_loader=None."
            )
        key = monitor_name[len("eval_"):]
        source = eval_stats
    else:
        if eval_stats is not None and monitor_name in eval_stats:
            key = monitor_name
            source = eval_stats
        else:
            key = monitor_name
            source = train_stats

    if key not in source:
        raise KeyError(
            f"Monitor key '{key}' not found in selected stats. "
            f"Available keys: {sorted(source.keys())[:50]}"
        )

    return float(source[key])


def _get_main_lr(optimizer: Any) -> float:
    if hasattr(optimizer, "adamw"):
        if optimizer.adamw.param_groups:
            return float(optimizer.adamw.param_groups[0]["lr"])
    if hasattr(optimizer, "param_groups") and optimizer.param_groups:
        return float(optimizer.param_groups[0]["lr"])
    return float("nan")


def _get_lr_message(optimizer: Any, scheduler: Optional[Any] = None) -> str:
    if scheduler is not None and hasattr(scheduler, "get_lr_dict"):
        d = scheduler.get_lr_dict()

        if "muon_lr" in d and "adamw_lr" in d:
            return f"muon_lr={d['muon_lr']:.2e} adamw_lr={d['adamw_lr']:.2e}"

        if "lr" in d:
            return f"lr={d['lr']:.2e}"

    if hasattr(optimizer, "muon") and hasattr(optimizer, "adamw"):
        muon_lr = optimizer.muon.param_groups[0]["lr"]
        adamw_lr = optimizer.adamw.param_groups[0]["lr"]
        return f"muon_lr={muon_lr:.2e} adamw_lr={adamw_lr:.2e}"

    return f"lr={_get_main_lr(optimizer):.2e}"


def _safe_float(x: Any, default: float = float("nan")) -> float:
    try:
        if torch.is_tensor(x):
            return float(x.detach().float().cpu().item())
        return float(x)
    except Exception:
        return default

# ============================================================
# PRETTY PRINT HELPERS FOR DEEPSEEK-V4 TRAINING
# ============================================================

import math
import time
import os
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

import torch
import torch.nn as nn


def ds_rule(width: int = 110, ch: str = "─") -> str:
    return ch * width


def ds_title(title: str, width: int = 110, ch: str = "═") -> None:
    print("\n" + ch * width)
    print(title)
    print(ch * width)


def ds_section(title: str, width: int = 110) -> None:
    print("\n" + title)
    print(ds_rule(width=width, ch="─"))


def fmt_hms(sec: float) -> str:
    m, s = divmod(int(sec), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}"


def _fmt_num(x, digits: int = 4, sci_low: float = 1e-3, sci_high: float = 1e4) -> str:
    try:
        x = float(x)
    except Exception:
        return "—"

    if not math.isfinite(x):
        return "—"

    ax = abs(x)

    if ax == 0:
        return "0"

    if ax < sci_low or ax >= sci_high:
        return f"{x:.2e}"

    return f"{x:.{digits}f}"


def _fmt_lr(x) -> str:
    try:
        return f"{float(x):.2e}"
    except Exception:
        return "—"


def _safe_float(x: Any, default: float = float("nan")) -> float:
    try:
        if torch.is_tensor(x):
            return float(x.detach().float().cpu().item())
        return float(x)
    except Exception:
        return default


def _get_main_lr(optimizer: Any) -> float:
    if hasattr(optimizer, "adamw"):
        if optimizer.adamw.param_groups:
            return float(optimizer.adamw.param_groups[0]["lr"])

    if hasattr(optimizer, "param_groups") and optimizer.param_groups:
        return float(optimizer.param_groups[0]["lr"])

    return float("nan")


def _get_optimizer_lrs(optimizer: Any) -> Dict[str, float]:
    if hasattr(optimizer, "muon") and hasattr(optimizer, "adamw"):
        return {
            "muon_lr": float(optimizer.muon.param_groups[0]["lr"]) if optimizer.muon.param_groups else float("nan"),
            "adamw_lr": float(optimizer.adamw.param_groups[0]["lr"]) if optimizer.adamw.param_groups else float("nan"),
        }

    return {
        "lr": _get_main_lr(optimizer),
    }


def _print_deepseek_run_header(
    *,
    run_name: str,
    model: nn.Module,
    device,
    precision: Dict[str, Any],
    optimizer_type: str,
    ema,
    epochs: int,
    start_epoch: int,
    global_step: int,
    total_steps: int,
    warmup_steps: int,
    monitor_name: str,
    monitor_mode: str,
    best_metric,
    grad_clip,
    grad_accum_steps: int,
    val_loader,
    eval_every: int,
    eval_max_batches,
    drive_ckpt_dir,
    fixed_drive_name: str,
) -> None:
    raw_model = model.module if hasattr(model, "module") else model
    cfg = getattr(raw_model, "config", None)

    ema_str = (
        f"{ema.decay:.6f}"
        if ema is not None and hasattr(ema, "decay")
        else "off"
    )

    ds_title(f"DeepSeek-V4 run: {run_name}")

    print(
        f"Device    : {device} | AMP: {precision['amp_enabled']} "
        f"({precision['amp_dtype_requested']} -> {precision['amp_dtype_effective']})"
    )
    print(
        f"Optimizer : {optimizer_type} | EMA: {ema_str} | "
        f"grad_clip: {grad_clip} | grad_accum_steps: {grad_accum_steps}"
    )
    print(
        f"Schedule  : epochs={epochs} | start_epoch={start_epoch} | "
        f"global_step={global_step} | total_steps={total_steps} | warmup_steps={warmup_steps}"
    )
    print(
        f"Monitor   : {monitor_name} ({monitor_mode}) | best_metric={best_metric}"
    )

    if cfg is not None:
        print(
            f"Model     : layers={getattr(cfg, 'n_layers', '?')} | "
            f"d_model={getattr(cfg, 'd_model', '?')} | "
            f"attention={getattr(cfg, 'attention_type', '?')} | "
            f"ffn={getattr(cfg, 'ffn_type', '?')} | "
            f"mHC={getattr(cfg, 'use_mhc', '?')} | "
            f"MTP={getattr(cfg, 'use_mtp', '?')}"
        )

    if val_loader is not None:
        print(f"Eval      : enabled | eval_every={eval_every} | eval_max_batches={eval_max_batches}")
    else:
        print("Eval      : disabled")

    if drive_ckpt_dir:
        print(f"Drive     : {drive_ckpt_dir} | fixed checkpoint={fixed_drive_name}")

    print(ds_rule())


def _print_in_epoch_header() -> None:
    print("\n┆ In-epoch training")
    print(
        "┆   {:>7} │ {:>7} │ {:>8} │ {:>8} │ {:>8} │ {:>9} │ {:>9} │ {:>9} │ {:>9} │ {:>9}".format(
            "step",
            "batch",
            "loss",
            "lm",
            "mtp",
            "moe_aux",
            "grad",
            "muon_lr",
            "adamw_lr",
            "time",
        )
    )
    print("┆   " + "─" * 106)


def _print_in_epoch_row(
    *,
    global_step: int,
    batch_idx: int,
    loss_val: float,
    lm_loss: float,
    mtp_loss: float,
    moe_aux_loss: float,
    grad_norm_value,
    optimizer,
    dt_ms: float,
) -> None:
    lrs = _get_optimizer_lrs(optimizer)

    muon_lr = lrs.get("muon_lr", float("nan"))
    adamw_lr = lrs.get("adamw_lr", lrs.get("lr", float("nan")))

    print(
        "┆   {:7d} │ {:7d} │ {:>8} │ {:>8} │ {:>8} │ {:>9} │ {:>9} │ {:>9} │ {:>9} │ {:7.1f}ms".format(
            int(global_step),
            int(batch_idx + 1),
            _fmt_num(loss_val),
            _fmt_num(lm_loss),
            _fmt_num(mtp_loss),
            _fmt_num(moe_aux_loss, digits=2),
            _fmt_num(grad_norm_value, digits=2),
            _fmt_lr(muon_lr),
            _fmt_lr(adamw_lr),
            float(dt_ms),
        )
    )


def _print_epoch_summary(
    *,
    epoch: int,
    global_step: int,
    sec: float,
    optimizer,
    train_stats: Dict[str, Any],
    eval_stats: Optional[Dict[str, Any]],
    monitor_name: str,
    current_metric: float,
    best_metric: float,
    improved: bool,
) -> None:
    lrs = _get_optimizer_lrs(optimizer)

    ds_section(f"Epoch {epoch:03d} summary")

    print(
        f"step={global_step} | time={fmt_hms(sec)} | "
        + " | ".join([f"{k}={_fmt_lr(v)}" for k, v in lrs.items()])
    )

    print(
        "train -> "
        f"loss={train_stats['loss']:.5f} | "
        f"lm={train_stats['lm_loss']:.5f} | "
        f"mtp={train_stats['mtp_loss']:.5f} | "
        f"moe_aux={train_stats['moe_aux_loss']:.3e} | "
        f"grad_norm={train_stats['grad_norm']:.3e} | "
        f"optim_steps={int(train_stats['n_optimizer_steps'])}"
    )

    if eval_stats is not None:
        print("eval  -> " + format_metrics(eval_stats, prefix="eval"))

    print(
        f"monitor -> {monitor_name}={current_metric:.6f} | "
        f"best={best_metric:.6f} | improved={improved}"
    )

    print(ds_rule())

# ============================================================
# END-OF-EPOCH MODULE DIAGNOSTICS HELPERS
# ============================================================

@torch.no_grad()
def compute_one_batch_deepseek_diagnostics(
    *,
    model: nn.Module,
    dataloader,
    device: torch.device,
    precision: Dict[str, Any],
    prefix: str = "train",
) -> Dict[str, float]:
    """
    Compute DeepSeek module diagnostics on one batch.

    Intended for:
        module_metrics_every=0
        -> print diagnostics once after train_one_epoch and before eval.
    """
    was_training = model.training
    model.eval()

    batch = next(iter(dataloader))
    batch = normalize_lm_batch(batch)
    batch = move_batch_to_device(batch, device)

    with autocast_ctx(
        device=device,
        enabled=precision["amp_enabled"],
        amp_dtype=precision["amp_dtype_requested"],
        cache_enabled=precision["cache_enabled"],
        fallback_bf16_to_fp16=precision["fallback_bf16_to_fp16"],
    ):
        outputs = call_model_for_training(
            model=model,
            batch=batch,
            return_aux=True,
            need_weights=False,
        )

    metrics = compute_deepseek_module_metrics(
        outputs=outputs,
        model=model,
        prefix=prefix,
    )

    if was_training:
        model.train()

    return metrics


def print_deepseek_top10_critical_metrics(
    metrics: Dict[str, float],
    *,
    prefix: str = "train",
    title: Optional[str] = None,
    num_experts: Optional[int] = None,
    n_layers: Optional[int] = None,
) -> None:
    """
    Minimal top-10 critical diagnostics across:
        loss, MoE, MTP, mHC.
    """
    if title is not None:
        print("\n" + "═" * 96)
        print(title)
        print("═" * 96)

    rows = [
        (
            f"{prefix}/loss",
            "total objective; lower is better",
        ),
        (
            f"{prefix}/lm_loss",
            "main next-token CE; lower is better",
        ),
        (
            f"{prefix}/perplexity_from_lm_loss",
            "exp(lm_loss); lower is better",
        ),
        (
            f"{prefix}/mtp/raw_mtp_loss",
            "MTP raw CE; should fall with training",
        ),
        (
            f"{prefix}/moe/router_entropy_mean",
            (
                f"max≈log(E)={math.log(num_experts):.3f}; low => router collapse"
                if num_experts is not None and num_experts > 1
                else "low => router collapse"
            ),
        ),
        (
            f"{prefix}/moe/expert_fraction_min",
            "near 0 => dead/underused expert",
        ),
        (
            f"{prefix}/moe/expert_fraction_max",
            "too high => expert overload/collapse",
        ),
        (
            f"{prefix}/moe/dead_experts_across_layers",
            "should be 0",
        ),
        (
            f"{prefix}/mhc/B_row_sum_error_mean",
            "should be near 0",
        ),
        (
            f"{prefix}/mhc/B_column_sum_error_mean",
            "should be near 0",
        ),
    ]

    available = [(k, note) for k, note in rows if k in metrics]

    if not available:
        print("No critical DeepSeek diagnostics available.")
        return

    max_name = max(len(k.split("/")[-1]) for k, _ in available)

    print("Top-10 critical DeepSeek metrics")
    print("─" * 96)

    for key, note in available:
        short = key.split("/")[-1]
        value = _fmt_num(metrics[key])
        print(f"  {short:<{max_name}} : {value:<12} # {note}")

    if title is not None:
        print("═" * 96)

In [175]:
# ============================================================
# TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    *,
    model: nn.Module,
    dataloader,
    optimizer: Any,
    device: str | torch.device,
    precision: Dict[str, Any],
    scheduler: Optional[Any] = None,
    ema: Optional[Any] = None,
    epoch: int = 0,
    global_step: int = 0,
    grad_clip: Optional[float] = 1.0,
    grad_accum_steps: int = 1,
    max_batches: Optional[int] = None,
    log_every: int = 10,
    module_metrics_every: Optional[int] = 50,
    print_module_diagnostics: bool = True,
    log_grad_norm: bool = True,
    log_mem: bool = False,
    on_oom: str = "skip",
    is_main_process: bool = True,
) -> Tuple[Dict[str, float], int]:
    """
    Train one full epoch for DeepSeekV4LM.

    Uses:
        - model internal loss: outputs["loss"]
        - AMP/autocast
        - AdamW or HybridMuonAdamW
        - scheduler
        - EMA
        - grad clipping
        - DeepSeek module diagnostics
    """
    device = torch.device(device)
    model.to(device)
    model.train()

    grad_accum_steps = max(1, int(grad_accum_steps))
    optimizer.zero_grad(set_to_none=True)

    running = {
        "loss": 0.0,
        "lm_loss": 0.0,
        "mtp_loss": 0.0,
        "moe_aux_loss": 0.0,
        "grad_norm": 0.0,
    }

    n_seen_batches = 0
    n_seen_samples = 0
    n_optimizer_steps = 0
    n_grad_logs = 0
    n_module_logs = 0

    t_epoch = time.time()

    if is_main_process and log_every:
        _print_in_epoch_header()

    for batch_idx, batch in enumerate(dataloader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        try:
            t0 = time.perf_counter()

            batch = normalize_lm_batch(batch)
            batch = move_batch_to_device(batch, device)

            B = int(batch["input_ids"].shape[0])
            n_seen_samples += B

            step_now = ((batch_idx + 1) % grad_accum_steps) == 0

            should_log_modules = (
                module_metrics_every is not None
                and module_metrics_every > 0
                and step_now
                and ((global_step + 1) % module_metrics_every == 0)
            )

            scaler = precision.get("scaler", None)

            with autocast_ctx(
                device=device,
                enabled=precision["amp_enabled"],
                amp_dtype=precision["amp_dtype_requested"],
                cache_enabled=precision["cache_enabled"],
                fallback_bf16_to_fp16=precision["fallback_bf16_to_fp16"],
            ):
                outputs = call_model_for_training(
                    model=model,
                    batch=batch,
                    return_aux=bool(should_log_modules),
                    need_weights=False,
                )

                loss = get_loss_from_outputs(outputs)
                loss_for_backward = loss / grad_accum_steps

            if scaler is not None:
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            optimizer_step_happened = False
            grad_norm_value = None

            if step_now:
                if scaler is not None:
                    unscale_optimizer_grads(scaler, optimizer)

                if log_grad_norm or (grad_clip is not None and grad_clip > 0):
                    grad_norm = torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=float(grad_clip) if grad_clip is not None else 1e9,
                    )
                    grad_norm_value = float(grad_norm.detach().float().cpu().item())

                if scaler is not None:
                    old_scale = scaler.get_scale()

                    scaler_step_optimizer(scaler, optimizer)
                    scaler.update()

                    new_scale = scaler.get_scale()
                    optimizer_step_happened = new_scale >= old_scale
                else:
                    optimizer.step()
                    optimizer_step_happened = True

                if optimizer_step_happened:
                    if scheduler is not None:
                        scheduler.step()

                    global_step += 1
                    n_optimizer_steps += 1

                    if ema is not None:
                        try:
                            ema.update(model, step=global_step)
                        except TypeError:
                            ema.update(model)

                optimizer.zero_grad(set_to_none=True)

            loss_val = _safe_float(loss)
            running["loss"] += loss_val

            if isinstance(outputs, dict):
                lm_loss = outputs.get("lm_loss", None)
                mtp_loss = outputs.get("mtp_loss", None)
                moe_aux_loss = outputs.get("moe_aux_loss", None)
            else:
                lm_loss = getattr(outputs, "lm_loss", None)
                mtp_loss = getattr(outputs, "mtp_loss", None)
                moe_aux_loss = getattr(outputs, "moe_aux_loss", None)

            lm_loss_val = _safe_float(lm_loss, 0.0)
            mtp_loss_val = _safe_float(mtp_loss, 0.0)
            moe_aux_loss_val = _safe_float(moe_aux_loss, 0.0)

            running["lm_loss"] += lm_loss_val
            running["mtp_loss"] += mtp_loss_val
            running["moe_aux_loss"] += moe_aux_loss_val

            if grad_norm_value is not None:
                running["grad_norm"] += grad_norm_value
                n_grad_logs += 1

            n_seen_batches += 1

            should_log = (
                log_every
                and step_now
                and optimizer_step_happened
                and (global_step % log_every == 0)
            )

            if should_log and is_main_process:
                dt_ms = (time.perf_counter() - t0) * 1000.0

                _print_in_epoch_row(
                    global_step=global_step,
                    batch_idx=batch_idx,
                    loss_val=loss_val,
                    lm_loss=lm_loss_val,
                    mtp_loss=mtp_loss_val,
                    moe_aux_loss=moe_aux_loss_val,
                    grad_norm_value=grad_norm_value,
                    optimizer=optimizer,
                    dt_ms=dt_ms,
                )

                if log_mem:
                    alloc, reserv = gpu_mem_mb(device)
                    print(f"┆        memory: allocated={alloc:.0f}MB | reserved={reserv:.0f}MB")

            if should_log_modules:
                n_module_logs += 1

                module_metrics = compute_deepseek_module_metrics(
                    outputs=outputs,
                    model=model,
                    prefix="train",
                )

                if is_main_process and print_module_diagnostics:
                    raw_model = model.module if hasattr(model, "module") else model
                    cfg = getattr(raw_model, "config", None)

                    print_deepseek_module_metrics(
                        module_metrics,
                        prefix="train",
                        precision=4,
                        title=f"DeepSeek-V4 module diagnostics | epoch={epoch} step={global_step}",
                        num_experts=getattr(cfg, "num_experts", None),
                        top_k_experts=getattr(cfg, "top_k_experts", None),
                        n_layers=getattr(cfg, "n_layers", None),
                    )

                    if log_every:
                        _print_in_epoch_header()

        except RuntimeError as e:
            if ("out of memory" in str(e).lower()) and (on_oom == "skip"):
                import gc

                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                optimizer.zero_grad(set_to_none=True)

                if is_main_process:
                    print(f"[WARN][OOM] Batch {batch_idx} omitido. Limpié cache y sigo.")

                continue

            raise

    denom = max(1, n_seen_batches)

    epoch_stats = {
        "loss": running["loss"] / denom,
        "lm_loss": running["lm_loss"] / denom,
        "mtp_loss": running["mtp_loss"] / denom,
        "moe_aux_loss": running["moe_aux_loss"] / denom,
        "grad_norm": running["grad_norm"] / max(1, n_grad_logs),
        "n_seen_batches": float(n_seen_batches),
        "n_seen_samples": float(n_seen_samples),
        "n_optimizer_steps": float(n_optimizer_steps),
        "n_module_logs": float(n_module_logs),
        "epoch_time_sec": float(time.time() - t_epoch),
    }

    return epoch_stats, global_step

In [180]:
# ============================================================
# EVAL ONE EPOCH
# DeepSeek-V4 Mini
# ============================================================

from __future__ import annotations

import math
import time
from typing import Any, Dict, Optional, Sequence

import torch
import torch.nn as nn


def _decode_ids(ids, tokenizer=None, id2tok_fn=None, max_tokens: Optional[int] = None) -> str:
    """
    Decode token ids using either:
      - id2tok_fn(list[int]) -> str
      - tokenizer.decode(list[int]) -> str
      - tokenizer.idx_to_token mapping
      - fallback: space-separated ids
    """
    if torch.is_tensor(ids):
        ids = ids.detach().cpu().tolist()

    ids = [int(x) for x in ids]

    if max_tokens is not None:
        ids = ids[:max_tokens]

    if id2tok_fn is not None:
        try:
            return id2tok_fn(ids)
        except Exception:
            pass

    if tokenizer is not None:
        if hasattr(tokenizer, "decode"):
            try:
                return tokenizer.decode(ids)
            except Exception:
                pass

        if hasattr(tokenizer, "idx_to_token"):
            try:
                return " ".join(tokenizer.idx_to_token.get(int(i), f"<{int(i)}>") for i in ids)
            except Exception:
                pass

    return " ".join(str(int(i)) for i in ids)


@torch.no_grad()
def autoregressive_preview(
    *,
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    device: torch.device,
    precision: Dict[str, Any],
    tokenizer=None,
    id2tok_fn=None,
    max_context_tokens: int = 48,
    max_new_tokens: int = 24,
    sample_idx: int = 0,
    temperature: float = 0.0,
) -> Dict[str, Any]:
    """
    Qualitative eval preview.

    Produces:
      1. teacher-forced argmax:
         CTX / REF / HYP over the existing validation sequence.

      2. short autoregressive rollout:
         takes a prefix from CTX and generates max_new_tokens greedily
         or with sampling if temperature > 0.

    Notes:
      - This is for qualitative inspection only.
      - It is intentionally small and runs only during eval.
    """
    raw_model = model.module if hasattr(model, "module") else model
    cfg = getattr(raw_model, "config", None)

    pad_token_id = getattr(cfg, "pad_token_id", None) if cfg is not None else None
    max_seq_len = getattr(cfg, "max_seq_len", None) if cfg is not None else None

    input_ids = batch["input_ids"]
    labels = batch.get("labels", None)

    sample_idx = min(int(sample_idx), input_ids.shape[0] - 1)

    x = input_ids[sample_idx:sample_idx + 1].to(device)

    y = None
    if labels is not None:
        y = labels[sample_idx:sample_idx + 1].to(device)

    # --------------------------------------------------------
    # Teacher-forced argmax preview
    # --------------------------------------------------------
    with autocast_ctx(
        device=device,
        enabled=precision["amp_enabled"],
        amp_dtype=precision["amp_dtype_requested"],
        cache_enabled=precision["cache_enabled"],
        fallback_bf16_to_fp16=precision["fallback_bf16_to_fp16"],
    ):
        outputs = call_model_for_training(
            model=model,
            batch={
                "input_ids": x,
                "labels": y,
                "attention_mask": None,
                "mtp_labels": None,
            },
            return_aux=False,
            need_weights=False,
        )

    logits = _get_logits_from_outputs(outputs)
    pred_ids = logits.argmax(dim=-1)

    show_len = min(max_context_tokens, x.shape[1])

    ctx_ids = x[0, :show_len]
    pred_show = pred_ids[0, :show_len]

    if y is not None:
        ref_show = y[0, :show_len]
    else:
        ref_show = None

    ctx_text = _decode_ids(ctx_ids, tokenizer=tokenizer, id2tok_fn=id2tok_fn)
    hyp_text = _decode_ids(pred_show, tokenizer=tokenizer, id2tok_fn=id2tok_fn)
    ref_text = _decode_ids(ref_show, tokenizer=tokenizer, id2tok_fn=id2tok_fn) if ref_show is not None else None

    # --------------------------------------------------------
    # Autoregressive rollout preview
    # --------------------------------------------------------
    prefix_len = min(max_context_tokens, x.shape[1])
    generated = x[:, :prefix_len].clone()

    for _ in range(max_new_tokens):
        model_input = generated

        if max_seq_len is not None and model_input.shape[1] > max_seq_len:
            model_input = model_input[:, -max_seq_len:]

        with autocast_ctx(
            device=device,
            enabled=precision["amp_enabled"],
            amp_dtype=precision["amp_dtype_requested"],
            cache_enabled=precision["cache_enabled"],
            fallback_bf16_to_fp16=precision["fallback_bf16_to_fp16"],
        ):
            out_gen = call_model_for_training(
                model=model,
                batch={"input_ids": model_input},
                return_aux=False,
                need_weights=False,
            )

        gen_logits = _get_logits_from_outputs(out_gen)
        next_logits = gen_logits[:, -1, :].float()

        if temperature is not None and temperature > 0:
            probs = torch.softmax(next_logits / float(temperature), dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
        else:
            next_id = torch.argmax(next_logits, dim=-1, keepdim=True)

        generated = torch.cat([generated, next_id], dim=1)

        if pad_token_id is not None and int(next_id.item()) == int(pad_token_id):
            # Do not necessarily stop on pad, but for tiny synthetic datasets this avoids ugly tails.
            pass

    generated_text = _decode_ids(
        generated[0],
        tokenizer=tokenizer,
        id2tok_fn=id2tok_fn,
        max_tokens=prefix_len + max_new_tokens,
    )

    return {
        "ctx": ctx_text,
        "ref": ref_text,
        "hyp": hyp_text,
        "generated": generated_text,
        "prefix_len": int(prefix_len),
        "max_new_tokens": int(max_new_tokens),
    }


def print_eval_preview(preview: Dict[str, Any], title: str = "Eval qualitative preview") -> None:
    print("\n" + "─" * 110)
    print(title)
    print("─" * 110)
    print("Teacher-forced argmax:")
    print(f"  CTX: {repr(preview.get('ctx', ''))}")

    if preview.get("ref", None) is not None:
        print(f"  REF: {repr(preview.get('ref', ''))}")

    print(f"  HYP: {repr(preview.get('hyp', ''))}")
    print()
    print(f"Autoregressive rollout ({preview.get('max_new_tokens', '?')} new tokens):")
    print(f"  GEN: {repr(preview.get('generated', ''))}")
    print("─" * 110)


@torch.no_grad()
def eval_one_epoch(
    *,
    model: nn.Module,
    dataloader,
    device: str | torch.device,
    precision: Dict[str, Any],
    epoch: int = 0,
    max_batches: Optional[int] = None,
    topk: Sequence[int] = (1, 5),
    ema: Optional[Any] = None,
    use_ema: bool = False,
    tokenizer=None,
    id2tok_fn=None,
    preview: bool = True,
    preview_batch_idx: int = 0,
    preview_sample_idx: int = 0,
    preview_max_context_tokens: int = 48,
    preview_max_new_tokens: int = 24,
    preview_temperature: float = 0.0,
    log_every: Optional[int] = None,
    is_main_process: bool = True,
) -> Dict[str, float]:
    """
    Evaluate one epoch for DeepSeekV4LM.

    Computes:
      - LM loss/perplexity/token accuracy/top-k/entropy
      - optional EMA eval
      - qualitative preview:
          CTX / REF / HYP teacher-forced argmax
          GEN autoregressive rollout

    Returns:
      eval_stats dict.
    """
    device = torch.device(device)

    was_training = model.training
    model.to(device)
    model.eval()

    averager = MetricAverager()
    aux_sums = {}
    aux_counts = {}

    t0 = time.time()
    n_batches = 0
    n_samples = 0
    preview_done = False

    def _eval_loop():
        nonlocal n_batches, n_samples, preview_done

        for batch_idx, batch in enumerate(dataloader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            batch = normalize_lm_batch(batch)
            batch = move_batch_to_device(batch, device)

            n_batches += 1
            n_samples += int(batch["input_ids"].shape[0])

            with autocast_ctx(
                device=device,
                enabled=precision["amp_enabled"],
                amp_dtype=precision["amp_dtype_requested"],
                cache_enabled=precision["cache_enabled"],
                fallback_bf16_to_fp16=precision["fallback_bf16_to_fp16"],
            ):
                outputs = call_model_for_training(
                    model=model,
                    batch=batch,
                    return_aux=False,
                    need_weights=False,
                )

            logits = _get_logits_from_outputs(outputs)
            labels = batch["labels"]

            metrics = compute_lm_metrics_from_logits(
                model=model,
                logits=logits,
                labels=labels,
                topk=topk,
            )

            averager.update(metrics)

            # Collect scalar model-side loss diagnostics if present.
            for key in [
                "loss",
                "lm_loss",
                "mtp_loss",
                "moe_aux_loss",
                "raw_mtp_loss",
                "weighted_mtp_loss",
            ]:
                value = _get_output_value(outputs, key, None)

                if value is None:
                    continue

                value = _safe_float(value, default=float("nan"))

                if math.isfinite(value):
                    aux_sums[key] = aux_sums.get(key, 0.0) + value
                    aux_counts[key] = aux_counts.get(key, 0) + 1

            # Qualitative preview only once per eval.
            if (
                preview
                and (not preview_done)
                and batch_idx == int(preview_batch_idx)
                and is_main_process
            ):
                preview_dict = autoregressive_preview(
                    model=model,
                    batch=batch,
                    device=device,
                    precision=precision,
                    tokenizer=tokenizer,
                    id2tok_fn=id2tok_fn,
                    max_context_tokens=preview_max_context_tokens,
                    max_new_tokens=preview_max_new_tokens,
                    sample_idx=preview_sample_idx,
                    temperature=preview_temperature,
                )

                print_eval_preview(
                    preview_dict,
                    title=f"Eval qualitative preview | epoch={epoch} | batch={batch_idx}",
                )

                preview_done = True

            if log_every is not None and log_every > 0 and is_main_process:
                if n_batches % log_every == 0:
                    partial = averager.compute()
                    print(
                        f"eval batch {n_batches:04d} | "
                        f"loss={partial['loss']:.4f} | "
                        f"ppl={partial['perplexity']:.2f} | "
                        f"tok_acc={partial.get('token_accuracy', float('nan')):.4f}"
                    )

    if use_ema:
        if ema is None:
            raise ValueError("use_ema=True but ema=None.")

        with ema.average_parameters(model):
            _eval_loop()
    else:
        _eval_loop()

    stats = averager.compute()

    for key, total in aux_sums.items():
        count = aux_counts.get(key, 0)
        if count > 0:
            stats[f"model_{key}"] = total / count

    stats["n_eval_batches"] = float(n_batches)
    stats["n_eval_samples"] = float(n_samples)
    stats["eval_time_sec"] = float(time.time() - t0)
    stats["used_ema"] = float(bool(use_ema))

    if was_training:
        model.train()

    return stats

In [185]:
# ============================================================
# TRAIN DEEPSEEK-V4
# High-level orchestration
# ============================================================

def train_deepseekv4(
    *,
    model: nn.Module,
    train_loader,
    val_loader=None,

    # Device / precision
    seed: int = 42,
    deterministic: bool = False,
    device: str = "auto",
    amp_enabled: bool = True,
    amp_dtype: str = "bf16",
    fallback_bf16_to_fp16: bool = True,

    # Optimizer
    optimizer_type: str = "adamw",  # "adamw", "muon_adamw"
    learning_rate: float = 3e-4,
    min_learning_rate: float = 3e-5,
    weight_decay: float = 0.1,
    betas: tuple[float, float] = (0.9, 0.95),
    eps: float = 1e-8,

    # Muon
    muon_lr: Optional[float] = None,
    muon_momentum: float = 0.95,
    muon_nesterov: bool = True,
    muon_ns_steps: int = 5,
    muon_eps: float = 1e-7,
    muon_weight_decay: float = 0.0,

    # Scheduler
    total_steps: Optional[int] = None,
    warmup_steps: int = 500,
    min_muon_lr: Optional[float] = None,

    # EMA
    use_ema: bool = False,
    ema_decay: float = 0.999,
    ema_device: Optional[str] = "cpu",
    ema_update_after_step: int = 10,
    ema_update_every: int = 1,

    # Training
    epochs: int = 1,
    start_epoch: int = 0,
    global_step: int = 0,
    grad_clip: Optional[float] = 1.0,
    grad_accum_steps: int = 1,
    max_batches_per_epoch: Optional[int] = None,
    log_every: int = 10,

    # Module diagnostics
    # module_metrics_every > 0:
    #     print diagnostics inside train_one_epoch every N optimizer steps.
    # module_metrics_every == 0:
    #     print diagnostics once after train_one_epoch and before eval.
    # verbose=1:
    #     full diagnostics.
    # verbose=0:
    #     top-10 critical diagnostics only.
    module_metrics_every: Optional[int] = 150,
    print_module_diagnostics: bool = True,
    verbose: int = 1,

    log_grad_norm: bool = True,
    log_mem: bool = False,
    on_oom: str = "skip",

    # Eval
    eval_every: int = 1,
    eval_max_batches: Optional[int] = 50,
    eval_use_ema: bool = False,
    eval_log_every: Optional[int] = None,

    # Eval qualitative preview
    eval_preview: bool = True,
    eval_preview_batch_idx: int = 0,
    eval_preview_sample_idx: int = 0,
    eval_preview_max_context_tokens: int = 48,
    eval_preview_max_new_tokens: int = 24,
    eval_preview_temperature: float = 0.0,
    tokenizer=None,
    id2tok_fn=None,

    # Checkpoints
    ckpt_dir: str = "checkpoints/deepseekv4_mini",
    run_name: str = "deepseekv4_mini",
    save_every: int = 1,
    save_last: bool = True,
    keep_last_n_checkpoints: int = 3,
    monitor_name: str = "eval_loss",
    monitor_mode: str = "min",
    best_metric: Optional[float] = None,
    resume_path: Optional[str] = None,
    strict_resume: bool = True,
    restore_rng_state: bool = False,

    # Drive / metrics
    metrics_jsonl_name: str = "metrics.jsonl",
    drive_ckpt_dir: Optional[str] = None,
    copy_fixed_to_drive: bool = True,
    fixed_drive_name: str = "latest_deepseekv4.pt",
    fixed_drive_metrics_name: str = "metrics_deepseekv4.jsonl",

    # Metadata
    config: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Full DeepSeekV4 training orchestration.

    High-level loop calls:
        - train_one_epoch(...)
        - eval_one_epoch(...)

    Responsibilities:
        - seed/device/precision
        - optimizer creation: AdamW or Muon+AdamW
        - warmup cosine scheduler
        - optional EMA
        - resume
        - epoch-level training
        - epoch-level eval
        - qualitative autoregressive preview during eval
        - DeepSeek module diagnostics
        - checkpointing
        - Drive mirroring
        - JSONL metric logging
    """

    # ========================================================
    # Validate
    # ========================================================

    if optimizer_type not in {"adamw", "muon_adamw"}:
        raise ValueError(
            f"optimizer_type must be 'adamw' or 'muon_adamw', got {optimizer_type}."
        )

    if monitor_name not in VALID_MONITOR_NAMES:
        raise ValueError(
            f"monitor_name='{monitor_name}' no es válido. "
            f"Usa uno de: {sorted(VALID_MONITOR_NAMES)}"
        )

    if monitor_mode not in {"min", "max"}:
        raise ValueError(
            f"monitor_mode must be 'min' or 'max', got {monitor_mode}."
        )

    if verbose not in {0, 1}:
        raise ValueError(
            f"verbose must be 0 or 1. Got {verbose}."
        )

    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_dir = str(ckpt_dir)
    metrics_path = Path(ckpt_dir) / metrics_jsonl_name

    # ========================================================
    # Seed / device / precision
    # ========================================================

    set_seed(seed, deterministic=deterministic)

    precision = setup_device_and_precision(
        device=device,
        amp_enabled=amp_enabled,
        amp_dtype=amp_dtype,
        fallback_bf16_to_fp16=fallback_bf16_to_fp16,
    )

    device_obj = precision["device"]
    model = model.to(device_obj)

    # ========================================================
    # Optimizer
    # ========================================================

    if optimizer_type == "adamw":
        optimizer, opt_info = build_adamw_optimizer(
            model=model,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            betas=betas,
            eps=eps,
            verbose=True,
        )

    elif optimizer_type == "muon_adamw":
        optimizer, opt_info = build_muon_adamw_optimizer(
            model=model,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            betas=betas,
            eps=eps,
            muon_lr=muon_lr,
            muon_momentum=muon_momentum,
            muon_nesterov=muon_nesterov,
            muon_ns_steps=muon_ns_steps,
            muon_eps=muon_eps,
            muon_weight_decay=muon_weight_decay,
            verbose=True,
        )

    # ========================================================
    # Scheduler
    # ========================================================

    if total_steps is None:
        steps_per_epoch = len(train_loader)

        if max_batches_per_epoch is not None:
            steps_per_epoch = min(steps_per_epoch, int(max_batches_per_epoch))

        optim_steps_per_epoch = math.ceil(
            steps_per_epoch / max(1, int(grad_accum_steps))
        )
        total_steps = max(1, optim_steps_per_epoch * int(epochs))

    scheduler = build_warmup_cosine_scheduler(
        optimizer=optimizer,
        total_steps=total_steps,
        warmup_steps=warmup_steps,
        min_lr=min_learning_rate,
        min_muon_lr=min_muon_lr,
    )

    # ========================================================
    # EMA
    # ========================================================

    ema = None

    if use_ema:
        ema = EMA(
            model=model,
            decay=ema_decay,
            device=ema_device,
            use_num_updates=True,
            update_after_step=ema_update_after_step,
            update_every=ema_update_every,
        )

    # ========================================================
    # Resume
    # ========================================================

    if resume_path is not None and os.path.exists(resume_path):
        ds_section("Resume checkpoint")

        state = load_checkpoint(
            checkpoint_path=resume_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=precision.get("scaler", None),
            ema=ema,
            map_location="cpu",
            strict=strict_resume,
            load_optimizer=True,
            load_scheduler=True,
            load_scaler=True,
            load_ema=True,
            load_rng_state=restore_rng_state,
        )

        start_epoch = int(state["epoch"]) + 1
        global_step = int(state["step"])
        best_metric = state["best_metric"]

        model = model.to(device_obj)

        print(f"[RESUME] path={resume_path}")
        print(
            f"[RESUME] start_epoch={start_epoch} | "
            f"global_step={global_step} | best_metric={best_metric}"
        )
        print(ds_rule())

    # ========================================================
    # Header
    # ========================================================

    _print_deepseek_run_header(
        run_name=run_name,
        model=model,
        device=device_obj,
        precision=precision,
        optimizer_type=optimizer_type,
        ema=ema,
        epochs=epochs,
        start_epoch=start_epoch,
        global_step=global_step,
        total_steps=total_steps,
        warmup_steps=warmup_steps,
        monitor_name=monitor_name,
        monitor_mode=monitor_mode,
        best_metric=best_metric,
        grad_clip=grad_clip,
        grad_accum_steps=grad_accum_steps,
        val_loader=val_loader,
        eval_every=max(1, int(eval_every)),
        eval_max_batches=eval_max_batches,
        drive_ckpt_dir=drive_ckpt_dir,
        fixed_drive_name=fixed_drive_name,
    )

    # ========================================================
    # Resolve module diagnostics policy
    # ========================================================

    # module_metrics_every > 0:
    #   train_one_epoch handles module diagnostics during the epoch.
    # module_metrics_every == 0:
    #   train_one_epoch does not print diagnostics;
    #   this wrapper prints once after train_one_epoch and before eval.
    if module_metrics_every is not None and module_metrics_every > 0:
        module_metrics_inside_epoch = int(module_metrics_every)
    else:
        module_metrics_inside_epoch = None

    print_full_module_diagnostics_inside_epoch = (
        bool(print_module_diagnostics)
        and int(verbose) >= 1
        and module_metrics_inside_epoch is not None
    )

    print_end_epoch_module_diagnostics = (
        bool(print_module_diagnostics)
        and module_metrics_every == 0
    )

    # ========================================================
    # Epoch loop
    # ========================================================

    total_time = 0.0
    train_stats = None
    eval_stats = None
    combined_metrics = {}

    eval_every = max(1, int(eval_every))

    for epoch in range(start_epoch, epochs):
        ds_title(f"Epoch {epoch:03d}/{epochs - 1:03d}", ch="─")

        t0 = time.time()

        # ----------------------------------------------------
        # Train one epoch
        # ----------------------------------------------------

        train_stats, global_step = train_one_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            device=device_obj,
            precision=precision,
            scheduler=scheduler,
            ema=ema,
            epoch=epoch,
            global_step=global_step,
            grad_clip=grad_clip,
            grad_accum_steps=grad_accum_steps,
            max_batches=max_batches_per_epoch,
            log_every=log_every,
            module_metrics_every=module_metrics_inside_epoch,
            print_module_diagnostics=print_full_module_diagnostics_inside_epoch,
            log_grad_norm=log_grad_norm,
            log_mem=log_mem,
            on_oom=on_oom,
            is_main_process=True,
        )

        # ----------------------------------------------------
        # Optional end-of-train-epoch module diagnostics
        # Happens before eval.
        # ----------------------------------------------------

        if print_end_epoch_module_diagnostics:
            raw_model = model.module if hasattr(model, "module") else model
            cfg = getattr(raw_model, "config", None)

            module_metrics = compute_one_batch_deepseek_diagnostics(
                model=model,
                dataloader=train_loader,
                device=device_obj,
                precision=precision,
                prefix="train",
            )

            if int(verbose) >= 1:
                print_deepseek_module_metrics(
                    module_metrics,
                    prefix="train",
                    precision=4,
                    title=(
                        f"DeepSeek-V4 module diagnostics | "
                        f"end of train epoch={epoch} step={global_step}"
                    ),
                    num_experts=getattr(cfg, "num_experts", None),
                    top_k_experts=getattr(cfg, "top_k_experts", None),
                    n_layers=getattr(cfg, "n_layers", None),
                )
            else:
                print_deepseek_top10_critical_metrics(
                    module_metrics,
                    prefix="train",
                    title=(
                        f"DeepSeek-V4 critical diagnostics | "
                        f"end of train epoch={epoch} step={global_step}"
                    ),
                    num_experts=getattr(cfg, "num_experts", None),
                    n_layers=getattr(cfg, "n_layers", None),
                )

        # ----------------------------------------------------
        # Eval one epoch
        # ----------------------------------------------------

        eval_stats = None

        if val_loader is not None and ((epoch - start_epoch) % eval_every) == 0:
            ds_section("Evaluation")

            eval_stats = eval_one_epoch(
                model=model,
                dataloader=val_loader,
                device=device_obj,
                precision=precision,
                epoch=epoch,
                max_batches=eval_max_batches,
                topk=(1, 5),
                ema=ema,
                use_ema=bool(eval_use_ema and ema is not None),
                tokenizer=tokenizer,
                id2tok_fn=id2tok_fn,
                preview=eval_preview,
                preview_batch_idx=eval_preview_batch_idx,
                preview_sample_idx=eval_preview_sample_idx,
                preview_max_context_tokens=eval_preview_max_context_tokens,
                preview_max_new_tokens=eval_preview_max_new_tokens,
                preview_temperature=eval_preview_temperature,
                log_every=eval_log_every,
                is_main_process=True,
            )

            print(format_metrics(eval_stats, prefix="eval"))

        # ----------------------------------------------------
        # Metrics / monitor
        # ----------------------------------------------------

        sec = time.time() - t0
        total_time += sec

        combined_metrics = {}
        combined_metrics.update(_prefixed_stats("train", train_stats))
        combined_metrics.update(_prefixed_stats("eval", eval_stats))

        if eval_stats is not None:
            combined_metrics.update(eval_stats)
        else:
            combined_metrics.update(train_stats)

        current_metric = _resolve_monitor_value(
            monitor_name=monitor_name,
            train_stats=train_stats,
            eval_stats=eval_stats,
        )

        improved = _is_better_metric(
            current=current_metric,
            best=best_metric,
            mode=monitor_mode,
        )

        if improved:
            best_metric = current_metric

        # ----------------------------------------------------
        # Epoch summary
        # ----------------------------------------------------

        _print_epoch_summary(
            epoch=epoch,
            global_step=global_step,
            sec=sec,
            optimizer=optimizer,
            train_stats=train_stats,
            eval_stats=eval_stats,
            monitor_name=monitor_name,
            current_metric=current_metric,
            best_metric=best_metric,
            improved=improved,
        )

        # ----------------------------------------------------
        # Metrics JSONL
        # ----------------------------------------------------

        metrics_record = {
            "epoch": int(epoch),
            "global_step": int(global_step),
            "time_sec": float(sec),
            "monitor_name": monitor_name,
            "monitor_value": float(current_metric),
            "best_metric": float(best_metric) if best_metric is not None else None,
            "improved": bool(improved),
            "optimizer_type": optimizer_type,
            **combined_metrics,
        }

        append_jsonl(metrics_path, metrics_record)

        if drive_ckpt_dir:
            copy_metrics_to_drive(
                src_path=metrics_path,
                drive_dir=drive_ckpt_dir,
                fixed_name=fixed_drive_metrics_name,
            )

        # ----------------------------------------------------
        # Checkpointing
        # ----------------------------------------------------

        ds_section("Checkpointing")

        if improved:
            best_path = save_checkpoint(
                checkpoint_dir=ckpt_dir,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=precision.get("scaler", None),
                ema=ema,
                epoch=epoch,
                step=global_step,
                best_metric=best_metric,
                config=config if config is not None else getattr(model, "config", None),
                extra_state={
                    "monitor_name": monitor_name,
                    "monitor_mode": monitor_mode,
                    "monitor_value": current_metric,
                    "train_stats": train_stats,
                    "eval_stats": eval_stats,
                    "optimizer_type": optimizer_type,
                    "opt_info": opt_info,
                },
                filename=f"{run_name}_best.pt",
                save_rng_state=True,
                keep_last_n=None,
                tag="best",
            )

            print(f"└─ [BEST] improved {monitor_name} -> {best_metric:.6f}")
            print(f"└─ [BEST] saved → {best_path}")

            if copy_fixed_to_drive and drive_ckpt_dir:
                copy_ckpt_to_drive_fixed(
                    src_path=best_path,
                    drive_dir=drive_ckpt_dir,
                    fixed_name=f"best_{fixed_drive_name}",
                )
        else:
            print("└─ [BEST] no improvement")

        should_save_epoch = (
            save_every is not None
            and save_every > 0
            and ((epoch % save_every == 0) or (epoch == epochs - 1))
        )

        if should_save_epoch:
            ckpt_path = save_checkpoint(
                checkpoint_dir=ckpt_dir,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=precision.get("scaler", None),
                ema=ema,
                epoch=epoch,
                step=global_step,
                best_metric=best_metric,
                config=config if config is not None else getattr(model, "config", None),
                extra_state={
                    "monitor_name": monitor_name,
                    "monitor_mode": monitor_mode,
                    "monitor_value": current_metric,
                    "train_stats": train_stats,
                    "eval_stats": eval_stats,
                    "optimizer_type": optimizer_type,
                    "opt_info": opt_info,
                },
                filename=f"{run_name}_e{epoch:03d}.pt",
                save_rng_state=True,
                keep_last_n=keep_last_n_checkpoints,
            )

            print(f"└─ [CKPT] saved → {ckpt_path}")

            if copy_fixed_to_drive and drive_ckpt_dir:
                copy_ckpt_to_drive_fixed(
                    src_path=ckpt_path,
                    drive_dir=drive_ckpt_dir,
                    fixed_name=fixed_drive_name,
                )
        else:
            print("└─ [CKPT] skipped by save_every")

    # ========================================================
    # Final checkpoint
    # ========================================================

    if save_last and train_stats is not None:
        ds_section("Final checkpoint")

        last_path = save_checkpoint(
            checkpoint_dir=ckpt_dir,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=precision.get("scaler", None),
            ema=ema,
            epoch=epochs - 1,
            step=global_step,
            best_metric=best_metric,
            config=config if config is not None else getattr(model, "config", None),
            extra_state={
                "monitor_name": monitor_name,
                "monitor_mode": monitor_mode,
                "train_stats": train_stats,
                "eval_stats": eval_stats,
                "optimizer_type": optimizer_type,
                "opt_info": opt_info,
            },
            filename=f"{run_name}_last_manual.pt",
            save_rng_state=True,
            keep_last_n=None,
            tag="last",
        )

        print(f"└─ [CKPT] final saved → {last_path}")

        if copy_fixed_to_drive and drive_ckpt_dir:
            copy_ckpt_to_drive_fixed(
                src_path=last_path,
                drive_dir=drive_ckpt_dir,
                fixed_name=fixed_drive_name,
            )

    # ========================================================
    # Done
    # ========================================================

    ds_title("Training complete")
    print(f"Total time : {fmt_hms(total_time)}")
    print(f"Final step : {global_step}")
    print(f"Best metric: {best_metric}")
    print(f"Metrics    : {metrics_path}")
    print(f"Checkpoints: {ckpt_dir}")
    print(ds_rule())

    return {
        "model": model,
        "optimizer": optimizer,
        "scheduler": scheduler,
        "ema": ema,
        "precision": precision,
        "opt_info": opt_info,
        "global_step": global_step,
        "best_metric": best_metric,
        "last_train_stats": train_stats,
        "last_eval_stats": eval_stats,
        "metrics_path": str(metrics_path),
        "checkpoint_dir": ckpt_dir,
    }

---

In [199]:
cfg = DeepSeekV4LMConfig(
    vocab_size=216,
    d_model=64,
    n_layers=4,
    max_seq_len=64,
    pad_token_id=0,
    ignore_index=-100,

    # ========================================================
    # Attention: hybrid CSA/HCA baseline
    # Pattern:
    #   layer 0 -> CSA
    #   layer 1 -> HCA
    #   layer 2 -> CSA
    #   layer 3 -> HCA
    # ========================================================
    attention_type="hybrid",
    attention_pattern=("csa", "hca"),

    n_heads=4,
    head_dim=16,
    rotary_dim=16,

    attention_dropout=0.0,
    residual_dropout=0.0,
    use_attention_bias=False,
    use_rope=True,
    rope_theta=10000.0,

    # ========================================================
    # CSA config
    # ========================================================
    compression_factor=4,
    top_k_blocks=2,
    window_size=8,
    indexer_dim=8,
    n_indexer_heads=2,
    query_compression_dim=16,

    use_attention_sink=True,
    use_grouped_output_projection=True,
    output_projection_groups=2,
    use_indexer_score_bias=False,
    use_separate_local_kv=True,

    # ========================================================
    # HCA config
    # HCA should compress more aggressively than CSA.
    # With max_seq_len=64, hca_compression_factor=8 is a sane
    # mini baseline. 16 also works but gives very few global blocks.
    # ========================================================
    hca_compression_factor=8,

    # ========================================================
    # FFN: DeepSeekMoE
    # ========================================================
    ffn_type="moe",
    num_experts=4,
    top_k_experts=2,

    expert_hidden_dim=128,
    expert_expansion_factor=4.0,
    expert_multiple_of=1,

    shared_experts=1,
    shared_hidden_dim=128,
    shared_expansion_factor=4.0,

    router_type="learned",
    router_score_fn="sqrt_softplus",
    normalize_topk_weights=True,
    topk_weight_scale=1.0,
    router_jitter_noise=0.0,
    hash_routing_stride=1,

    routed_scale=1.0,
    shared_scale=1.0,

    balance_loss_weight=0.01,
    sequence_balance_loss_weight=0.01,

    # ========================================================
    # mHC
    # ========================================================
    use_mhc=True,
    n_hc=4,
    mhc_sinkhorn_iters=20,
    mhc_eps=1e-6,
    mhc_dynamic=True,
    mhc_expand_mode="first",
    mhc_collapse_mode="readout",

    mhc_use_log_sinkhorn=False,
    mhc_sinkhorn_fp32=True,
    mhc_init_alpha=1e-3,
    mhc_alpha_max=1.0,
    mhc_bounded_alpha=True,

    # ========================================================
    # MTP
    # ========================================================
    use_mtp=True,
    mtp_depth=2,
    mtp_hidden_dim=64,
    use_mtp_transform=True,
    mtp_activation="silu",
    mtp_dropout=0.0,
    mtp_loss_weight=0.3,
    mtp_tie_with_lm_head=False,
    mtp_depth_loss_weights=None,
    mtp_validate_label_range=True,

    # ========================================================
    # Embedding / norm / init
    # ========================================================
    embedding_dropout=0.0,
    scale_embeddings=False,
    tie_word_embeddings=True,

    rms_norm_eps=1e-6,
    init_std=0.02,

    # ========================================================
    # Loss semantics
    #   input_ids = ids[:-1]
    #   labels    = ids[1:]
    # ========================================================
    labels_are_shifted=True,
    ignore_pad_token_in_loss=True,)

model = DeepSeekV4LM(cfg)
model.to('cuda')

for i, block in enumerate(model.blocks):
    print(i, block.attention_type, type(block.attention).__name__)

0 csa CSAAttention
1 hca HCAAttention
2 csa CSAAttention
3 hca HCAAttention


In [201]:
result = train_deepseekv4(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer,

    optimizer_type="muon_adamw",
    learning_rate=3e-4,
    min_learning_rate=3e-5,
    weight_decay=0.1,

    epochs=15,
    max_batches_per_epoch=200,
    log_every=20,
    module_metrics_every=0, # Para imrpimir al final solo
    verbose=1,

    eval_every=1,
    eval_max_batches=50,
    eval_use_ema=True,
    eval_preview=True,
    eval_preview_max_context_tokens=48,
    eval_preview_max_new_tokens=24,
    eval_preview_temperature=0.0,

    use_ema=True,
    ema_decay=0.999,
    ema_device="cpu",

    ckpt_dir="checkpoints/deepseekv4_mini",
    run_name="deepseekv4_mini_muon",
    monitor_name="eval_loss",
    monitor_mode="min",

    drive_ckpt_dir=None,
)

Hybrid Muon + AdamW parameter groups
Muon tensors          : 108
AdamW decay tensors   : 10
AdamW no-decay tensors: 127
--------------------------------------------------------------------------------
Muon params           : 548,608
AdamW params          : 94,556
Total trainable params: 643,164

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
DeepSeek-V4 run: deepseekv4_mini_muon
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Device    : cuda | AMP: True (bf16 -> torch.bfloat16)
Optimizer : muon_adamw | EMA: 0.999000 | grad_clip: 1.0 | grad_accum_steps: 1
Schedule  : epochs=15 | start_epoch=0 | global_step=0 | total_steps=3000 | warmup_steps=500
Monitor   : eval_loss (min) | best_metric=None
Model     : layers=4 | d_model=64 | attention=hybrid | ffn=moe | mHC=True | MTP=True
Eval      : enabled | eval_every=1 | eval_max_batches=50
──────────────────────────────

KeyboardInterrupt: 